# DIMER Notebook: Image Segmentation with Text, Points, and Boxes

**Which pixels belong to the target, and how does the information we give the model change its answer?**

**Profile:** `E2E` · **Mode:** `WORKSHOP` · **Notebook specification:** `2.2` · **Status:** Candidate — fresh Colab GPU qualification pending.

This modular, self-paced Level 2 Applied Tasks unit fits after Open-Vocabulary Object Detection. Basic Python and Colab familiarity are sufficient; no prior segmentation knowledge is required.

**Classification → a label. Detection → a box. Segmentation → a pixel mask.** We explore **Image + Prompt → Model → Mask**, compare two SAM models under identical geometric prompts, and adapt the CLIPSeg decoder using labelled food images. Only CLIPSeg is trained; both SAM models stay frozen.

By the end you should be able to interpret masks and overlap metrics, identify unequal prompting assistance, run a controlled comparison, and explain what an exported adapter and fresh reload establish.

**Run all:** select Runtime → Change runtime type → T4 GPU. The default workflow is automatic: no file upload, token, manual restart, repository clone or DIMER service is required. This file carries its implementations, manifests and dependency lock. It trains for four bounded epochs, evaluates held-out images, exports an adapter and verifies it in a new process.

The 212-image sample has **120 train / 40 validation / 40 test / 12 activity** images. Downloads include three model snapshots and a GPU framework environment; allow several GiB of disk. A fresh Colab T4 `Run all` of the previous revision took about 6.5 minutes (393 s, recorded 2026-09-27); peak memory and disk use have not been measured yet. One model is resident at a time.

**Roadmap:** orient → validate images → learn metrics → adapt CLIPSeg → reload → compare SAMs → change box precision → conclude → optional own data.

**How to use this notebook**

- Choose **Runtime → Run all**. Every default cell runs in order; nothing needs editing.
- Cells titled **Infrastructure** (and the collapsed carried-source cell) prepare the environment. Run them without studying them; they are not part of the lesson.
- Form fields appear only in the optional Sections 9 and 10. Leave them off for the default run; set them afterwards and run only that cell.
- Each run writes to its own folder under `outputs/image_segmentation/`. Section 8 prints the results bundle's path; download it from Colab's Files sidebar.

Personal learning notes are optional and do not need to be submitted.

<details><summary>Glossary</summary>

- **Semantic class mask:** every pixel of one ingredient class, even in several separate pieces. An **instance** is one object.
- **Reference mask:** the labelled answer used for scoring. **Predicted mask:** the model's answer after thresholding.
- **Logits / probability map:** CLIPSeg's per-pixel score before and after the sigmoid; not a calibrated probability.
- **Threshold:** the cut (0.5 here) that turns a probability map into a mask.
- **Decoder:** CLIPSeg's small network that turns CLIP features into a mask; the only part trained here.
- **Adapter:** the exported decoder weights plus a manifest naming the base model they belong to.
- **Oracle prompt:** a point or box computed from the reference mask, as if an informed user supplied it.
- **Candidate mask / predicted quality:** SAM models return several candidate masks with their own quality estimates; the highest estimate is chosen. The estimate is not measured IoU.
- **IoU / Dice:** overlap between predicted and reference masks (Section 3).

</details>

## 1. Prepare an isolated runtime

**Input:** the selected GPU runtime and carried files. **System:** an isolated Python 3.12.12 environment. **Output:** pinned, hash-verified dependencies without replacing Colab's already-loaded NumPy or torch.

The next three cells are **Infrastructure** and are collapsed: run them without studying them. The setup verifies the uv 0.12.15 bootstrap wheel, uses uv's verified managed interpreter download, and installs the complete Linux dependency lock. Model files are later checked against the carried per-file manifests before loading. An optional course Hugging Face token is not needed here.

**What to notice:** the GPU name, Python version and installation completion. An integrity error is a reason to stop and investigate, not to disable verification.

In [ ]:
# @title Infrastructure: check the runtime and create this run's folder
import hashlib
import json
import os
import platform
from pathlib import Path
import subprocess
import time
import urllib.request
import uuid
import zipfile

SESSION_START = time.perf_counter()
if platform.system() != 'Linux' or platform.machine() != 'x86_64':
    raise RuntimeError('The canonical environment is Linux x86_64 Google Colab with a T4 GPU.')
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
except FileNotFoundError as exc:
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU before Run all.') from exc
if gpu.returncode:
    raise RuntimeError('Select a fresh T4 GPU runtime; CUDA was not detected.')
print(gpu.stdout.strip())
ROOT = Path.cwd() / 'outputs' / 'image_segmentation' / uuid.uuid4().hex[:12]
ROOT.mkdir(parents=True)

In [ ]:
CARRIED_FILES = {
    'clipseg_reference/__init__.py': (
        '"""DIMER pipeline package for the pinned ``CIDAS/clipseg-rd64-refined`` checkpoint: text-prompted binary\n'
        'segmentation, corpus evaluation of labelled (image, phrase, mask) records, bounded fine-tuning of the CLIPSeg\n'
        'decoder, and a verified adapter artifact."""\n'
        '\n'
        'from .metrics import (\n'
        '    METRIC_DEFINITIONS,\n'
        '    empty_baseline,\n'
        '    full_baseline,\n'
        '    mask_metrics,\n'
        '    segmentation_metrics,\n'
        ')\n'
        'from .pipeline import (\n'
        '    ADAPTER_MANIFEST,\n'
        '    ADAPTER_WEIGHTS,\n'
        '    ARTIFACT_FORMAT,\n'
        '    DECODER_PARAMETERS,\n'
        '    DEFAULT_WEIGHTS_DIR,\n'
        '    EVAL_BATCH_SIZE,\n'
        '    EXTRACT_LAYERS,\n'
        '    GRAD_CLIP,\n'
        '    INPUT_SCHEMA,\n'
        '    LOGIT_SIZE,\n'
        '    MASK_THRESHOLD,\n'
        '    MAX_EVAL_RECORDS,\n'
        '    MAX_IMAGE_SIDE,\n'
        '    MAX_PROMPT_CHARS,\n'
        '    MAX_PROMPTS,\n'
        '    MAX_TEXT_TOKENS,\n'
        '    MIN_IMAGE_SIDE,\n'
        '    MIN_SCORED_RECORDS,\n'
        '    MODEL_ID,\n'
        '    MODEL_KEY,\n'
        '    MODEL_LICENSE,\n'
        '    MODEL_REVISION,\n'
        '    PARAMETER_COUNT,\n'
        '    WEIGHTS_FILE,\n'
        '    ClipSegSegmentationPipeline,\n'
        '    evaluation_report,\n'
        '    format_prompts,\n'
        '    mask_bbox,\n'
        '    mask_iou,\n'
        '    stage_missing_files,\n'
        '    validate_image,\n'
        '    validate_inputs,\n'
        '    verify_snapshot,\n'
        ')\n'
        'from .samples import (\n'
        '    CORPUS_LICENSE,\n'
        '    CORPUS_NAME,\n'
        '    CORPUS_REPO,\n'
        '    CORPUS_REVISION,\n'
        '    EXCLUDED_CLASS_IDS,\n'
        '    FOODSEG103_CLASSES,\n'
        '    MAX_RECORDS,\n'
        '    MIN_RECORDS,\n'
        '    ROW_GROUP_PINS,\n'
        '    SAMPLE_DIGEST,\n'
        '    SAMPLE_SEED,\n'
        '    SAMPLE_SPLIT,\n'
        '    build_sample_dataset,\n'
        '    check_split_disjoint,\n'
        '    coerce_mask,\n'
        '    dataset_digest,\n'
        '    fetch_corpus,\n'
        '    fetch_sample_dataset,\n'
        '    image_digest,\n'
        '    largest_class,\n'
        '    load_byod_dataset,\n'
        '    mask_digest,\n'
        '    read_corpus,\n'
        '    split_dataset,\n'
        '    validate_dataset,\n'
        '    write_dataset_csv,\n'
        ')\n'
        '\n'
        '__all__ = [\n'
        '    "ADAPTER_MANIFEST",\n'
        '    "ADAPTER_WEIGHTS",\n'
        '    "ARTIFACT_FORMAT",\n'
        '    "CORPUS_LICENSE",\n'
        '    "CORPUS_NAME",\n'
        '    "CORPUS_REPO",\n'
        '    "CORPUS_REVISION",\n'
        '    "DECODER_PARAMETERS",\n'
        '    "DEFAULT_WEIGHTS_DIR",\n'
        '    "EVAL_BATCH_SIZE",\n'
        '    "EXCLUDED_CLASS_IDS",\n'
        '    "EXTRACT_LAYERS",\n'
        '    "FOODSEG103_CLASSES",\n'
        '    "GRAD_CLIP",\n'
        '    "INPUT_SCHEMA",\n'
        '    "LOGIT_SIZE",\n'
        '    "MASK_THRESHOLD",\n'
        '    "MAX_EVAL_RECORDS",\n'
        '    "MAX_IMAGE_SIDE",\n'
        '    "MAX_PROMPTS",\n'
        '    "MAX_PROMPT_CHARS",\n'
        '    "MAX_RECORDS",\n'
        '    "MAX_TEXT_TOKENS",\n'
        '    "METRIC_DEFINITIONS",\n'
        '    "MIN_IMAGE_SIDE",\n'
        '    "MIN_RECORDS",\n'
        '    "MIN_SCORED_RECORDS",\n'
        '    "MODEL_ID",\n'
        '    "MODEL_KEY",\n'
        '    "MODEL_LICENSE",\n'
        '    "MODEL_REVISION",\n'
        '    "PARAMETER_COUNT",\n'
        '    "ROW_GROUP_PINS",\n'
        '    "SAMPLE_DIGEST",\n'
        '    "SAMPLE_SEED",\n'
        '    "SAMPLE_SPLIT",\n'
        '    "WEIGHTS_FILE",\n'
        '    "ClipSegSegmentationPipeline",\n'
        '    "build_sample_dataset",\n'
        '    "check_split_disjoint",\n'
        '    "coerce_mask",\n'
        '    "dataset_digest",\n'
        '    "empty_baseline",\n'
        '    "evaluation_report",\n'
        '    "fetch_corpus",\n'
        '    "fetch_sample_dataset",\n'
        '    "format_prompts",\n'
        '    "full_baseline",\n'
        '    "image_digest",\n'
        '    "largest_class",\n'
        '    "load_byod_dataset",\n'
        '    "mask_bbox",\n'
        '    "mask_digest",\n'
        '    "mask_iou",\n'
        '    "mask_metrics",\n'
        '    "read_corpus",\n'
        '    "segmentation_metrics",\n'
        '    "split_dataset",\n'
        '    "stage_missing_files",\n'
        '    "validate_dataset",\n'
        '    "validate_image",\n'
        '    "validate_inputs",\n'
        '    "verify_snapshot",\n'
        '    "write_dataset_csv",\n'
        ']\n'
    ),
    'clipseg_reference/metrics.py': (
        '"""Corpus-level measures for text-prompted binary segmentation: per-record IoU and Dice at a stated threshold,\n'
        'their means (macro), the pixel-pooled IoU (micro), pixel precision and recall, and two non-adapted baselines\n'
        '(empty mask, full-image mask).\n'
        '\n'
        'Every rate is computed over the supplied records at the supplied threshold; the sigmoid is uncalibrated and no\n'
        'dispersion is estimated (one seeded split of one sample gives one number)."""\n'
        '# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'from collections.abc import Mapping, Sequence\n'
        'from typing import Any\n'
        '\n'
        'import numpy as np\n'
        '\n'
        'METRIC_DEFINITIONS = {\n'
        '    "miou": "mean over records of the intersection-over-union between the thresholded mask and the reference mask (macro IoU; an empty prediction against a non-empty reference scores 0)",\n'
        '    "iou_micro": "total intersection pixels over total union pixels across all records (micro IoU; large masks weigh more)",\n'
        '    "dice": "mean over records of 2·|A∩B| / (|A| + |B|)",\n'
        '    "pixel_precision": "total true-positive pixels over total predicted pixels",\n'
        '    "pixel_recall": "total true-positive pixels over total reference pixels",\n'
        '    "baselines": "empty = no pixel predicted (IoU 0 by construction); full = every pixel predicted (IoU = the reference\'s area fraction)",\n'
        '}\n'
        '\n'
        '\n'
        'def mask_metrics(prediction: np.ndarray, reference: np.ndarray) -> dict[str, float]:\n'
        '    """IoU, Dice, precision and recall of one boolean prediction against one boolean reference."""\n'
        '    a = np.asarray(prediction, dtype=bool)\n'
        '    b = np.asarray(reference, dtype=bool)\n'
        '    if a.shape != b.shape:\n'
        '        raise ValueError(f"prediction shape {a.shape} != reference shape {b.shape}")\n'
        '    inter = int(np.logical_and(a, b).sum())\n'
        '    union = int(np.logical_or(a, b).sum())\n'
        '    pred, ref = int(a.sum()), int(b.sum())\n'
        '    return {\n'
        '        "iou": inter / union if union else 0.0,\n'
        '        "dice": 2 * inter / (pred + ref) if (pred + ref) else 0.0,\n'
        '        "precision": inter / pred if pred else 0.0,\n'
        '        "recall": inter / ref if ref else 0.0,\n'
        '        "intersection": inter,\n'
        '        "union": union,\n'
        '        "predicted": pred,\n'
        '        "reference": ref,\n'
        '    }\n'
        '\n'
        '\n'
        'def segmentation_metrics(predictions: Sequence[np.ndarray], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n'
        '    """Score one boolean mask per record against the record\'s `mask`."""\n'
        '    if len(predictions) != len(records):\n'
        '        raise ValueError(f"{len(predictions)} predictions for {len(records)} records")\n'
        '    if not records:\n'
        '        raise ValueError("no records to score")\n'
        '    rows = []\n'
        '    for prediction, record in zip(predictions, records, strict=True):\n'
        '        row = mask_metrics(prediction, record["mask"])\n'
        '        row["id"] = record["id"]\n'
        '        row["prompt"] = record["prompt"]\n'
        '        rows.append(row)\n'
        '    inter = sum(r["intersection"] for r in rows)\n'
        '    union = sum(r["union"] for r in rows)\n'
        '    pred = sum(r["predicted"] for r in rows)\n'
        '    ref = sum(r["reference"] for r in rows)\n'
        '    return {\n'
        '        "n": len(rows),\n'
        '        "miou": sum(r["iou"] for r in rows) / len(rows),\n'
        '        "iou_micro": inter / union if union else 0.0,\n'
        '        "dice": sum(r["dice"] for r in rows) / len(rows),\n'
        '        "pixel_precision": inter / pred if pred else 0.0,\n'
        '        "pixel_recall": inter / ref if ref else 0.0,\n'
        '        "predicted_pixels": pred,\n'
        '        "reference_pixels": ref,\n'
        '        "rows": rows,\n'
        '        "definitions": dict(METRIC_DEFINITIONS),\n'
        '    }\n'
        '\n'
        '\n'
        'def empty_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n'
        '    """Predict no pixel for every record: IoU, Dice and recall 0 by construction."""\n'
        '    out = segmentation_metrics([np.zeros(np.asarray(r["mask"]).shape, dtype=bool) for r in records], records)\n'
        '    out["baseline"] = "empty mask (no pixel predicted)"\n'
        '    return out\n'
        '\n'
        '\n'
        'def full_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n'
        '    """Predict every pixel for every record: IoU equals the reference\'s area fraction, recall 1."""\n'
        '    out = segmentation_metrics([np.ones(np.asarray(r["mask"]).shape, dtype=bool) for r in records], records)\n'
        '    out["baseline"] = "full-image mask (every pixel predicted)"\n'
        '    return out\n'
    ),
    'clipseg_reference/pipeline.py': (
        '"""Text-prompted (zero-shot) image segmentation with the pinned ``CIDAS/clipseg-rd64-refined`` checkpoint, plus\n'
        'the adaptation contract for labelled (image, phrase, mask) records: corpus evaluation at a stated threshold,\n'
        'bounded fine-tuning of the CLIPSeg decoder on cached CLIP activations, and a verified adapter artifact.\n'
        '\n'
        'The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)\n'
        'or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with\n'
        '``trust_remote_code=False``: the CLIPSeg architecture comes from the pinned ``transformers`` release,\n'
        'the weights are SafeTensors, and no model-repository code is executed.\n'
        '"""\n'
        '# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import hashlib\n'
        'import json\n'
        'import random\n'
        'import time\n'
        'from collections.abc import Callable, Mapping, Sequence\n'
        'from dataclasses import dataclass\n'
        'from pathlib import Path\n'
        'from typing import Any\n'
        '\n'
        'import numpy as np\n'
        'from PIL import Image\n'
        '\n'
        'MODEL_ID = "CIDAS/clipseg-rd64-refined"\n'
        'MODEL_REVISION = "999e0328d9e10b484360c477313983f9afdd7050"\n'
        'MODEL_LICENSE = "apache-2.0"\n'
        'MODEL_KEY = "clipseg-rd64-refined"\n'
        'DEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / "weights" / MODEL_KEY\n'
        'MANIFEST_NAME = "dimer-base-manifest.json"\n'
        '\n'
        '# Threshold on the per-pixel sigmoid of the decoder logits. 0.5 is the natural cut of a sigmoid and\n'
        '# the value the smoke run used; the sigmoid is not calibrated, so the deployment owns tuning it on its\n'
        '# own labelled masks.\n'
        'MASK_THRESHOLD = 0.5\n'
        '# Decoder output resolution: logits are 352x352 for every input (preprocessor_config.json resizes to\n'
        '# 352x352 without preserving aspect ratio); the pipeline resamples the probability map back to the\n'
        '# input size bilinearly.\n'
        'LOGIT_SIZE = 352\n'
        '# Input ceilings. Image cost is bounded by the fixed resize; the side ceiling only guards memory during\n'
        '# decoding and resampling. Each prompt is one CLIP text query (77-token context); phrases are short.\n'
        'MAX_IMAGE_SIDE = 4096\n'
        'MIN_IMAGE_SIDE = 16\n'
        'MAX_PROMPTS = 16\n'
        'MAX_PROMPT_CHARS = 64\n'
        'MAX_TEXT_TOKENS = 77\n'
        'WEIGHTS_FILE = "model.safetensors"\n'
        '\n'
        '# Adaptation contract: the CLIPSeg decoder (three transformer layers over the CLIP activations extracted at\n'
        '# `EXTRACT_LAYERS`, the FiLM conditioning on the phrase embedding, the reduce projections and the transposed\n'
        '# convolution) is the adapter — the part the upstream authors trained; the CLIP vision and text towers stay\n'
        '# frozen, so their outputs are computed once per record and cached (fp16 on the host).\n'
        'PARAMETER_COUNT = 150_747_746\n'
        'DECODER_PARAMETERS = 1_127_009\n'
        'EXTRACT_LAYERS = (3, 6, 9)\n'
        '_TRAINABLE_PREFIXES = ("decoder.",)\n'
        'ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"\n'
        'ARTIFACT_VERSION = 1\n'
        'ADAPTER_WEIGHTS = "adapter.safetensors"\n'
        'ADAPTER_MANIFEST = "manifest.json"\n'
        'MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample\n'
        'MAX_EVAL_RECORDS = 5_000\n'
        'EVAL_BATCH_SIZE = 8\n'
        'GRAD_CLIP = 1.0\n'
        '\n'
        '\n'
        'def _sha256(path: Path) -> str:\n'
        '    digest = hashlib.sha256()\n'
        '    with open(path, "rb") as fh:\n'
        '        for chunk in iter(lambda: fh.read(1 << 20), b""):\n'
        '            digest.update(chunk)\n'
        '    return digest.hexdigest()\n'
        '\n'
        '\n'
        'def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n'
        '    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""\n'
        '    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID:\n'
        '        raise ValueError(f"manifest modelId {manifest.get(\'modelId\')!r} != {MODEL_ID!r}")\n'
        '    if manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(f"manifest revision {manifest.get(\'revision\')!r} != {MODEL_REVISION!r}")\n'
        '    for entry in manifest["files"]:\n'
        '        file_path = root / entry["path"]\n'
        '        if not file_path.is_file():\n'
        '            raise FileNotFoundError(f"snapshot file missing: {file_path}")\n'
        '        size = file_path.stat().st_size\n'
        '        if size != entry["bytes"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: size {size} != manifest {entry[\'bytes\']}")\n'
        '        digest = _sha256(file_path)\n'
        '        if digest != entry["sha256"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: sha256 {digest} != manifest {entry[\'sha256\']}")\n'
        '    return {\n'
        '        "path": str(root),\n'
        '        "model_id": manifest["modelId"],\n'
        '        "revision": manifest["revision"],\n'
        '        "files": len(manifest["files"]),\n'
        '        "total_bytes": manifest.get("totalBytes"),\n'
        '    }\n'
        '\n'
        '\n'
        'def _weight_digest(root: Path) -> str | None:\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        return None\n'
        '    with open(manifest_path, encoding="utf-8") as handle:\n'
        '        entries = json.load(handle).get("files", [])\n'
        '    return next((e["sha256"] for e in entries if e["path"] == WEIGHTS_FILE), None)\n'
        '\n'
        '\n'
        'def _trainable_names(model: Any) -> list[str]:\n'
        '    """The decoder\'s tensors; the CLIP vision and text towers and their projections stay frozen."""\n'
        '    return [name for name, _ in model.named_parameters() if name.startswith(_TRAINABLE_PREFIXES)]\n'
        '\n'
        '\n'
        'def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:\n'
        '    """Refuse an adapter that names another base, another format or a file that does not match its digest."""\n'
        '    if manifest.get("format") != ARTIFACT_FORMAT:\n'
        '        raise ValueError(f"artifact format {manifest.get(\'format\')!r} != {ARTIFACT_FORMAT!r}")\n'
        '    base = manifest.get("base", {})\n'
        '    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(f"artifact was trained on {base.get(\'model_id\')}@{base.get(\'revision\')}, not {MODEL_ID}@{MODEL_REVISION}")\n'
        '    if base.get("weight_sha256") != base_sha256:\n'
        '        raise ValueError("artifact base weight digest does not match the verified snapshot")\n'
        '    files = manifest.get("files") or []\n'
        '    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:\n'
        '        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")\n'
        '    weights = artifact_dir / ADAPTER_WEIGHTS\n'
        '    if not weights.is_file():\n'
        '        raise FileNotFoundError(f"artifact weights missing: {weights}")\n'
        '    size = weights.stat().st_size\n'
        '    if size != files[0].get("bytes"):\n'
        '        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get(\'bytes\')}")\n'
        '    digest = _sha256(weights)\n'
        '    if digest != files[0].get("sha256"):\n'
        '        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get(\'sha256\')}")\n'
        '    names = manifest.get("tensors") or []\n'
        '    if not names or any(not str(n).startswith(_TRAINABLE_PREFIXES) for n in names):\n'
        '        raise ValueError("artifact tensors must all belong to the CLIPSeg decoder")\n'
        '    adapter = manifest.get("adapter") or {}\n'
        '    threshold = adapter.get("threshold")\n'
        '    if isinstance(threshold, bool) or not isinstance(threshold, int | float) or not 0.0 <= float(threshold) <= 1.0:\n'
        '        raise ValueError("artifact manifest must record adapter.threshold, the mask threshold the epoch was selected at")\n'
        '\n'
        '\n'
        'def _hub_download(relative_path: str, root: Path) -> None:\n'
        '    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""\n'
        '    from huggingface_hub import hf_hub_download\n'
        '\n'
        '    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n'
        '\n'
        '\n'
        'def stage_missing_files(\n'
        '    path: str | Path | None = None,\n'
        '    *,\n'
        '    allow_download: bool = False,\n'
        '    downloader: Callable[[str, Path], None] | None = None,\n'
        ') -> list[str]:\n'
        '    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but\n'
        '    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""\n'
        '    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(\n'
        '            f"manifest names {manifest.get(\'modelId\')}@{manifest.get(\'revision\')}, "\n'
        '            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"\n'
        '        )\n'
        '    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]\n'
        '    if not missing:\n'
        '        return []\n'
        '    if not allow_download:\n'
        '        raise FileNotFoundError(\n'
        '            f"snapshot at {root} is missing {missing}; "\n'
        '            f"pass allow_download=True to fetch them at {MODEL_REVISION}"\n'
        '        )\n'
        '    fetch = downloader or _hub_download\n'
        '    for relative_path in missing:\n'
        '        fetch(relative_path, root)\n'
        '    return missing\n'
        '\n'
        '\n'
        'def mask_iou(a: np.ndarray, b: np.ndarray) -> float:\n'
        '    """Intersection-over-union of two boolean masks of the same shape; the building block for any mIoU."""\n'
        '    a_bool, b_bool = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)\n'
        '    if a_bool.shape != b_bool.shape:\n'
        '        raise ValueError(f"mask shapes differ: {a_bool.shape} vs {b_bool.shape}")\n'
        '    union = np.logical_or(a_bool, b_bool).sum()\n'
        '    return float(np.logical_and(a_bool, b_bool).sum() / union) if union else 0.0\n'
        '\n'
        '\n'
        'def mask_bbox(mask: np.ndarray) -> list[int] | None:\n'
        '    """Tight xyxy pixel box around the true pixels of a mask, or ``None`` for an empty mask."""\n'
        '    rows = np.flatnonzero(np.asarray(mask, dtype=bool).any(axis=1))\n'
        '    cols = np.flatnonzero(np.asarray(mask, dtype=bool).any(axis=0))\n'
        '    if rows.size == 0 or cols.size == 0:\n'
        '        return None\n'
        '    return [int(cols[0]), int(rows[0]), int(cols[-1]) + 1, int(rows[-1]) + 1]\n'
        '\n'
        '\n'
        'def format_prompts(prompts: Sequence[str]) -> list[str]:\n'
        '    """Validate a list of phrases and normalise them: stripped, whitespace-collapsed, lower-cased,\n'
        "    trailing full stop removed, distinct. The pipeline passes the caller's phrases through otherwise\n"
        '    unchanged (the upstream examples use plain noun phrases such as "a cat")."""\n'
        '    if isinstance(prompts, str) or not isinstance(prompts, Sequence):\n'
        '        raise TypeError("prompts must be a list of phrases, not a single string")\n'
        '    if not 1 <= len(prompts) <= MAX_PROMPTS:\n'
        '        raise ValueError(f"prompt count {len(prompts)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")\n'
        '    cleaned: list[str] = []\n'
        '    for phrase in prompts:\n'
        '        if not isinstance(phrase, str):\n'
        '            raise TypeError(f"prompt must be str, got {type(phrase).__name__}")\n'
        '        text = " ".join(phrase.split()).strip().rstrip(".").strip().lower()\n'
        '        if not text:\n'
        '            raise ValueError("prompt phrases must not be empty")\n'
        '        if len(text) > MAX_PROMPT_CHARS:\n'
        '            raise ValueError(\n'
        '                f"prompt {text[:12]!r}... is {len(text)} chars > MAX_PROMPT_CHARS {MAX_PROMPT_CHARS}"\n'
        '            )\n'
        '        cleaned.append(text)\n'
        '    if len(set(cleaned)) != len(cleaned):\n'
        '        raise ValueError("prompt phrases must be distinct after normalisation")\n'
        '    return cleaned\n'
        '\n'
        '\n'
        'def validate_image(image: Any) -> Image.Image:\n'
        '    if not isinstance(image, Image.Image):\n'
        '        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")\n'
        '    width, height = image.size\n'
        '    if min(width, height) < MIN_IMAGE_SIDE:\n'
        '        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")\n'
        '    if max(width, height) > MAX_IMAGE_SIDE:\n'
        '        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")\n'
        '    return image.convert("RGB")\n'
        '\n'
        '\n'
        'def _check_threshold(name: str, value: Any) -> float:\n'
        '    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:\n'
        '        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")\n'
        '    return float(value)\n'
        '\n'
        '\n'
        'INPUT_SCHEMA: dict[str, Any] = {\n'
        '    "input": "one PIL.Image.Image (any mode, converted to RGB) plus 1..MAX_PROMPTS free-text phrases",\n'
        '    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],\n'
        '    "prompts": [1, MAX_PROMPTS],\n'
        '    "prompt_chars": [1, MAX_PROMPT_CHARS],\n'
        '    "prompt_tokens_per_query": [1, MAX_TEXT_TOKENS],\n'
        '    "threshold": [0.0, 1.0],\n'
        '    "preprocessing": (\n'
        '        f"image converted to RGB and resized to {LOGIT_SIZE}x{LOGIT_SIZE} (aspect ratio not preserved, "\n'
        '        "ImageNet mean/std); phrases normalised into one CLIP text query each (format_prompts); the "\n'
        '        f"decoder\'s {LOGIT_SIZE}x{LOGIT_SIZE} logits are passed through a sigmoid and resampled "\n'
        '        "bilinearly to the input size; the mask is probability >= threshold"\n'
        '    ),\n'
        '    "output": (\n'
        '        "per prompt: a boolean mask and a float32 probability map at input resolution, the mask\'s area "\n'
        '        "fraction, tight box and maximum probability; probabilities are uncalibrated sigmoids"\n'
        '    ),\n'
        '}\n'
        '\n'
        '\n'
        'def _check_inputs(image: Any, prompts: Any, threshold: Any) -> tuple[Image.Image, list[str], float]:\n'
        '    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.\n'
        '\n'
        '    ``segment`` and ``validate_inputs`` both route through this function so their acceptance\n'
        '    criteria cannot diverge.\n'
        '    """\n'
        '    rgb = validate_image(image)\n'
        '    queries = format_prompts(prompts)\n'
        '    checked = _check_threshold("threshold", threshold)\n'
        '    return rgb, queries, checked\n'
        '\n'
        '\n'
        'def validate_inputs(\n'
        '    image: Image.Image,\n'
        '    prompts: Sequence[str],\n'
        '    *,\n'
        '    threshold: float = MASK_THRESHOLD,\n'
        '    names: Sequence[str] | None = None,\n'
        ') -> dict[str, Any]:\n'
        '    """Validation stage: return the input manifest (schema, observations, request, verdict).\n'
        '\n'
        '    Rejection is reported by raising exactly as ``segment`` would; a caller that wants the finding\n'
        '    recorded catches the exception and stores ``str(exc)`` under ``findings``.\n'
        '    """\n'
        '    _rgb, queries, checked = _check_inputs(image, prompts, threshold)\n'
        '    if names is not None and len(names) != 1:\n'
        '        raise ValueError("names must have exactly one entry (segment takes one image)")\n'
        '    return {\n'
        '        "schema": dict(INPUT_SCHEMA),\n'
        '        "inputs": [\n'
        '            {\n'
        '                "id": names[0] if names else "image-0",\n'
        '                "mode": image.mode,\n'
        '                "size": list(image.size),\n'
        '                "n_prompts": len(prompts),\n'
        '            }\n'
        '        ],\n'
        '        "queries": queries,\n'
        '        "threshold": checked,\n'
        '        "verdict": "accepted",\n'
        '        "findings": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '\n'
        '\n'
        'def evaluation_report(\n'
        '    result: Mapping[str, Any],\n'
        '    reference_masks: Mapping[str, np.ndarray] | None = None,\n'
        '    *,\n'
        '    sample_kind: str = "synthetic",\n'
        ') -> dict[str, Any]:\n'
        '    """Evaluation stage: a machine-readable report even when nothing is measurable.\n'
        '\n'
        '    With ``reference_masks`` (phrase -> boolean mask at input resolution) the report carries one\n'
        '    ``mask_iou`` entry per reference and their mean (``miou``) as sample-sanity evidence; without them\n'
        '    the verdict is ``not-measurable`` and the report says what labelled data would make the task\n'
        '    measurable.\n'
        '    """\n'
        '    segments = list(result["segments"])\n'
        '    by_prompt = {segment["prompt"]: segment for segment in segments}\n'
        '    base = {\n'
        '        "task": "zero-shot (text-prompted) binary segmentation, one mask per phrase",\n'
        '        "decision_rule": (\n'
        '            "each phrase yields a per-pixel sigmoid over the decoder logits; a pixel belongs to the mask "\n'
        '            "when that sigmoid reaches the threshold; the sigmoid is uncalibrated and masks of different "\n'
        '            "phrases are independent (they may overlap or leave pixels unassigned)"\n'
        '        ),\n'
        '        "threshold": result.get("threshold", MASK_THRESHOLD),\n'
        '        "sample_kind": sample_kind,\n'
        '        "n_prompts": len(segments),\n'
        '        "area_fractions": {segment["prompt"]: segment["area_fraction"] for segment in segments},\n'
        '        "baselines": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '    if not reference_masks:\n'
        '        return {\n'
        '            **base,\n'
        '            "metrics": [],\n'
        '            "verdict": "not-measurable",\n'
        '            "reason": "no reference masks were supplied for the evaluated image",\n'
        '            "needs": (\n'
        '                "labelled masks on your own images with a phrase vocabulary matching the prompts, scored "\n'
        '                "per phrase with mask_iou and aggregated into mean IoU at a stated threshold; no such "\n'
        '                "labelled set ships with this repository"\n'
        '            ),\n'
        '        }\n'
        '    metrics = []\n'
        '    for phrase, reference in reference_masks.items():\n'
        '        key = format_prompts([phrase])[0]\n'
        '        if key not in by_prompt:\n'
        '            raise ValueError(f"reference phrase {phrase!r} was not among the segmented prompts")\n'
        '        metrics.append(\n'
        '            {\n'
        '                "id": "mask_iou",\n'
        '                "reference": key,\n'
        '                "value": mask_iou(by_prompt[key]["mask"], reference),\n'
        '                "reference_area_fraction": float(np.asarray(reference, dtype=bool).mean()),\n'
        '                "predicted_area_fraction": by_prompt[key]["area_fraction"],\n'
        '                "estimation": "one reference mask per phrase on a single scene, no dispersion estimate",\n'
        '            }\n'
        '        )\n'
        '    miou = sum(entry["value"] for entry in metrics) / len(metrics)\n'
        '    metrics.append(\n'
        '        {\n'
        '            "id": "miou",\n'
        '            "value": miou,\n'
        '            "estimation": f"mean of {len(metrics)} mask_iou value(s) on one scene, no dispersion estimate",\n'
        '        }\n'
        '    )\n'
        '    return {\n'
        '        **base,\n'
        '        "metrics": metrics,\n'
        '        "verdict": "sample-sanity",\n'
        '        "reason": (\n'
        '            f"{len(metrics) - 1} reference mask(s) on one tutorial sample; geometry sanity evidence, "\n'
        '            "not a segmentation benchmark"\n'
        '        ),\n'
        '        "needs": (\n'
        '            "a labelled mask set from the deployment domain with a matching phrase vocabulary for any "\n'
        '            "mean-IoU claim"\n'
        '        ),\n'
        '    }\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class ClipSegSegmentationPipeline:\n'
        '    """Text-prompted (zero-shot) binary segmentation over the pinned CLIPSeg rd64-refined checkpoint."""\n'
        '\n'
        '    _runner: Callable[[Image.Image, list[str]], np.ndarray]\n'
        '    device: str\n'
        '    _batch_runner: Callable[[list[Image.Image], list[str]], list[np.ndarray]] | None = None\n'
        '    _model: Any = None\n'
        '    _processor: Any = None\n'
        '    weight_sha256: str | None = None\n'
        '    adapter: dict[str, Any] | None = None\n'
        '\n'
        '    @classmethod\n'
        '    def from_pretrained(\n'
        '        cls,\n'
        '        device: str | None = None,\n'
        '        weights_dir: str | Path | None = None,\n'
        '        allow_download: bool = False,\n'
        '    ) -> ClipSegSegmentationPipeline:\n'
        '        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR\n'
        '        if (root / MANIFEST_NAME).is_file():\n'
        '            stage_missing_files(root, allow_download=allow_download)\n'
        '            verify_snapshot(root)\n'
        '            source, kwargs = str(root), {"local_files_only": True}\n'
        '        elif allow_download:\n'
        '            source, kwargs = MODEL_ID, {}\n'
        '        else:\n'
        '            raise FileNotFoundError(\n'
        '                f"no verified snapshot at {root} and allow_download=False; "\n'
        '                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"\n'
        '            )\n'
        '        # Refuse invalid snapshots before importing model libraries.\n'
        '        import torch\n'
        '        from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor\n'
        '\n'
        '        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n'
        '        processor = CLIPSegProcessor.from_pretrained(\n'
        '            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs\n'
        '        )\n'
        '        model = CLIPSegForImageSegmentation.from_pretrained(\n'
        '            source, revision=MODEL_REVISION, trust_remote_code=False, dtype=torch.float32, **kwargs\n'
        '        )\n'
        '        model = model.to(resolved_device).eval()\n'
        '        for param in model.parameters():\n'
        '            param.requires_grad_(False)\n'
        '        weight_sha256 = _weight_digest(root) if (root / MANIFEST_NAME).is_file() else None\n'
        '\n'
        '        def runner(image: Image.Image, queries: list[str]) -> np.ndarray:\n'
        '            # One image copy per phrase: CLIPSeg conditions the decoder on each text query separately.\n'
        '            inputs = processor(\n'
        '                text=queries, images=[image] * len(queries), padding=True, return_tensors="pt"\n'
        '            ).to(resolved_device)\n'
        '            with torch.inference_mode():\n'
        '                logits = model(**inputs).logits\n'
        '            if logits.dim() == 2:  # a single prompt may come back squeezed\n'
        '                logits = logits.unsqueeze(0)\n'
        '            probs = torch.sigmoid(logits).unsqueeze(1)\n'
        '            probs = torch.nn.functional.interpolate(\n'
        '                probs, size=(image.height, image.width), mode="bilinear", align_corners=False\n'
        '            )\n'
        '            return probs.squeeze(1).float().cpu().numpy()\n'
        '\n'
        '        def batch_runner(images: list[Image.Image], queries: list[str]) -> list[np.ndarray]:\n'
        '            # One phrase per image: each (image, phrase) pair is one decoder query.\n'
        '            inputs = processor(text=queries, images=images, padding=True, return_tensors="pt").to(resolved_device)\n'
        '            with torch.inference_mode():\n'
        '                logits = model(**inputs).logits\n'
        '            if logits.dim() == 2:\n'
        '                logits = logits.unsqueeze(0)\n'
        '            probs = torch.sigmoid(logits).unsqueeze(1)\n'
        '            out = []\n'
        '            for k, image in enumerate(images):\n'
        '                resampled = torch.nn.functional.interpolate(probs[k : k + 1], size=(image.height, image.width), mode="bilinear", align_corners=False)\n'
        '                out.append(resampled[0, 0].float().cpu().numpy())\n'
        '            return out\n'
        '\n'
        '        return cls(runner, resolved_device, batch_runner, model, processor, weight_sha256, None)\n'
        '\n'
        '    def _require_model(self) -> tuple[Any, Any]:\n'
        '        if self._model is None or self._processor is None:\n'
        '            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained")\n'
        '        return self._model, self._processor\n'
        '\n'
        '    def segment(\n'
        '        self,\n'
        '        image: Image.Image,\n'
        '        prompts: Sequence[str],\n'
        '        *,\n'
        '        threshold: float = MASK_THRESHOLD,\n'
        '    ) -> dict[str, Any]:\n'
        '        """Segment each phrase in ``prompts``; masks and probability maps are at input resolution."""\n'
        '        rgb, queries, checked = _check_inputs(image, prompts, threshold)\n'
        '        probs = np.asarray(self._runner(rgb, queries), dtype=np.float32)\n'
        '        if probs.shape != (len(queries), rgb.height, rgb.width):\n'
        '            raise RuntimeError(\n'
        '                f"backend returned probability maps of shape {probs.shape}, "\n'
        '                f"expected {(len(queries), rgb.height, rgb.width)}"\n'
        '            )\n'
        '        if probs.min() < 0.0 or probs.max() > 1.0:\n'
        '            raise RuntimeError("backend returned probabilities outside [0, 1]")\n'
        '        segments = []\n'
        '        for query, prob in zip(queries, probs, strict=True):\n'
        '            mask = prob >= checked\n'
        '            segments.append(\n'
        '                {\n'
        '                    "prompt": query,\n'
        '                    "mask": mask,\n'
        '                    "probability": prob,\n'
        '                    "area_fraction": float(mask.mean()),\n'
        '                    "max_probability": float(prob.max()),\n'
        '                    "bbox": mask_bbox(mask),\n'
        '                }\n'
        '            )\n'
        '        return {\n'
        '            "segments": segments,\n'
        '            "queries": queries,\n'
        '            "threshold": checked,\n'
        '            "width": rgb.width,\n'
        '            "height": rgb.height,\n'
        '            "model_id": MODEL_ID,\n'
        '            "model_revision": MODEL_REVISION,\n'
        '        }\n'
        '\n'
        '    # ------------------------------------------------------------------------------------------------------\n'
        '    # Adaptation contract: batched (image, phrase) segmentation, corpus evaluation, bounded fine-tuning, artifacts\n'
        '    # ------------------------------------------------------------------------------------------------------\n'
        '\n'
        '    def segment_batch(\n'
        '        self,\n'
        '        pairs: Sequence[tuple[Image.Image, str]],\n'
        '        *,\n'
        '        threshold: float = MASK_THRESHOLD,\n'
        '        batch_size: int = EVAL_BATCH_SIZE,\n'
        '        progress: Callable[[int, int], None] | None = None,\n'
        '    ) -> list[dict[str, Any]]:\n'
        '        """Segment one phrase per image for many (image, phrase) pairs, `batch_size` pairs per forward; one\n'
        '        ``{prompt, mask, probability, area_fraction, max_probability, bbox}`` per pair, in order. With an injected\n'
        '        runner and no batch runner the pairs are segmented one by one through the runner."""\n'
        '        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:\n'
        '            raise ValueError("batch_size must be an int in 1..64")\n'
        '        checked = [_check_inputs(image, [prompt], threshold) for image, prompt in pairs]\n'
        '        cut = _check_threshold("threshold", threshold)\n'
        '        out: list[dict[str, Any]] = []\n'
        '        for start in range(0, len(checked), batch_size):\n'
        '            batch = checked[start : start + batch_size]\n'
        '            if self._batch_runner is not None:\n'
        '                probs = self._batch_runner([rgb for rgb, _, _ in batch], [q[0] for _, q, _ in batch])\n'
        '            else:\n'
        '                probs = [np.asarray(self._runner(rgb, q), dtype=np.float32)[0] for rgb, q, _ in batch]\n'
        '            if len(probs) != len(batch):\n'
        '                raise RuntimeError(f"backend returned {len(probs)} probability maps for {len(batch)} pairs")\n'
        '            for (rgb, queries, _), prob in zip(batch, probs, strict=True):\n'
        '                prob = np.asarray(prob, dtype=np.float32)\n'
        '                if prob.shape != (rgb.height, rgb.width):\n'
        '                    raise RuntimeError(f"backend returned a probability map of shape {prob.shape}, expected {(rgb.height, rgb.width)}")\n'
        '                if prob.min() < 0.0 or prob.max() > 1.0:\n'
        '                    raise RuntimeError("backend returned probabilities outside [0, 1]")\n'
        '                mask = prob >= cut\n'
        '                out.append({"prompt": queries[0], "mask": mask, "probability": prob, "area_fraction": float(mask.mean()), "max_probability": float(prob.max()), "bbox": mask_bbox(mask)})\n'
        '            if progress is not None:\n'
        '                progress(len(out), len(checked))\n'
        '        return out\n'
        '\n'
        '    def evaluate(\n'
        '        self,\n'
        '        records: Sequence[Mapping[str, Any]],\n'
        '        *,\n'
        '        threshold: float = MASK_THRESHOLD,\n'
        '        batch_size: int = EVAL_BATCH_SIZE,\n'
        '        progress: Callable[[int, int], None] | None = None,\n'
        '    ) -> dict[str, Any]:\n'
        '        """Segment every validated record\'s phrase on its image and score the thresholded masks against the record\n'
        '        masks with ``metrics.segmentation_metrics`` (mean IoU, micro IoU, Dice, pixel precision and recall). Works\n'
        '        with an injected runner too."""\n'
        '        from .metrics import segmentation_metrics\n'
        '        from .samples import validate_dataset\n'
        '\n'
        '        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]\n'
        '        started = time.perf_counter()\n'
        '        items = self.segment_batch([(r["image"], r["prompt"]) for r in checked], threshold=threshold, batch_size=batch_size, progress=progress)\n'
        '        metrics = segmentation_metrics([item["mask"] for item in items], checked)\n'
        '        metrics.update(\n'
        '            {\n'
        '                "threshold": float(threshold),\n'
        '                "predicted_area_fraction": sum(item["area_fraction"] for item in items) / len(items),\n'
        '                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",\n'
        '                "adapted": self.adapter is not None,\n'
        '                "seconds": round(time.perf_counter() - started, 3),\n'
        '                "model_id": MODEL_ID,\n'
        '                "model_revision": MODEL_REVISION,\n'
        '            }\n'
        '        )\n'
        '        return metrics\n'
        '\n'
        '    def _encode(self, records: Sequence[Mapping[str, Any]], batch_size: int, progress: Callable[[int, int], None] | None = None) -> tuple[list[Any], Any, Any]:\n'
        '        """Run the frozen towers once per record: the CLIP vision activations at `EXTRACT_LAYERS` (stored fp16 on\n'
        '        the host), the phrase embeddings, and the reference masks resampled to the decoder\'s logit grid."""\n'
        '        model, processor = self._require_model()\n'
        '        import torch\n'
        '\n'
        '        activations: list[list[Any]] = [[] for _ in EXTRACT_LAYERS]\n'
        '        conditionals, targets = [], []\n'
        '        with torch.no_grad():\n'
        '            for start in range(0, len(records), batch_size):\n'
        '                batch = records[start : start + batch_size]\n'
        '                inputs = processor(text=[r["prompt"] for r in batch], images=[r["image"] for r in batch], padding=True, return_tensors="pt").to(self.device)\n'
        '                vision = model.clip.vision_model(pixel_values=inputs["pixel_values"], output_hidden_states=True)\n'
        '                for slot, layer in enumerate(EXTRACT_LAYERS):\n'
        '                    activations[slot].append(vision.hidden_states[layer + 1].to("cpu", torch.float16))\n'
        '                conditionals.append(model.clip.get_text_features(inputs["input_ids"], attention_mask=inputs["attention_mask"]).to("cpu"))\n'
        '                resized = [torch.nn.functional.interpolate(torch.from_numpy(np.asarray(r["mask"], dtype=np.float32))[None, None], size=(LOGIT_SIZE, LOGIT_SIZE), mode="nearest")[0, 0] for r in batch]\n'
        '                targets.append(torch.stack(resized))\n'
        '                if progress is not None:\n'
        '                    progress(min(start + batch_size, len(records)), len(records))\n'
        '        return [torch.cat(slot) for slot in activations], torch.cat(conditionals), torch.cat(targets)\n'
        '\n'
        '    def _decoder_logits(self, activations: Sequence[Any], conditionals: Any) -> Any:\n'
        '        """The decoder on cached tower outputs: exactly what `CLIPSegForImageSegmentation.forward` computes after\n'
        '        its frozen CLIP steps (parity with the full forward is asserted by the model-backed tests)."""\n'
        '        model, _ = self._require_model()\n'
        '        import torch\n'
        '\n'
        '        return model.decoder([a.to(self.device, torch.float32) for a in activations], conditionals.to(self.device, torch.float32)).logits\n'
        '\n'
        '    def adapt(\n'
        '        self,\n'
        '        train: Sequence[Mapping[str, Any]],\n'
        '        val: Sequence[Mapping[str, Any]] | None,\n'
        '        *,\n'
        '        epochs: int = 8,\n'
        '        lr: float = 3e-4,\n'
        '        batch_size: int = 8,\n'
        '        seed: int = 0,\n'
        '        threshold: float = MASK_THRESHOLD,\n'
        '        progress: Callable[[Mapping[str, Any]], None] | None = None,\n'
        '    ) -> dict[str, Any]:\n'
        '        """Bounded fine-tuning of the CLIPSeg decoder on labelled (image, phrase, mask) records with the per-pixel\n'
        "        binary cross-entropy over the decoder's logit grid — the loss the upstream authors trained the decoder\n"
        '        with. The frozen CLIP towers are run once per record under no gradient and their outputs cached (the\n'
        '        vision activations at `EXTRACT_LAYERS` and the phrase embedding), so each step runs only the decoder; the\n'
        "        logits equal the full model's exactly. AdamW (no weight decay), gradient clipping at `GRAD_CLIP`, seeded\n"
        "        shuffling, no scheduler, no augmentation. Epoch 0 records the frozen model's validation rates at\n"
        '        `threshold`; the epoch with the highest validation mean IoU (the earliest on ties) is kept. On any\n'
        '        exception the frozen decoder is restored."""\n'
        '        from .metrics import segmentation_metrics\n'
        '        from .samples import validate_dataset\n'
        '\n'
        '        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 100:\n'
        '            raise ValueError("epochs must be an int in 1..100")\n'
        '        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 128:\n'
        '            raise ValueError("batch_size must be an int in 1..128")\n'
        '        if not isinstance(lr, int | float) or isinstance(lr, bool) or not 0 < lr <= 1e-2:\n'
        '            raise ValueError("lr must be a number in (0, 1e-2]")\n'
        '        cut = _check_threshold("threshold", threshold)\n'
        '        train_checked = validate_dataset(train)["records"]\n'
        '        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None\n'
        '        model, _processor = self._require_model()\n'
        '        import torch\n'
        '\n'
        '        started = time.perf_counter()\n'
        '        names = _trainable_names(model)\n'
        '        params = {name: param for name, param in model.named_parameters() if name in set(names)}\n'
        '        n_trainable = sum(p.numel() for p in params.values())\n'
        '        backup = {name: param.detach().clone() for name, param in params.items()}\n'
        '        previous_adapter = self.adapter\n'
        '        cudnn_flags = torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark\n'
        '        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False\n'
        '        try:\n'
        '            model.eval()\n'
        '            activations, conditionals, targets = self._encode(train_checked, EVAL_BATCH_SIZE)\n'
        '            cached_val = self._encode(val_checked, EVAL_BATCH_SIZE) if val_checked is not None else None\n'
        '            cache_seconds = round(time.perf_counter() - started, 3)\n'
        '\n'
        '            def score_val() -> dict[str, Any] | None:\n'
        '                if val_checked is None or cached_val is None:\n'
        '                    return None\n'
        '                with torch.no_grad():\n'
        '                    masks = []\n'
        '                    for start in range(0, len(val_checked), EVAL_BATCH_SIZE):\n'
        '                        logits = self._decoder_logits([a[start : start + EVAL_BATCH_SIZE].float() for a in cached_val[0]], cached_val[1][start : start + EVAL_BATCH_SIZE])\n'
        '                        probs = torch.sigmoid(logits).unsqueeze(1)\n'
        '                        for k, record in enumerate(val_checked[start : start + EVAL_BATCH_SIZE]):\n'
        '                            prob = torch.nn.functional.interpolate(probs[k : k + 1], size=(record["image"].height, record["image"].width), mode="bilinear", align_corners=False)[0, 0]\n'
        '                            masks.append((prob >= cut).cpu().numpy())\n'
        '                m = segmentation_metrics(masks, val_checked)\n'
        '                return {k: m[k] for k in ("miou", "iou_micro", "dice", "pixel_precision", "pixel_recall", "n")}\n'
        '\n'
        '            for name, param in model.named_parameters():\n'
        '                param.requires_grad_(name in params)\n'
        '            history: list[dict[str, Any]] = [{"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}]\n'
        '            if progress is not None:\n'
        '                progress(history[-1])\n'
        '            best_epoch, best_score = 0, (history[0]["val"] or {}).get("miou", -1.0)\n'
        '            best_state = {name: param.detach().clone() for name, param in params.items()}\n'
        '            optimizer = torch.optim.AdamW(list(params.values()), lr=lr, weight_decay=0.0)\n'
        '            rng = random.Random(seed)\n'
        '            torch.manual_seed(seed)\n'
        '            order = list(range(len(train_checked)))\n'
        '            for epoch in range(1, epochs + 1):\n'
        '                rng.shuffle(order)\n'
        '                model.decoder.train()\n'
        '                total, steps = 0.0, 0\n'
        '                for start in range(0, len(order), batch_size):\n'
        '                    idx = torch.tensor(order[start : start + batch_size])\n'
        '                    optimizer.zero_grad(set_to_none=True)\n'
        '                    logits = self._decoder_logits([a[idx].float() for a in activations], conditionals[idx])\n'
        '                    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets[idx].to(self.device))\n'
        '                    loss.backward()\n'
        '                    torch.nn.utils.clip_grad_norm_(list(params.values()), GRAD_CLIP)\n'
        '                    optimizer.step()\n'
        '                    total += float(loss.detach())\n'
        '                    steps += 1\n'
        '                model.eval()\n'
        '                entry = {"epoch": epoch, "train_loss": round(total / max(steps, 1), 5), "val": score_val()}\n'
        '                history.append(entry)\n'
        '                if progress is not None:\n'
        '                    progress(entry)\n'
        '                score = (entry["val"] or {}).get("miou")\n'
        '                if val_checked is None or (score is not None and score > best_score):\n'
        '                    best_epoch, best_score = epoch, score if score is not None else best_score\n'
        '                    best_state = {name: param.detach().clone() for name, param in params.items()}\n'
        '            with torch.no_grad():\n'
        '                for name, param in params.items():\n'
        '                    param.copy_(best_state[name])\n'
        '        except BaseException:\n'
        '            with torch.no_grad():\n'
        '                for name, param in params.items():\n'
        '                    param.copy_(backup[name])\n'
        '            model.eval()\n'
        '            self.adapter = previous_adapter\n'
        '            raise\n'
        '        finally:\n'
        '            for param in model.parameters():\n'
        '                param.requires_grad_(False)\n'
        '            model.eval()\n'
        '            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = cudnn_flags\n'
        '        self.adapter = {\n'
        '            "threshold": cut,\n'
        '            "trainable_names": names,\n'
        '            "n_trainable": n_trainable,\n'
        '            "n_total": sum(p.numel() for p in model.parameters()),\n'
        '            "extract_layers": list(EXTRACT_LAYERS),\n'
        '            "epochs": epochs,\n'
        '            "batch_size": batch_size,\n'
        '            "best_epoch": best_epoch,\n'
        '            "selection": "highest validation mean IoU" if val_checked is not None else "final epoch (no validation split)",\n'
        '            "loss": f"per-pixel binary cross-entropy over the {LOGIT_SIZE}x{LOGIT_SIZE} decoder logits against the reference mask resampled to that grid; computed on cached CLIP activations",\n'
        '            "lr": float(lr),\n'
        '            "seed": seed,\n'
        '            "n_train": len(train_checked),\n'
        '            "n_val": len(val_checked) if val_checked is not None else 0,\n'
        '            "cache_seconds": cache_seconds,\n'
        '            "history": history,\n'
        '            "seconds": round(time.perf_counter() - started, 3),\n'
        '        }\n'
        '        return dict(self.adapter)\n'
        '\n'
        '    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:\n'
        '        """Write the trained tensors as safetensors plus a manifest naming the base, the digests, the threshold and\n'
        '        the training configuration. Requires a prior `adapt`."""\n'
        '        model, _processor = self._require_model()  # refuse before importing torch\n'
        '        import torch\n'
        '        from safetensors.torch import save_file\n'
        '\n'
        '        if self.adapter is None:\n'
        '            raise RuntimeError("nothing to save: call adapt() first")\n'
        '        out = Path(output_dir)\n'
        '        out.mkdir(parents=True, exist_ok=True)\n'
        '        names = list(self.adapter["trainable_names"])\n'
        '        state = model.state_dict()\n'
        '        tensors = {name: state[name].detach().cpu().contiguous() for name in names}\n'
        '        weights = out / ADAPTER_WEIGHTS\n'
        '        save_file(tensors, str(weights), metadata={"format": "pt"})\n'
        '        manifest = {\n'
        '            "format": ARTIFACT_FORMAT,\n'
        '            "version": ARTIFACT_VERSION,\n'
        '            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHTS_FILE, "weight_sha256": self.weight_sha256},\n'
        '            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},\n'
        '            "history": self.adapter["history"],\n'
        '            "tensors": names,\n'
        '            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],\n'
        '            "torch": torch.__version__,\n'
        '            "metadata": dict(metadata or {}),\n'
        '        }\n'
        '        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:\n'
        '            json.dump(manifest, handle, indent=2, ensure_ascii=False)\n'
        '        return out\n'
        '\n'
        '    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:\n'
        '        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact\n'
        '        tensor set. Refuses tensors outside the decoder."""\n'
        '        model, _processor = self._require_model()  # refuse before importing safetensors\n'
        '        from safetensors.torch import load_file\n'
        '\n'
        '        artifact = Path(artifact_dir)\n'
        '        manifest_path = artifact / ADAPTER_MANIFEST\n'
        '        if not manifest_path.is_file():\n'
        '            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")\n'
        '        with open(manifest_path, encoding="utf-8") as handle:\n'
        '            manifest = json.load(handle)\n'
        '        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")\n'
        '        expected = _trainable_names(model)\n'
        '        if sorted(manifest["tensors"]) != sorted(expected):\n'
        '            raise ValueError("artifact tensor set does not match its recorded configuration")\n'
        '        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))\n'
        '        if sorted(tensors) != sorted(expected):\n'
        '            raise ValueError("artifact tensor names differ from the manifest")\n'
        '        state = model.state_dict()\n'
        '        for name, tensor in tensors.items():\n'
        '            if tuple(tensor.shape) != tuple(state[name].shape):\n'
        '                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")\n'
        '        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)\n'
        '        model.eval()\n'
        '        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}\n'
        '        return dict(self.adapter)\n'
        '\n'
        '    @classmethod\n'
        '    def from_artifact(\n'
        '        cls,\n'
        '        artifact_dir: str | Path,\n'
        '        *,\n'
        '        device: str | None = None,\n'
        '        weights_dir: str | Path | None = None,\n'
        '        allow_download: bool = False,\n'
        '    ) -> ClipSegSegmentationPipeline:\n'
        '        """Check the adapter manifest against the base snapshot\'s recorded weight digest, load the verified base, then\n'
        '        overlay the adapter (checked again, and the tensor set, before deserialising). A refused manifest never loads\n'
        '        a model."""\n'
        '        artifact = Path(artifact_dir)\n'
        '        manifest_path = artifact / ADAPTER_MANIFEST\n'
        '        if not manifest_path.is_file():\n'
        '            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")\n'
        '        with open(manifest_path, encoding="utf-8") as handle:\n'
        '            manifest = json.load(handle)\n'
        '        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR\n'
        '        _check_artifact_manifest(manifest, artifact, _weight_digest(root) or "")\n'
        '        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)\n'
        '        pipe.load_artifact(artifact_dir)\n'
        '        return pipe\n'
    ),
    'clipseg_reference/samples.py': (
        '"""Labelled (image, phrase, mask) datasets for the adaptation contract: the digest-pinned FoodSeg103 sample, the\n'
        'record contract and its structural validation, image-disjoint splitting, and the BYOD loader.\n'
        '\n'
        "A record is ``{id, image, prompt, mask}`` where ``image`` is a PIL image (sides within the pipeline's ceilings),\n"
        '``prompt`` the phrase the mask answers (normalised like ``format_prompts`` normalises a query) and ``mask`` a\n'
        "boolean array (or a Pillow-decodable image whose non-zero pixels are the mask) of the image's height × width\n"
        'with at least one true pixel.\n'
        '\n'
        'The default sample is drawn from FoodSeg103 (Wu et al. 2021, LARC-CMU-SMU; **Apache-2.0**; a curated sample of\n'
        'Recipe1M food photographs with pixel-wise ingredient masks) as converted to parquet by the Hugging Face Hub at an\n'
        'immutable revision: the first ``CORPUS_ROW_GROUPS`` row groups of the validation shard are read with HTTPS range\n'
        "requests (about 5 MB each; the shard's declared size is checked first and every row group's content is refused\n"
        'unless its SHA-256 matches the pin). Each image yields one record: the ingredient class that covers the most\n'
        "pixels becomes the phrase (its FoodSeg103 name, e.g. ``bread``, ``chicken duck``, ``steak``) and that class's\n"
        'pixels the mask; images whose largest class is ``background`` or ``other ingredients`` are skipped (none of the\n'
        "800 are). The domain gap to the model's PhraseCut training distribution is the point of the sample: the frozen\n"
        'model must already know what the phrase means, and the adapter can only sharpen where it draws the boundary.\n'
        '"""\n'
        '# ruff: noqa: E501  -- record and pin literals are kept on single lines\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import csv\n'
        'import hashlib\n'
        'import io\n'
        'import random\n'
        'import re\n'
        'import urllib.request\n'
        'import zipfile\n'
        'from collections.abc import Mapping, Sequence\n'
        'from pathlib import Path\n'
        'from typing import Any\n'
        '\n'
        'import numpy as np\n'
        'from PIL import Image\n'
        '\n'
        'from .pipeline import MODEL_ID, format_prompts, validate_image\n'
        '\n'
        'CORPUS_NAME = "FoodSeg103 (validation split), first eight parquet row groups"\n'
        'CORPUS_REPO = "EduardoPacheco/FoodSeg103"\n'
        'CORPUS_REVISION = "176acc3edd2432ee126bda6fb01469eadeb018df"  # refs/convert/parquet commit on the Hub\n'
        'CORPUS_FILE = "default/validation/0000.parquet"\n'
        'CORPUS_BYTES = 115_186_173\n'
        'CORPUS_ROWS = 2_135\n'
        'CORPUS_ROW_GROUPS = 8  # of 22; 100 images each\n'
        'CORPUS_LICENSE = "Apache-2.0 (FoodSeg103, LARC-CMU-SMU; Wu, Fu, Liu, Lim, Hoi, Sun, ACM MM 2021; images curated from Recipe1M)"\n'
        'CORPUS_URL = f"https://huggingface.co/datasets/{CORPUS_REPO}/resolve/{CORPUS_REVISION}/{CORPUS_FILE}"\n'
        '# SHA-256 over the concatenated image bytes + label-mask bytes + UTF-8 id of each row, in row order, and that byte total.\n'
        'ROW_GROUP_PINS: dict[int, tuple[str, int]] = {\n'
        '    0: ("931827f16c33a0548ad26c456c8495fc3949c79f7950869eabaff44f748e8f78", 5_090_593),\n'
        '    1: ("a784bb0f23e413eef8d0dad90e504f398bde1a44d2110dced06bdea8953f08bf", 5_207_382),\n'
        '    2: ("f8a8e15d28c6678d344324b3c972008e642878b25ebff9f481196707c39fc32e", 5_467_905),\n'
        '    3: ("adebbbc59740e87c5eb0578f029174872bd95b7fa5b1178c708e61c766b73fa5", 6_071_430),\n'
        '    4: ("fd181120d4966d20e47742232e4102d45247b3f14cf7e2a8733b9d08d1aa152d", 4_758_783),\n'
        '    5: ("c3af4dda9af15abf54d5f4494fd07df925d57c1d9ec654eb30a6c1bd30f50b61", 5_020_081),\n'
        '    6: ("0597cb610e859eb400c746dfba37abc446afd34d897b573b1c68453c5c9d80d9", 4_784_703),\n'
        '    7: ("9e9aac90865e09124fa6c3e89cd676ef2dcd65f87532c002574a545671bae4a9", 5_899_238),\n'
        '}\n'
        'DEFAULT_CACHE_DIR = Path("weights") / "foodseg103"\n'
        '\n'
        '# The FoodSeg103 class vocabulary (dataset card, ids 0..103); 0 = background and 103 = other ingredients never become a phrase.\n'
        'FOODSEG103_CLASSES = ("background", "candy", "egg tart", "french fries", "chocolate", "biscuit", "popcorn", "pudding", "ice cream", "cheese butter", "cake", "wine", "milkshake", "coffee", "juice", "milk", "tea", "almond", "red beans", "cashew", "dried cranberries", "soy", "walnut", "peanut", "egg", "apple", "date", "apricot", "avocado", "banana", "strawberry", "cherry", "blueberry", "raspberry", "mango", "olives", "peach", "lemon", "pear", "fig", "pineapple", "grape", "kiwi", "melon", "orange", "watermelon", "steak", "pork", "chicken duck", "sausage", "fried meat", "lamb", "sauce", "crab", "fish", "shellfish", "shrimp", "soup", "bread", "corn", "hamburg", "pizza", "hanamaki baozi", "wonton dumplings", "pasta", "noodles", "rice", "pie", "tofu", "eggplant", "potato", "garlic", "cauliflower", "tomato", "kelp", "seaweed", "spring onion", "rape", "ginger", "okra", "lettuce", "pumpkin", "cucumber", "white radish", "carrot", "asparagus", "bamboo shoots", "broccoli", "celery stick", "cilantro '
        'mint", "snow peas", "cabbage", "bean sprouts", "onion", "pepper", "green beans", "French beans", "king oyster mushroom", "shiitake", "enoki mushroom", "oyster mushroom", "white button mushroom", "salad", "other ingredients")\n'
        'EXCLUDED_CLASS_IDS = (0, 103)\n'
        '\n'
        'SAMPLE_SEED = 42\n'
        'SAMPLE_SPLIT = {"train": 600, "validation": 60, "test": 140}  # of the 800 images the eight row groups hold\n'
        'SAMPLE_DIGEST = "d8123852b1f1c8dd7054c77ab36cf408738f48c7902678109d620165c2a6d4bf"  # dataset_digest over the three default splits together; tests pin it\n'
        'MIN_RECORDS = 8\n'
        'MAX_RECORDS = 5_000\n'
        'MIN_MASK_PIXELS = 1\n'
        '_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")\n'
        '\n'
        '\n'
        'def _sha256_bytes(data: bytes) -> str:\n'
        '    return hashlib.sha256(data).hexdigest()\n'
        '\n'
        '\n'
        'class _HttpRangeFile(io.RawIOBase):\n'
        '    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to read a\n'
        '    parquet footer and a few row groups without downloading the file)."""\n'
        '\n'
        '    def __init__(self, url: str, size: int) -> None:\n'
        '        self.url, self.size, self.pos = url, size, 0\n'
        '        self.fetched = 0\n'
        '\n'
        '    def readable(self) -> bool:\n'
        '        return True\n'
        '\n'
        '    def seekable(self) -> bool:\n'
        '        return True\n'
        '\n'
        '    def tell(self) -> int:\n'
        '        return self.pos\n'
        '\n'
        '    def seek(self, offset: int, whence: int = 0) -> int:\n'
        '        base = {0: 0, 1: self.pos, 2: self.size}[whence]\n'
        '        self.pos = max(0, base + offset)\n'
        '        return self.pos\n'
        '\n'
        '    def read(self, n: int = -1) -> bytes:\n'
        '        if n is None or n < 0:\n'
        '            n = self.size - self.pos\n'
        '        if n <= 0 or self.pos >= self.size:\n'
        '            return b""\n'
        '        end = min(self.size, self.pos + n) - 1\n'
        '        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}", "User-Agent": "clipseg-segmentation-pipeline"})\n'
        '        with urllib.request.urlopen(request, timeout=300) as response:  # noqa: S310 (pinned https URL)\n'
        '            if response.status != 206:\n'
        '                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")\n'
        '            data = response.read()\n'
        '        self.fetched += len(data)\n'
        '        self.pos += len(data)\n'
        '        return data\n'
        '\n'
        '    def readinto(self, buffer: Any) -> int:\n'
        '        data = self.read(len(buffer))\n'
        '        buffer[: len(data)] = data\n'
        '        return len(data)\n'
        '\n'
        '\n'
        'def _declared_size(url: str) -> int:\n'
        '    request = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "clipseg-segmentation-pipeline"})\n'
        '    with urllib.request.urlopen(request, timeout=60) as response:  # noqa: S310 (pinned https URL)\n'
        '        length = response.headers.get("Content-Length")\n'
        '    if length is None:\n'
        '        raise ValueError(f"{url}: no Content-Length in the HEAD response")\n'
        '    return int(length)\n'
        '\n'
        '\n'
        'def _group_digest(rows: Sequence[Mapping[str, Any]]) -> tuple[str, int]:\n'
        '    digest, total = hashlib.sha256(), 0\n'
        '    for row in rows:\n'
        '        for chunk in (row["image"]["bytes"], row["label"]["bytes"], str(row["id"]).encode("utf-8")):\n'
        '            digest.update(chunk)\n'
        '            total += len(chunk)\n'
        '    return digest.hexdigest(), total\n'
        '\n'
        '\n'
        'def fetch_corpus(\n'
        '    *, cache_dir: str | Path | None = None, groups: Sequence[int] | None = None, opener: Any = None\n'
        ') -> dict[int, list[dict[str, Any]]]:\n'
        '    """Return the pinned row groups as lists of `{image, label, id}` (JPEG/PNG bytes, label-mask PNG bytes, source\n'
        '    id), from the cache (one parquet file per row group) or the Hub (footer + the row groups it needs, over range\n'
        "    requests). Every row group's decoded content is refused unless its SHA-256 and byte total match\n"
        '    `ROW_GROUP_PINS`; a fresh fetch also checks the shard\'s declared size and row count."""\n'
        '    import pyarrow.parquet as pq\n'
        '\n'
        '    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR\n'
        '    cache.mkdir(parents=True, exist_ok=True)\n'
        '    wanted = list(groups) if groups is not None else sorted(ROW_GROUP_PINS)\n'
        '    out: dict[int, list[dict[str, Any]]] = {}\n'
        '    reader = None\n'
        '    for group in wanted:\n'
        '        if group not in ROW_GROUP_PINS:\n'
        '            raise ValueError(f"row group {group} has no pin; pinned groups are {sorted(ROW_GROUP_PINS)}")\n'
        '        local = cache / f"validation-rg{group}.parquet"\n'
        '        rows: list[dict[str, Any]] | None = None\n'
        '        if local.is_file():\n'
        '            rows = pq.read_table(local).to_pylist()\n'
        '            if _group_digest(rows) != ROW_GROUP_PINS[group]:\n'
        '                rows = None  # stale or corrupt cache: refetch\n'
        '        if rows is None:\n'
        '            if reader is None:\n'
        '                if opener is not None:\n'
        '                    reader = pq.ParquetFile(opener(CORPUS_URL))\n'
        '                else:\n'
        '                    declared = _declared_size(CORPUS_URL)\n'
        '                    if declared != CORPUS_BYTES:\n'
        '                        raise ValueError(f"{CORPUS_FILE}: declared size {declared} != pinned {CORPUS_BYTES}")\n'
        '                    reader = pq.ParquetFile(io.BufferedReader(_HttpRangeFile(CORPUS_URL, CORPUS_BYTES), buffer_size=1 << 20))\n'
        '                if reader.metadata.num_rows != CORPUS_ROWS:\n'
        '                    raise ValueError(f"{CORPUS_FILE}: {reader.metadata.num_rows} rows, pinned {CORPUS_ROWS}")\n'
        '            table = reader.read_row_group(group, columns=["image", "label", "id"])\n'
        '            rows = table.to_pylist()\n'
        '            digest, total = _group_digest(rows)\n'
        '            if (digest, total) != ROW_GROUP_PINS[group]:\n'
        '                raise ValueError(f"{CORPUS_FILE} row group {group}: sha256 {digest} / {total} bytes != pinned {ROW_GROUP_PINS[group]}")\n'
        '            pq.write_table(table, local)\n'
        '        out[group] = [{"image": r["image"]["bytes"], "label": r["label"]["bytes"], "id": int(r["id"])} for r in rows]\n'
        '    return out\n'
        '\n'
        '\n'
        'def largest_class(label: np.ndarray) -> int | None:\n'
        '    """The FoodSeg103 class id covering the most pixels, background and \'other ingredients\' excluded; None if no\n'
        '    other class is present."""\n'
        '    ids, counts = np.unique(np.asarray(label), return_counts=True)\n'
        '    candidates = [(int(c), int(i)) for i, c in zip(ids, counts, strict=True) if int(i) not in EXCLUDED_CLASS_IDS and 0 <= int(i) < len(FOODSEG103_CLASSES)]\n'
        '    if not candidates:\n'
        '        return None\n'
        '    return max(candidates)[1]\n'
        '\n'
        '\n'
        'def read_corpus(groups: Mapping[int, Sequence[Mapping[str, Any]]]) -> list[dict[str, Any]]:\n'
        '    """Decode the verified row groups into records: one per image, the largest ingredient class as the phrase and\n'
        '    its pixels as the mask (images with no ingredient class besides background / other are skipped)."""\n'
        '    out = []\n'
        '    for group in sorted(groups):\n'
        '        for index, row in enumerate(groups[group]):\n'
        '            label = np.asarray(Image.open(io.BytesIO(row["label"])))\n'
        '            class_id = largest_class(label)\n'
        '            if class_id is None:\n'
        '                continue\n'
        '            image = Image.open(io.BytesIO(row["image"]))\n'
        '            image.load()\n'
        '            out.append(\n'
        '                {\n'
        '                    "id": f"foodseg103-val-{group * 100 + index}",\n'
        '                    "image": image.convert("RGB"),\n'
        '                    "prompt": FOODSEG103_CLASSES[class_id],\n'
        '                    "mask": label == class_id,\n'
        '                    "class_id": class_id,\n'
        '                    "source_id": row["id"],\n'
        '                    "source_row_group": group,\n'
        '                }\n'
        '            )\n'
        '    return out\n'
        '\n'
        '\n'
        'def build_sample_dataset(\n'
        '    records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] | None = None\n'
        ') -> dict[str, list[dict[str, Any]]]:\n'
        '    """Seeded image-level draw: shuffle the records and cut `sizes` (train / validation / test) in order."""\n'
        '    sizes = dict(sizes or SAMPLE_SPLIT)\n'
        '    pool = [dict(r) for r in records]\n'
        '    random.Random(seed).shuffle(pool)\n'
        '    needed = sum(sizes.values())\n'
        '    if len(pool) < needed:\n'
        '        raise ValueError(f"only {len(pool)} records available, need {needed}")\n'
        '    out, cursor = {}, 0\n'
        '    for name, count in sizes.items():\n'
        '        out[name] = pool[cursor : cursor + count]\n'
        '        cursor += count\n'
        '    return out\n'
        '\n'
        '\n'
        'def fetch_sample_dataset(*, cache_dir: str | Path | None = None, seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:\n'
        '    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir)), seed=seed)\n'
        '\n'
        '\n'
        '# ---------------------------------------------------------------------------------------------------------\n'
        '# Record contract\n'
        '# ---------------------------------------------------------------------------------------------------------\n'
        '\n'
        '\n'
        'def _open(image: Any, where: str) -> Image.Image:\n'
        '    if isinstance(image, str | Path):\n'
        '        path = Path(image)\n'
        '        if not path.is_file():\n'
        '            raise ValueError(f"{where}: image file not found: {path}")\n'
        '        image = Image.open(path)\n'
        '        image.load()\n'
        '    if not isinstance(image, Image.Image):\n'
        '        raise ValueError(f"{where}: must be a PIL.Image.Image or a file path")\n'
        '    return image\n'
        '\n'
        '\n'
        'def coerce_mask(mask: Any, size: tuple[int, int], where: str = "mask") -> np.ndarray:\n'
        '    """A boolean height × width array from a boolean/integer array or a Pillow-decodable image (non-zero = true)."""\n'
        '    if isinstance(mask, str | Path):\n'
        '        mask = _open(mask, where)\n'
        '    if isinstance(mask, Image.Image):\n'
        '        array = np.asarray(mask.convert("L")) > 0\n'
        '    else:\n'
        '        try:\n'
        '            array = np.asarray(mask)\n'
        '        except Exception as exc:  # noqa: BLE001\n'
        '            raise ValueError(f"{where}: must be a boolean array or an image") from exc\n'
        '        if array.dtype != bool:\n'
        '            if not np.issubdtype(array.dtype, np.number):\n'
        '                raise ValueError(f"{where}: must be a boolean or numeric array")\n'
        '            array = array != 0\n'
        '    width, height = size\n'
        '    if array.ndim != 2 or array.shape != (height, width):\n'
        '        raise ValueError(f"{where}: shape {array.shape} does not match the image\'s height x width {(height, width)}")\n'
        '    if int(array.sum()) < MIN_MASK_PIXELS:\n'
        '        raise ValueError(f"{where}: the mask has no true pixel")\n'
        '    return array\n'
        '\n'
        '\n'
        'def _check_record(record: Any, index: int) -> dict[str, Any]:\n'
        '    where = f"records[{index}]"\n'
        '    if not isinstance(record, Mapping):\n'
        '        raise ValueError(f"{where} must be a mapping with id/image/prompt/mask")\n'
        '    for key in ("id", "image", "prompt", "mask"):\n'
        '        if key not in record:\n'
        '            raise ValueError(f"{where} is missing {key!r}")\n'
        '    rid = record["id"]\n'
        '    if not isinstance(rid, str) or not _ID_RE.match(rid):\n'
        '        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")\n'
        '    try:\n'
        '        image = validate_image(_open(record["image"], f"{where}.image"))\n'
        '    except (TypeError, ValueError) as exc:\n'
        '        raise ValueError(f"{where}: {exc}") from exc\n'
        '    if not isinstance(record["prompt"], str):\n'
        '        raise ValueError(f"{where}: prompt must be a str")\n'
        '    try:\n'
        '        prompt = format_prompts([record["prompt"]])[0]\n'
        '    except (TypeError, ValueError) as exc:\n'
        '        raise ValueError(f"{where}: {exc}") from exc\n'
        '    mask = coerce_mask(record["mask"], image.size, f"{where}.mask")\n'
        '    item = {"id": rid, "image": image, "prompt": prompt, "mask": mask}\n'
        '    for key in ("class_id", "source_id", "source_row_group"):\n'
        '        if key in record:\n'
        '            item[key] = record[key]\n'
        '    return item\n'
        '\n'
        '\n'
        'def validate_dataset(\n'
        '    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS\n'
        ') -> dict[str, Any]:\n'
        '    """Structural validation of a labelled-mask dataset; raises ValueError before any model import."""\n'
        '    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):\n'
        '        raise ValueError("records must be a list of {id, image, prompt, mask} mappings")\n'
        '    if not min_records <= len(records) <= max_records:\n'
        '        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")\n'
        '    checked, ids = [], set()\n'
        '    for index, record in enumerate(records):\n'
        '        item = _check_record(record, index)\n'
        '        if item["id"] in ids:\n'
        '            raise ValueError(f"duplicate id {item[\'id\']!r}")\n'
        '        ids.add(item["id"])\n'
        '        checked.append(item)\n'
        '    areas = [float(r["mask"].mean()) for r in checked]\n'
        '    widths = [r["image"].width for r in checked]\n'
        '    heights = [r["image"].height for r in checked]\n'
        '    prompts = sorted({r["prompt"] for r in checked})\n'
        '    return {\n'
        '        "records": checked,\n'
        '        "n_records": len(checked),\n'
        '        "n_prompts": len(prompts),\n'
        '        "prompts": prompts,\n'
        '        "mask_area_fraction": {"min": min(areas), "max": max(areas), "mean": sum(areas) / len(areas)},\n'
        '        "image_width": {"min": min(widths), "max": max(widths)},\n'
        '        "image_height": {"min": min(heights), "max": max(heights)},\n'
        '        "digest": dataset_digest(checked),\n'
        '        "model_id": MODEL_ID,\n'
        '    }\n'
        '\n'
        '\n'
        'def image_digest(image: Image.Image) -> str:\n'
        '    """SHA-256 of the decoded RGB pixels (size-prefixed) — the identity a split is made disjoint on."""\n'
        '    rgb = image.convert("RGB")\n'
        '    return _sha256_bytes(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes())\n'
        '\n'
        '\n'
        'def mask_digest(mask: np.ndarray) -> str:\n'
        '    array = np.asarray(mask, dtype=bool)\n'
        '    return _sha256_bytes(f"{array.shape[1]}x{array.shape[0]}:".encode() + np.packbits(array).tobytes())\n'
        '\n'
        '\n'
        'def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:\n'
        '    """Order-independent SHA-256 over (id, image digest, prompt, mask digest)."""\n'
        '    parts = sorted(f"{r[\'id\']}:{image_digest(r[\'image\'])}:{r[\'prompt\']}:{mask_digest(r[\'mask\'])}" for r in records)\n'
        '    return _sha256_bytes("\\n".join(parts).encode("utf-8"))\n'
        '\n'
        '\n'
        'def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:\n'
        '    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""\n'
        '    seen: dict[str, str] = {}\n'
        '    for name, records in splits.items():\n'
        '        for record in records:\n'
        '            key = image_digest(record["image"])\n'
        '            if key in seen and seen[key] != name:\n'
        '                raise ValueError(f"image {record[\'id\']!r} appears in both {seen[key]} and {name}")\n'
        '            seen[key] = name\n'
        '    return {name: len(records) for name, records in splits.items()}\n'
        '\n'
        '\n'
        'def split_dataset(\n'
        '    records: Sequence[Mapping[str, Any]], *, val_fraction: float = 0.15, test_fraction: float = 0.2, seed: int = 0\n'
        ') -> dict[str, list[dict[str, Any]]]:\n'
        '    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating images."""\n'
        '    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):\n'
        '        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")\n'
        '    checked = validate_dataset(records)["records"]\n'
        '    seen: set[str] = set()\n'
        '    unique = []\n'
        '    for record in checked:\n'
        '        key = image_digest(record["image"])\n'
        '        if key not in seen:\n'
        '            seen.add(key)\n'
        '            unique.append(record)\n'
        '    random.Random(seed).shuffle(unique)\n'
        '    n = len(unique)\n'
        '    n_test = max(1, round(n * test_fraction))\n'
        '    n_val = round(n * val_fraction)\n'
        '    if n - n_test - n_val < 1:\n'
        '        raise ValueError(f"{n} distinct images are too few to split into train/validation/test")\n'
        '    return {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}\n'
        '\n'
        '\n'
        'def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:\n'
        '    """Records from a directory or zip holding images, mask images and a `masks.csv` with the columns `file`,\n'
        '    `mask` and `prompt` (and optionally `id`); every listed file must exist and every image file must be listed\n'
        '    (mask files are those a `mask` column names)."""\n'
        '    source = Path(path)\n'
        '    members: dict[str, bytes] = {}\n'
        '    if source.is_dir():\n'
        '        for file in sorted(source.rglob("*")):\n'
        '            if file.is_file():\n'
        '                members[file.name] = file.read_bytes()\n'
        '    elif zipfile.is_zipfile(source):\n'
        '        with zipfile.ZipFile(source) as archive:\n'
        '            for info in archive.infolist():\n'
        '                if not info.is_dir():\n'
        '                    members[Path(info.filename).name] = archive.read(info)  # flattened; no extractall\n'
        '    else:\n'
        '        raise ValueError(f"{source} is neither a directory nor a zip file")\n'
        '    if "masks.csv" not in members:\n'
        '        raise ValueError("BYOD data must include masks.csv with the columns file, mask and prompt")\n'
        '    rows = list(csv.DictReader(io.StringIO(members["masks.csv"].decode("utf-8-sig"))))\n'
        '    if not rows or any(column not in rows[0] for column in ("file", "mask", "prompt")):\n'
        '        raise ValueError("masks.csv must have the columns file, mask and prompt")\n'
        '    out = []\n'
        '    for row in rows:\n'
        '        name = Path(str(row.get("file", "")).strip()).name\n'
        '        mask_name = Path(str(row.get("mask", "")).strip()).name\n'
        '        for needed in (name, mask_name):\n'
        '            if needed not in members:\n'
        '                raise ValueError(f"masks.csv names a missing file: {needed}")\n'
        '        try:\n'
        '            image = Image.open(io.BytesIO(members[name]))\n'
        '            image.load()\n'
        '            mask = Image.open(io.BytesIO(members[mask_name]))\n'
        '            mask.load()\n'
        '        except Exception as exc:  # noqa: BLE001\n'
        '            raise ValueError(f"BYOD file is not a decodable image: {name} / {mask_name}") from exc\n'
        '        rid = str(row.get("id", "") or "").strip()\n'
        '        out.append({"id": rid or re.sub(r"[^A-Za-z0-9_.:-]", "_", Path(name).stem)[:64], "image": image.convert("RGB"), "prompt": str(row.get("prompt", "")), "mask": mask})\n'
        '    listed = {Path(str(r.get(k, "")).strip()).name for r in rows for k in ("file", "mask")}\n'
        '    unlisted = [n for n in members if n != "masks.csv" and n not in listed]\n'
        '    if unlisted:\n'
        '        raise ValueError(f"{len(unlisted)} file(s) have no masks.csv row, e.g. {unlisted[0]}")\n'
        '    return out\n'
        '\n'
        '\n'
        'def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:\n'
        '    """A summary table (id, image size, prompt, mask area fraction, provenance) in the BYOD `masks.csv` column\n'
        '    layout plus extras (`file`/`mask` name the id; the images themselves are not written)."""\n'
        '    out = Path(path)\n'
        '    out.parent.mkdir(parents=True, exist_ok=True)\n'
        '    with open(out, "w", encoding="utf-8", newline="") as handle:\n'
        '        writer = csv.writer(handle)\n'
        '        writer.writerow(["id", "file", "mask", "prompt", "width", "height", "mask_area_fraction", "class_id", "source_id", "source_row_group"])\n'
        '        for r in records:\n'
        '            writer.writerow([r["id"], f"{r[\'id\']}.jpg", f"{r[\'id\']}_mask.png", r["prompt"], r["image"].width, r["image"].height, round(float(np.asarray(r["mask"], dtype=bool).mean()), 4), r.get("class_id", ""), r.get("source_id", ""), r.get("source_row_group", "")])\n'
        '    return out\n'
    ),
    'sam_reference.py': (
        '"""Promptable image segmentation over the pinned ``facebook/sam-vit-base`` (SAM v1, ViT-B) checkpoint.\n'
        '\n'
        'Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,\n'
        'from the Hugging Face Hub at the pinned revision. One task method, ``segment``: one object per call from point\n'
        "clicks and/or one box, returning boolean masks at input resolution plus the model's predicted IoU per mask.\n"
        '"""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        '# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width\n'
        'import hashlib\n'
        'import json\n'
        'import random\n'
        'import time\n'
        'from collections.abc import Callable, Mapping, Sequence\n'
        'from dataclasses import dataclass, field\n'
        'from pathlib import Path\n'
        'from typing import Any\n'
        '\n'
        'import numpy as np\n'
        'from PIL import Image\n'
        '\n'
        'MODEL_ID = "facebook/sam-vit-base"\n'
        'MODEL_REVISION = "70c1a07f894ebb5b307fd9eaaee97b9dfc16068f"\n'
        'MODEL_LICENSE = "apache-2.0"\n'
        'MODEL_KEY = "sam-vit-base"\n'
        'DEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / "weights" / MODEL_KEY\n'
        'MANIFEST_NAME = "dimer-base-manifest.json"\n'
        '\n'
        '# Input ceilings. The processor resizes the longest edge to 1024 and pads to 1024x1024 (see\n'
        "# preprocessor_config.json), so model cost is fixed; the caller's resolution only sets the size of the\n"
        '# up-sampled output masks. Prompts are one object per call: up to MAX_PROMPTS point clicks (label 1 =\n'
        '# foreground, 0 = background) and/or one xyxy box.\n'
        'MAX_IMAGE_SIDE = 4096\n'
        'MIN_IMAGE_SIDE = 16\n'
        'MAX_PROMPTS = 16\n'
        'NUM_MULTIMASK_OUTPUTS = 3  # config.json mask_decoder_config.num_multimask_outputs\n'
        'MASK_THRESHOLD = 0.0  # logits above this become True in the binarised masks (processor default)\n'
        'PARAMETER_COUNT = 93_735_472\n'
        'MASK_DECODER_PARAMETERS = 4_058_340\n'
        'ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"\n'
        'ARTIFACT_VERSION = "1.0"\n'
        'ADAPTER_WEIGHTS = "adapter.safetensors"\n'
        'ADAPTER_MANIFEST = "manifest.json"\n'
        'WEIGHT_FILE = "model.safetensors"\n'
        'MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample\n'
        'MAX_EVAL_RECORDS = 5_000\n'
        'PROMPT_KINDS = ("point", "box", "mixed")\n'
        '\n'
        '\n'
        'def _sha256(path: Path) -> str:\n'
        '    digest = hashlib.sha256()\n'
        '    with open(path, "rb") as fh:\n'
        '        for chunk in iter(lambda: fh.read(1 << 20), b""):\n'
        '            digest.update(chunk)\n'
        '    return digest.hexdigest()\n'
        '\n'
        '\n'
        'def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n'
        '    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""\n'
        '    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID:\n'
        '        raise ValueError(f"manifest modelId {manifest.get(\'modelId\')!r} != {MODEL_ID!r}")\n'
        '    if manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(f"manifest revision {manifest.get(\'revision\')!r} != {MODEL_REVISION!r}")\n'
        '    for entry in manifest["files"]:\n'
        '        file_path = root / entry["path"]\n'
        '        if not file_path.is_file():\n'
        '            raise FileNotFoundError(f"snapshot file missing: {file_path}")\n'
        '        size = file_path.stat().st_size\n'
        '        if size != entry["bytes"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: size {size} != manifest {entry[\'bytes\']}")\n'
        '        digest = _sha256(file_path)\n'
        '        if digest != entry["sha256"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: sha256 {digest} != manifest {entry[\'sha256\']}")\n'
        '    return {\n'
        '        "path": str(root),\n'
        '        "model_id": manifest["modelId"],\n'
        '        "revision": manifest["revision"],\n'
        '        "files": len(manifest["files"]),\n'
        '        "total_bytes": manifest.get("totalBytes"),\n'
        '    }\n'
        '\n'
        '\n'
        'def _hub_download(relative_path: str, root: Path) -> None:\n'
        '    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""\n'
        '    from huggingface_hub import hf_hub_download\n'
        '\n'
        '    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n'
        '\n'
        '\n'
        'def stage_missing_files(\n'
        '    path: str | Path | None = None,\n'
        '    *,\n'
        '    allow_download: bool = False,\n'
        '    downloader: Callable[[str, Path], None] | None = None,\n'
        ') -> list[str]:\n'
        '    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but\n'
        '    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""\n'
        '    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(\n'
        '            f"manifest names {manifest.get(\'modelId\')}@{manifest.get(\'revision\')}, "\n'
        '            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"\n'
        '        )\n'
        '    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]\n'
        '    if not missing:\n'
        '        return []\n'
        '    if not allow_download:\n'
        '        raise FileNotFoundError(\n'
        '            f"snapshot at {root} is missing {missing}; "\n'
        '            f"pass allow_download=True to fetch them at {MODEL_REVISION}"\n'
        '        )\n'
        '    fetch = downloader or _hub_download\n'
        '    for relative_path in missing:\n'
        '        fetch(relative_path, root)\n'
        '    return missing\n'
        '\n'
        '\n'
        'def mask_iou(a: np.ndarray, b: np.ndarray) -> float:\n'
        '    """Intersection-over-union of two boolean masks of identical shape; the primitive behind any mIoU."""\n'
        '    a = np.asarray(a)\n'
        '    b = np.asarray(b)\n'
        '    if a.shape != b.shape:\n'
        '        raise ValueError(f"shape mismatch: {a.shape} vs {b.shape}")\n'
        '    if a.dtype != np.bool_ or b.dtype != np.bool_:\n'
        '        raise TypeError("mask_iou expects boolean arrays")\n'
        '    union = np.logical_or(a, b).sum()\n'
        '    return float(np.logical_and(a, b).sum() / union) if union else 0.0\n'
        '\n'
        '\n'
        'def validate_image(image: Any) -> Image.Image:\n'
        '    if not isinstance(image, Image.Image):\n'
        '        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")\n'
        '    width, height = image.size\n'
        '    if min(width, height) < MIN_IMAGE_SIDE:\n'
        '        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")\n'
        '    if max(width, height) > MAX_IMAGE_SIDE:\n'
        '        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")\n'
        '    return image.convert("RGB")\n'
        '\n'
        '\n'
        'def validate_prompts(\n'
        '    width: int,\n'
        '    height: int,\n'
        '    points: Sequence[Sequence[float]] | None,\n'
        '    point_labels: Sequence[int] | None,\n'
        '    box: Sequence[float] | None,\n'
        ') -> tuple[list[list[float]] | None, list[int] | None, list[float] | None]:\n'
        '    """Check one object\'s prompts: points inside the image with 0/1 labels, and/or one xyxy box inside it."""\n'
        '    if points is None and box is None:\n'
        '        raise ValueError("provide at least one of points or box")\n'
        '    clean_points = clean_labels = None\n'
        '    if points is not None:\n'
        '        if isinstance(points, str) or not isinstance(points, Sequence):\n'
        '            raise TypeError("points must be a sequence of [x, y] pairs")\n'
        '        if not 1 <= len(points) <= MAX_PROMPTS:\n'
        '            raise ValueError(f"point count {len(points)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")\n'
        '        if point_labels is None or len(point_labels) != len(points):\n'
        '            raise ValueError("point_labels must be given with one 0/1 entry per point")\n'
        '        clean_points, clean_labels = [], []\n'
        '        for (x, y), label in zip(points, point_labels, strict=True):\n'
        '            if not (0 <= x < width and 0 <= y < height):\n'
        '                raise ValueError(f"point ({x}, {y}) outside image {width}x{height}")\n'
        '            if label not in (0, 1) or isinstance(label, bool):\n'
        '                raise ValueError(f"point label must be 0 or 1, got {label!r}")\n'
        '            clean_points.append([float(x), float(y)])\n'
        '            clean_labels.append(int(label))\n'
        '    elif point_labels is not None:\n'
        '        raise ValueError("point_labels given without points")\n'
        '    clean_box = None\n'
        '    if box is not None:\n'
        '        if len(box) != 4:\n'
        '            raise ValueError("box must be [x0, y0, x1, y1]")\n'
        '        x0, y0, x1, y1 = (float(v) for v in box)\n'
        '        if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):\n'
        '            raise ValueError(f"box {box!r} is not a non-empty xyxy box inside image {width}x{height}")\n'
        '        clean_box = [x0, y0, x1, y1]\n'
        '    return clean_points, clean_labels, clean_box\n'
        '\n'
        '\n'
        'INPUT_SCHEMA: dict[str, Any] = {\n'
        '    "input": (\n'
        '        "one PIL.Image.Image (any mode, converted to RGB) plus one object\'s prompts: point clicks "\n'
        '        "and/or one xyxy box"\n'
        '    ),\n'
        '    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],\n'
        '    "points": [1, MAX_PROMPTS],\n'
        '    "point_labels": "one per point, 1 = foreground and 0 = background",\n'
        '    "box": "at most one [x0, y0, x1, y1] inside the image with x0 < x1 and y0 < y1",\n'
        '    "objects_per_call": 1,\n'
        '    "multimask_outputs": NUM_MULTIMASK_OUTPUTS,\n'
        '    "preprocessing": (\n'
        '        "image converted to RGB; the processor resizes the longest edge to 1024 and pads to 1024x1024; "\n'
        '        "returned masks are up-sampled "\n'
        '        f"to the input resolution and binarised at logit MASK_THRESHOLD={MASK_THRESHOLD}"\n'
        '    ),\n'
        '}\n'
        '\n'
        '\n'
        'def _check_inputs(\n'
        '    image: Any,\n'
        '    points: Any,\n'
        '    point_labels: Any,\n'
        '    box: Any,\n'
        '    multimask: Any,\n'
        ') -> tuple[Image.Image, list[list[float]] | None, list[int] | None, list[float] | None]:\n'
        '    """Raise TypeError/ValueError naming the first violated ceiling; return the cleaned request.\n'
        '\n'
        '    ``segment`` and ``validate_inputs`` both route through this function so their acceptance\n'
        '    criteria cannot diverge.\n'
        '    """\n'
        '    rgb = validate_image(image)\n'
        '    clean_points, clean_labels, clean_box = validate_prompts(\n'
        '        rgb.width, rgb.height, points, point_labels, box\n'
        '    )\n'
        '    if not isinstance(multimask, bool):\n'
        '        raise TypeError("multimask must be a bool")\n'
        '    return rgb, clean_points, clean_labels, clean_box\n'
        '\n'
        '\n'
        'def validate_inputs(\n'
        '    image: Image.Image,\n'
        '    *,\n'
        '    points: Sequence[Sequence[float]] | None = None,\n'
        '    point_labels: Sequence[int] | None = None,\n'
        '    box: Sequence[float] | None = None,\n'
        '    multimask: bool = True,\n'
        '    names: Sequence[str] | None = None,\n'
        ') -> dict[str, Any]:\n'
        '    """Validation stage: return the input manifest (schema, observations, request, verdict).\n'
        '\n'
        '    Rejection is reported by raising exactly as ``segment`` would; a caller that wants the finding\n'
        '    recorded catches the exception and stores ``str(exc)`` under ``findings``.\n'
        '    """\n'
        '    _rgb, clean_points, clean_labels, clean_box = _check_inputs(\n'
        '        image, points, point_labels, box, multimask\n'
        '    )\n'
        '    if names is not None and len(names) != 1:\n'
        '        raise ValueError("names must have exactly one entry (segment takes one image)")\n'
        '    return {\n'
        '        "schema": dict(INPUT_SCHEMA),\n'
        '        "inputs": [\n'
        '            {\n'
        '                "id": names[0] if names else "image-0",\n'
        '                "mode": image.mode,\n'
        '                "size": list(image.size),\n'
        '                "n_points": 0 if clean_points is None else len(clean_points),\n'
        '                "has_box": clean_box is not None,\n'
        '            }\n'
        '        ],\n'
        '        "points": clean_points,\n'
        '        "point_labels": clean_labels,\n'
        '        "box": clean_box,\n'
        '        "multimask": multimask,\n'
        '        "verdict": "accepted",\n'
        '        "findings": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '\n'
        '\n'
        'def evaluation_report(\n'
        '    result: Mapping[str, Any],\n'
        '    reference_mask: Any = None,\n'
        '    *,\n'
        '    sample_kind: str = "synthetic",\n'
        ') -> dict[str, Any]:\n'
        '    """Evaluation stage: a machine-readable report even when nothing is measurable.\n'
        '\n'
        '    With a boolean ``reference_mask`` of the same shape as the returned masks the report carries one\n'
        '    ``mask_iou`` entry per candidate as sample-sanity geometry evidence; without one the verdict is\n'
        '    ``not-measurable`` and the report says what labelled data would make the task measurable.\n'
        '    """\n'
        '    masks = np.asarray(result["masks"])\n'
        '    scores = [float(v) for v in result["iou_scores"]]\n'
        '    best = int(np.argmax(scores)) if scores else None\n'
        '    base = {\n'
        '        "task": "promptable single-object image segmentation (point and/or box prompts)",\n'
        '        "decision_rule": (\n'
        '            "keep the candidate with the highest model-predicted IoU; the pipeline ships no "\n'
        '            "acceptance threshold and does not choose for the caller"\n'
        '        ),\n'
        '        "score_semantics": (\n'
        '            "iou_scores are the model\'s own uncalibrated predicted IoU for each candidate, not a "\n'
        '            "measured overlap and not a probability; the regression head is unclipped, so a value "\n'
        '            "may exceed 1.0"\n'
        '        ),\n'
        '        "sample_kind": sample_kind,\n'
        '        "n_masks": int(masks.shape[0]) if masks.ndim == 3 else 0,\n'
        '        "best_candidate": best,\n'
        '        "iou_scores_model_predicted": scores,\n'
        '        "mask_areas_px": [int(mask.sum()) for mask in masks] if masks.ndim == 3 else [],\n'
        '        "baselines": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '    if reference_mask is None:\n'
        '        return {\n'
        '            **base,\n'
        '            "metrics": [],\n'
        '            "verdict": "not-measurable",\n'
        '            "reason": "no ground-truth mask was supplied for the evaluated image",\n'
        '            "needs": (\n'
        '                "hand-labelled boolean masks for the prompted objects on your own images, scored with "\n'
        '                "mask_iou per object and averaged into a mean IoU over a held-out set; no labelled "\n'
        '                "mask set ships with this repository"\n'
        '            ),\n'
        '        }\n'
        '    reference = np.asarray(reference_mask)\n'
        '    return {\n'
        '        **base,\n'
        '        "metrics": [\n'
        '            {\n'
        '                "id": "mask_iou",\n'
        '                "candidate": index,\n'
        '                "value": mask_iou(masks[index], reference),\n'
        '                "selected": index == best,\n'
        '                "estimation": "one reference mask on a single scene, no dispersion estimate",\n'
        '            }\n'
        '            for index in range(masks.shape[0])\n'
        '        ],\n'
        '        "reference_area_px": int(reference.sum()),\n'
        '        "verdict": "sample-sanity",\n'
        '        "reason": (\n'
        '            "one reference mask on one tutorial sample; geometry sanity evidence, not a segmentation "\n'
        '            "benchmark"\n'
        '        ),\n'
        '        "needs": (\n'
        '            "a labelled mask set from the deployment domain for any mean-IoU or boundary-quality claim"\n'
        '        ),\n'
        '    }\n'
        '\n'
        '\n'
        'def _trainable_names(model: Any) -> list[str]:\n'
        '    """Every parameter of the mask decoder (transformer, upscaling and IoU-prediction heads). The image encoder\n'
        '    and the prompt encoder stay frozen."""\n'
        '    return [name for name, _ in model.named_parameters() if name.startswith("mask_decoder.")]\n'
        '\n'
        '\n'
        'def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:\n'
        '    """Refuse an adapter that names another base, another format or a file that does not match its digest."""\n'
        '    if manifest.get("format") != ARTIFACT_FORMAT:\n'
        '        raise ValueError(f"artifact format {manifest.get(\'format\')!r} != {ARTIFACT_FORMAT!r}")\n'
        '    base = manifest.get("base", {})\n'
        '    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(f"artifact was trained on {base.get(\'model_id\')}@{base.get(\'revision\')}, not {MODEL_ID}@{MODEL_REVISION}")\n'
        '    if base.get("weight_sha256") != base_sha256:\n'
        '        raise ValueError("artifact base weight digest does not match the verified snapshot")\n'
        '    files = manifest.get("files") or []\n'
        '    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:\n'
        '        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")\n'
        '    weights = artifact_dir / ADAPTER_WEIGHTS\n'
        '    if not weights.is_file():\n'
        '        raise FileNotFoundError(f"artifact weights missing: {weights}")\n'
        '    size = weights.stat().st_size\n'
        '    if size != files[0].get("bytes"):\n'
        '        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get(\'bytes\')}")\n'
        '    digest = _sha256(weights)\n'
        '    if digest != files[0].get("sha256"):\n'
        '        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get(\'sha256\')}")\n'
        '    names = manifest.get("tensors") or []\n'
        '    if not names or any(not str(n).startswith("mask_decoder.") for n in names):\n'
        '        raise ValueError("artifact tensors must all belong to the mask decoder")\n'
        '    if (manifest.get("adapter") or {}).get("prompts") not in PROMPT_KINDS:\n'
        '        raise ValueError(f"artifact adapter.prompts must be one of {PROMPT_KINDS}")\n'
        '\n'
        '\n'
        'def _low_res_target(mask: np.ndarray) -> Any:\n'
        '    """A record\'s boolean mask in the model\'s low-resolution frame: resized so the longest side is 1024, padded to\n'
        '    1024x1024 at the bottom/right (as the processor pads the image), then downsampled to the 256x256 decoder grid."""\n'
        '    import torch\n'
        '\n'
        '    target = torch.from_numpy(np.asarray(mask, dtype=np.float32))[None, None]\n'
        '    height, width = target.shape[2], target.shape[3]\n'
        '    scale = 1024 / max(height, width)\n'
        '    new_h, new_w = int(round(height * scale)), int(round(width * scale))\n'
        '    resized = torch.nn.functional.interpolate(target, size=(new_h, new_w), mode="bilinear", align_corners=False)\n'
        '    canvas = torch.zeros(1, 1, 1024, 1024)\n'
        '    canvas[:, :, :new_h, :new_w] = resized\n'
        '    return torch.nn.functional.interpolate(canvas, size=(256, 256), mode="bilinear", align_corners=False)[0]\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class SAMViTSegmentationPipeline:\n'
        '    """Promptable image segmentation (points/box -> masks) over the pinned SAM ViT-B checkpoint.\n'
        '\n'
        '    Prompted mode only (`SamModel` + `SamProcessor`); automatic grid-prompt mask generation is not exposed."""\n'
        '\n'
        '    _runner: Callable[..., tuple[np.ndarray, list[float]]]\n'
        '    device: str\n'
        '    _model: Any = field(default=None, repr=False)\n'
        '    _processor: Any = field(default=None, repr=False)\n'
        '    weight_sha256: str | None = None\n'
        '    adapter: dict[str, Any] | None = None\n'
        '\n'
        '    @classmethod\n'
        '    def from_pretrained(\n'
        '        cls,\n'
        '        device: str | None = None,\n'
        '        weights_dir: str | Path | None = None,\n'
        '        allow_download: bool = False,\n'
        '    ) -> SAMViTSegmentationPipeline:\n'
        '        import torch\n'
        '        from transformers import SamModel, SamProcessor\n'
        '\n'
        '        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n'
        '        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR\n'
        '        weight_sha256 = None\n'
        '        if (root / MANIFEST_NAME).is_file():\n'
        '            stage_missing_files(root, allow_download=allow_download)\n'
        '            verify_snapshot(root)\n'
        '            with open(root / MANIFEST_NAME, encoding="utf-8") as handle:\n'
        '                entries = json.load(handle).get("files", [])\n'
        '            weight_sha256 = next((e["sha256"] for e in entries if e["path"] == WEIGHT_FILE), None)\n'
        '            source, kwargs = str(root), {"local_files_only": True}\n'
        '        elif allow_download:\n'
        '            source, kwargs = MODEL_ID, {}\n'
        '        else:\n'
        '            raise FileNotFoundError(\n'
        '                f"no verified snapshot at {root} and allow_download=False; "\n'
        '                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"\n'
        '            )\n'
        '        processor = SamProcessor.from_pretrained(\n'
        '            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs\n'
        '        )\n'
        '        model = SamModel.from_pretrained(\n'
        '            source, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs\n'
        '        )\n'
        '        model = model.to(resolved_device).eval()\n'
        '        for param in model.parameters():\n'
        '            param.requires_grad_(False)\n'
        '\n'
        '        def runner(image, points, labels, box, multimask) -> tuple[np.ndarray, list[float]]:\n'
        '            prompt_kwargs: dict[str, Any] = {}\n'
        '            if points is not None:\n'
        '                prompt_kwargs["input_points"] = [[points]]\n'
        '                prompt_kwargs["input_labels"] = [[labels]]\n'
        '            if box is not None:\n'
        '                prompt_kwargs["input_boxes"] = [[box]]\n'
        '            inputs = processor(images=image, return_tensors="pt", **prompt_kwargs).to(resolved_device)\n'
        '            with torch.inference_mode():\n'
        '                outputs = model(**inputs, multimask_output=multimask)\n'
        '            # SAM v1 pads the resized image to 1024x1024; reshaped_input_sizes lets post-processing strip it.\n'
        '            masks = processor.post_process_masks(\n'
        '                outputs.pred_masks.cpu(),\n'
        '                inputs["original_sizes"].cpu(),\n'
        '                inputs["reshaped_input_sizes"].cpu(),\n'
        '                mask_threshold=MASK_THRESHOLD,\n'
        '            )[0]\n'
        '            return masks[0].numpy().astype(np.bool_), [float(v) for v in outputs.iou_scores[0, 0].tolist()]\n'
        '\n'
        '        return cls(runner, resolved_device, model, processor, weight_sha256)\n'
        '\n'
        '    def segment(\n'
        '        self,\n'
        '        image: Image.Image,\n'
        '        *,\n'
        '        points: Sequence[Sequence[float]] | None = None,\n'
        '        point_labels: Sequence[int] | None = None,\n'
        '        box: Sequence[float] | None = None,\n'
        '        multimask: bool = True,\n'
        '    ) -> dict[str, Any]:\n'
        '        """Segment one object; returns K boolean masks (K = 3 with multimask, else 1) at input resolution."""\n'
        '        rgb, clean_points, clean_labels, clean_box = _check_inputs(\n'
        '            image, points, point_labels, box, multimask\n'
        '        )\n'
        '        masks, iou_scores = self._runner(rgb, clean_points, clean_labels, clean_box, multimask)\n'
        '        masks = np.asarray(masks)\n'
        '        expected = (NUM_MULTIMASK_OUTPUTS if multimask else 1, rgb.height, rgb.width)\n'
        '        if masks.dtype != np.bool_ or masks.shape != expected or len(iou_scores) != expected[0]:\n'
        '            raise RuntimeError(f"backend returned {masks.shape} {masks.dtype}, {len(iou_scores)} scores")\n'
        '        return {\n'
        '            "masks": masks,\n'
        '            "iou_scores": [float(v) for v in iou_scores],\n'
        '            "multimask": multimask,\n'
        '            "points": clean_points,\n'
        '            "point_labels": clean_labels,\n'
        '            "box": clean_box,\n'
        '            "width": rgb.width,\n'
        '            "height": rgb.height,\n'
        '            "model_id": MODEL_ID,\n'
        '            "model_revision": MODEL_REVISION,\n'
        '        }\n'
        '\n'
        '    def _require_model(self) -> tuple[Any, Any]:\n'
        '        if self._model is None or self._processor is None:\n'
        '            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained for evaluate/adapt")\n'
        '        return self._model, self._processor\n'
        '\n'
        '\n'
        '    def _prompt_kwargs(self, record: Mapping[str, Any], prompt: str) -> dict[str, Any]:\n'
        '        if prompt == "point":\n'
        '            return {"points": [record["point"]], "point_labels": [1]}\n'
        '        if prompt == "box":\n'
        '            return {"box": record["box"]}\n'
        '        raise ValueError(f"prompt must be \'point\' or \'box\', got {prompt!r}")\n'
        '\n'
        '\n'
        '    def predict_mask(self, record: Mapping[str, Any], *, prompt: str = "point") -> np.ndarray:\n'
        '        """One boolean mask for a record: `segment` with the record\'s point (label 1) or box, keeping the multimask\n'
        '        output the model itself scores highest — the single-mask policy the corpus measures use."""\n'
        '        result = self.segment(record["image"], multimask=True, **self._prompt_kwargs(record, prompt))\n'
        '        return result["masks"][int(np.argmax(result["iou_scores"]))]\n'
        '\n'
        '\n'
        '    def evaluate(self, records: Sequence[Mapping[str, Any]], *, prompt: str = "point", progress: Callable[[int, int], None] | None = None) -> dict[str, Any]:\n'
        '        """Segment every validated record from one prompt kind and score the masks with `metrics.segmentation_metrics`\n'
        '        (mean IoU and hit rate, overall and per category)."""\n'
        '        from .metrics import segmentation_metrics\n'
        '        from .samples import validate_dataset\n'
        '\n'
        '        if prompt not in ("point", "box"):\n'
        '            raise ValueError("prompt must be \'point\' or \'box\'")\n'
        '        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]\n'
        '        started = time.perf_counter()\n'
        '        masks = []\n'
        '        for i, record in enumerate(checked):\n'
        '            masks.append(self.predict_mask(record, prompt=prompt))\n'
        '            if progress is not None:\n'
        '                progress(i + 1, len(checked))\n'
        '        metrics = segmentation_metrics(masks, checked)\n'
        '        metrics.update(\n'
        '            {\n'
        '                "prompt": prompt,\n'
        '                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",\n'
        '                "adapted": self.adapter is not None,\n'
        '                "seconds": round(time.perf_counter() - started, 3),\n'
        '                "model_id": MODEL_ID,\n'
        '                "model_revision": MODEL_REVISION,\n'
        '            }\n'
        '        )\n'
        '        return metrics\n'
        '\n'
        '\n'
        '    def adapt(\n'
        '        self,\n'
        '        train: Sequence[Mapping[str, Any]],\n'
        '        val: Sequence[Mapping[str, Any]] | None = None,\n'
        '        *,\n'
        '        epochs: int = 6,\n'
        '        lr: float = 5e-5,\n'
        '        prompts: str = "mixed",\n'
        '        seed: int = 0,\n'
        '        progress: Callable[[Mapping[str, Any]], None] | None = None,\n'
        '    ) -> dict[str, Any]:\n'
        '        """Bounded fine-tuning of the mask decoder only: the frozen image encoder embeds every training image once\n'
        "        (cached), the frozen prompt encoder embeds the record's point or box (`prompts`: 'point', 'box' or 'mixed' —\n"
        "        a seeded coin per record and epoch), and the decoder's single-mask logits are trained against the target mask\n"
        '        in the 256x256 low-resolution frame with binary cross-entropy plus a soft Dice term. AdamW (no weight decay),\n'
        '        gradient clipping at 1.0, one record per step, seeded shuffling, no scheduler. Epoch 0 records the frozen\n'
        "        model's validation metrics; the epoch with the highest mean of the validation point- and box-prompt IoU is kept\n"
        '        (the final one\n'
        '        without a validation split). On any exception the frozen weights are restored."""\n'
        '        model, processor = self._require_model()  # refuse before importing torch\n'
        '        import torch\n'
        '\n'
        '        from .samples import validate_dataset\n'
        '\n'
        '        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 50:\n'
        '            raise ValueError("epochs must be an int in 1..50")\n'
        '        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-2:\n'
        '            raise ValueError("lr must be in (0, 1e-2]")\n'
        '        if prompts not in PROMPT_KINDS:\n'
        '            raise ValueError(f"prompts must be one of {PROMPT_KINDS}")\n'
        '        train_checked = validate_dataset(train)["records"]\n'
        '        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None\n'
        '        names = _trainable_names(model)\n'
        '        name_set = set(names)\n'
        '        device = torch.device(self.device)\n'
        '        frozen_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}\n'
        '        previous_adapter = self.adapter\n'
        '        cudnn_flags = (torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark)\n'
        '        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False  # repeatable on one device\n'
        '        history: list[dict[str, Any]] = []\n'
        '        started = time.perf_counter()\n'
        '\n'
        '        def _val() -> dict[str, Any] | None:\n'
        '            if val_checked is None:\n'
        '                return None\n'
        '            point = self.evaluate(val_checked, prompt="point")\n'
        '            box = self.evaluate(val_checked, prompt="box")\n'
        '            return {\n'
        '                "iou": point["iou"],\n'
        '                "hit_rate": point["hit_rate"],\n'
        '                "box_iou": box["iou"],\n'
        '                "box_hit_rate": box["hit_rate"],\n'
        '                "score": (point["iou"] + box["iou"]) / 2.0,\n'
        '                "n": point["n"],\n'
        '            }\n'
        '\n'
        '        try:\n'
        '            for param in model.parameters():\n'
        '                param.requires_grad_(False)\n'
        '            params = []\n'
        '            for name, param in model.named_parameters():\n'
        '                if name in name_set:\n'
        '                    param.requires_grad_(True)\n'
        '                    params.append(param)\n'
        '            n_trainable = sum(p.numel() for p in params)\n'
        '            embed_started = time.perf_counter()\n'
        '            embeddings: dict[str, Any] = {}\n'
        '            with torch.no_grad():  # not inference_mode: the cached embeddings feed a backward pass\n'
        '                for record in train_checked:\n'
        '                    pixel_values = processor(images=record["image"], return_tensors="pt")["pixel_values"].to(device)\n'
        '                    embeddings[record["id"]] = model.get_image_embeddings(pixel_values).clone()\n'
        '            embed_seconds = round(time.perf_counter() - embed_started, 3)\n'
        '            targets = {record["id"]: _low_res_target(record["mask"]).to(device) for record in train_checked}\n'
        '            entry = {"epoch": 0, "train_loss": None, "val": _val(), "note": "frozen model"}\n'
        '            history.append(entry)\n'
        '            if progress is not None:\n'
        '                progress(entry)\n'
        '            best_epoch, best_score = 0, (history[0]["val"] or {}).get("score", -1.0)\n'
        '            best_state = frozen_state\n'
        '            optimizer = torch.optim.AdamW(params, lr=float(lr), weight_decay=0.0)\n'
        '            rng = random.Random(seed)\n'
        '            torch.manual_seed(seed)\n'
        '            for epoch in range(1, epochs + 1):\n'
        '                model.train()\n'
        '                order = list(train_checked)\n'
        '                rng.shuffle(order)\n'
        '                losses = []\n'
        '                for record in order:\n'
        '                    use_box = prompts == "box" or (prompts == "mixed" and rng.random() < 0.5)\n'
        '                    if use_box:\n'
        '                        encoded = processor(images=record["image"], input_boxes=[[record["box"]]], return_tensors="pt")\n'
        '                        output = model(image_embeddings=embeddings[record["id"]], input_boxes=encoded["input_boxes"].to(device), multimask_output=False)\n'
        '                    else:\n'
        '                        encoded = processor(images=record["image"], input_points=[[[record["point"]]]], input_labels=[[[1]]], return_tensors="pt")\n'
        '                        output = model(image_embeddings=embeddings[record["id"]], input_points=encoded["input_points"].to(device), input_labels=encoded["input_labels"].to(device), multimask_output=False)\n'
        '                    logits = output.pred_masks[0, 0]\n'
        '                    target = targets[record["id"]]\n'
        '                    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, target)\n'
        '                    prob = torch.sigmoid(logits)\n'
        '                    dice = 1.0 - (2.0 * (prob * target).sum() + 1.0) / (prob.sum() + target.sum() + 1.0)\n'
        '                    loss = bce + dice\n'
        '                    optimizer.zero_grad(set_to_none=True)\n'
        '                    loss.backward()\n'
        '                    torch.nn.utils.clip_grad_norm_(params, 1.0)\n'
        '                    optimizer.step()\n'
        '                    losses.append(float(loss.detach()))\n'
        '                model.eval()\n'
        '                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": _val()}\n'
        '                history.append(entry)\n'
        '                if progress is not None:\n'
        '                    progress(entry)\n'
        '                if val_checked is None or entry["val"]["score"] > best_score:\n'
        '                    best_epoch, best_score = epoch, (entry["val"] or {}).get("score", -1.0)\n'
        '                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}\n'
        '            model.load_state_dict(best_state, strict=False)\n'
        '            for param in model.parameters():\n'
        '                param.requires_grad_(False)\n'
        '            model.eval()\n'
        '        except BaseException:\n'
        '            model.load_state_dict(frozen_state, strict=False)\n'
        '            for param in model.parameters():\n'
        '                param.requires_grad_(False)\n'
        '            model.eval()\n'
        '            self.adapter = previous_adapter\n'
        '            raise\n'
        '        finally:\n'
        '            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = cudnn_flags\n'
        '        self.adapter = {\n'
        '            "prompts": prompts,\n'
        '            "trainable_names": names,\n'
        '            "n_trainable": n_trainable,\n'
        '            "n_total": sum(p.numel() for p in model.parameters()),\n'
        '            "epochs": epochs,\n'
        '            "best_epoch": best_epoch,\n'
        '            "selection": "highest mean of validation point- and box-prompt IoU" if val_checked is not None else "final epoch (no validation split)",\n'
        '            "loss": "binary cross-entropy + soft Dice on the 256x256 single-mask logits",\n'
        '            "lr": float(lr),\n'
        '            "seed": seed,\n'
        '            "n_train": len(train_checked),\n'
        '            "n_val": len(val_checked) if val_checked is not None else 0,\n'
        '            "embedding_seconds": embed_seconds,\n'
        '            "history": history,\n'
        '            "seconds": round(time.perf_counter() - started, 3),\n'
        '        }\n'
        '        return dict(self.adapter)\n'
        '\n'
        '\n'
        '    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:\n'
        '        """Write the trained mask-decoder tensors as safetensors plus a manifest naming the base, the digests and the\n'
        '        training configuration. Requires a prior `adapt`."""\n'
        '        model, _processor = self._require_model()  # refuse before importing torch\n'
        '        import torch\n'
        '        from safetensors.torch import save_file\n'
        '\n'
        '        if self.adapter is None:\n'
        '            raise RuntimeError("nothing to save: call adapt() first")\n'
        '        out = Path(output_dir)\n'
        '        out.mkdir(parents=True, exist_ok=True)\n'
        '        names = list(self.adapter["trainable_names"])\n'
        '        state = model.state_dict()\n'
        '        tensors = {name: state[name].detach().cpu().contiguous() for name in names}\n'
        '        weights = out / ADAPTER_WEIGHTS\n'
        '        save_file(tensors, str(weights), metadata={"format": "pt"})\n'
        '        manifest = {\n'
        '            "format": ARTIFACT_FORMAT,\n'
        '            "version": ARTIFACT_VERSION,\n'
        '            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHT_FILE, "weight_sha256": self.weight_sha256},\n'
        '            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},\n'
        '            "history": self.adapter["history"],\n'
        '            "tensors": names,\n'
        '            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],\n'
        '            "torch": torch.__version__,\n'
        '            "metadata": dict(metadata or {}),\n'
        '        }\n'
        '        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:\n'
        '            json.dump(manifest, handle, indent=2, ensure_ascii=False)\n'
        '        return out\n'
        '\n'
        '\n'
        '    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:\n'
        '        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact\n'
        '        tensor set. Refuses tensors outside the mask decoder."""\n'
        '        model, _processor = self._require_model()  # refuse before importing safetensors\n'
        '        from safetensors.torch import load_file\n'
        '\n'
        '        artifact = Path(artifact_dir)\n'
        '        manifest_path = artifact / ADAPTER_MANIFEST\n'
        '        if not manifest_path.is_file():\n'
        '            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")\n'
        '        with open(manifest_path, encoding="utf-8") as handle:\n'
        '            manifest = json.load(handle)\n'
        '        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")\n'
        '        expected = _trainable_names(model)\n'
        '        if sorted(manifest["tensors"]) != sorted(expected):\n'
        '            raise ValueError("artifact tensor set does not match its recorded configuration")\n'
        '        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))\n'
        '        if sorted(tensors) != sorted(expected):\n'
        '            raise ValueError("artifact tensor names differ from the manifest")\n'
        '        state = model.state_dict()\n'
        '        for name, tensor in tensors.items():\n'
        '            if tuple(tensor.shape) != tuple(state[name].shape):\n'
        '                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")\n'
        '        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)\n'
        '        model.eval()\n'
        '        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}\n'
        '        return dict(self.adapter)\n'
        '\n'
        '\n'
        '    @classmethod\n'
        '    def from_artifact(\n'
        '        cls,\n'
        '        artifact_dir: str | Path,\n'
        '        *,\n'
        '        device: str | None = None,\n'
        '        weights_dir: str | Path | None = None,\n'
        '        allow_download: bool = False,\n'
        '    ) -> SAMViTSegmentationPipeline:\n'
        '        """Load the verified base snapshot, then overlay the adapter (verified before deserialising)."""\n'
        '        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)\n'
        '        pipe.load_artifact(artifact_dir)\n'
        '        return pipe\n'
    ),
    'weights/sam/dimer-base-manifest.json': (
        '{\n'
        '  "format": "dimer_hf_snapshot",\n'
        '  "formatVersion": 1,\n'
        '  "modelKey": "sam-vit-base",\n'
        '  "modelId": "facebook/sam-vit-base",\n'
        '  "revision": "70c1a07f894ebb5b307fd9eaaee97b9dfc16068f",\n'
        '  "files": [\n'
        '    {\n'
        '      "path": "README.md",\n'
        '      "bytes": 6727,\n'
        '      "sha256": "7a8579105233ff1e8fdc6c8387e102e88458e94c873463f640c416de01c921dd"\n'
        '    },\n'
        '    {\n'
        '      "path": "config.json",\n'
        '      "bytes": 6566,\n'
        '      "sha256": "5ebd0d8643b486f3a716bf17c2a15531eb818b2b96ff0c6c5dcc88fa015161af"\n'
        '    },\n'
        '    {\n'
        '      "path": "model.safetensors",\n'
        '      "bytes": 374979480,\n'
        '      "sha256": "892c410e496344e527255ccdcb2cb7244a609acb5389c7c4fdba1288f861c579"\n'
        '    },\n'
        '    {\n'
        '      "path": "preprocessor_config.json",\n'
        '      "bytes": 466,\n'
        '      "sha256": "225545a743c654e3c495ec6f545a0eaba57c8ba3fbbd8483b3cb1c0fc58db517"\n'
        '    }\n'
        '  ],\n'
        '  "totalBytes": 374993239\n'
        '}'
    ),
    'licenses/sam.txt': (
        '                                 Apache License\n'
        '                           Version 2.0, January 2004\n'
        '                        http://www.apache.org/licenses/\n'
        '\n'
        '   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n'
        '\n'
        '   1. Definitions.\n'
        '\n'
        '      "License" shall mean the terms and conditions for use, reproduction,\n'
        '      and distribution as defined by Sections 1 through 9 of this document.\n'
        '\n'
        '      "Licensor" shall mean the copyright owner or entity authorized by\n'
        '      the copyright owner that is granting the License.\n'
        '\n'
        '      "Legal Entity" shall mean the union of the acting entity and all\n'
        '      other entities that control, are controlled by, or are under common\n'
        '      control with that entity. For the purposes of this definition,\n'
        '      "control" means (i) the power, direct or indirect, to cause the\n'
        '      direction or management of such entity, whether by contract or\n'
        '      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n'
        '      outstanding shares, or (iii) beneficial ownership of such entity.\n'
        '\n'
        '      "You" (or "Your") shall mean an individual or Legal Entity\n'
        '      exercising permissions granted by this License.\n'
        '\n'
        '      "Source" form shall mean the preferred form for making modifications,\n'
        '      including but not limited to software source code, documentation\n'
        '      source, and configuration files.\n'
        '\n'
        '      "Object" form shall mean any form resulting from mechanical\n'
        '      transformation or translation of a Source form, including but\n'
        '      not limited to compiled object code, generated documentation,\n'
        '      and conversions to other media types.\n'
        '\n'
        '      "Work" shall mean the work of authorship, whether in Source or\n'
        '      Object form, made available under the License, as indicated by a\n'
        '      copyright notice that is included in or attached to the work\n'
        '      (an example is provided in the Appendix below).\n'
        '\n'
        '      "Derivative Works" shall mean any work, whether in Source or Object\n'
        '      form, that is based on (or derived from) the Work and for which the\n'
        '      editorial revisions, annotations, elaborations, or other modifications\n'
        '      represent, as a whole, an original work of authorship. For the purposes\n'
        '      of this License, Derivative Works shall not include works that remain\n'
        '      separable from, or merely link (or bind by name) to the interfaces of,\n'
        '      the Work and Derivative Works thereof.\n'
        '\n'
        '      "Contribution" shall mean any work of authorship, including\n'
        '      the original version of the Work and any modifications or additions\n'
        '      to that Work or Derivative Works thereof, that is intentionally\n'
        '      submitted to Licensor for inclusion in the Work by the copyright owner\n'
        '      or by an individual or Legal Entity authorized to submit on behalf of\n'
        '      the copyright owner. For the purposes of this definition, "submitted"\n'
        '      means any form of electronic, verbal, or written communication sent\n'
        '      to the Licensor or its representatives, including but not limited to\n'
        '      communication on electronic mailing lists, source code control systems,\n'
        '      and issue tracking systems that are managed by, or on behalf of, the\n'
        '      Licensor for the purpose of discussing and improving the Work, but\n'
        '      excluding communication that is conspicuously marked or otherwise\n'
        '      designated in writing by the copyright owner as "Not a Contribution."\n'
        '\n'
        '      "Contributor" shall mean Licensor and any individual or Legal Entity\n'
        '      on behalf of whom a Contribution has been received by Licensor and\n'
        '      subsequently incorporated within the Work.\n'
        '\n'
        '   2. Grant of Copyright License. Subject to the terms and conditions of\n'
        '      this License, each Contributor hereby grants to You a perpetual,\n'
        '      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n'
        '      copyright license to reproduce, prepare Derivative Works of,\n'
        '      publicly display, publicly perform, sublicense, and distribute the\n'
        '      Work and such Derivative Works in Source or Object form.\n'
        '\n'
        '   3. Grant of Patent License. Subject to the terms and conditions of\n'
        '      this License, each Contributor hereby grants to You a perpetual,\n'
        '      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n'
        '      (except as stated in this section) patent license to make, have made,\n'
        '      use, offer to sell, sell, import, and otherwise transfer the Work,\n'
        '      where such license applies only to those patent claims licensable\n'
        '      by such Contributor that are necessarily infringed by their\n'
        '      Contribution(s) alone or by combination of their Contribution(s)\n'
        '      with the Work to which such Contribution(s) was submitted. If You\n'
        '      institute patent litigation against any entity (including a\n'
        '      cross-claim or counterclaim in a lawsuit) alleging that the Work\n'
        '      or a Contribution incorporated within the Work constitutes direct\n'
        '      or contributory patent infringement, then any patent licenses\n'
        '      granted to You under this License for that Work shall terminate\n'
        '      as of the date such litigation is filed.\n'
        '\n'
        '   4. Redistribution. You may reproduce and distribute copies of the\n'
        '      Work or Derivative Works thereof in any medium, with or without\n'
        '      modifications, and in Source or Object form, provided that You\n'
        '      meet the following conditions:\n'
        '\n'
        '      (a) You must give any other recipients of the Work or Derivative\n'
        '          Works a copy of this License; and\n'
        '\n'
        '      (b) You must cause any modified files to carry prominent notices\n'
        '          stating that You changed the files; and\n'
        '\n'
        '      (c) You must retain, in the Source form of any Derivative Works\n'
        '          that You distribute, all copyright, patent, trademark, and\n'
        '          attribution notices from the Source form of the Work,\n'
        '          excluding those notices that do not pertain to any part of\n'
        '          the Derivative Works; and\n'
        '\n'
        '      (d) If the Work includes a "NOTICE" text file as part of its\n'
        '          distribution, then any Derivative Works that You distribute must\n'
        '          include a readable copy of the attribution notices contained\n'
        '          within such NOTICE file, excluding those notices that do not\n'
        '          pertain to any part of the Derivative Works, in at least one\n'
        '          of the following places: within a NOTICE text file distributed\n'
        '          as part of the Derivative Works; within the Source form or\n'
        '          documentation, if provided along with the Derivative Works; or,\n'
        '          within a display generated by the Derivative Works, if and\n'
        '          wherever such third-party notices normally appear. The contents\n'
        '          of the NOTICE file are for informational purposes only and\n'
        '          do not modify the License. You may add Your own attribution\n'
        '          notices within Derivative Works that You distribute, alongside\n'
        '          or as an addendum to the NOTICE text from the Work, provided\n'
        '          that such additional attribution notices cannot be construed\n'
        '          as modifying the License.\n'
        '\n'
        '      You may add Your own copyright statement to Your modifications and\n'
        '      may provide additional or different license terms and conditions\n'
        '      for use, reproduction, or distribution of Your modifications, or\n'
        '      for any such Derivative Works as a whole, provided Your use,\n'
        '      reproduction, and distribution of the Work otherwise complies with\n'
        '      the conditions stated in this License.\n'
        '\n'
        '   5. Submission of Contributions. Unless You explicitly state otherwise,\n'
        '      any Contribution intentionally submitted for inclusion in the Work\n'
        '      by You to the Licensor shall be under the terms and conditions of\n'
        '      this License, without any additional terms or conditions.\n'
        '      Notwithstanding the above, nothing herein shall supersede or modify\n'
        '      the terms of any separate license agreement you may have executed\n'
        '      with Licensor regarding such Contributions.\n'
        '\n'
        '   6. Trademarks. This License does not grant permission to use the trade\n'
        '      names, trademarks, service marks, or product names of the Licensor,\n'
        '      except as required for reasonable and customary use in describing the\n'
        '      origin of the Work and reproducing the content of the NOTICE file.\n'
        '\n'
        '   7. Disclaimer of Warranty. Unless required by applicable law or\n'
        '      agreed to in writing, Licensor provides the Work (and each\n'
        '      Contributor provides its Contributions) on an "AS IS" BASIS,\n'
        '      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n'
        '      implied, including, without limitation, any warranties or conditions\n'
        '      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n'
        '      PARTICULAR PURPOSE. You are solely responsible for determining the\n'
        '      appropriateness of using or redistributing the Work and assume any\n'
        '      risks associated with Your exercise of permissions under this License.\n'
        '\n'
        '   8. Limitation of Liability. In no event and under no legal theory,\n'
        '      whether in tort (including negligence), contract, or otherwise,\n'
        '      unless required by applicable law (such as deliberate and grossly\n'
        '      negligent acts) or agreed to in writing, shall any Contributor be\n'
        '      liable to You for damages, including any direct, indirect, special,\n'
        '      incidental, or consequential damages of any character arising as a\n'
        '      result of this License or out of the use or inability to use the\n'
        '      Work (including but not limited to damages for loss of goodwill,\n'
        '      work stoppage, computer failure or malfunction, or any and all\n'
        '      other commercial damages or losses), even if such Contributor\n'
        '      has been advised of the possibility of such damages.\n'
        '\n'
        '   9. Accepting Warranty or Additional Liability. While redistributing\n'
        '      the Work or Derivative Works thereof, You may choose to offer,\n'
        '      and charge a fee for, acceptance of support, warranty, indemnity,\n'
        '      or other liability obligations and/or rights consistent with this\n'
        '      License. However, in accepting such obligations, You may act only\n'
        '      on Your own behalf and on Your sole responsibility, not on behalf\n'
        '      of any other Contributor, and only if You agree to indemnify,\n'
        '      defend, and hold each Contributor harmless for any liability\n'
        '      incurred by, or claims asserted against, such Contributor by reason\n'
        '      of your accepting any such warranty or additional liability.\n'
        '\n'
        '   END OF TERMS AND CONDITIONS\n'
        '\n'
        '   APPENDIX: How to apply the Apache License to your work.\n'
        '\n'
        '      To apply the Apache License to your work, attach the following\n'
        '      boilerplate notice, with the fields enclosed by brackets "[]"\n'
        "      replaced with your own identifying information. (Don't include\n"
        '      the brackets!)  The text should be enclosed in the appropriate\n'
        '      comment syntax for the file format. We also recommend that a\n'
        '      file or class name and description of purpose be included on the\n'
        '      same "printed page" as the copyright notice for easier\n'
        '      identification within third-party archives.\n'
        '\n'
        '   Copyright 2026 Kurt Valcorza\n'
        '\n'
        '   Licensed under the Apache License, Version 2.0 (the "License");\n'
        '   you may not use this file except in compliance with the License.\n'
        '   You may obtain a copy of the License at\n'
        '\n'
        '       http://www.apache.org/licenses/LICENSE-2.0\n'
        '\n'
        '   Unless required by applicable law or agreed to in writing, software\n'
        '   distributed under the License is distributed on an "AS IS" BASIS,\n'
        '   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n'
        '   See the License for the specific language governing permissions and\n'
        '   limitations under the License.\n'
    ),
    'sam2_reference.py': (
        'from __future__ import annotations\n'
        '\n'
        'import copy\n'
        'import hashlib\n'
        'import json\n'
        'import math\n'
        'from collections.abc import Callable, Mapping, Sequence\n'
        'from dataclasses import dataclass, field\n'
        'from datetime import UTC, datetime\n'
        'from pathlib import Path\n'
        'from typing import Any\n'
        '\n'
        'import numpy as np\n'
        'from PIL import Image\n'
        '\n'
        'MODEL_ID = "facebook/sam2.1-hiera-small"\n'
        'MODEL_REVISION = "ee5bba1d82bb8749febdf90f45e84b687142ba03"\n'
        'MODEL_LICENSE = "apache-2.0"\n'
        'MODEL_KEY = "sam2.1-hiera-small"\n'
        'DEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / "weights" / MODEL_KEY\n'
        'MANIFEST_NAME = "dimer-base-manifest.json"\n'
        'ARTIFACT_FORMAT = "org.valcorza.sam2.mask-decoder-adapter"\n'
        'ARTIFACT_FORMAT_VERSION = "1.0"\n'
        'ARTIFACT_MANIFEST_NAME = "manifest.json"\n'
        'ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"\n'
        'TRAINABLE_PREFIXES = ("mask_decoder.output_hypernetworks_mlps.0.",)\n'
        'ADAPTATION_METHOD = "frozen-backbone-mask-hypernetwork-gradient-adaptation"\n'
        'MAX_ADAPTATION_RECORDS = 128\n'
        'MAX_ADAPTATION_PIXELS = 32 * 1024 * 1024\n'
        '\n'
        '_ADAPTATION_RUN_FIELDS = (\n'
        '    "epochs",\n'
        '    "learning_rate",\n'
        '    "batch_size",\n'
        '    "seed",\n'
        '    "loss",\n'
        '    "train_manifest",\n'
        '    "validation_manifest",\n'
        '    "baseline_validation_mask_iou",\n'
        '    "history",\n'
        '    "weight_delta_l2",\n'
        ')\n'
        '_DATASET_MANIFEST_FIELDS = {\n'
        '    "records",\n'
        '    "unique_ids",\n'
        '    "positive_pixels",\n'
        '    "dataset_sha256",\n'
        '    "verdict",\n'
        '}\n'
        '_ADAPTATION_BASE_FIELDS = {\n'
        '    "method",\n'
        '    "trainable_prefixes",\n'
        '    "trainable_parameters",\n'
        '    "frozen_parameters",\n'
        '}\n'
        '_ADAPTATION_METADATA_FIELDS = {\n'
        '    *_ADAPTATION_BASE_FIELDS,\n'
        '    *_ADAPTATION_RUN_FIELDS,\n'
        '    "training_runs",\n'
        '    "artifact_lineage",\n'
        '}\n'
        '_ARTIFACT_LINEAGE_FIELDS = {"manifest_sha256", "weights_sha256", "producer"}\n'
        '\n'
        '# Input ceilings. The processor resizes every image to 1024x1024 (preprocessor_config.json), so model cost is\n'
        "# fixed; the caller's resolution only sets the size of the up-sampled output masks. Prompts are one object per\n"
        '# call: up to MAX_PROMPTS point clicks (label 1 = foreground, 0 = background) and/or one xyxy box.\n'
        'MAX_IMAGE_SIDE = 4096\n'
        'MIN_IMAGE_SIDE = 16\n'
        'MAX_PROMPTS = 16\n'
        'NUM_MULTIMASK_OUTPUTS = 3  # config.json mask_decoder_config.num_multimask_outputs\n'
        'MASK_THRESHOLD = 0.0  # logits above this become True in the binarised masks (processor default)\n'
        '\n'
        '\n'
        'def _sha256(path: Path) -> str:\n'
        '    digest = hashlib.sha256()\n'
        '    with open(path, "rb") as fh:\n'
        '        for chunk in iter(lambda: fh.read(1 << 20), b""):\n'
        '            digest.update(chunk)\n'
        '    return digest.hexdigest()\n'
        '\n'
        '\n'
        'def _is_positive_finite_number(value: Any) -> bool:\n'
        '    return (\n'
        '        isinstance(value, int | float)\n'
        '        and not isinstance(value, bool)\n'
        '        and math.isfinite(float(value))\n'
        '        and value > 0\n'
        '    )\n'
        '\n'
        '\n'
        'def _validate_dataset_manifest(manifest: Any, *, label: str) -> None:\n'
        '    if not isinstance(manifest, Mapping):\n'
        '        raise ValueError(f"{label} must be a mapping")\n'
        '    if set(manifest) != _DATASET_MANIFEST_FIELDS:\n'
        '        raise ValueError(f"{label} does not match the dataset manifest schema")\n'
        '    records = manifest["records"]\n'
        '    unique_ids = manifest["unique_ids"]\n'
        '    positive_pixels = manifest["positive_pixels"]\n'
        '    if (\n'
        '        not isinstance(records, int)\n'
        '        or isinstance(records, bool)\n'
        '        or not 2 <= records <= MAX_ADAPTATION_RECORDS\n'
        '    ):\n'
        '        raise ValueError(f"{label} has an invalid record count")\n'
        '    if not isinstance(unique_ids, int) or isinstance(unique_ids, bool) or unique_ids != records:\n'
        '        raise ValueError(f"{label} has an invalid unique-id count")\n'
        '    if (\n'
        '        not isinstance(positive_pixels, int)\n'
        '        or isinstance(positive_pixels, bool)\n'
        '        or positive_pixels <= 0\n'
        '    ):\n'
        '        raise ValueError(f"{label} has an invalid positive-pixel count")\n'
        '    digest = manifest["dataset_sha256"]\n'
        '    if (\n'
        '        not isinstance(digest, str)\n'
        '        or len(digest) != 64\n'
        '        or any(character not in "0123456789abcdef" for character in digest)\n'
        '    ):\n'
        '        raise ValueError(f"{label} has an invalid dataset SHA-256")\n'
        '    if manifest["verdict"] != "accepted":\n'
        '        raise ValueError(f"{label} has an invalid verdict")\n'
        '\n'
        '\n'
        'def _validate_adaptation_run(run: Any, *, label: str) -> None:\n'
        '    if not isinstance(run, Mapping):\n'
        '        raise ValueError(f"{label} must be a mapping")\n'
        '    if set(run) != set(_ADAPTATION_RUN_FIELDS):\n'
        '        raise ValueError(f"{label} does not match the adaptation run schema")\n'
        '    missing = [key for key in _ADAPTATION_RUN_FIELDS if key not in run]\n'
        '    if missing:\n'
        '        raise ValueError(f"{label} is missing required fields: {missing}")\n'
        '    epochs = run["epochs"]\n'
        '    batch_size = run["batch_size"]\n'
        '    seed = run["seed"]\n'
        '    if not isinstance(epochs, int) or isinstance(epochs, bool) or epochs <= 0:\n'
        '        raise ValueError(f"{label} has invalid epochs")\n'
        '    if not _is_positive_finite_number(run["learning_rate"]):\n'
        '        raise ValueError(f"{label} has invalid learning rate")\n'
        '    if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size <= 0:\n'
        '        raise ValueError(f"{label} has invalid batch size")\n'
        '    if not isinstance(seed, int) or isinstance(seed, bool):\n'
        '        raise ValueError(f"{label} has invalid seed")\n'
        '    if run["loss"] != "binary-cross-entropy-plus-soft-dice":\n'
        '        raise ValueError(f"{label} has an unsupported loss")\n'
        '    _validate_dataset_manifest(run["train_manifest"], label=f"{label} train manifest")\n'
        '    validation_manifest = run["validation_manifest"]\n'
        '    if validation_manifest is not None:\n'
        '        _validate_dataset_manifest(\n'
        '            validation_manifest, label=f"{label} validation manifest"\n'
        '        )\n'
        '    baseline = run["baseline_validation_mask_iou"]\n'
        '    if validation_manifest is None and baseline is not None:\n'
        '        raise ValueError(f"{label} has invalid validation baseline")\n'
        '    if baseline is not None and (\n'
        '        not isinstance(baseline, int | float)\n'
        '        or isinstance(baseline, bool)\n'
        '        or not math.isfinite(float(baseline))\n'
        '        or not 0 <= float(baseline) <= 1\n'
        '    ):\n'
        '        raise ValueError(f"{label} has invalid validation baseline")\n'
        '    history = run["history"]\n'
        '    if not isinstance(history, list) or not history:\n'
        '        raise ValueError(f"{label} requires non-empty training history")\n'
        '    if len(history) != epochs:\n'
        '        raise ValueError(f"{label} history length does not match epochs")\n'
        '    expected_history_fields = {"epoch", "train_loss", "optimizer_steps"}\n'
        '    if validation_manifest is not None:\n'
        '        expected_history_fields.add("val_mask_iou")\n'
        '    expected_steps = math.ceil(run["train_manifest"]["records"] / batch_size)\n'
        '    for expected_epoch, item in enumerate(history, start=1):\n'
        '        if not isinstance(item, Mapping) or set(item) != expected_history_fields:\n'
        '            raise ValueError(f"{label} history entry schema is invalid")\n'
        '        if item["epoch"] != expected_epoch or isinstance(item["epoch"], bool):\n'
        '            raise ValueError(f"{label} history has an invalid epoch sequence")\n'
        '        train_loss = item["train_loss"]\n'
        '        if (\n'
        '            not isinstance(train_loss, int | float)\n'
        '            or isinstance(train_loss, bool)\n'
        '            or not math.isfinite(float(train_loss))\n'
        '            or train_loss < 0\n'
        '        ):\n'
        '            raise ValueError(f"{label} history has an invalid train loss")\n'
        '        optimizer_steps = item["optimizer_steps"]\n'
        '        if (\n'
        '            not isinstance(optimizer_steps, int)\n'
        '            or isinstance(optimizer_steps, bool)\n'
        '            or optimizer_steps != expected_steps\n'
        '        ):\n'
        '            raise ValueError(f"{label} history has an invalid optimizer step count")\n'
        '        if validation_manifest is not None:\n'
        '            val_mask_iou = item["val_mask_iou"]\n'
        '            if (\n'
        '                not isinstance(val_mask_iou, int | float)\n'
        '                or isinstance(val_mask_iou, bool)\n'
        '                or not math.isfinite(float(val_mask_iou))\n'
        '                or not 0 <= float(val_mask_iou) <= 1\n'
        '            ):\n'
        '                raise ValueError(f"{label} history has an invalid validation mask IoU")\n'
        '    if not _is_positive_finite_number(run["weight_delta_l2"]):\n'
        '        raise ValueError(f"{label} requires a positive finite weight delta")\n'
        '\n'
        '\n'
        'def _validate_completed_adaptation_metadata(adaptation: Any) -> None:\n'
        '    """Reject metadata that cannot identify a completed adapter before weights are applied."""\n'
        '    if not isinstance(adaptation, Mapping):\n'
        '        raise ValueError("artifact adaptation metadata must be a mapping")\n'
        '    unknown_fields = set(adaptation) - _ADAPTATION_METADATA_FIELDS\n'
        '    if unknown_fields:\n'
        '        raise ValueError(\n'
        '            f"artifact adaptation metadata contains unknown fields: {sorted(unknown_fields)}"\n'
        '        )\n'
        '    if adaptation.get("method") != ADAPTATION_METHOD:\n'
        '        raise ValueError("artifact adaptation metadata has an unsupported or missing method")\n'
        '    if adaptation.get("trainable_prefixes") != list(TRAINABLE_PREFIXES):\n'
        '        raise ValueError("artifact adaptation metadata has incomplete trainable prefixes")\n'
        '    trainable = adaptation.get("trainable_parameters")\n'
        '    frozen = adaptation.get("frozen_parameters")\n'
        '    if not isinstance(trainable, int) or isinstance(trainable, bool) or trainable <= 0:\n'
        '        raise ValueError("artifact adaptation metadata has invalid trainable parameter count")\n'
        '    if not isinstance(frozen, int) or isinstance(frozen, bool) or frozen < 0:\n'
        '        raise ValueError("artifact adaptation metadata has invalid frozen parameter count")\n'
        '    _validate_adaptation_run(\n'
        '        _adaptation_run_snapshot(adaptation),\n'
        '        label="artifact adaptation metadata current run",\n'
        '    )\n'
        '    if "training_runs" in adaptation:\n'
        '        training_runs = adaptation["training_runs"]\n'
        '        if not isinstance(training_runs, list) or not training_runs:\n'
        '            raise ValueError("artifact adaptation metadata training_runs must be a non-empty list")\n'
        '        for index, run in enumerate(training_runs):\n'
        '            _validate_adaptation_run(run, label=f"artifact adaptation metadata training_runs[{index}]")\n'
        '        if training_runs[-1] != _adaptation_run_snapshot(adaptation):\n'
        '            raise ValueError("artifact adaptation metadata latest training run does not match current run")\n'
        '    if "artifact_lineage" in adaptation:\n'
        '        artifact_lineage = adaptation["artifact_lineage"]\n'
        '        if not isinstance(artifact_lineage, list):\n'
        '            raise ValueError("artifact adaptation metadata artifact_lineage must be a list of mappings")\n'
        '        for index, item in enumerate(artifact_lineage):\n'
        '            if not isinstance(item, Mapping) or set(item) != _ARTIFACT_LINEAGE_FIELDS:\n'
        '                raise ValueError(\n'
        '                    f"artifact adaptation metadata artifact_lineage[{index}] has invalid fields"\n'
        '                )\n'
        '            for digest_field in ("manifest_sha256", "weights_sha256"):\n'
        '                digest = item[digest_field]\n'
        '                if (\n'
        '                    not isinstance(digest, str)\n'
        '                    or len(digest) != 64\n'
        '                    or any(character not in "0123456789abcdef" for character in digest)\n'
        '                ):\n'
        '                    raise ValueError(\n'
        '                        f"artifact adaptation metadata artifact_lineage[{index}] has an invalid digest"\n'
        '                    )\n'
        '            producer = item["producer"]\n'
        '            if (\n'
        '                not isinstance(producer, Mapping)\n'
        '                or set(producer) != {"pipelineId", "revision"}\n'
        '                or producer["pipelineId"] != "sam2-segmentation-pipeline"\n'
        '                or not isinstance(producer["revision"], str)\n'
        '                or len(producer["revision"]) != 40\n'
        '                or any(\n'
        '                    character not in "0123456789abcdef" for character in producer["revision"]\n'
        '                )\n'
        '            ):\n'
        '                raise ValueError(\n'
        '                    f"artifact adaptation metadata artifact_lineage[{index}] has an invalid producer"\n'
        '                )\n'
        '\n'
        '\n'
        'def _adaptation_run_snapshot(adaptation: Mapping[str, Any]) -> dict[str, Any]:\n'
        '    """Copy the per-run fields retained in cumulative adapter provenance."""\n'
        '    return {\n'
        '        key: json.loads(json.dumps(adaptation[key]))\n'
        '        for key in _ADAPTATION_RUN_FIELDS\n'
        '        if key in adaptation\n'
        '    }\n'
        '\n'
        '\n'
        'def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n'
        '    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""\n'
        '    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID:\n'
        '        raise ValueError(f"manifest modelId {manifest.get(\'modelId\')!r} != {MODEL_ID!r}")\n'
        '    if manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(f"manifest revision {manifest.get(\'revision\')!r} != {MODEL_REVISION!r}")\n'
        '    for entry in manifest["files"]:\n'
        '        file_path = root / entry["path"]\n'
        '        if not file_path.is_file():\n'
        '            raise FileNotFoundError(f"snapshot file missing: {file_path}")\n'
        '        size = file_path.stat().st_size\n'
        '        if size != entry["bytes"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: size {size} != manifest {entry[\'bytes\']}")\n'
        '        digest = _sha256(file_path)\n'
        '        if digest != entry["sha256"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: sha256 {digest} != manifest {entry[\'sha256\']}")\n'
        '    return {\n'
        '        "path": str(root),\n'
        '        "model_id": manifest["modelId"],\n'
        '        "revision": manifest["revision"],\n'
        '        "files": len(manifest["files"]),\n'
        '        "total_bytes": manifest.get("totalBytes"),\n'
        '    }\n'
        '\n'
        '\n'
        'def _hub_download(relative_path: str, root: Path) -> None:\n'
        '    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""\n'
        '    from huggingface_hub import hf_hub_download\n'
        '\n'
        '    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n'
        '\n'
        '\n'
        'def stage_missing_files(\n'
        '    path: str | Path | None = None,\n'
        '    *,\n'
        '    allow_download: bool = False,\n'
        '    downloader: Callable[[str, Path], None] | None = None,\n'
        ') -> list[str]:\n'
        '    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but\n'
        '    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""\n'
        '    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(\n'
        '            f"manifest names {manifest.get(\'modelId\')}@{manifest.get(\'revision\')}, "\n'
        '            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"\n'
        '        )\n'
        '    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]\n'
        '    if not missing:\n'
        '        return []\n'
        '    if not allow_download:\n'
        '        raise FileNotFoundError(\n'
        '            f"snapshot at {root} is missing {missing}; "\n'
        '            f"pass allow_download=True to fetch them at {MODEL_REVISION}"\n'
        '        )\n'
        '    fetch = downloader or _hub_download\n'
        '    for relative_path in missing:\n'
        '        fetch(relative_path, root)\n'
        '    return missing\n'
        '\n'
        '\n'
        'def mask_iou(a: np.ndarray, b: np.ndarray) -> float:\n'
        '    """Intersection-over-union of two boolean masks of identical shape; the primitive behind any mIoU."""\n'
        '    a = np.asarray(a)\n'
        '    b = np.asarray(b)\n'
        '    if a.shape != b.shape:\n'
        '        raise ValueError(f"shape mismatch: {a.shape} vs {b.shape}")\n'
        '    if a.dtype != np.bool_ or b.dtype != np.bool_:\n'
        '        raise TypeError("mask_iou expects boolean arrays")\n'
        '    union = np.logical_or(a, b).sum()\n'
        '    return float(np.logical_and(a, b).sum() / union) if union else 0.0\n'
        '\n'
        '\n'
        'def validate_image(image: Any) -> Image.Image:\n'
        '    if not isinstance(image, Image.Image):\n'
        '        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")\n'
        '    width, height = image.size\n'
        '    if min(width, height) < MIN_IMAGE_SIDE:\n'
        '        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")\n'
        '    if max(width, height) > MAX_IMAGE_SIDE:\n'
        '        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")\n'
        '    return image.convert("RGB")\n'
        '\n'
        '\n'
        'def validate_prompts(\n'
        '    width: int,\n'
        '    height: int,\n'
        '    points: Sequence[Sequence[float]] | None,\n'
        '    point_labels: Sequence[int] | None,\n'
        '    box: Sequence[float] | None,\n'
        ') -> tuple[list[list[float]] | None, list[int] | None, list[float] | None]:\n'
        '    """Check one object\'s prompts: points inside the image with 0/1 labels, and/or one xyxy box inside it."""\n'
        '    if points is None and box is None:\n'
        '        raise ValueError("provide at least one of points or box")\n'
        '    clean_points = clean_labels = None\n'
        '    if points is not None:\n'
        '        if isinstance(points, str) or not isinstance(points, Sequence):\n'
        '            raise TypeError("points must be a sequence of [x, y] pairs")\n'
        '        if not 1 <= len(points) <= MAX_PROMPTS:\n'
        '            raise ValueError(f"point count {len(points)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")\n'
        '        if point_labels is None or len(point_labels) != len(points):\n'
        '            raise ValueError("point_labels must be given with one 0/1 entry per point")\n'
        '        clean_points, clean_labels = [], []\n'
        '        for (x, y), label in zip(points, point_labels, strict=True):\n'
        '            if not (0 <= x < width and 0 <= y < height):\n'
        '                raise ValueError(f"point ({x}, {y}) outside image {width}x{height}")\n'
        '            if label not in (0, 1) or isinstance(label, bool):\n'
        '                raise ValueError(f"point label must be 0 or 1, got {label!r}")\n'
        '            clean_points.append([float(x), float(y)])\n'
        '            clean_labels.append(int(label))\n'
        '    elif point_labels is not None:\n'
        '        raise ValueError("point_labels given without points")\n'
        '    clean_box = None\n'
        '    if box is not None:\n'
        '        if len(box) != 4:\n'
        '            raise ValueError("box must be [x0, y0, x1, y1]")\n'
        '        x0, y0, x1, y1 = (float(v) for v in box)\n'
        '        if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):\n'
        '            raise ValueError(f"box {box!r} is not a non-empty xyxy box inside image {width}x{height}")\n'
        '        clean_box = [x0, y0, x1, y1]\n'
        '    return clean_points, clean_labels, clean_box\n'
        '\n'
        '\n'
        'INPUT_SCHEMA: dict[str, Any] = {\n'
        '    "input": (\n'
        '        "one PIL.Image.Image (any mode, converted to RGB) plus one object\'s prompts: point clicks "\n'
        '        "and/or one xyxy box"\n'
        '    ),\n'
        '    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],\n'
        '    "points": [1, MAX_PROMPTS],\n'
        '    "point_labels": "one per point, 1 = foreground and 0 = background",\n'
        '    "box": "at most one [x0, y0, x1, y1] inside the image with x0 < x1 and y0 < y1",\n'
        '    "adapted_multimask": (\n'
        '        "base pipelines default to three candidates; adapted pipelines default to one mask and reject "\n'
        '        "multimask=True because the adapter trains only the single-mask output hypernetwork"\n'
        '    ),\n'
        '    "objects_per_call": 1,\n'
        '    "multimask_outputs": NUM_MULTIMASK_OUTPUTS,\n'
        '    "preprocessing": (\n'
        '        "image converted to RGB; the processor resizes it to 1024x1024; returned masks are up-sampled "\n'
        '        f"to the input resolution and binarised at logit MASK_THRESHOLD={MASK_THRESHOLD}"\n'
        '    ),\n'
        '}\n'
        '\n'
        '\n'
        'def _check_inputs(\n'
        '    image: Any,\n'
        '    points: Any,\n'
        '    point_labels: Any,\n'
        '    box: Any,\n'
        '    multimask: Any,\n'
        ') -> tuple[Image.Image, list[list[float]] | None, list[int] | None, list[float] | None]:\n'
        '    """Raise TypeError/ValueError naming the first violated ceiling; return the cleaned request.\n'
        '\n'
        '    ``segment`` and ``validate_inputs`` both route through this function so their acceptance\n'
        '    criteria cannot diverge.\n'
        '    """\n'
        '    rgb = validate_image(image)\n'
        '    clean_points, clean_labels, clean_box = validate_prompts(\n'
        '        rgb.width, rgb.height, points, point_labels, box\n'
        '    )\n'
        '    if not isinstance(multimask, bool):\n'
        '        raise TypeError("multimask must be a bool")\n'
        '    return rgb, clean_points, clean_labels, clean_box\n'
        '\n'
        '\n'
        'def validate_inputs(\n'
        '    image: Image.Image,\n'
        '    *,\n'
        '    points: Sequence[Sequence[float]] | None = None,\n'
        '    point_labels: Sequence[int] | None = None,\n'
        '    box: Sequence[float] | None = None,\n'
        '    multimask: bool = True,\n'
        '    names: Sequence[str] | None = None,\n'
        ') -> dict[str, Any]:\n'
        '    """Validation stage: return the input manifest (schema, observations, request, verdict).\n'
        '\n'
        '    Rejection is reported by raising exactly as ``segment`` would; a caller that wants the finding\n'
        '    recorded catches the exception and stores ``str(exc)`` under ``findings``.\n'
        '    """\n'
        '    _rgb, clean_points, clean_labels, clean_box = _check_inputs(\n'
        '        image, points, point_labels, box, multimask\n'
        '    )\n'
        '    if names is not None and len(names) != 1:\n'
        '        raise ValueError("names must have exactly one entry (segment takes one image)")\n'
        '    return {\n'
        '        "schema": dict(INPUT_SCHEMA),\n'
        '        "inputs": [\n'
        '            {\n'
        '                "id": names[0] if names else "image-0",\n'
        '                "mode": image.mode,\n'
        '                "size": list(image.size),\n'
        '                "n_points": 0 if clean_points is None else len(clean_points),\n'
        '                "has_box": clean_box is not None,\n'
        '            }\n'
        '        ],\n'
        '        "points": clean_points,\n'
        '        "point_labels": clean_labels,\n'
        '        "box": clean_box,\n'
        '        "multimask": multimask,\n'
        '        "verdict": "accepted",\n'
        '        "findings": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        'def evaluation_report(\n'
        '    result: Mapping[str, Any],\n'
        '    reference_mask: Any = None,\n'
        '    *,\n'
        '    sample_kind: str = "synthetic",\n'
        ') -> dict[str, Any]:\n'
        '    """Evaluation stage: a machine-readable report even when nothing is measurable.\n'
        '\n'
        '    With a boolean ``reference_mask`` of the same shape as the returned masks the report carries one\n'
        '    ``mask_iou`` entry per candidate as sample-sanity geometry evidence; without one the verdict is\n'
        '    ``not-measurable`` and the report says what labelled data would make the task measurable.\n'
        '    """\n'
        '    masks = np.asarray(result["masks"])\n'
        '    scores = [float(v) for v in result["iou_scores"]]\n'
        '    best = int(np.argmax(scores)) if scores else None\n'
        '    base = {\n'
        '        "task": "promptable single-object image segmentation (point and/or box prompts)",\n'
        '        "decision_rule": (\n'
        '            "keep the candidate with the highest model-predicted IoU; the pipeline ships no "\n'
        '            "acceptance threshold and does not choose for the caller"\n'
        '        ),\n'
        '        "score_semantics": (\n'
        '            "iou_scores are the model\'s own uncalibrated predicted IoU for each candidate, not a "\n'
        '            "measured overlap and not a probability; the regression head is unclipped, so a value "\n'
        '            "may exceed 1.0"\n'
        '        ),\n'
        '        "sample_kind": sample_kind,\n'
        '        "n_masks": int(masks.shape[0]) if masks.ndim == 3 else 0,\n'
        '        "best_candidate": best,\n'
        '        "iou_scores_model_predicted": scores,\n'
        '        "mask_areas_px": [int(mask.sum()) for mask in masks] if masks.ndim == 3 else [],\n'
        '        "baselines": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '    if reference_mask is None:\n'
        '        return {\n'
        '            **base,\n'
        '            "metrics": [],\n'
        '            "verdict": "not-measurable",\n'
        '            "reason": "no ground-truth mask was supplied for the evaluated image",\n'
        '            "needs": (\n'
        '                "hand-labelled boolean masks for the prompted objects on your own images, scored with "\n'
        '                "mask_iou per object and averaged into a mean IoU over a held-out set; no labelled "\n'
        '                "mask set ships with this repository"\n'
        '            ),\n'
        '        }\n'
        '    reference = np.asarray(reference_mask)\n'
        '    return {\n'
        '        **base,\n'
        '        "metrics": [\n'
        '            {\n'
        '                "id": "mask_iou",\n'
        '                "candidate": index,\n'
        '                "value": mask_iou(masks[index], reference),\n'
        '                "selected": index == best,\n'
        '                "estimation": "one reference mask on a single scene, no dispersion estimate",\n'
        '            }\n'
        '            for index in range(masks.shape[0])\n'
        '        ],\n'
        '        "reference_area_px": int(reference.sum()),\n'
        '        "verdict": "sample-sanity",\n'
        '        "reason": (\n'
        '            "one reference mask on one tutorial sample; geometry sanity evidence, not a segmentation "\n'
        '            "benchmark"\n'
        '        ),\n'
        '        "needs": (\n'
        '            "a labelled mask set from the deployment domain for any mean-IoU or boundary-quality claim"\n'
        '        ),\n'
        '    }\n'
        '\n'
        '\n'
        'def validate_segmentation_dataset(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n'
        '    """Validate paired images, prompts, and boolean masks for bounded adaptation."""\n'
        '    if isinstance(records, str | bytes) or not isinstance(records, Sequence):\n'
        '        raise TypeError("records must be a sequence of mappings")\n'
        '    if not 2 <= len(records) <= MAX_ADAPTATION_RECORDS:\n'
        '        raise ValueError(f"record count {len(records)} outside 2..{MAX_ADAPTATION_RECORDS}")\n'
        '\n'
        '    ids: list[str] = []\n'
        '    digest = hashlib.sha256()\n'
        '    positive_pixels = 0\n'
        '    for index, record in enumerate(records):\n'
        '        if not isinstance(record, Mapping):\n'
        '            raise TypeError(f"record {index} must be a mapping")\n'
        '        record_id = record.get("id")\n'
        '        if not isinstance(record_id, str) or not record_id.strip():\n'
        '            raise ValueError(f"record {index} id must be a non-empty string")\n'
        '        if record_id in ids:\n'
        '            raise ValueError(f"duplicate record id: {record_id}")\n'
        '        ids.append(record_id)\n'
        '\n'
        '        image = validate_image(record.get("image"))\n'
        '        mask = np.asarray(record.get("mask"))\n'
        '        if mask.dtype != np.bool_:\n'
        '            raise TypeError(f"record {record_id} mask must be boolean")\n'
        '        if mask.shape != (image.height, image.width):\n'
        '            raise ValueError(\n'
        '                f"record {record_id} mask shape {mask.shape} != image {(image.height, image.width)}"\n'
        '            )\n'
        '        area = int(mask.sum())\n'
        '        if area == 0 or area == mask.size:\n'
        '            raise ValueError(f"record {record_id} mask must contain foreground and background")\n'
        '        clean_points, clean_labels, clean_box = validate_prompts(\n'
        '            image.width,\n'
        '            image.height,\n'
        '            record.get("points"),\n'
        '            record.get("point_labels"),\n'
        '            record.get("box"),\n'
        '        )\n'
        '        if clean_points is not None and clean_labels is not None:\n'
        '            for (x, y), label in zip(clean_points, clean_labels, strict=True):\n'
        '                row = min(int(y), image.height - 1)\n'
        '                column = min(int(x), image.width - 1)\n'
        '                pixel_is_foreground = bool(mask[row, column])\n'
        '                if pixel_is_foreground != bool(label):\n'
        '                    raise ValueError(\n'
        '                        f"record {record_id} point ({x}, {y}) label {label} contradicts target mask"\n'
        '                    )\n'
        '        if clean_box is not None:\n'
        '            x0, y0 = (math.floor(value) for value in clean_box[:2])\n'
        '            x1, y1 = (math.ceil(value) for value in clean_box[2:])\n'
        '            if not bool(mask[y0:y1, x0:x1].any()):\n'
        '                raise ValueError(f"record {record_id} box does not overlap target mask foreground")\n'
        '        positive_pixels += area\n'
        '        digest.update(record_id.encode("utf-8"))\n'
        '        digest.update(\n'
        '            json.dumps(\n'
        '                {"image_shape": [image.height, image.width, 3], "mask_shape": list(mask.shape)},\n'
        '                sort_keys=True,\n'
        '                separators=(",", ":"),\n'
        '            ).encode("utf-8")\n'
        '        )\n'
        '        digest.update(np.asarray(image, dtype=np.uint8).tobytes())\n'
        '        digest.update(mask.tobytes())\n'
        '        digest.update(\n'
        '            json.dumps(\n'
        '                {"points": clean_points, "point_labels": clean_labels, "box": clean_box},\n'
        '                sort_keys=True,\n'
        '                separators=(",", ":"),\n'
        '            ).encode("utf-8")\n'
        '        )\n'
        '\n'
        '    return {\n'
        '        "records": len(records),\n'
        '        "unique_ids": len(ids),\n'
        '        "positive_pixels": positive_pixels,\n'
        '        "dataset_sha256": digest.hexdigest(),\n'
        '        "verdict": "accepted",\n'
        '    }\n'
        '\n'
        '\n'
        'def _segmentation_record_content_sha256(record: Mapping[str, Any]) -> str:\n'
        '    """Fingerprint one validated image/mask sample without its ID or mutable prompts."""\n'
        '    image = validate_image(record["image"])\n'
        '    mask = np.asarray(record["mask"], dtype=np.bool_)\n'
        '    digest = hashlib.sha256()\n'
        '    digest.update(\n'
        '        json.dumps(\n'
        '            {"image_shape": [image.height, image.width, 3], "mask_shape": list(mask.shape)},\n'
        '            sort_keys=True,\n'
        '            separators=(",", ":"),\n'
        '        ).encode("utf-8")\n'
        '    )\n'
        '    digest.update(np.asarray(image, dtype=np.uint8).tobytes())\n'
        '    digest.update(mask.tobytes())\n'
        '    return digest.hexdigest()\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class SAM2SegmentationPipeline:\n'
        '    """Promptable image segmentation (points/box -> masks) over the pinned SAM 2.1 Hiera-Small checkpoint.\n'
        '\n'
        '    Image mode only: `Sam2Model` + `Sam2Processor`. Video tracking (`Sam2VideoModel`) is not exposed."""\n'
        '\n'
        '    _runner: Callable[..., tuple[np.ndarray, list[float]]]\n'
        '    device: str\n'
        '    model: Any | None = None\n'
        '    processor: Any | None = None\n'
        '    adaptation_config: dict[str, Any] = field(default_factory=dict)\n'
        '\n'
        '    @classmethod\n'
        '    def from_pretrained(\n'
        '        cls,\n'
        '        device: str | None = None,\n'
        '        weights_dir: str | Path | None = None,\n'
        '        allow_download: bool = False,\n'
        '    ) -> SAM2SegmentationPipeline:\n'
        '        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR\n'
        '        if (root / MANIFEST_NAME).is_file():\n'
        '            stage_missing_files(root, allow_download=allow_download)\n'
        '            verify_snapshot(root)\n'
        '            source, kwargs = str(root), {"local_files_only": True}\n'
        '        elif allow_download:\n'
        '            source, kwargs = MODEL_ID, {}\n'
        '        else:\n'
        '            raise FileNotFoundError(\n'
        '                f"no verified snapshot at {root} and allow_download=False; "\n'
        '                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"\n'
        '            )\n'
        '        # Refuse invalid snapshots before importing model libraries.\n'
        '        import torch\n'
        '        from transformers import Sam2Model, Sam2Processor\n'
        '        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n'
        '        processor = Sam2Processor.from_pretrained(\n'
        '            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs\n'
        '        )\n'
        '        # config.json declares Sam2VideoModel; image-mode Sam2Model loads the same weights (upstream README).\n'
        '        model = Sam2Model.from_pretrained(source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs)\n'
        '        model = model.to(resolved_device).eval()\n'
        '\n'
        '        def runner(image, points, labels, box, multimask) -> tuple[np.ndarray, list[float]]:\n'
        '            prompt_kwargs: dict[str, Any] = {}\n'
        '            if points is not None:\n'
        '                prompt_kwargs["input_points"] = [[points]]\n'
        '                prompt_kwargs["input_labels"] = [[labels]]\n'
        '            if box is not None:\n'
        '                prompt_kwargs["input_boxes"] = [[box]]\n'
        '            inputs = processor(images=image, return_tensors="pt", **prompt_kwargs).to(resolved_device)\n'
        '            with torch.inference_mode():\n'
        '                outputs = model(**inputs, multimask_output=multimask)\n'
        '            masks = processor.post_process_masks(\n'
        '                outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(), mask_threshold=MASK_THRESHOLD\n'
        '            )[0]\n'
        '            return masks[0].numpy().astype(np.bool_), [float(v) for v in outputs.iou_scores[0, 0].tolist()]\n'
        '\n'
        '        return cls(runner, resolved_device, model=model, processor=processor)\n'
        '\n'
        '    def freeze_for_adaptation(self) -> dict[str, int]:\n'
        '        """Freeze the base model and enable the first mask-token hypernetwork only."""\n'
        '        if self.model is None:\n'
        '            raise RuntimeError("cannot configure adaptation without an underlying torch model")\n'
        '        previous_config = dict(self.adaptation_config)\n'
        '        trainable = frozen = 0\n'
        '        for name, parameter in self.model.named_parameters():\n'
        '            parameter.requires_grad = name.startswith(TRAINABLE_PREFIXES)\n'
        '            if parameter.requires_grad:\n'
        '                trainable += parameter.numel()\n'
        '            else:\n'
        '                frozen += parameter.numel()\n'
        '        if trainable == 0:\n'
        '            raise RuntimeError("SAM2 adaptation selected no trainable parameters")\n'
        '        base_config = {\n'
        '            "method": ADAPTATION_METHOD,\n'
        '            "trainable_prefixes": list(TRAINABLE_PREFIXES),\n'
        '            "trainable_parameters": trainable,\n'
        '            "frozen_parameters": frozen,\n'
        '        }\n'
        '        if any(key in previous_config for key in _ADAPTATION_RUN_FIELDS):\n'
        '            _validate_completed_adaptation_metadata(previous_config)\n'
        '            previous_config.update(base_config)\n'
        '            self.adaptation_config = previous_config\n'
        '        else:\n'
        '            self.adaptation_config = base_config\n'
        '        return {"trainable_parameters": trainable, "frozen_parameters": frozen}\n'
        '\n'
        '    def finetune(\n'
        '        self,\n'
        '        train_records: Sequence[Mapping[str, Any]],\n'
        '        val_records: Sequence[Mapping[str, Any]] | None = None,\n'
        '        *,\n'
        '        epochs: int = 2,\n'
        '        learning_rate: float = 2e-5,\n'
        '        seed: int = 42,\n'
        '    ) -> list[dict[str, Any]]:\n'
        '        """Run bounded mask-hypernetwork fine-tuning with BCE plus soft-Dice loss."""\n'
        '        import random\n'
        '\n'
        '        import torch\n'
        '        import torch.nn.functional as F\n'
        '        from torch.optim import AdamW\n'
        '\n'
        '        if self.model is None or self.processor is None:\n'
        '            raise RuntimeError("cannot fine-tune without the underlying model and processor")\n'
        '        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:\n'
        '            raise ValueError("epochs must be an int in 1..20")\n'
        '        if not isinstance(learning_rate, int | float) or isinstance(learning_rate, bool):\n'
        '            raise TypeError("learning_rate must be numeric")\n'
        '        if not 0 < float(learning_rate) <= 1e-2:\n'
        '            raise ValueError("learning_rate must be in (0, 1e-2]")\n'
        '        if not isinstance(seed, int) or isinstance(seed, bool):\n'
        '            raise ValueError("seed must be an int")\n'
        '        train_manifest = validate_segmentation_dataset(train_records)\n'
        '        val_manifest = validate_segmentation_dataset(val_records) if val_records else None\n'
        '        adaptation_pixels = sum(\n'
        '            record["image"].width * record["image"].height for record in train_records\n'
        '        ) + sum(record["image"].width * record["image"].height for record in (val_records or ()))\n'
        '        if adaptation_pixels > MAX_ADAPTATION_PIXELS:\n'
        '            raise ValueError(\n'
        '                f"adaptation aggregate pixel count {adaptation_pixels} exceeds "\n'
        '                f"MAX_ADAPTATION_PIXELS {MAX_ADAPTATION_PIXELS}"\n'
        '            )\n'
        '        if val_records:\n'
        '            train_ids = {str(record["id"]) for record in train_records}\n'
        '            val_ids = {str(record["id"]) for record in val_records}\n'
        '            overlapping_ids = sorted(train_ids & val_ids)\n'
        '            if overlapping_ids:\n'
        '                raise ValueError(\n'
        '                    f"train and validation records overlap by id: {overlapping_ids[:5]}"\n'
        '                )\n'
        '            train_content = {\n'
        '                _segmentation_record_content_sha256(record) for record in train_records\n'
        '            }\n'
        '            val_content = {\n'
        '                _segmentation_record_content_sha256(record) for record in val_records\n'
        '            }\n'
        '            if train_content & val_content:\n'
        '                raise ValueError("train and validation records overlap by image/mask content")\n'
        '\n'
        '        entry_config = copy.deepcopy(self.adaptation_config)\n'
        '        entry_requires_grad = {\n'
        '            name: parameter.requires_grad for name, parameter in self.model.named_parameters()\n'
        '        }\n'
        '        entry_parameter_devices = {\n'
        '            name: parameter.device for name, parameter in self.model.named_parameters()\n'
        '        }\n'
        '        entry_parameter_grads = {\n'
        '            name: None if parameter.grad is None else parameter.grad.detach().clone()\n'
        '            for name, parameter in self.model.named_parameters()\n'
        '        }\n'
        '        entry_buffer_devices = {\n'
        '            name: buffer.device for name, buffer in self.model.named_buffers()\n'
        '        }\n'
        '        entry_training = self.model.training\n'
        '\n'
        '        def restore_entry_state() -> None:\n'
        '            self.adaptation_config = copy.deepcopy(entry_config)\n'
        '            with torch.no_grad():\n'
        '                for name, parameter in self.model.named_parameters():\n'
        '                    parameter.data = parameter.data.to(entry_parameter_devices[name])\n'
        '                    entry_grad = entry_parameter_grads[name]\n'
        '                    parameter.grad = (\n'
        '                        None\n'
        '                        if entry_grad is None\n'
        '                        else entry_grad.to(entry_parameter_devices[name]).clone()\n'
        '                    )\n'
        '                    parameter.requires_grad = entry_requires_grad[name]\n'
        '                for name, buffer in self.model.named_buffers():\n'
        '                    buffer.data = buffer.data.to(entry_buffer_devices[name])\n'
        '            self.model.train(entry_training)\n'
        '\n'
        '        device = torch.device(self.device)\n'
        '\n'
        '        def prepare(\n'
        '            record: Mapping[str, Any],\n'
        '        ) -> tuple[list[torch.Tensor], dict[str, torch.Tensor], torch.Tensor, torch.Tensor]:\n'
        '            prompt_kwargs: dict[str, Any] = {}\n'
        '            if record.get("points") is not None:\n'
        '                prompt_kwargs["input_points"] = [[record["points"]]]\n'
        '                prompt_kwargs["input_labels"] = [[record["point_labels"]]]\n'
        '            if record.get("box") is not None:\n'
        '                prompt_kwargs["input_boxes"] = [[record["box"]]]\n'
        '            batch = self.processor(\n'
        '                images=record["image"], return_tensors="pt", **prompt_kwargs\n'
        '            ).to(device)\n'
        '            with torch.inference_mode():\n'
        '                embeddings = [\n'
        '                    item.detach().cpu()\n'
        '                    for item in self.model.get_image_embeddings(batch["pixel_values"])\n'
        '                ]\n'
        '            prompts = {\n'
        '                key: batch[key].detach().cpu()\n'
        '                for key in ("input_points", "input_labels", "input_boxes")\n'
        '                if key in batch\n'
        '            }\n'
        '            target = torch.from_numpy(np.asarray(record["mask"], dtype=np.float32))\n'
        '            original_size = torch.tensor([[record["image"].height, record["image"].width]])\n'
        '            return embeddings, prompts, target, original_size\n'
        '\n'
        '        def score(\n'
        '            cached: Sequence[\n'
        '                tuple[list[torch.Tensor], dict[str, torch.Tensor], torch.Tensor, torch.Tensor]\n'
        '            ],\n'
        '        ) -> float:\n'
        '            values: list[float] = []\n'
        '            self.model.eval()\n'
        '            with torch.inference_mode():\n'
        '                for embeddings, prompts, target, original_size in cached:\n'
        '                    device_embeddings = [item.to(device) for item in embeddings]\n'
        '                    device_prompts = {key: value.to(device) for key, value in prompts.items()}\n'
        '                    device_target = target.to(device)\n'
        '                    outputs = self.model(\n'
        '                        image_embeddings=device_embeddings,\n'
        '                        multimask_output=False,\n'
        '                        **device_prompts,\n'
        '                    )\n'
        '                    public_mask = self.processor.post_process_masks(\n'
        '                        outputs.pred_masks.cpu(),\n'
        '                        original_size,\n'
        '                        mask_threshold=MASK_THRESHOLD,\n'
        '                    )[0][0, 0]\n'
        '                    values.append(mask_iou(public_mask.numpy(), device_target.cpu().numpy().astype(bool)))\n'
        '            return float(np.mean(values)) if values else 0.0\n'
        '\n'
        '        try:\n'
        '            torch.manual_seed(seed)\n'
        '            self.model.to(device).eval()\n'
        '            cached_train = [prepare(record) for record in train_records]\n'
        '            cached_val = [prepare(record) for record in val_records] if val_records else []\n'
        '            baseline_val_iou = score(cached_val) if cached_val else None\n'
        '        except BaseException:\n'
        '            restore_entry_state()\n'
        '            raise\n'
        '\n'
        '        try:\n'
        '            if not self.adaptation_config:\n'
        '                self.freeze_for_adaptation()\n'
        '            if any(\n'
        '                parameter.requires_grad and not name.startswith(TRAINABLE_PREFIXES)\n'
        '                for name, parameter in self.model.named_parameters()\n'
        '            ):\n'
        '                raise RuntimeError("parameters outside the declared SAM2 adapter surface are trainable")\n'
        '            trainable = {\n'
        '                name: parameter\n'
        '                for name, parameter in self.model.named_parameters()\n'
        '                if parameter.requires_grad\n'
        '            }\n'
        '            if not trainable:\n'
        '                raise RuntimeError("model has no trainable parameters")\n'
        '            before = {\n'
        '                name: parameter.detach().cpu().clone() for name, parameter in trainable.items()\n'
        '            }\n'
        '            optimizer = AdamW(list(trainable.values()), lr=float(learning_rate))\n'
        '            previous_config = copy.deepcopy(self.adaptation_config)\n'
        '            prior_runs = previous_config.get("training_runs")\n'
        '            if prior_runs is None:\n'
        '                prior_runs = (\n'
        '                    [_adaptation_run_snapshot(previous_config)]\n'
        '                    if _is_positive_finite_number(previous_config.get("weight_delta_l2"))\n'
        '                    and isinstance(previous_config.get("history"), list)\n'
        '                    and previous_config["history"]\n'
        '                    else []\n'
        '                )\n'
        '            elif not isinstance(prior_runs, list) or not all(\n'
        '                isinstance(run, Mapping) for run in prior_runs\n'
        '            ):\n'
        '                raise RuntimeError("adaptation training_runs metadata is malformed")\n'
        '            else:\n'
        '                prior_runs = [json.loads(json.dumps(run)) for run in prior_runs]\n'
        '            for key in (\n'
        '                "epochs",\n'
        '                "learning_rate",\n'
        '                "batch_size",\n'
        '                "seed",\n'
        '                "loss",\n'
        '                "train_manifest",\n'
        '                "validation_manifest",\n'
        '                "baseline_validation_mask_iou",\n'
        '                "history",\n'
        '                "weight_delta_l2",\n'
        '            ):\n'
        '                self.adaptation_config.pop(key, None)\n'
        '        except BaseException:\n'
        '            restore_entry_state()\n'
        '            raise\n'
        '\n'
        '        history: list[dict[str, Any]] = []\n'
        '        try:\n'
        '            for epoch in range(1, epochs + 1):\n'
        '                order = list(range(len(cached_train)))\n'
        '                random.Random(seed + epoch * 17).shuffle(order)\n'
        '                total_loss = 0.0\n'
        '                for index in order:\n'
        '                    embeddings, prompts, target, _original_size = cached_train[index]\n'
        '                    device_embeddings = [item.to(device) for item in embeddings]\n'
        '                    device_prompts = {key: value.to(device) for key, value in prompts.items()}\n'
        '                    device_target = target.to(device)\n'
        '                    optimizer.zero_grad(set_to_none=True)\n'
        '                    logits = self.model(\n'
        '                        image_embeddings=device_embeddings,\n'
        '                        multimask_output=False,\n'
        '                        **device_prompts,\n'
        '                    ).pred_masks[0, 0, 0]\n'
        '                    target_low = (\n'
        '                        F.interpolate(\n'
        '                            device_target[None, None], size=logits.shape, mode="area"\n'
        '                        )[0, 0]\n'
        '                        > 0\n'
        '                    ).to(logits.dtype)\n'
        '                    probabilities = logits.sigmoid()\n'
        '                    bce = F.binary_cross_entropy_with_logits(logits, target_low)\n'
        '                    dice = 1.0 - (2.0 * (probabilities * target_low).sum() + 1.0) / (\n'
        '                        probabilities.sum() + target_low.sum() + 1.0\n'
        '                    )\n'
        '                    loss = bce + dice\n'
        '                    if not bool(torch.isfinite(loss)):\n'
        '                        raise RuntimeError("fine-tuning produced a non-finite loss")\n'
        '                    loss.backward()\n'
        '                    optimizer.step()\n'
        '                    if any(\n'
        '                        not bool(torch.isfinite(parameter).all())\n'
        '                        for parameter in trainable.values()\n'
        '                    ):\n'
        '                        raise RuntimeError("fine-tuning produced non-finite adapter weights")\n'
        '                    total_loss += float(loss.item())\n'
        '                epoch_data: dict[str, Any] = {\n'
        '                    "epoch": epoch,\n'
        '                    "train_loss": round(total_loss / len(cached_train), 6),\n'
        '                    "optimizer_steps": len(cached_train),\n'
        '                }\n'
        '                if cached_val:\n'
        '                    epoch_data["val_mask_iou"] = round(score(cached_val), 6)\n'
        '                history.append(epoch_data)\n'
        '\n'
        '            delta_sq = 0.0\n'
        '            for name, parameter in trainable.items():\n'
        '                delta_sq += float(\n'
        '                    torch.sum((parameter.detach().cpu() - before[name]) ** 2).item()\n'
        '                )\n'
        '            weight_delta_l2 = delta_sq**0.5\n'
        '            if not math.isfinite(weight_delta_l2):\n'
        '                raise RuntimeError("fine-tuning produced a non-finite weight delta")\n'
        '            if weight_delta_l2 == 0.0:\n'
        '                raise RuntimeError("fine-tuning completed without changing adapter weights")\n'
        '            self.model.eval()\n'
        '            completed_run = {\n'
        '                "epochs": epochs,\n'
        '                "learning_rate": float(learning_rate),\n'
        '                "batch_size": 1,\n'
        '                "seed": seed,\n'
        '                "loss": "binary-cross-entropy-plus-soft-dice",\n'
        '                "train_manifest": train_manifest,\n'
        '                "validation_manifest": val_manifest,\n'
        '                "baseline_validation_mask_iou": baseline_val_iou,\n'
        '                "history": history,\n'
        '                "weight_delta_l2": weight_delta_l2,\n'
        '            }\n'
        '            stored_run = json.loads(json.dumps(completed_run))\n'
        '            self.adaptation_config.update(stored_run)\n'
        '            self.adaptation_config["training_runs"] = [\n'
        '                *prior_runs,\n'
        '                json.loads(json.dumps(stored_run)),\n'
        '            ]\n'
        '        except BaseException:\n'
        '            with torch.no_grad():\n'
        '                for name, parameter in trainable.items():\n'
        '                    parameter.copy_(before[name].to(device=parameter.device, dtype=parameter.dtype))\n'
        '            restore_entry_state()\n'
        '            raise\n'
        '        return json.loads(json.dumps(history))\n'
        '\n'
        '    def evaluate_adaptation(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n'
        '        """Evaluate adapted masks against ground truth and a prompt-box baseline."""\n'
        '        validate_segmentation_dataset(records)\n'
        '        model_ious: list[float] = []\n'
        '        box_ious: list[float] = []\n'
        '        box_model_ious: list[float] = []\n'
        '        for record in records:\n'
        '            result = self.segment(\n'
        '                record["image"],\n'
        '                points=record.get("points"),\n'
        '                point_labels=record.get("point_labels"),\n'
        '                box=record.get("box"),\n'
        '                multimask=False,\n'
        '            )\n'
        '            target = np.asarray(record["mask"], dtype=np.bool_)\n'
        '            model_iou = mask_iou(result["masks"][0], target)\n'
        '            model_ious.append(model_iou)\n'
        '            box = record.get("box")\n'
        '            if box is not None:\n'
        '                baseline = np.zeros_like(target)\n'
        '                x0, y0 = (math.floor(float(value)) for value in box[:2])\n'
        '                x1, y1 = (math.ceil(float(value)) for value in box[2:])\n'
        '                baseline[y0:y1, x0:x1] = True\n'
        '                box_ious.append(mask_iou(baseline, target))\n'
        '                box_model_ious.append(model_iou)\n'
        '        box_baseline_mean_iou = float(np.mean(box_ious)) if box_ious else None\n'
        '        box_prompt_model_mean_iou = float(np.mean(box_model_ious)) if box_model_ious else None\n'
        '        return {\n'
        '            "records": len(records),\n'
        '            "mean_mask_iou": float(np.mean(model_ious)),\n'
        '            "per_record_mask_iou": model_ious,\n'
        '            "box_baseline_records": len(box_ious),\n'
        '            "box_prompt_model_mean_iou": box_prompt_model_mean_iou,\n'
        '            "box_baseline_mean_iou": box_baseline_mean_iou,\n'
        '            "delta_over_box_baseline": (\n'
        '                None\n'
        '                if box_baseline_mean_iou is None\n'
        '                else float(box_prompt_model_mean_iou - box_baseline_mean_iou)\n'
        '            ),\n'
        '        }\n'
        '\n'
        '    def save_artifact(self, output_dir: str | Path, *, producer_revision: str) -> Path:\n'
        '        """Write a safe mask-hypernetwork adapter plus a closed integrity manifest."""\n'
        '        from safetensors.torch import save_file\n'
        '\n'
        '        if self.model is None:\n'
        '            raise RuntimeError("artifact export requires an underlying model")\n'
        '        try:\n'
        '            _validate_completed_adaptation_metadata(self.adaptation_config)\n'
        '        except ValueError as exc:\n'
        '            raise RuntimeError("artifact export requires completed fine-tuning metadata") from exc\n'
        '        if len(producer_revision) != 40 or any(ch not in "0123456789abcdef" for ch in producer_revision):\n'
        '            raise ValueError("producer_revision must be a lowercase 40-hex Git commit")\n'
        '        state = {}\n'
        '        for name, tensor in self.model.state_dict().items():\n'
        '            if not name.startswith(TRAINABLE_PREFIXES):\n'
        '                continue\n'
        '            if not bool(tensor.isfinite().all()):\n'
        '                raise RuntimeError(f"cannot export non-finite adapter tensor: {name}")\n'
        '            state[name] = tensor.detach().cpu().contiguous()\n'
        '        if not state:\n'
        '            raise RuntimeError("no adapter tensors selected for export")\n'
        '        root = Path(output_dir)\n'
        '        root.mkdir(parents=True, exist_ok=True)\n'
        '        if any(root.iterdir()):\n'
        '            raise FileExistsError(f"artifact directory is not empty: {root}")\n'
        '        weights_path = root / ARTIFACT_WEIGHTS_NAME\n'
        '        save_file(state, str(weights_path))\n'
        '        manifest = {\n'
        '            "artifactSpec": "1.0",\n'
        '            "format": ARTIFACT_FORMAT,\n'
        '            "formatVersion": ARTIFACT_FORMAT_VERSION,\n'
        '            "artifactClass": "ADAPTER",\n'
        '            "artifactKind": "sam2-mask-hypernetwork-adapter",\n'
        '            "producer": {"pipelineId": "sam2-segmentation-pipeline", "revision": producer_revision},\n'
        '            "createdAtUtc": datetime.now(UTC).isoformat(),\n'
        '            "baseModel": {"id": MODEL_ID, "revision": MODEL_REVISION},\n'
        '            "files": [\n'
        '                {\n'
        '                    "path": ARTIFACT_WEIGHTS_NAME,\n'
        '                    "bytes": weights_path.stat().st_size,\n'
        '                    "sha256": _sha256(weights_path),\n'
        '                }\n'
        '            ],\n'
        '            "adaptation": copy.deepcopy(self.adaptation_config),\n'
        '            "trainablePrefixes": list(TRAINABLE_PREFIXES),\n'
        '            "retainedData": {"containsTrainingRecords": False, "containsSupportRecords": False},\n'
        '            "serialization": "safetensors",\n'
        '        }\n'
        '        (root / ARTIFACT_MANIFEST_NAME).write_text(\n'
        '            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"\n'
        '        )\n'
        '        return root\n'
        '\n'
        '    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:\n'
        '        """Verify and load a SAM2 adapter without code-capable deserialization."""\n'
        '        from safetensors.torch import load_file\n'
        '\n'
        '        if self.model is None:\n'
        '            raise RuntimeError("cannot load an artifact without an underlying torch model")\n'
        '        root = Path(artifact_dir)\n'
        '        manifest_path = root / ARTIFACT_MANIFEST_NAME\n'
        '        if not manifest_path.is_file():\n'
        '            raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")\n'
        '        expected_files = {ARTIFACT_MANIFEST_NAME, ARTIFACT_WEIGHTS_NAME}\n'
        '        actual_entries = {path.name for path in root.iterdir()}\n'
        '        if actual_entries != expected_files:\n'
        '            raise ValueError(\n'
        '                f"artifact directory must contain exactly {sorted(expected_files)}, "\n'
        '                f"found {sorted(actual_entries)}"\n'
        '            )\n'
        '        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n'
        '        if manifest.get("format") != ARTIFACT_FORMAT:\n'
        '            raise ValueError(f"unrecognized artifact format: {manifest.get(\'format\')}")\n'
        '        if manifest.get("formatVersion") != ARTIFACT_FORMAT_VERSION:\n'
        '            raise ValueError(f"unsupported artifact formatVersion: {manifest.get(\'formatVersion\')}")\n'
        '        if manifest.get("baseModel") != {"id": MODEL_ID, "revision": MODEL_REVISION}:\n'
        '            raise ValueError("artifact base model identity is incompatible")\n'
        '        if manifest.get("trainablePrefixes") != list(TRAINABLE_PREFIXES):\n'
        '            raise ValueError("artifact trainable prefixes do not match this pipeline")\n'
        '        producer = manifest.get("producer")\n'
        '        if (\n'
        '            not isinstance(producer, Mapping)\n'
        '            or set(producer) != {"pipelineId", "revision"}\n'
        '            or producer["pipelineId"] != "sam2-segmentation-pipeline"\n'
        '            or not isinstance(producer["revision"], str)\n'
        '            or len(producer["revision"]) != 40\n'
        '            or any(character not in "0123456789abcdef" for character in producer["revision"])\n'
        '        ):\n'
        '            raise ValueError("artifact producer identity is invalid")\n'
        '        files = manifest.get("files")\n'
        '        if not isinstance(files, list) or len(files) != 1:\n'
        '            raise ValueError("artifact manifest must inventory exactly one weights file")\n'
        '        entry = files[0]\n'
        '        if entry.get("path") != ARTIFACT_WEIGHTS_NAME:\n'
        '            raise ValueError("artifact manifest names an unexpected weights path")\n'
        '        weights_path = root / ARTIFACT_WEIGHTS_NAME\n'
        '        if not weights_path.is_file():\n'
        '            raise FileNotFoundError(f"artifact weights not found: {weights_path}")\n'
        '        if weights_path.stat().st_size != entry.get("bytes") or _sha256(weights_path) != entry.get("sha256"):\n'
        '            raise ValueError("artifact weights failed size or SHA-256 verification")\n'
        '        adaptation = manifest.get("adaptation")\n'
        '        _validate_completed_adaptation_metadata(adaptation)\n'
        '        state = load_file(str(weights_path), device=self.device)\n'
        '        expected = {\n'
        '            name for name in self.model.state_dict() if name.startswith(TRAINABLE_PREFIXES)\n'
        '        }\n'
        '        if set(state) != expected:\n'
        '            raise ValueError("artifact tensor inventory does not match the declared adapter surface")\n'
        '        destination = self.model.state_dict()\n'
        '        for name, tensor in state.items():\n'
        '            expected_tensor = destination[name]\n'
        '            if tensor.shape != expected_tensor.shape or tensor.dtype != expected_tensor.dtype:\n'
        '                raise ValueError(\n'
        '                    f"artifact tensor {name} has shape/dtype {tuple(tensor.shape)}/{tensor.dtype}; "\n'
        '                    f"expected {tuple(expected_tensor.shape)}/{expected_tensor.dtype}"\n'
        '                )\n'
        '            if not bool(tensor.isfinite().all()):\n'
        '                raise ValueError(f"artifact tensor {name} contains non-finite values")\n'
        '        self.model.load_state_dict(state, strict=False)\n'
        '        self.model.to(self.device).eval()\n'
        '        # A fresh base model starts fully trainable. Reapply the declared adapter surface so a\n'
        '        # verified artifact can be continued with ``finetune`` without exposing base parameters.\n'
        '        self.freeze_for_adaptation()\n'
        '        loaded_adaptation = copy.deepcopy(adaptation)\n'
        '        prior_artifacts = loaded_adaptation.get("artifact_lineage", [])\n'
        '        loaded_adaptation["artifact_lineage"] = [\n'
        '            *json.loads(json.dumps(prior_artifacts)),\n'
        '            {\n'
        '                "manifest_sha256": _sha256(manifest_path),\n'
        '                "weights_sha256": entry["sha256"],\n'
        '                "producer": copy.deepcopy(producer),\n'
        '            },\n'
        '        ]\n'
        '        self.adaptation_config = loaded_adaptation\n'
        '        return manifest\n'
        '\n'
        '    @classmethod\n'
        '    def from_artifact(\n'
        '        cls,\n'
        '        artifact_dir: str | Path,\n'
        '        *,\n'
        '        device: str | None = None,\n'
        '        weights_dir: str | Path | None = None,\n'
        '        allow_download: bool = False,\n'
        '    ) -> SAM2SegmentationPipeline:\n'
        '        """Construct a fresh pinned base model and attach a verified adapter."""\n'
        '        pipeline = cls.from_pretrained(\n'
        '            device=device, weights_dir=weights_dir, allow_download=allow_download\n'
        '        )\n'
        '        pipeline.load_artifact(artifact_dir)\n'
        '        return pipeline\n'
        '\n'
        '    def segment(\n'
        '        self,\n'
        '        image: Image.Image,\n'
        '        *,\n'
        '        points: Sequence[Sequence[float]] | None = None,\n'
        '        point_labels: Sequence[int] | None = None,\n'
        '        box: Sequence[float] | None = None,\n'
        '        multimask: bool | None = None,\n'
        '    ) -> dict[str, Any]:\n'
        '        """Segment one object, defaulting adapted pipelines to their trained single-mask head."""\n'
        '        resolved_multimask = not bool(self.adaptation_config) if multimask is None else multimask\n'
        '        rgb, clean_points, clean_labels, clean_box = _check_inputs(\n'
        '            image, points, point_labels, box, resolved_multimask\n'
        '        )\n'
        '        if self.adaptation_config and resolved_multimask:\n'
        '            raise ValueError(\n'
        '                "adapted pipelines require multimask=False because the adapter trains only "\n'
        '                "the single-mask output hypernetwork"\n'
        '            )\n'
        '        masks, iou_scores = self._runner(\n'
        '            rgb, clean_points, clean_labels, clean_box, resolved_multimask\n'
        '        )\n'
        '        masks = np.asarray(masks)\n'
        '        expected = (NUM_MULTIMASK_OUTPUTS if resolved_multimask else 1, rgb.height, rgb.width)\n'
        '        if masks.dtype != np.bool_ or masks.shape != expected or len(iou_scores) != expected[0]:\n'
        '            raise RuntimeError(f"backend returned {masks.shape} {masks.dtype}, {len(iou_scores)} scores")\n'
        '        return {\n'
        '            "masks": masks,\n'
        '            "iou_scores": [float(v) for v in iou_scores],\n'
        '            "multimask": resolved_multimask,\n'
        '            "points": clean_points,\n'
        '            "point_labels": clean_labels,\n'
        '            "box": clean_box,\n'
        '            "width": rgb.width,\n'
        '            "height": rgb.height,\n'
        '            "model_id": MODEL_ID,\n'
        '            "model_revision": MODEL_REVISION,\n'
        '        }\n'
    ),
    'weights/sam2/dimer-base-manifest.json': (
        '{\n'
        '  "format": "dimer_hf_snapshot",\n'
        '  "formatVersion": 1,\n'
        '  "modelKey": "sam2.1-hiera-small",\n'
        '  "modelId": "facebook/sam2.1-hiera-small",\n'
        '  "revision": "ee5bba1d82bb8749febdf90f45e84b687142ba03",\n'
        '  "files": [\n'
        '    {\n'
        '      "path": "README.md",\n'
        '      "bytes": 19974,\n'
        '      "sha256": "6ec2d54879e41ad876d8cded0d641e8bc6ab74d5e095a73afc3f8406372ee6e9"\n'
        '    },\n'
        '    {\n'
        '      "path": "config.json",\n'
        '      "bytes": 5698,\n'
        '      "sha256": "97ff9f65b76d107acda4247885f0a5555d0048850ae3c5f97183df289aaecde9"\n'
        '    },\n'
        '    {\n'
        '      "path": "model.safetensors",\n'
        '      "bytes": 184305280,\n'
        '      "sha256": "0a4067b11ce1e23d5229203f11c718a823060d15a4b23fa2372a7d4b77cbbc60"\n'
        '    },\n'
        '    {\n'
        '      "path": "preprocessor_config.json",\n'
        '      "bytes": 683,\n'
        '      "sha256": "6ebf229ee259368ce4a8d4f2fe893a72b053023710853e257253939e601f583d"\n'
        '    },\n'
        '    {\n'
        '      "path": "processor_config.json",\n'
        '      "bytes": 95,\n'
        '      "sha256": "f8a68e865cfad115c1c2763f3d93eca7b1c622da06da2a9273eb437fb2389b6d"\n'
        '    },\n'
        '    {\n'
        '      "path": "sam2.1_hiera_s.yaml",\n'
        '      "bytes": 3761,\n'
        '      "sha256": "632e5cd0104f5ab6cd4f9d2dfd80a8e7240e481ad7960a13cad2ae3504b88dbd"\n'
        '    },\n'
        '    {\n'
        '      "path": "video_preprocessor_config.json",\n'
        '      "bytes": 705,\n'
        '      "sha256": "9fccfe5f464ec38c2f236d0e6a68e95511c80c22132fc2fa4b9f7b65f24fad95"\n'
        '    }\n'
        '  ],\n'
        '  "totalBytes": 184336196\n'
        '}'
    ),
    'licenses/sam2.txt': (
        '                                 Apache License\n'
        '                           Version 2.0, January 2004\n'
        '                        http://www.apache.org/licenses/\n'
        '\n'
        '   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n'
        '\n'
        '   1. Definitions.\n'
        '\n'
        '      "License" shall mean the terms and conditions for use, reproduction,\n'
        '      and distribution as defined by Sections 1 through 9 of this document.\n'
        '\n'
        '      "Licensor" shall mean the copyright owner or entity authorized by\n'
        '      the copyright owner that is granting the License.\n'
        '\n'
        '      "Legal Entity" shall mean the union of the acting entity and all\n'
        '      other entities that control, are controlled by, or are under common\n'
        '      control with that entity. For the purposes of this definition,\n'
        '      "control" means (i) the power, direct or indirect, to cause the\n'
        '      direction or management of such entity, whether by contract or\n'
        '      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n'
        '      outstanding shares, or (iii) beneficial ownership of such entity.\n'
        '\n'
        '      "You" (or "Your") shall mean an individual or Legal Entity\n'
        '      exercising permissions granted by this License.\n'
        '\n'
        '      "Source" form shall mean the preferred form for making modifications,\n'
        '      including but not limited to software source code, documentation\n'
        '      source, and configuration files.\n'
        '\n'
        '      "Object" form shall mean any form resulting from mechanical\n'
        '      transformation or translation of a Source form, including but\n'
        '      not limited to compiled object code, generated documentation,\n'
        '      and conversions to other media types.\n'
        '\n'
        '      "Work" shall mean the work of authorship, whether in Source or\n'
        '      Object form, made available under the License, as indicated by a\n'
        '      copyright notice that is included in or attached to the work\n'
        '      (an example is provided in the Appendix below).\n'
        '\n'
        '      "Derivative Works" shall mean any work, whether in Source or Object\n'
        '      form, that is based on (or derived from) the Work and for which the\n'
        '      editorial revisions, annotations, elaborations, or other modifications\n'
        '      represent, as a whole, an original work of authorship. For the purposes\n'
        '      of this License, Derivative Works shall not include works that remain\n'
        '      separable from, or merely link (or bind by name) to the interfaces of,\n'
        '      the Work and Derivative Works thereof.\n'
        '\n'
        '      "Contribution" shall mean any work of authorship, including\n'
        '      the original version of the Work and any modifications or additions\n'
        '      to that Work or Derivative Works thereof, that is intentionally\n'
        '      submitted to Licensor for inclusion in the Work by the copyright owner\n'
        '      or by an individual or Legal Entity authorized to submit on behalf of\n'
        '      the copyright owner. For the purposes of this definition, "submitted"\n'
        '      means any form of electronic, verbal, or written communication sent\n'
        '      to the Licensor or its representatives, including but not limited to\n'
        '      communication on electronic mailing lists, source code control systems,\n'
        '      and issue tracking systems that are managed by, or on behalf of, the\n'
        '      Licensor for the purpose of discussing and improving the Work, but\n'
        '      excluding communication that is conspicuously marked or otherwise\n'
        '      designated in writing by the copyright owner as "Not a Contribution."\n'
        '\n'
        '      "Contributor" shall mean Licensor and any individual or Legal Entity\n'
        '      on behalf of whom a Contribution has been received by Licensor and\n'
        '      subsequently incorporated within the Work.\n'
        '\n'
        '   2. Grant of Copyright License. Subject to the terms and conditions of\n'
        '      this License, each Contributor hereby grants to You a perpetual,\n'
        '      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n'
        '      copyright license to reproduce, prepare Derivative Works of,\n'
        '      publicly display, publicly perform, sublicense, and distribute the\n'
        '      Work and such Derivative Works in Source or Object form.\n'
        '\n'
        '   3. Grant of Patent License. Subject to the terms and conditions of\n'
        '      this License, each Contributor hereby grants to You a perpetual,\n'
        '      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n'
        '      (except as stated in this section) patent license to make, have made,\n'
        '      use, offer to sell, sell, import, and otherwise transfer the Work,\n'
        '      where such license applies only to those patent claims licensable\n'
        '      by such Contributor that are necessarily infringed by their\n'
        '      Contribution(s) alone or by combination of their Contribution(s)\n'
        '      with the Work to which such Contribution(s) was submitted. If You\n'
        '      institute patent litigation against any entity (including a\n'
        '      cross-claim or counterclaim in a lawsuit) alleging that the Work\n'
        '      or a Contribution incorporated within the Work constitutes direct\n'
        '      or contributory patent infringement, then any patent licenses\n'
        '      granted to You under this License for that Work shall terminate\n'
        '      as of the date such litigation is filed.\n'
        '\n'
        '   4. Redistribution. You may reproduce and distribute copies of the\n'
        '      Work or Derivative Works thereof in any medium, with or without\n'
        '      modifications, and in Source or Object form, provided that You\n'
        '      meet the following conditions:\n'
        '\n'
        '      (a) You must give any other recipients of the Work or Derivative\n'
        '          Works a copy of this License; and\n'
        '\n'
        '      (b) You must cause any modified files to carry prominent notices\n'
        '          stating that You changed the files; and\n'
        '\n'
        '      (c) You must retain, in the Source form of any Derivative Works\n'
        '          that You distribute, all copyright, patent, trademark, and\n'
        '          attribution notices from the Source form of the Work,\n'
        '          excluding those notices that do not pertain to any part of\n'
        '          the Derivative Works; and\n'
        '\n'
        '      (d) If the Work includes a "NOTICE" text file as part of its\n'
        '          distribution, then any Derivative Works that You distribute must\n'
        '          include a readable copy of the attribution notices contained\n'
        '          within such NOTICE file, excluding those notices that do not\n'
        '          pertain to any part of the Derivative Works, in at least one\n'
        '          of the following places: within a NOTICE text file distributed\n'
        '          as part of the Derivative Works; within the Source form or\n'
        '          documentation, if provided along with the Derivative Works; or,\n'
        '          within a display generated by the Derivative Works, if and\n'
        '          wherever such third-party notices normally appear. The contents\n'
        '          of the NOTICE file are for informational purposes only and\n'
        '          do not modify the License. You may add Your own attribution\n'
        '          notices within Derivative Works that You distribute, alongside\n'
        '          or as an addendum to the NOTICE text from the Work, provided\n'
        '          that such additional attribution notices cannot be construed\n'
        '          as modifying the License.\n'
        '\n'
        '      You may add Your own copyright statement to Your modifications and\n'
        '      may provide additional or different license terms and conditions\n'
        '      for use, reproduction, or distribution of Your modifications, or\n'
        '      for any such Derivative Works as a whole, provided Your use,\n'
        '      reproduction, and distribution of the Work otherwise complies with\n'
        '      the conditions stated in this License.\n'
        '\n'
        '   5. Submission of Contributions. Unless You explicitly state otherwise,\n'
        '      any Contribution intentionally submitted for inclusion in the Work\n'
        '      by You to the Licensor shall be under the terms and conditions of\n'
        '      this License, without any additional terms or conditions.\n'
        '      Notwithstanding the above, nothing herein shall supersede or modify\n'
        '      the terms of any separate license agreement you may have executed\n'
        '      with Licensor regarding such Contributions.\n'
        '\n'
        '   6. Trademarks. This License does not grant permission to use the trade\n'
        '      names, trademarks, service marks, or product names of the Licensor,\n'
        '      except as required for reasonable and customary use in describing the\n'
        '      origin of the Work and reproducing the content of the NOTICE file.\n'
        '\n'
        '   7. Disclaimer of Warranty. Unless required by applicable law or\n'
        '      agreed to in writing, Licensor provides the Work (and each\n'
        '      Contributor provides its Contributions) on an "AS IS" BASIS,\n'
        '      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n'
        '      implied, including, without limitation, any warranties or conditions\n'
        '      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n'
        '      PARTICULAR PURPOSE. You are solely responsible for determining the\n'
        '      appropriateness of using or redistributing the Work and assume any\n'
        '      risks associated with Your exercise of permissions under this License.\n'
        '\n'
        '   8. Limitation of Liability. In no event and under no legal theory,\n'
        '      whether in tort (including negligence), contract, or otherwise,\n'
        '      unless required by applicable law (such as deliberate and grossly\n'
        '      negligent acts) or agreed to in writing, shall any Contributor be\n'
        '      liable to You for damages, including any direct, indirect, special,\n'
        '      incidental, or consequential damages of any character arising as a\n'
        '      result of this License or out of the use or inability to use the\n'
        '      Work (including but not limited to damages for loss of goodwill,\n'
        '      work stoppage, computer failure or malfunction, or any and all\n'
        '      other commercial damages or losses), even if such Contributor\n'
        '      has been advised of the possibility of such damages.\n'
        '\n'
        '   9. Accepting Warranty or Additional Liability. While redistributing\n'
        '      the Work or Derivative Works thereof, You may choose to offer,\n'
        '      and charge a fee for, acceptance of support, warranty, indemnity,\n'
        '      or other liability obligations and/or rights consistent with this\n'
        '      License. However, in accepting such obligations, You may act only\n'
        '      on Your own behalf and on Your sole responsibility, not on behalf\n'
        '      of any other Contributor, and only if You agree to indemnify,\n'
        '      defend, and hold each Contributor harmless for any liability\n'
        '      incurred by, or claims asserted against, such Contributor by reason\n'
        '      of your accepting any such warranty or additional liability.\n'
        '\n'
        '   END OF TERMS AND CONDITIONS\n'
        '\n'
        '   APPENDIX: How to apply the Apache License to your work.\n'
        '\n'
        '      To apply the Apache License to your work, attach the following\n'
        '      boilerplate notice, with the fields enclosed by brackets "[]"\n'
        "      replaced with your own identifying information. (Don't include\n"
        '      the brackets!)  The text should be enclosed in the appropriate\n'
        '      comment syntax for the file format. We also recommend that a\n'
        '      file or class name and description of purpose be included on the\n'
        '      same "printed page" as the copyright notice for easier\n'
        '      identification within third-party archives.\n'
        '\n'
        '   Copyright 2026 Kurt Valcorza\n'
        '\n'
        '   Licensed under the Apache License, Version 2.0 (the "License");\n'
        '   you may not use this file except in compliance with the License.\n'
        '   You may obtain a copy of the License at\n'
        '\n'
        '       http://www.apache.org/licenses/LICENSE-2.0\n'
        '\n'
        '   Unless required by applicable law or agreed to in writing, software\n'
        '   distributed under the License is distributed on an "AS IS" BASIS,\n'
        '   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n'
        '   See the License for the specific language governing permissions and\n'
        '   limitations under the License.\n'
    ),
    'vendor_provenance.json': (
        '{\n'
        '  "sam": {\n'
        '    "repository": "kurtvalcorza/sam-vit-segmentation-pipeline",\n'
        '    "revision": "ed74a93f0c743dd184efe4f88b23f71b772e4d38",\n'
        '    "files": {\n'
        '      "sam_reference.py": {\n'
        '        "sha256": "ecdf92a7985ba182d716648545c7ba76223eac91cdd1753487d0350daf37d714",\n'
        '        "source": "src/sam_vit_segmentation_pipeline/pipeline.py"\n'
        '      },\n'
        '      "sam_manifest.json": {\n'
        '        "sha256": "3b51600ae3133cf24fe8ffaed771d94185f3935c9803d2c57c8b4a8c74c71dd4",\n'
        '        "source": "weights/sam-vit-base/dimer-base-manifest.json"\n'
        '      },\n'
        '      "sam_LICENSE": {\n'
        '        "sha256": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138",\n'
        '        "source": "LICENSE"\n'
        '      }\n'
        '    }\n'
        '  },\n'
        '  "sam2": {\n'
        '    "repository": "kurtvalcorza/sam2-segmentation-pipeline",\n'
        '    "revision": "df023e1af8c0f49e74af3f308e3e50b236e15850",\n'
        '    "files": {\n'
        '      "sam2_reference.py": {\n'
        '        "sha256": "8c2a3aac5444f2bebe2a56202fd42ae80e539e1a1921e5eb9f7b8c12c98d0a3b",\n'
        '        "source": "src/sam2_segmentation_pipeline/pipeline.py"\n'
        '      },\n'
        '      "sam2_manifest.json": {\n'
        '        "sha256": "ccbf7408250a591a6dbef9782ef2a262997bdab4ee97512516b54797769b9a11",\n'
        '        "source": "weights/sam2.1-hiera-small/dimer-base-manifest.json"\n'
        '      },\n'
        '      "sam2_LICENSE": {\n'
        '        "sha256": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138",\n'
        '        "source": "LICENSE"\n'
        '      }\n'
        '    }\n'
        '  }\n'
        '}'
    ),
    'weights/clipseg/dimer-base-manifest.json': (
        '{\n'
        '  "format": "dimer_hf_snapshot",\n'
        '  "formatVersion": 1,\n'
        '  "modelKey": "clipseg-rd64-refined",\n'
        '  "modelId": "CIDAS/clipseg-rd64-refined",\n'
        '  "revision": "999e0328d9e10b484360c477313983f9afdd7050",\n'
        '  "files": [\n'
        '    {\n'
        '      "path": "README.md",\n'
        '      "bytes": 596,\n'
        '      "sha256": "c1e26ac022542015a04d804460bdc1159d753c4349b628e086f127475d2fcf25"\n'
        '    },\n'
        '    {\n'
        '      "path": "config.json",\n'
        '      "bytes": 4732,\n'
        '      "sha256": "c023375966d31b3b1392764f7bd91df47098ce19f62f11b0263d8eedcf708bcd"\n'
        '    },\n'
        '    {\n'
        '      "path": "merges.txt",\n'
        '      "bytes": 524619,\n'
        '      "sha256": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a"\n'
        '    },\n'
        '    {\n'
        '      "path": "model.safetensors",\n'
        '      "bytes": 603049496,\n'
        '      "sha256": "d00ca85d6b859f9d07b7cfb8ef26fe9771cb275b34c9368f2ecf603139307f55"\n'
        '    },\n'
        '    {\n'
        '      "path": "preprocessor_config.json",\n'
        '      "bytes": 380,\n'
        '      "sha256": "4fb09ebcfd7651205ca8299b993c30088e5535ef350a72d46c5c4580eeac0440"\n'
        '    },\n'
        '    {\n'
        '      "path": "special_tokens_map.json",\n'
        '      "bytes": 472,\n'
        '      "sha256": "c4864a9376a8401918425bed71fc14fc0e81f9b59ec45c1cf96cccb2df508eac"\n'
        '    },\n'
        '    {\n'
        '      "path": "tokenizer_config.json",\n'
        '      "bytes": 974,\n'
        '      "sha256": "4c75d57fd9bd0be8478ad2d6f8b9cebdd4a45338eb108547329c2b6333476ca6"\n'
        '    },\n'
        '    {\n'
        '      "path": "vocab.json",\n'
        '      "bytes": 1059962,\n'
        '      "sha256": "e089ad92ba36837a0d31433e555c8f45fe601ab5c221d4f607ded32d9f7a4349"\n'
        '    }\n'
        '  ],\n'
        '  "totalBytes": 604641231\n'
        '}\n'
    ),
    'workshop.py': (
        '"""Standalone staged workshop worker; heavy dependencies live in an isolated process.\n'
        '\n'
        'Reference modules and manifests are embedded by the notebook generator. No repository\n'
        'or executable source is downloaded. All metric grids retain the decoded input size.\n'
        '"""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import argparse\n'
        'import csv\n'
        'import hashlib\n'
        'import json\n'
        'import math\n'
        'import os\n'
        'import shutil\n'
        'import subprocess\n'
        'import sys\n'
        'import time\n'
        'import urllib.error\n'
        'import uuid\n'
        'import zipfile\n'
        'from pathlib import Path\n'
        '\n'
        'import numpy as np\n'
        'from PIL import Image, ImageDraw\n'
        '\n'
        'ROLES = ("train", "validation", "test", "activity")\n'
        'MINIMUM = dict(zip(ROLES, (8, 2, 2, 1), strict=True))\n'
        'DEFAULT_COUNTS = dict(zip(ROLES, (120, 40, 40, 12), strict=True))\n'
        'PARITY_ATOL = 1e-5\n'
        '\n'
        '\n'
        'def read_json(path: Path) -> dict | list:\n'
        '    return json.loads(path.read_text(encoding="utf-8"))\n'
        '\n'
        '\n'
        'def write_json(path: Path, value: object) -> None:\n'
        '    path.parent.mkdir(parents=True, exist_ok=True)\n'
        '    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n'
        '\n'
        '\n'
        'def digest(path: Path) -> str:\n'
        '    h = hashlib.sha256()\n'
        '    with path.open("rb") as f:\n'
        '        for block in iter(lambda: f.read(1024 * 1024), b""):\n'
        '            h.update(block)\n'
        '    return h.hexdigest()\n'
        '\n'
        '\n'
        'def peak_host_rss() -> int | None:\n'
        '    try:\n'
        '        import resource\n'
        '    except ImportError:\n'
        '        return None  # CPU validation on Windows; canonical Colab records Linux KiB.\n'
        '    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n'
        '\n'
        '\n'
        'def overlap(prediction: np.ndarray, reference: np.ndarray) -> dict:\n'
        '    p, r = np.asarray(prediction), np.asarray(reference)\n'
        '    if p.dtype != bool or r.dtype != bool or p.ndim != 2 or p.shape != r.shape:\n'
        '        raise ValueError("Scoring needs aligned two-dimensional boolean masks")\n'
        '    tp, fp, fn = (int(x) for x in ((p & r).sum(), (p & ~r).sum(), (~p & r).sum()))\n'
        '    union = tp + fp + fn\n'
        '    return {\n'
        '        "tp": tp,\n'
        '        "fp": fp,\n'
        '        "fn": fn,\n'
        '        "pixels": int(p.size),\n'
        '        "iou": tp / union if union else None,\n'
        '        "dice": 2 * tp / (2 * tp + fp + fn) if union else None,\n'
        '        "precision": tp / (tp + fp) if tp + fp else None,\n'
        '        "recall": tp / (tp + fn) if tp + fn else None,\n'
        '        "predicted_area_fraction": float(p.mean()),\n'
        '        "precision_note": None if tp + fp else "Undefined: no predicted foreground pixels",\n'
        '        "overlap_note": None if union else "Undefined: reference and prediction both empty",\n'
        '    }\n'
        '\n'
        '\n'
        'def aggregate(rows: list[dict]) -> dict:\n'
        '    def mean(key: str) -> float | None:\n'
        '        values = [r[key] for r in rows if r[key] is not None]\n'
        '        return float(np.mean(values)) if values else None\n'
        '\n'
        '    tp, fp, fn = (sum(r[k] for r in rows) for k in ("tp", "fp", "fn"))\n'
        '    return {\n'
        '        "n": len(rows),\n'
        '        "mean_target_iou": mean("iou"),\n'
        '        "mean_dice": mean("dice"),\n'
        '        "pooled_iou": tp / (tp + fp + fn) if tp + fp + fn else None,\n'
        '        "pixel_precision": tp / (tp + fp) if tp + fp else None,\n'
        '        "pixel_recall": tp / (tp + fn) if tp + fn else None,\n'
        '        "overlap_defined_n": sum(r["iou"] is not None for r in rows),\n'
        '        "empty_prediction_n": sum(r["tp"] + r["fp"] == 0 for r in rows),\n'
        '        "empty_reference_n": sum(r["tp"] + r["fn"] == 0 for r in rows),\n'
        '        "empty_reference_false_positive_n": sum(r["tp"] + r["fn"] == 0 and r["fp"] > 0 for r in rows),\n'
        '        "failed_calls": 0,\n'
        '        "policy": "A failed call aborts the stage; never silently excluded",\n'
        '    }\n'
        '\n'
        '\n'
        'def verify_parity(before: np.ndarray, after: np.ndarray) -> dict:\n'
        '    a, b = np.asarray(before), np.asarray(after)\n'
        '    if a.shape != b.shape or not np.isfinite(a).all() or not np.isfinite(b).all():\n'
        '        raise ValueError("Parity arrays must be aligned and finite")\n'
        '    difference = float(np.max(np.abs(a - b)))\n'
        '    equal = bool(np.array_equal(a >= 0.5, b >= 0.5))\n'
        '    if difference > PARITY_ATOL or not equal:\n'
        '        raise ValueError("Fresh reload probability or binary mask parity failed")\n'
        '    return {"max_abs_probability_difference": difference, "identical_masks": equal}\n'
        '\n'
        '\n'
        'def prompts(mask: np.ndarray) -> dict:\n'
        '    """Four-connected largest component; ties and distance maxima use row-major order."""\n'
        '    from scipy.ndimage import distance_transform_edt, label\n'
        '\n'
        '    a = np.asarray(mask, dtype=bool)\n'
        '    if a.ndim != 2 or not a.any():\n'
        '        raise ValueError("A geometric prompt requires a positive two-dimensional mask")\n'
        '    components, n = label(a)\n'
        '    sizes = np.bincount(components.ravel())[1:]\n'
        '    component = components == (int(np.argmax(sizes)) + 1)\n'
        '    # Padding makes the image edge an actual boundary, including full-image masks.\n'
        '    distances = distance_transform_edt(np.pad(component, 1))[1:-1, 1:-1]\n'
        '    y, x = np.unravel_index(int(np.argmax(distances)), a.shape)\n'
        '    ys, xs = np.nonzero(a)\n'
        '    box = [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]\n'
        '    return {\n'
        '        "point": [int(x), int(y)],\n'
        '        "box": box,\n'
        '        "components": int(n),\n'
        '        "coordinate_contract": "xyxy pixel edges, upper bounds exclusive; wrapper receives unchanged",\n'
        '        "assistance": "Reference-derived simulated informed click / oracle localization",\n'
        '    }\n'
        '\n'
        '\n'
        'def expand_box(box: list, size: tuple[int, int], fraction: float) -> list[float]:\n'
        '    x0, y0, x1, y1 = box\n'
        '    w, h = size\n'
        '    dx, dy = fraction * (x1 - x0), fraction * (y1 - y0)\n'
        '    return [max(0.0, x0 - dx), max(0.0, y0 - dy), min(float(w), x1 + dx), min(float(h), y1 + dy)]\n'
        '\n'
        '\n'
        'SIDES = ("left", "top", "right", "bottom")\n'
        '\n'
        '\n'
        'def box_change(base: list, actual: list, size: tuple[int, int], fraction: float) -> dict:\n'
        '    """Requested versus realised expansion of one box; a side is clipped when the image edge stopped it."""\n'
        '    x0, y0, x1, y1 = base\n'
        '    w, h = size\n'
        '    dx, dy = fraction * (x1 - x0), fraction * (y1 - y0)\n'
        '    requested = [dx, dy, dx, dy]\n'
        '    realised = [x0 - actual[0], y0 - actual[1], actual[2] - x1, actual[3] - y1]\n'
        '    return {\n'
        '        "base_box": base,\n'
        '        "requested_fraction": fraction,\n'
        '        "requested_expansion_pixels": requested,\n'
        '        # actual_expansion_pixels is the field name used before revision 0.2.0; both hold the same values.\n'
        '        "actual_expansion_pixels": realised,\n'
        '        "realised_expansion_pixels": realised,\n'
        '        "clipped_sides": [s for s, want, got in zip(SIDES, requested, realised, strict=True) if got < want - 1e-9],\n'
        '        "whole_image_box": [float(v) for v in actual] == [0.0, 0.0, float(w), float(h)],\n'
        '        "box_area_fraction": (actual[2] - actual[0]) * (actual[3] - actual[1]) / (w * h),\n'
        '    }\n'
        '\n'
        '\n'
        'def change_counts(paired: list[dict]) -> dict:\n'
        '    return {\n'
        '        "n": len(paired),\n'
        '        "improved": sum(r["delta_iou"] > 0 for r in paired),\n'
        '        "unchanged": sum(r["delta_iou"] == 0 for r in paired),\n'
        '        "worsened": sum(r["delta_iou"] < 0 for r in paired),\n'
        '    }\n'
        '\n'
        '\n'
        'def validate_roles(records: list[dict], *, default: bool = False) -> dict:\n'
        '    from clipseg_reference import samples\n'
        '\n'
        '    ids, identities, groups = set(), set(), {}\n'
        '    splits = {role: [] for role in ROLES}\n'
        '    for index, entry in enumerate(records):\n'
        '        if not isinstance(entry, dict):\n'
        '            raise ValueError(f"records[{index}] must be an object with id, image, mask, prompt and role")\n'
        '        where = f"records[{index}] (id {entry.get(\'id\')!r})"\n'
        '        role = entry.get("role")\n'
        '        if role not in splits:\n'
        '            raise ValueError(f"{where}: unknown role {role!r}; expected one of {ROLES}")\n'
        '        try:\n'
        '            checked = samples.validate_dataset([entry], min_records=1)["records"][0]\n'
        '        except ValueError as exc:\n'
        '            # The record is validated on its own, so the library numbers it records[0]; restore its position.\n'
        '            raise ValueError(str(exc).replace("records[0]", where, 1)) from exc\n'
        '        if checked["image"].getexif().get(274, 1) != 1:\n'
        '            raise ValueError(f"{where}: normalize EXIF orientation together with the mask before using this image")\n'
        '        identity = samples.image_digest(checked["image"])\n'
        '        if checked["id"] in ids:\n'
        '            raise ValueError(f"{where}: duplicate id; every record needs a unique id")\n'
        '        if identity in identities:\n'
        '            raise ValueError(f"{where}: the decoded image duplicates an earlier record; roles must contain distinct images")\n'
        '        ids.add(checked["id"])\n'
        '        identities.add(identity)\n'
        '        group = entry.get("group_id", entry.get("group", entry.get("source_id", checked["id"])))\n'
        '        if group in groups and groups[group] != role:\n'
        '            raise ValueError(f"{where}: related group {group!r} is in both {groups[group]!r} and {role!r}")\n'
        '        groups[group] = role\n'
        '        checked.update(role=role, group=group)\n'
        '        for key, actual in (\n'
        '            ("image_sha256", identity),\n'
        '            ("mask_sha256", samples.mask_digest(checked["mask"])),\n'
        '        ):\n'
        '            if key in entry and entry[key] != actual:\n'
        '                raise ValueError(f"{checked[\'id\']}: {key} integrity mismatch")\n'
        '            checked[key] = actual\n'
        '        splits[role].append(checked)\n'
        '    for role, values in splits.items():\n'
        '        expected = DEFAULT_COUNTS[role] if default else MINIMUM[role]\n'
        '        if len(values) < expected or (default and len(values) != expected):\n'
        '            raise ValueError(f"{role}: expected {\'exactly\' if default else \'at least\'} {expected} records")\n'
        '    samples.check_split_disjoint(splits)\n'
        '    return splits\n'
        '\n'
        '\n'
        'def load_records(root: Path) -> dict:\n'
        '    entries = read_json(root / "data/records.json")["records"]\n'
        '    for item in entries:\n'
        '        item["image"] = root / item["image"]\n'
        '        item["mask"] = root / item["mask"]\n'
        '    return validate_roles(entries)\n'
        '\n'
        '\n'
        'def persist_records(root: Path, records: list[dict], *, default: bool) -> None:\n'
        '    splits = validate_roles(records, default=default)\n'
        '    exported, prompt_manifest = [], []\n'
        '    for role in ROLES:\n'
        '        for record in splits[role]:\n'
        '            rid = record["id"]\n'
        '            image_path, mask_path = f"data/images/{rid}.png", f"data/masks/{rid}.png"\n'
        '            (root / image_path).parent.mkdir(parents=True, exist_ok=True)\n'
        '            (root / mask_path).parent.mkdir(parents=True, exist_ok=True)\n'
        '            record["image"].save(root / image_path)\n'
        '            Image.fromarray(record["mask"].astype(np.uint8) * 255).save(root / mask_path)\n'
        '            exported.append(\n'
        '                {\n'
        '                    **{k: v for k, v in record.items() if k not in ("image", "mask")},\n'
        '                    "image": image_path,\n'
        '                    "mask": mask_path,\n'
        '                }\n'
        '            )\n'
        '            prompt_manifest.append(\n'
        '                {"id": rid, "role": role, "text": record["prompt"], **prompts(record["mask"])}\n'
        '            )\n'
        '    write_json(root / "data/records.json", {"records": exported, "private_byod": not default})\n'
        '    write_json(root / "outputs/sample_manifest.json", {"records": exported, "private_byod": not default})\n'
        '    write_json(root / "outputs/prompt_manifest.json", {"records": prompt_manifest})\n'
        '    baseline_rows = []\n'
        '    for role in ("validation", "test", "activity"):\n'
        '        for record in splits[role]:\n'
        '            box = prompts(record["mask"])["box"]\n'
        '            filled = np.zeros_like(record["mask"])\n'
        '            x0, y0, x1, y1 = box\n'
        '            filled[y0:y1, x0:x1] = True\n'
        '            for name, mask in (("empty", np.zeros_like(filled)), ("oracle_filled_box", filled)):\n'
        '                baseline_rows.append(\n'
        '                    {\n'
        '                        "id": record["id"],\n'
        '                        "role": role,\n'
        '                        "model": name,\n'
        '                        "protocol": "unassisted" if name == "empty" else "oracle_box",\n'
        '                        **overlap(mask, record["mask"]),\n'
        '                    }\n'
        '                )\n'
        '    write_json(root / "outputs/baselines.json", baseline_rows)\n'
        '    write_json(\n'
        '        root / "outputs/prepare.json",\n'
        '        {\n'
        '            "counts": {r: len(v) for r, v in splits.items()},\n'
        '            "data_manifest_sha256": digest(root / "data/records.json"),\n'
        '            "invalid_records": 0,\n'
        '            "duplicates": 0,\n'
        '            "policy": "Invalid or duplicate input aborts; no silent exclusions",\n'
        '        },\n'
        '    )\n'
        '    preview(root, splits["activity"][0], "reference_first")\n'
        '    disconnected = next((r for r in splits["activity"] if prompts(r["mask"])["components"] > 1), None)\n'
        '    if disconnected is not None:\n'
        '        preview(root, disconnected, "reference_disconnected")\n'
        '    write_json(\n'
        '        root / "outputs/frozen_experiment.json",\n'
        '        {\n'
        '            "schema": "segmentation-workshop-v1",\n'
        '            "data_sha256": digest(root / "data/records.json"),\n'
        '            "prompts_sha256": digest(root / "outputs/prompt_manifest.json"),\n'
        '            "seed": 42,\n'
        '            "epochs": 4,\n'
        '            "batch_size": 4,\n'
        '            "learning_rate": 3e-4,\n'
        '            "clipseg_threshold": 0.5,\n'
        '            "selection": "highest validation mean target IoU, earliest tie including epoch zero",\n'
        '            "parity_atol": PARITY_ATOL,\n'
        '            "parity_rtol": 0,\n'
        '            "identical_masks_required": True,\n'
        '            "sam_candidate_selection": "argmax predicted quality; earliest tie",\n'
        '            "box_expansions": [0, 0.1, 0.25],\n'
        '        },\n'
        '    )\n'
        '\n'
        '\n'
        'def preview(root: Path, record: dict, name: str, predictions: list[str] | None = None) -> None:\n'
        '    panels = [("Original", None), ("Reference: " + record["prompt"], record["mask"])]\n'
        '    for model in predictions or []:\n'
        '        path = root / "outputs/predictions" / model / f"{record[\'id\']}.png"\n'
        '        panels.append((model, np.asarray(Image.open(path)) > 0))\n'
        '    canvas = Image.new("RGB", (280 * len(panels), 310), "white")\n'
        '    draw = ImageDraw.Draw(canvas)\n'
        '    for index, (label, mask) in enumerate(panels):\n'
        '        rgb = np.asarray(record["image"]).copy()\n'
        '        if mask is not None:\n'
        '            rgb[mask] = (rgb[mask] * 0.5 + np.array([0, 220, 120]) * 0.5).astype(np.uint8)\n'
        '        image = Image.fromarray(rgb)\n'
        '        image.thumbnail((280, 280))\n'
        '        canvas.paste(image, (280 * index, 30))\n'
        '        draw.text((280 * index + 3, 4), label, fill="black")\n'
        '    canvas.save(root / "outputs" / f"{name}.png")\n'
        '\n'
        '\n'
        'def prepare(root: Path, byod: Path | None = None) -> None:\n'
        '    from clipseg_reference import samples\n'
        '\n'
        '    if byod is not None:\n'
        '        manifest_path = byod / "manifest.json"\n'
        '        if not manifest_path.is_file():\n'
        '            raise ValueError(f"BYOD_PATH must be a directory containing manifest.json; not found: {manifest_path}")\n'
        '        if manifest_path.stat().st_size > 1024**2:\n'
        '            raise ValueError("BYOD manifest exceeds 1 MiB")\n'
        '        data = read_json(manifest_path)\n'
        '        records = data.get("records") if isinstance(data, dict) else None\n'
        '        if not isinstance(records, list) or not 13 <= len(records) <= 256:\n'
        '            raise ValueError(\'manifest.json must be {"records": [...]} with 13–256 records across the four roles\')\n'
        '        total_pixels = 0\n'
        '        for index, r in enumerate(records):\n'
        '            if not isinstance(r, dict):\n'
        '                raise ValueError(f"records[{index}] must be an object with id, image, mask, prompt and role")\n'
        '            where = f"records[{index}] (id {r.get(\'id\')!r})"\n'
        '            missing = [k for k in ("id", "image", "mask", "prompt", "role") if k not in r]\n'
        '            if missing:\n'
        '                raise ValueError(f"{where}: missing required field(s) {missing}")\n'
        '            for key in ("image", "mask"):\n'
        '                text = r[key]\n'
        '                if not isinstance(text, str):\n'
        '                    raise ValueError(f"{where}: {key} must be a relative path string")\n'
        '                relative = Path(text)\n'
        '                if relative.is_absolute() or ".." in relative.parts or "\\\\" in text or ":" in text:\n'
        '                    raise ValueError(f"{where}: {key} {text!r} must be a simple relative path")\n'
        '                candidate = (byod / text).resolve()\n'
        '                if not candidate.is_relative_to(byod.resolve()):\n'
        '                    raise ValueError(f"{where}: {key} {text!r} must remain inside the selected data directory")\n'
        '                if not candidate.is_file():\n'
        '                    raise ValueError(f"{where}: {key} file not found: {text}")\n'
        '                r[key] = candidate\n'
        '                if candidate.stat().st_size > 20 * 1024**2:\n'
        '                    raise ValueError(f"{where}: {key} {text} exceeds 20 MiB")\n'
        '                try:\n'
        '                    header = Image.open(candidate)\n'
        '                except OSError as exc:\n'
        '                    raise ValueError(f"{where}: {key} {text} is not a readable image") from exc\n'
        '                with header:\n'
        '                    if not all(16 <= side <= 4096 for side in header.size):\n'
        '                        raise ValueError(f"{where}: {key} {text} is {header.size[0]}x{header.size[1]}; sides must be 16–4096 pixels")\n'
        '                    if header.getexif().get(274, 1) != 1:\n'
        '                        raise ValueError(f"{where}: normalize the EXIF orientation of {key} {text} before use")\n'
        '                    if key == "image":\n'
        '                        total_pixels += header.width * header.height\n'
        '                        if total_pixels > 64 * 1024**2:\n'
        '                            raise ValueError(f"{where}: BYOD exceeds 64 million decoded image pixels in total")\n'
        '            with Image.open(r["mask"]) as mask:\n'
        '                array = np.asarray(mask)\n'
        '                if array.ndim != 2 or not set(np.unique(array)).issubset({0, 1, 255}):\n'
        '                    raise ValueError(f"{where}: mask must be a single-channel binary 0/1 or 0/255 image")\n'
        '        persist_records(root, records, default=False)\n'
        '        return\n'
        '    manifest = read_json(root / "sample.json")\n'
        '    for attempt in range(3):\n'
        '        print(f"Verifying and loading pinned FoodSeg103 row groups (attempt {attempt + 1}/3)", flush=True)\n'
        '        try:\n'
        '            corpus = samples.read_corpus(samples.fetch_corpus(cache_dir=root / "cache"))\n'
        '            break\n'
        '        except (urllib.error.URLError, TimeoutError, ConnectionError):\n'
        '            if attempt == 2:\n'
        '                raise\n'
        '            time.sleep(attempt + 1)\n'
        '    by_id = {r["id"]: r for r in corpus}\n'
        '    chosen = []\n'
        '    for pin in manifest["records"]:\n'
        '        original = by_id[pin["id"]]\n'
        '        for key in ("prompt", "class_id", "source_id", "source_row_group"):\n'
        '            if original[key] != pin[key]:\n'
        '                raise ValueError(f"Selection identity mismatch: {pin[\'id\']} {key}")\n'
        '        chosen.append({**original, **pin})\n'
        '    persist_records(root, chosen, default=True)\n'
        '\n'
        '\n'
        'def device() -> str:\n'
        '    import torch\n'
        '\n'
        '    if not torch.cuda.is_available():\n'
        '        raise RuntimeError(\n'
        '            "Select a Colab GPU runtime, then start the notebook again. CPU qualification is unsupported."\n'
        '        )\n'
        '    torch.cuda.reset_peak_memory_stats()\n'
        '    return "cuda"\n'
        '\n'
        '\n'
        'def timed(call):\n'
        '    import torch\n'
        '\n'
        '    torch.cuda.synchronize()\n'
        '    start = time.perf_counter()\n'
        '    value = call()\n'
        '    torch.cuda.synchronize()\n'
        '    return value, time.perf_counter() - start\n'
        '\n'
        '\n'
        'def save_prediction(root: Path, name: str, record: dict, mask: np.ndarray, probability=None) -> None:\n'
        '    target = root / "outputs/predictions" / name\n'
        '    target.mkdir(parents=True, exist_ok=True)\n'
        '    Image.fromarray(mask.astype(np.uint8) * 255).save(target / f"{record[\'id\']}.png")\n'
        '    if probability is not None:\n'
        '        if not np.isfinite(probability).all():\n'
        '            raise ValueError("Non-finite CLIPSeg probabilities")\n'
        '        np.save(target / f"{record[\'id\']}.npy", probability, allow_pickle=False)\n'
        '\n'
        '\n'
        'def clipseg_rows(root: Path, pipe, records: list[dict], name: str) -> list[dict]:\n'
        '    rows = []\n'
        '    for r in records:\n'
        '        result, elapsed = timed(lambda r=r: pipe.segment(r["image"], [r["prompt"]], threshold=0.5))\n'
        '        segment = result["segments"][0]\n'
        '        save_prediction(root, name, r, segment["mask"], segment["probability"])\n'
        '        rows.append(\n'
        '            {\n'
        '                "id": r["id"],\n'
        '                "role": r["role"],\n'
        '                "model": name,\n'
        '                "protocol": "text",\n'
        '                "seconds": elapsed,\n'
        '                **overlap(segment["mask"], r["mask"]),\n'
        '            }\n'
        '        )\n'
        '    return rows\n'
        '\n'
        '\n'
        'def clipseg(root: Path) -> None:\n'
        '    import torch\n'
        '    from clipseg_reference.pipeline import ClipSegSegmentationPipeline\n'
        '\n'
        '    splits = load_records(root)\n'
        '    pipe = ClipSegSegmentationPipeline.from_pretrained(\n'
        '        device=device(), weights_dir=root / "weights/clipseg", allow_download=True\n'
        '    )\n'
        '    pipe.segment(splits["validation"][0]["image"], [splits["validation"][0]["prompt"]])  # untimed warm-up\n'
        '    evaluate = splits["validation"] + splits["test"] + splits["activity"]\n'
        '    original = {\n'
        '        n: p.detach().cpu().clone() for n, p in pipe._model.named_parameters() if n.startswith("decoder.")\n'
        '    }\n'
        '    updates = []\n'
        '\n'
        '    def progress(entry):\n'
        '        maximum = max(\n'
        '            float((p.detach().cpu() - original[n]).abs().max())\n'
        '            for n, p in pipe._model.named_parameters()\n'
        '            if n in original\n'
        '        )\n'
        '        updates.append({"epoch": entry["epoch"], "max_abs_decoder_change": maximum})\n'
        '        print(json.dumps(entry), flush=True)\n'
        '\n'
        '    step_count = 0\n'
        '    original_optimizer = torch.optim.AdamW\n'
        '\n'
        '    def observe_step(*_args, **_kwargs):\n'
        '        nonlocal step_count\n'
        '        step_count += 1\n'
        '\n'
        '    def observed_optimizer(*args, **kwargs):\n'
        '        optimizer = original_optimizer(*args, **kwargs)\n'
        '        permitted = {id(p) for n, p in pipe._model.named_parameters() if n in original}\n'
        '        if any(id(p) not in permitted for group in optimizer.param_groups for p in group["params"]):\n'
        '            raise RuntimeError("Adaptation optimizer includes a frozen tower parameter")\n'
        '        optimizer.register_step_post_hook(observe_step)\n'
        '        return optimizer\n'
        '\n'
        '    try:\n'
        '        torch.optim.AdamW = observed_optimizer\n'
        '        adaptation = pipe.adapt(\n'
        '            splits["train"],\n'
        '            splits["validation"],\n'
        '            epochs=4,\n'
        '            batch_size=4,\n'
        '            lr=3e-4,\n'
        '            seed=42,\n'
        '            threshold=0.5,\n'
        '            progress=progress,\n'
        '        )\n'
        '    finally:\n'
        '        torch.optim.AdamW = original_optimizer\n'
        '    if len(updates) != 5 or not any(x["max_abs_decoder_change"] > 0 for x in updates[1:]):\n'
        '        raise RuntimeError("Four adaptation epochs and actual decoder updates were not observed")\n'
        '    adaptation["observed_epoch_updates"] = updates\n'
        '    adaptation["optimizer_steps"] = step_count\n'
        '    if step_count != 4 * math.ceil(len(splits["train"]) / 4):\n'
        '        raise RuntimeError("Observed optimizer step count differs from the bounded recipe")\n'
        '    adaptation["selected_frozen_epoch_zero"] = adaptation["best_epoch"] == 0\n'
        '    pipe.save_artifact(\n'
        '        root / "outputs/adapter",\n'
        '        metadata={"data_manifest_sha256": digest(root / "data/records.json"), "observed_updates": updates},\n'
        '    )\n'
        '    # Selection is now persisted. Evaluate the same final test images with the exact\n'
        '    # frozen decoder, then restore the selected decoder; no second resident model.\n'
        '    write_json(\n'
        '        root / "outputs/selection.json",\n'
        '        {\n'
        '            "best_epoch": adaptation["best_epoch"],\n'
        '            "threshold": 0.5,\n'
        '            "adapter_manifest_sha256": digest(root / "outputs/adapter/manifest.json"),\n'
        '            "test_used_for_selection": False,\n'
        '        },\n'
        '    )\n'
        '    selected_state = {n: p.detach().cpu().clone() for n, p in pipe._model.named_parameters() if n in original}\n'
        '    with torch.no_grad():\n'
        '        for n, p in pipe._model.named_parameters():\n'
        '            if n in original:\n'
        '                p.copy_(original[n].to(p.device))\n'
        '    try:\n'
        '        baseline = clipseg_rows(root, pipe, evaluate, "clipseg_frozen")\n'
        '    finally:\n'
        '        with torch.no_grad():\n'
        '            for n, p in pipe._model.named_parameters():\n'
        '                if n in selected_state:\n'
        '                    p.copy_(selected_state[n].to(p.device))\n'
        '    write_json(root / "outputs/clipseg_frozen.json", baseline)\n'
        '    selected = clipseg_rows(root, pipe, evaluate, "clipseg_selected_live")\n'
        '    write_json(root / "outputs/clipseg_selected_live.json", selected)\n'
        '    write_json(root / "outputs/training_history.json", adaptation)\n'
        '    preview(root, splits["test"][0], "clipseg_first", ["clipseg_frozen", "clipseg_selected_live"])\n'
        '    write_json(\n'
        '        root / "outputs/clipseg.json",\n'
        '        {\n'
        '            "pid": os.getpid(),\n'
        '            "peak_gpu_bytes": torch.cuda.max_memory_allocated(),\n'
        '            "gpu": torch.cuda.get_device_name(),\n'
        '            "precision": "float32 inference and decoder; fp16 cached encoder activations",\n'
        '            "warmup": "One validation image, excluded from timed calls",\n'
        '            "selected_epoch": adaptation["best_epoch"],\n'
        '        },\n'
        '    )\n'
        '\n'
        '\n'
        'def reload_stage(root: Path) -> None:\n'
        '    from clipseg_reference.pipeline import ClipSegSegmentationPipeline\n'
        '\n'
        '    if read_json(root / "outputs/clipseg.json")["pid"] == os.getpid():\n'
        '        raise RuntimeError("Reload qualification must execute in a fresh process")\n'
        '    splits = load_records(root)\n'
        '    pipe = ClipSegSegmentationPipeline.from_artifact(\n'
        '        root / "outputs/adapter", device=device(), weights_dir=root / "weights/clipseg"\n'
        '    )\n'
        '    pipe.segment(splits["validation"][0]["image"], [splits["validation"][0]["prompt"]])\n'
        '    records = splits["validation"] + splits["test"] + splits["activity"]\n'
        '    rows = clipseg_rows(root, pipe, records, "clipseg_reloaded")\n'
        '    parity = []\n'
        '    for r in records:\n'
        '        before = root / "outputs/predictions/clipseg_selected_live" / r["id"]\n'
        '        after = root / "outputs/predictions/clipseg_reloaded" / r["id"]\n'
        '        a, b = (\n'
        '            np.load(Path(str(before) + ".npy"), allow_pickle=False),\n'
        '            np.load(Path(str(after) + ".npy"), allow_pickle=False),\n'
        '        )\n'
        '        try:\n'
        '            checked = verify_parity(a, b)\n'
        '        except ValueError as exc:\n'
        '            write_json(\n'
        '                root / "outputs/reload_parity.json", {"passed": False, "atol": PARITY_ATOL, "records": parity}\n'
        '            )\n'
        '            raise RuntimeError(\n'
        '                f"Fresh reload parity failed for {r[\'id\']}; diagnose rather than widening tolerance"\n'
        '            ) from exc\n'
        '        parity.append({"id": r["id"], **checked})\n'
        '    live = read_json(root / "outputs/clipseg_selected_live.json")\n'
        '    for a, b in zip(live, rows, strict=True):\n'
        '        if (a["id"], a["iou"], a["dice"]) != (b["id"], b["iou"], b["dice"]):\n'
        '            raise RuntimeError("Fresh reload metric parity failed")\n'
        '    write_json(root / "outputs/clipseg_reloaded.json", rows)\n'
        '    write_json(\n'
        '        root / "outputs/reload_parity.json",\n'
        '        {\n'
        '            "passed": True,\n'
        '            "atol": PARITY_ATOL,\n'
        '            "rtol": 0,\n'
        '            "pid": os.getpid(),\n'
        '            "metric_parity": True,\n'
        '            "records": parity,\n'
        '        },\n'
        '    )\n'
        '\n'
        '\n'
        'def sam_stage(root: Path, name: str) -> None:\n'
        '    import torch\n'
        '\n'
        '    if name == "sam":\n'
        '        from sam_reference import SAMViTSegmentationPipeline as Pipeline\n'
        '    else:\n'
        '        from sam2_reference import SAM2SegmentationPipeline as Pipeline\n'
        '    splits = load_records(root)\n'
        '    prompt_by_id = {r["id"]: r for r in read_json(root / "outputs/prompt_manifest.json")["records"]}\n'
        '    pipe = Pipeline.from_pretrained(device=device(), weights_dir=root / "weights" / name, allow_download=True)\n'
        '    first = splits["validation"][0]\n'
        '    pipe.segment(first["image"], box=prompt_by_id[first["id"]]["box"], multimask=True)\n'
        '    rows, experiments = [], []\n'
        '    for r in splits["validation"] + splits["test"] + splits["activity"]:\n'
        '        prompt = prompt_by_id[r["id"]]\n'
        '        variants = [\n'
        '            ("point", {"points": [prompt["point"]], "point_labels": [1]}),\n'
        '            ("box", {"box": prompt["box"]}),\n'
        '        ]\n'
        '        if name == "sam2" and r["role"] == "activity":\n'
        '            variants += [\n'
        '                (f"box_expand_{fraction}", {"box": expand_box(prompt["box"], r["image"].size, fraction)})\n'
        '                for fraction in (0.0, 0.1, 0.25)\n'
        '            ]\n'
        '        for protocol, kwargs in variants:\n'
        '            result, elapsed = timed(\n'
        '                lambda r=r, kwargs=kwargs: pipe.segment(r["image"], **kwargs, multimask=True)\n'
        '            )\n'
        '            qualities = np.asarray(result["iou_scores"])\n'
        '            if not np.isfinite(qualities).all():\n'
        '                raise ValueError("Non-finite predicted mask quality")\n'
        '            selected = int(np.argmax(qualities))\n'
        '            mask = result["masks"][selected]\n'
        '            save_prediction(root, f"{name}_{protocol}", r, mask)\n'
        '            row = {\n'
        '                "id": r["id"],\n'
        '                "role": r["role"],\n'
        '                "model": name,\n'
        '                "protocol": protocol,\n'
        '                "candidate_index": selected,\n'
        '                "predicted_quality": float(qualities[selected]),\n'
        '                "seconds": elapsed,\n'
        '                "prompt": kwargs,\n'
        '                **overlap(mask, r["mask"]),\n'
        '            }\n'
        '            rows.append(row)\n'
        '            if protocol.startswith("box_expand"):\n'
        '                fraction = float(protocol.rsplit("_", 1)[1])\n'
        '                experiments.append(\n'
        '                    {\n'
        '                        **row,\n'
        '                        "phrase": r["prompt"],\n'
        '                        **box_change(prompt["box"], kwargs["box"], r["image"].size, fraction),\n'
        '                    }\n'
        '                )\n'
        '    write_json(root / f"outputs/{name}_metrics.json", rows)\n'
        '    write_json(\n'
        '        root / f"outputs/{name}.json",\n'
        '        {\n'
        '            "peak_gpu_bytes": torch.cuda.max_memory_allocated(),\n'
        '            "gpu": torch.cuda.get_device_name(),\n'
        '            "warmup": "One validation box, excluded from timing",\n'
        '            "precision": "float32",\n'
        '            "prompt_manifest_sha256": digest(root / "outputs/prompt_manifest.json"),\n'
        '        },\n'
        '    )\n'
        '    if name == "sam2":\n'
        '        write_json(root / "outputs/controlled_experiment.json", experiments)\n'
        '        deltas = []\n'
        '        baseline = {r["id"]: r for r in experiments if r["protocol"] == "box_expand_0.0"}\n'
        '        for fraction in (0.1, 0.25):\n'
        '            paired = [\n'
        '                {\n'
        '                    "id": r["id"],\n'
        '                    "phrase": r["phrase"],\n'
        '                    "iou_tight_box": baseline[r["id"]]["iou"],\n'
        '                    "iou_expanded_box": r["iou"],\n'
        '                    "delta_iou": r["iou"] - baseline[r["id"]]["iou"],\n'
        '                    "delta_dice": r["dice"] - baseline[r["id"]]["dice"],\n'
        '                    "clipped_sides": r["clipped_sides"],\n'
        '                    "whole_image_box": r["whole_image_box"],\n'
        '                    "box_area_fraction": r["box_area_fraction"],\n'
        '                    "realised_expansion_pixels": r["realised_expansion_pixels"],\n'
        '                }\n'
        '                for r in experiments\n'
        '                if r["protocol"] == f"box_expand_{fraction}"\n'
        '            ]\n'
        '            counts = change_counts(paired)\n'
        '            deltas.append(\n'
        '                {\n'
        '                    "expansion": fraction,\n'
        '                    "records": paired,\n'
        '                    "improved": counts["improved"],\n'
        '                    "unchanged": counts["unchanged"],\n'
        '                    "worsened": counts["worsened"],\n'
        '                    # A box stopped by an image edge grew on fewer sides than requested; keep it visible.\n'
        '                    "clipped_records": sum(bool(r["clipped_sides"]) for r in paired),\n'
        '                    "whole_image_records": sum(r["whole_image_box"] for r in paired),\n'
        '                    "by_clipping": {\n'
        '                        "clipped": change_counts([r for r in paired if r["clipped_sides"]]),\n'
        '                        "not_clipped": change_counts([r for r in paired if not r["clipped_sides"]]),\n'
        '                    },\n'
        '                }\n'
        '            )\n'
        '        write_json(root / "outputs/box_experiment.json", deltas)\n'
        '        preview(\n'
        '            root,\n'
        '            splits["activity"][0],\n'
        '            "activity_expansion",\n'
        '            ["sam2_box_expand_0.0", "sam2_box_expand_0.1", "sam2_box_expand_0.25"],\n'
        '        )\n'
        '    preview(root, splits["test"][0], f"{name}_first", [f"{name}_point", f"{name}_box"])\n'
        '\n'
        '\n'
        'def report(root: Path) -> None:\n'
        '    rows = []\n'
        '    for filename in (\n'
        '        "baselines.json",\n'
        '        "clipseg_frozen.json",\n'
        '        "clipseg_reloaded.json",\n'
        '        "sam_metrics.json",\n'
        '        "sam2_metrics.json",\n'
        '    ):\n'
        '        rows.extend(read_json(root / "outputs" / filename))\n'
        '    groups = sorted({(r["model"], r["protocol"], r["role"]) for r in rows})\n'
        '    summaries = [\n'
        '        {\n'
        '            "model": model,\n'
        '            "protocol": protocol,\n'
        '            "role": role,\n'
        '            **aggregate(\n'
        '                [r for r in rows if (r["model"], r["protocol"], r["role"]) == (model, protocol, role)]\n'
        '            ),\n'
        '        }\n'
        '        for model, protocol, role in groups\n'
        '    ]\n'
        '    write_json(root / "outputs/metrics.json", {"rows": rows, "summary": summaries})\n'
        '    frozen = {r["id"]: r for r in rows if r["model"] == "clipseg_frozen" and r["role"] == "test"}\n'
        '    changes = [\n'
        '        {"id": r["id"], "iou_change": r["iou"] - frozen[r["id"]]["iou"]}\n'
        '        for r in rows\n'
        '        if r["model"] == "clipseg_reloaded" and r["role"] == "test"\n'
        '    ]\n'
        '    paired_counts = {\n'
        '        "improved": sum(r["iou_change"] > 0 for r in changes),\n'
        '        "unchanged": sum(r["iou_change"] == 0 for r in changes),\n'
        '        "worsened": sum(r["iou_change"] < 0 for r in changes),\n'
        '    }\n'
        '    write_json(root / "outputs/paired_changes.json", {"records": changes, **paired_counts})\n'
        '    history = read_json(root / "outputs/training_history.json")\n'
        '    reload = read_json(root / "outputs/reload_parity.json")\n'
        '    markdown = [\n'
        '        "# Segmentation results",\n'
        '        "",\n'
        '        "Small-sample evidence; compare within equal prompt protocols.",\n'
        '        "",\n'
        '    ]\n'
        '    for title, models in (\n'
        '        ("CLIPSeg: identical text prompts", {"clipseg_frozen", "clipseg_reloaded", "empty"}),\n'
        '        ("SAM models: reference-derived geometric prompts", {"sam", "sam2", "oracle_filled_box"}),\n'
        '    ):\n'
        '        markdown += [\n'
        '            "## " + title,\n'
        '            "",\n'
        '            "| Model | Protocol | n | Mean target IoU | Dice |",\n'
        '            "|---|---|---:|---:|---:|",\n'
        '        ]\n'
        '        for row in summaries:\n'
        '            if row["role"] == "test" and row["model"] in models:\n'
        '                markdown.append(\n'
        '                    f"| {row[\'model\']} | {row[\'protocol\']} | {row[\'n\']} | "\n'
        '                    f"{row[\'mean_target_iou\']:.4f} | {row[\'mean_dice\']:.4f} |"\n'
        '                )\n'
        '        markdown += [""]\n'
        '    worst = max((r["max_abs_probability_difference"] for r in reload["records"]), default=None)\n'
        '    markdown += [\n'
        '        "Epoch selected on validation: "\n'
        '        + str(history["best_epoch"])\n'
        '        + (" (the frozen decoder: no adapted epoch beat it)" if history["best_epoch"] == 0 else ""),\n'
        '        "",\n'
        '        f"Fresh reload: {\'passed\' if reload[\'passed\'] else \'FAILED\'} on {len(reload[\'records\'])} records in a new "\n'
        '        f"process (maximum probability difference {worst}, tolerance {reload[\'atol\']}; identical masks: "\n'
        '        f"{all(r[\'identical_masks\'] for r in reload[\'records\'])}; metric parity: {reload.get(\'metric_parity\')}).",\n'
        '        "",\n'
        '        f"CLIPSeg test images, frozen to adapted (reloaded) IoU: {paired_counts[\'improved\']} improved, "\n'
        '        f"{paired_counts[\'unchanged\']} unchanged, {paired_counts[\'worsened\']} worsened (of {len(changes)}).",\n'
        '        "",\n'
        '    ]\n'
        '    box_path = root / "outputs/box_experiment.json"\n'
        '    if box_path.is_file():\n'
        '        markdown += [\n'
        '            "## SAM 2 box-precision activity (activity images)",\n'
        '            "",\n'
        '            "| Expansion per side | Images | Improved | Unchanged | Worsened | Clipped by an image edge | Whole-image box |",\n'
        '            "|---:|---:|---:|---:|---:|---:|---:|",\n'
        '        ]\n'
        '        for entry in read_json(box_path):\n'
        '            markdown.append(\n'
        '                f"| {entry[\'expansion\']:.0%} | {len(entry[\'records\'])} | {entry[\'improved\']} | {entry[\'unchanged\']} | "\n'
        '                f"{entry[\'worsened\']} | {entry.get(\'clipped_records\', \'n/a\')} | {entry.get(\'whole_image_records\', \'n/a\')} |"\n'
        '            )\n'
        '        markdown += [""]\n'
        '    markdown += [\n'
        '        "A reference-derived box is localization assistance, not autonomous recognition.",\n'
        '        "Personal learning notes are optional; no submission is required.",\n'
        '    ]\n'
        '    (root / "outputs/summary.md").write_text("\\n".join(markdown) + "\\n", encoding="utf-8")\n'
        '    keys = sorted(set().union(*(r.keys() for r in rows)))\n'
        '    with (root / "outputs/per_image_metrics.csv").open("w", newline="", encoding="utf-8") as f:\n'
        '        writer = csv.DictWriter(f, fieldnames=keys)\n'
        '        writer.writeheader()\n'
        '        writer.writerows(\n'
        '            {k: json.dumps(v) if isinstance(v, (dict, list)) else v for k, v in r.items()} for r in rows\n'
        '        )\n'
        '    with (root / "outputs/per_image_metrics.csv").open(encoding="utf-8", newline="") as f:\n'
        '        restored = list(csv.DictReader(f))\n'
        '    if len(restored) != len(rows) or any(\n'
        '        saved["id"] != row["id"]\n'
        '        or saved["model"] != row["model"]\n'
        '        or saved["role"] != row["role"]\n'
        '        or float(saved["iou"]) != row["iou"]\n'
        '        for saved, row in zip(restored, rows, strict=True)\n'
        '    ):\n'
        '        raise RuntimeError("Exported per-image metric table failed reload verification")\n'
        '    data = load_records(root)\n'
        '    private = read_json(root / "data/records.json")["private_byod"]\n'
        '    # Fixed first case plus deterministic best/worst changes and a disconnected activity case.\n'
        '    chosen = {\n'
        '        data["test"][0]["id"],\n'
        '        min(changes, key=lambda r: (r["iou_change"], r["id"]))["id"],\n'
        '        max(changes, key=lambda r: (r["iou_change"], r["id"]))["id"],\n'
        '    }\n'
        '    disconnected = next(\n'
        '        (r for r in data["activity"] if prompts(r["mask"])["components"] > 1), data["activity"][0]\n'
        '    )\n'
        '    chosen.add(disconnected["id"])\n'
        '    figure_files = []\n'
        '    for record in data["test"] + data["activity"]:\n'
        '        if record["id"] not in chosen:\n'
        '            continue\n'
        '        panels = [("Reference", record["mask"])]\n'
        '        for name in ("clipseg_frozen", "clipseg_reloaded", "sam_box", "sam2_box"):\n'
        '            panels.append(\n'
        '                (\n'
        '                    name,\n'
        '                    np.asarray(Image.open(root / "outputs/predictions" / name / f"{record[\'id\']}.png")) > 0,\n'
        '                )\n'
        '            )\n'
        '        canvas = Image.new("RGB", (240 * len(panels), 270), "white")\n'
        '        draw = ImageDraw.Draw(canvas)\n'
        '        for index, (name, mask) in enumerate(panels):\n'
        '            pixels = np.asarray(record["image"]).copy()\n'
        '            pixels[mask] = (pixels[mask] * 0.5 + np.array([0, 220, 120]) * 0.5).astype(np.uint8)\n'
        '            panel = Image.fromarray(pixels)\n'
        '            panel.thumbnail((240, 240))\n'
        '            canvas.paste(panel, (index * 240, 30))\n'
        '            draw.text((index * 240 + 3, 5), name, fill="black")\n'
        '        relative = f"figures/{record[\'id\']}.png"\n'
        '        (root / "outputs/figures").mkdir(exist_ok=True)\n'
        '        canvas.save(root / "outputs" / relative)\n'
        '        figure_files.append(relative)\n'
        '    summary = {\n'
        '        "status": "completed_current_run",\n'
        '        "release_qualification": "Requires maintainer review of exact executed notebook and outputs",\n'
        '        "counts": {r: len(v) for r, v in data.items()},\n'
        '        "adaptation": read_json(root / "outputs/training_history.json"),\n'
        '        "reload": read_json(root / "outputs/reload_parity.json"),\n'
        '        "metric_summary": summaries,\n'
        '        "source_manifest_sha256": digest(root / "data/records.json"),\n'
        '        "figures": figure_files,\n'
        '        "comparisons": "CLIPSeg frozen/adapted text; SAM/SAM2 matched oracle prompts; no overall winner.",\n'
        '        "private_images_exported": False,\n'
        '        "private_byod": private,\n'
        '    }\n'
        '    write_json(root / "outputs/run_summary.json", summary)\n'
        '    # Explicit generated-file allowlist: no arbitrary directory walk, tokens, raw BYOD data or npy files.\n'
        '    filenames = [\n'
        '        "run_summary.json",\n'
        '        "sample_manifest.json",\n'
        '        "prompt_manifest.json",\n'
        '        "per_image_metrics.csv",\n'
        '        "metrics.json",\n'
        '        "paired_changes.json",\n'
        '        "training_history.json",\n'
        '        "reload_parity.json",\n'
        '        "controlled_experiment.json",\n'
        '        "prepare.json",\n'
        '        "clipseg.json",\n'
        '        "sam.json",\n'
        '        "sam2.json",\n'
        '        "frozen_experiment.json",\n'
        '        "selection.json",\n'
        '        "box_experiment.json",\n'
        '        "summary.md",\n'
        '    ]\n'
        '    import importlib.metadata\n'
        '    import platform\n'
        '\n'
        '    provenance = {\n'
        '        "python": platform.python_version(),\n'
        '        "source": read_json(root / "source.json"),\n'
        '        "dependency_lock_sha256": digest(root / "requirements.txt"),\n'
        '        "packages": {d.metadata["Name"]: d.version for d in importlib.metadata.distributions()},\n'
        '        "report_process_peak_host_rss_kib": peak_host_rss(),\n'
        '        "stage_receipt_scope": "Snapshot before report completed; prior stages must be complete",\n'
        '        "stages": read_json(root / "outputs/stage_receipts.json"),\n'
        '        "models": {\n'
        '            name: read_json(root / "weights" / name / "dimer-base-manifest.json")\n'
        '            for name in ("clipseg", "sam", "sam2")\n'
        '        },\n'
        '    }\n'
        '    write_json(root / "outputs/provenance.json", provenance)\n'
        '    filenames.append("provenance.json")\n'
        '    from clipseg_reference.pipeline import ADAPTER_MANIFEST, ADAPTER_WEIGHTS\n'
        '\n'
        '    filenames += [f"adapter/{ADAPTER_MANIFEST}", f"adapter/{ADAPTER_WEIGHTS}"]\n'
        '    if not private:\n'
        '        filenames += figure_files\n'
        '    write_json(root / "outputs/checksums.json", {p: digest(root / "outputs" / p) for p in filenames})\n'
        '    filenames.append("checksums.json")\n'
        '    with zipfile.ZipFile(root / "outputs/segmentation_results.zip", "w", zipfile.ZIP_DEFLATED) as bundle:\n'
        '        for name in filenames:\n'
        '            bundle.write(root / "outputs" / name, name)\n'
        '    with zipfile.ZipFile(root / "outputs/segmentation_results.zip") as bundle:\n'
        '        for name, expected in read_json(root / "outputs/checksums.json").items():\n'
        '            if hashlib.sha256(bundle.read(name)).hexdigest() != expected:\n'
        '                raise RuntimeError("Report ZIP reload integrity failed")\n'
        '    write_json(\n'
        '        root / "outputs/report_identity.json",\n'
        '        {"sha256": digest(root / "outputs/segmentation_results.zip"), "reload_verified": True},\n'
        '    )\n'
        '    print(\n'
        '        json.dumps({"summary": summaries, "bundle": str(root / "outputs/segmentation_results.zip")}),\n'
        '        flush=True,\n'
        '    )\n'
        '\n'
        '\n'
        'def begin_stage(root: Path, stage: str) -> dict:\n'
        '    path = root / "outputs/stage_receipts.json"\n'
        '    if stage == "prepare":\n'
        '        receipts = {"run_id": uuid.uuid4().hex, "stages": {}}\n'
        '    else:\n'
        '        receipts = read_json(path)\n'
        '        needed = {\n'
        '            "clipseg": ["prepare"],\n'
        '            "reload": ["prepare", "clipseg"],\n'
        '            "sam": ["prepare"],\n'
        '            "sam2": ["prepare"],\n'
        '            "report": ["prepare", "clipseg", "reload", "sam", "sam2"],\n'
        '        }[stage]\n'
        '        for prior in needed:\n'
        '            if receipts["stages"].get(prior, {}).get("status") != "complete":\n'
        '                raise RuntimeError(f"Current run needs a completed {prior} stage before {stage}")\n'
        '        for name, key in (\n'
        '            ("data/records.json", "data_sha256"),\n'
        '            ("outputs/prompt_manifest.json", "prompts_sha256"),\n'
        '        ):\n'
        '            if digest(root / name) != receipts[key]:\n'
        '                raise RuntimeError("Data or prompt manifest changed: rerun prepare and all downstream stages")\n'
        '    # Invalidate descendants when repeating a stage so old qualification cannot survive.\n'
        '    descendants = {\n'
        '        "clipseg": ("reload", "report"),\n'
        '        "reload": ("report",),\n'
        '        "sam": ("report",),\n'
        '        "sam2": ("report",),\n'
        '    }\n'
        '    for child in descendants.get(stage, ()):\n'
        '        receipts["stages"].pop(child, None)\n'
        '    receipts["stages"][stage] = {"status": "running", "pid": os.getpid()}\n'
        '    write_json(path, receipts)\n'
        '    return receipts\n'
        '\n'
        '\n'
        'def finish_stage(root: Path, stage: str, receipts: dict, elapsed: float) -> None:\n'
        '    if stage == "prepare":\n'
        '        receipts["data_sha256"] = digest(root / "data/records.json")\n'
        '        receipts["prompts_sha256"] = digest(root / "outputs/prompt_manifest.json")\n'
        '    receipts["stages"][stage] = {"status": "complete", "pid": os.getpid(), "elapsed_seconds": elapsed}\n'
        '    receipts["stages"][stage]["peak_host_rss_kib"] = peak_host_rss()\n'
        '    write_json(root / "outputs/stage_receipts.json", receipts)\n'
        '\n'
        '\n'
        'def infer_unlabelled(root: Path, image_path: Path, phrase: str, point: list | None, box: list | None) -> None:\n'
        '    import gc\n'
        '\n'
        '    import sam2_reference\n'
        '    import sam_reference\n'
        '    from clipseg_reference import pipeline as clip\n'
        '\n'
        '    if not image_path.is_file() or image_path.stat().st_size > 20 * 1024**2:\n'
        '        raise ValueError("Choose an image file up to 20 MiB")\n'
        '    with Image.open(image_path) as raw:\n'
        '        if not all(16 <= side <= 4096 for side in raw.size) or raw.getexif().get(274, 1) != 1:\n'
        '            raise ValueError("Expected an oriented image with sides 16–4096 pixels")\n'
        '        image = raw.convert("RGB")\n'
        '    kwargs = {"points": [point], "point_labels": [1]} if point is not None else {"box": box}\n'
        '    clip.validate_inputs(image, [phrase])\n'
        '    sam_reference.validate_inputs(image, **kwargs)\n'
        '    sam2_reference.validate_inputs(image, **kwargs)\n'
        '    selected_device = device()\n'
        '    output = root / "unlabelled" / uuid.uuid4().hex[:12]\n'
        '    output.mkdir(parents=True)\n'
        '    results = []\n'
        '    for name, cls in (\n'
        '        ("clipseg", clip.ClipSegSegmentationPipeline),\n'
        '        ("sam", sam_reference.SAMViTSegmentationPipeline),\n'
        '        ("sam2", sam2_reference.SAM2SegmentationPipeline),\n'
        '    ):\n'
        '        pipe = (\n'
        '            cls.from_artifact(\n'
        '                root / "outputs/adapter", device=selected_device, weights_dir=root / "weights/clipseg"\n'
        '            )\n'
        '            if name == "clipseg"\n'
        '            else cls.from_pretrained(\n'
        '                device=selected_device, weights_dir=root / "weights" / name, allow_download=True\n'
        '            )\n'
        '        )\n'
        '        result = (\n'
        '            pipe.segment(image, [phrase], threshold=0.5)\n'
        '            if name == "clipseg"\n'
        '            else pipe.segment(image, **kwargs, multimask=True)\n'
        '        )\n'
        '        mask = (\n'
        '            result["segments"][0]["mask"]\n'
        '            if name == "clipseg"\n'
        '            else result["masks"][int(np.argmax(result["iou_scores"]))]\n'
        '        )\n'
        '        Image.fromarray(mask.astype(np.uint8) * 255).save(output / f"{name}.png")\n'
        '        results.append(\n'
        '            {\n'
        '                "model": name,\n'
        '                "area_fraction": float(mask.mean()),\n'
        '                "iou": None,\n'
        '                "dice": None,\n'
        '                "reason": "not-measurable: no reference mask",\n'
        '                "prompt": phrase if name == "clipseg" else kwargs,\n'
        '            }\n'
        '        )\n'
        '        del pipe\n'
        '        gc.collect()\n'
        '        import torch\n'
        '\n'
        '        torch.cuda.empty_cache()\n'
        '    write_json(output / "results.json", {"input_sha256": digest(image_path), "results": results})\n'
        '    write_json(root / "unlabelled_output.json", {"path": str(output)})\n'
        '    print(json.dumps({"path": str(output), "results": results}))\n'
        '\n'
        '\n'
        'def main() -> None:\n'
        '    parser = argparse.ArgumentParser()\n'
        '    parser.add_argument("--root", type=Path, required=True)\n'
        '    parser.add_argument(\n'
        '        "--stage",\n'
        '        choices=("prepare", "clipseg", "reload", "sam", "sam2", "report", "byod", "infer"),\n'
        '        required=True,\n'
        '    )\n'
        '    parser.add_argument("--byod", type=Path)\n'
        '    parser.add_argument("--image", type=Path)\n'
        '    parser.add_argument("--prompt", default="")\n'
        '    parser.add_argument("--point", type=json.loads)\n'
        '    parser.add_argument("--box", type=json.loads)\n'
        '    args = parser.parse_args()\n'
        '    root = args.root.resolve()\n'
        '    if args.stage == "infer":\n'
        '        if args.image is None or (args.point is None) == (args.box is None):\n'
        '            raise ValueError(\n'
        '                "Unlabelled inference requires --image, --prompt and exactly one --point or --box"\n'
        '            )\n'
        '        infer_unlabelled(root, args.image, args.prompt, args.point, args.box)\n'
        '        return\n'
        '    receipts = begin_stage(root, args.stage) if args.stage != "byod" else None\n'
        '    started = time.perf_counter()\n'
        '    if args.stage == "prepare":\n'
        '        prepare(root, args.byod)\n'
        '    elif args.stage == "clipseg":\n'
        '        clipseg(root)\n'
        '    elif args.stage == "reload":\n'
        '        reload_stage(root)\n'
        '    elif args.stage in ("sam", "sam2"):\n'
        '        sam_stage(root, args.stage)\n'
        '    elif args.stage == "report":\n'
        '        report(root)\n'
        '    else:\n'
        '        if args.byod is None:\n'
        '            raise ValueError("--byod must name a directory containing manifest.json")\n'
        '        child = root / "byod_runs" / uuid.uuid4().hex[:12]\n'
        '        child.mkdir(parents=True)\n'
        '        (child / "weights").symlink_to(root / "weights", target_is_directory=True)\n'
        '        for name in ("source.json", "requirements.txt"):\n'
        '            shutil.copyfile(root / name, child / name)\n'
        '        write_json(root / "byod_output.json", {"path": str(child)})\n'
        '        for stage in ("prepare", "clipseg", "reload", "sam", "sam2", "report"):\n'
        '            command = [sys.executable, __file__, "--root", str(child), "--stage", stage]\n'
        '            if stage == "prepare":\n'
        '                command += ["--byod", str(args.byod.resolve())]\n'
        '            subprocess.run(command, check=True)\n'
        '    if receipts is not None:\n'
        '        finish_stage(root, args.stage, receipts, time.perf_counter() - started)\n'
        '\n'
        '\n'
        'if __name__ == "__main__":\n'
        '    main()\n'
    ),
    'sample.json': (
        '{\n'
        '  "schema": "dimer-segmentation-sample-v1",\n'
        '  "seed": 42,\n'
        '  "dataset": "EduardoPacheco/FoodSeg103",\n'
        '  "revision": "176acc3edd2432ee126bda6fb01469eadeb018df",\n'
        '  "source": "FoodSeg103 (validation split), first eight parquet row groups",\n'
        '  "license": "Apache-2.0 (FoodSeg103, LARC-CMU-SMU; Wu, Fu, Liu, Lim, Hoi, Sun, ACM MM 2021; images curated from Recipe1M)",\n'
        '  "roles": {\n'
        '    "train": 120,\n'
        '    "validation": 40,\n'
        '    "test": 40,\n'
        '    "activity": 12\n'
        '  },\n'
        '  "source_count": 800,\n'
        '  "unique_count": 800,\n'
        '  "exclusions": [],\n'
        '  "selection": "stable ID order; decoded-image deduplication; Random(42) shuffle",\n'
        '  "target": "largest eligible ingredient class union; never largest connected component",\n'
        '  "records": [\n'
        '    {\n'
        '      "id": "foodseg103-val-758",\n'
        '      "prompt": "french fries",\n'
        '      "class_id": 3,\n'
        '      "source_id": 758,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "21452a2594212da09112ee220ac2226a8e43b4b182c3031e43239ec93daee993",\n'
        '      "mask_sha256": "4c29f81ba3dd500772720979ab9c33bd09bf5613f0da1d0ece2de91979ac6a65"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-435",\n'
        '      "prompt": "asparagus",\n'
        '      "class_id": 85,\n'
        '      "source_id": 435,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "bace96be0479d14f4f15bd4d5fc7f4af6e4825d8f16f78ec1136e48a5f7ba417",\n'
        '      "mask_sha256": "4972b0e7cc1a7585dad97292c9ebaab601d9cc2d1ad17dbbf476404432f395db"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-614",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 614,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "24e051113414acd1b866db7bf0c1316f9998577aaadd94dc770286a0183bb168",\n'
        '      "mask_sha256": "effa1c0e37d35a69bf65918de4658c2172fa2418c40ed67732b18f3d42e110de"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-54",\n'
        '      "prompt": "grape",\n'
        '      "class_id": 41,\n'
        '      "source_id": 54,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "d22470cc406354d02278741a73bd5385cbb952d9349713b8c2b322caeb861bfd",\n'
        '      "mask_sha256": "30ddfbe721b05a904da8bf25a81335819b50410f58d92e5f9c10c2194e80a302"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-681",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 681,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "b693b2d3d99f264531dbc8b1d0bb4c15d93c42c53cc5781e21cba62f6dce4b2b",\n'
        '      "mask_sha256": "1c9f85c2317e9b6fd59697e5bdb51206af106b3230f7232fcb2ad4766284bdfc"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-577",\n'
        '      "prompt": "rice",\n'
        '      "class_id": 66,\n'
        '      "source_id": 577,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "4c4af2f7915b170ccb72a457b70d11308e21e7718d52a732d4924aac3fbfe523",\n'
        '      "mask_sha256": "7f74af6fdc19e3aafc2bdc9da02939ac689f490172265ba9c1cbb9b574e58581"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-420",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 420,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "26c656b4c33ced07646d14998fdaf3058f8b0cc247fab1181630709022f5e602",\n'
        '      "mask_sha256": "29715da89fc3ef373f1aef6854b5af562a1abd06a846213e0094a9a7753227f5"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-683",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 683,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "852adbaaa0362f9bbbe95089cb7d105c1e9a68176bc5dc0aa9d79f96231fbb33",\n'
        '      "mask_sha256": "d8ead58172f2b718905ce8eca576c460027567a38eaa54bd43221cff3b910f61"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-503",\n'
        '      "prompt": "coffee",\n'
        '      "class_id": 13,\n'
        '      "source_id": 503,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "0f9bbd636769c8daa4ac17fc45fd8c70a94b6adc686881e81321dd69fcbb5b0a",\n'
        '      "mask_sha256": "2b0ca9f98ce482edaac38cda9866a086543fae08ba89edb1ca8cff6af5393ae6"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-795",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 795,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "d8bea9f98fc4321a1d32aec301b76a0465e71c54fefb3667ee7f0e207ba5369d",\n'
        '      "mask_sha256": "f06693dc1f5f887d46d3920006d5c9a351cf0928498c10f7dc23216e0d8c5721"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-544",\n'
        '      "prompt": "pie",\n'
        '      "class_id": 67,\n'
        '      "source_id": 544,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "74491e03957b1de79c922cce776dd65784039d0d57ad8d6161eb745f048681e0",\n'
        '      "mask_sha256": "4ce3e95db026686725dc51172bdaed69a3cc529374de3f4877dc6d9f4f3f4e84"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-679",\n'
        '      "prompt": "carrot",\n'
        '      "class_id": 84,\n'
        '      "source_id": 679,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "05f6881b1aac5bbd6656a0785d41602beb46b106655c664287e8615e55f5cd54",\n'
        '      "mask_sha256": "bf8e99cacc648328aeb2f8eed029a07953df81cdc141e011f6ed3815e2319425"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-89",\n'
        '      "prompt": "cucumber",\n'
        '      "class_id": 82,\n'
        '      "source_id": 89,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "a12d77f2b3d4f5c13ee37c5d1996496b073971c7cf4024221c14e14449eeed83",\n'
        '      "mask_sha256": "49c79e517d9f15b8723644e5383adf3d569f8946c3780be61f636bd52ea01c22"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-401",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 401,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "d914ccc30c7956863afd2c0f0d0876dbd35d4d471206c3f9ad002fa1e7a51632",\n'
        '      "mask_sha256": "6788258a1371a1cf321f32c11f9263f5602ef29aee94b463893909035263b7b5"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-12",\n'
        '      "prompt": "broccoli",\n'
        '      "class_id": 87,\n'
        '      "source_id": 12,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "3dbf82fd8644a461152620a457a12a771f86b502ee7dc4e3bf2d5fd7ed2b073c",\n'
        '      "mask_sha256": "5f3773514560acefabb3701accaefe0c518ec0981e7c4232427be31ebcd68e01"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-91",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 91,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "13714c535c6dbf7658b20730a0241c64a42c0c3702445ae6ef2069c100b50c45",\n'
        '      "mask_sha256": "02a64b9e1133c66093269565d8150a85d5aa2f075af71fea3166cf5725d06c4c"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-258",\n'
        '      "prompt": "french fries",\n'
        '      "class_id": 3,\n'
        '      "source_id": 258,\n'
        '      "source_row_group": 2,\n'
        '      "role": "train",\n'
        '      "image_sha256": "a86d198ee695c9457698c23aef82d659f4417b05e89e0e669b6ef8e844d7c3ef",\n'
        '      "mask_sha256": "bc36cbdc7204e22fac0a6cf7d99f3a7db6beafee52e245e4b393324377d98338"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-439",\n'
        '      "prompt": "pie",\n'
        '      "class_id": 67,\n'
        '      "source_id": 439,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "3cbb2128a3db2cad8104c0cae898ef0576d882638872f9b1fe8063c7416f1353",\n'
        '      "mask_sha256": "c9688c0452b43c92a5d83c637bf4eda2dbce966d587371beaf742e9453d23f38"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-404",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 404,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "7c37f284a0e653ab90aab553abdc1c3031613b78ab27edef86795f848df3b4e6",\n'
        '      "mask_sha256": "2a7892e43a5ec457b338f70472940d51a4b405145d3381b1566ff37edefeb9c3"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-193",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 193,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "cf201ca85447c222e35794c38916c49d6ed52dac8aef941de39e5aba3f133e63",\n'
        '      "mask_sha256": "2d623d2942148e9de7c5ac83470c2e670397db4eccf6a10551a91c50f1c3a685"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-457",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 457,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "844bddb7ed6d52a790d92367a8f52486fb6e281f2468373096b1415c143bddd2",\n'
        '      "mask_sha256": "5a62ecbd1e5ad20e0089c4e73a40f48fe80e53685652d8441ba3ff660121b82b"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-178",\n'
        '      "prompt": "pork",\n'
        '      "class_id": 47,\n'
        '      "source_id": 178,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "e56ec8eb6caa94b1af650de24b4b63a807ecb3ed828b207e146b10a5fae0fdb2",\n'
        '      "mask_sha256": "d09cb1d55111a287ffd350477ab9680b11fa217466e5a676973235eceb38ca64"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-652",\n'
        '      "prompt": "corn",\n'
        '      "class_id": 59,\n'
        '      "source_id": 652,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "a552926a811e70ae500acbb4b2556648de110d2d207aac3ca3e63352fcfb315f",\n'
        '      "mask_sha256": "979cbbade731a5e57bcd504264f7966615692f159aa740eb71ccd3f8542bb3c8"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-478",\n'
        '      "prompt": "pie",\n'
        '      "class_id": 67,\n'
        '      "source_id": 478,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "3d2c2a7686503bc2766fd5bd05e53fb93317b39dd018e3f4ed66503d3d00a770",\n'
        '      "mask_sha256": "6991922e7de8635a67f1ca491d909050ddd87ab53853b32062b94048e5e57122"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-279",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 279,\n'
        '      "source_row_group": 2,\n'
        '      "role": "train",\n'
        '      "image_sha256": "87455768fbcfc9d696b5db1852c894dea8a0121cda90d0997860d71bf6e4c707",\n'
        '      "mask_sha256": "750e00f15d85085f387884774b00a32fe1f64d2702e01e72f16e59ee76f632c6"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-28",\n'
        '      "prompt": "french fries",\n'
        '      "class_id": 3,\n'
        '      "source_id": 28,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "c3cae4e888a78252a6bd3e9dabd269afe964f2aeb3fee3828149d997b6a6a001",\n'
        '      "mask_sha256": "aedcf733a5ee724930f761e2cfda78a64cc7bd463008b0e008c7954f3020140a"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-539",\n'
        '      "prompt": "noodles",\n'
        '      "class_id": 65,\n'
        '      "source_id": 539,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "8b431d388bade7d0f77d116237f59baf020d22cf9410bf6035f1ac8d6aa60d67",\n'
        '      "mask_sha256": "ca99607f4630a21f3500fdc386b539db780ce2dd3b0900c57ee20575f8fd9c2f"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-306",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 306,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "36af5e74e38c2c1efd239cf28ef587d6144f794245e5a6a8c808bc381cadb08c",\n'
        '      "mask_sha256": "f54a267b1a3fdc3cc32afff524c9b01a531b9a2b0c33a14ebacb7f1b8661afab"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-313",\n'
        '      "prompt": "green beans",\n'
        '      "class_id": 95,\n'
        '      "source_id": 313,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "fe80a5b1bd8a06fd68b1ef529bf833b1c8be1c84be1da969ae3e5e3f5bbea4a5",\n'
        '      "mask_sha256": "4dde98519255fa853a28e5db135327eaa5b1ec28097307fa0d3c791578e88173"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-437",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 437,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "3a299e5f38a5337faa805686167b7497605d37bf4de3a8d7062b78d5f62980ca",\n'
        '      "mask_sha256": "382a9a21b6adea0b3f5f1fafaddda6523e58954eff37e6a8a278602444c1d14c"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-156",\n'
        '      "prompt": "asparagus",\n'
        '      "class_id": 85,\n'
        '      "source_id": 156,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "6f398c343dbb3f8982395640ac5e693ca327c174012985351a128e4d1f78cd78",\n'
        '      "mask_sha256": "b71ec819b4d936b9a27d169ae8a802b48f38e4f2744568d09e507c6063006b4b"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-76",\n'
        '      "prompt": "shrimp",\n'
        '      "class_id": 56,\n'
        '      "source_id": 76,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "e9250015589d1583a3581272ce20e5d15a872acf6d123af5f79cb6efb643f360",\n'
        '      "mask_sha256": "3ab65be052b1bc5db34265b18cd74cd66fcfe021cd2b19a498394206797996c8"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-756",\n'
        '      "prompt": "carrot",\n'
        '      "class_id": 84,\n'
        '      "source_id": 756,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "da4c7dc943369ac540c7777c1edd91ea576dac410420094a511b1ed7f85f09d6",\n'
        '      "mask_sha256": "477f3a914526f125c189e368d6d677ae7af9f5e1a92718de0c4e99d2a93a916a"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-304",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 304,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "ac3394144cf94d6b1704fe27dd3c0731a57b424a85063dd92b635a874bab35ef",\n'
        '      "mask_sha256": "d6ac1ce4e40652bbf80c041deafbf0d0d7fbd9f3f37e05e06196b4ab369e1c24"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-589",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 589,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "6def589d4b3221f59362f7a99127387b811be9b71aa9d3a1757d5383ef2b76bf",\n'
        '      "mask_sha256": "ff5177513627cfc665dd53ef3224e129b379e914c5381bd1903eeabafe5493ee"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-65",\n'
        '      "prompt": "sausage",\n'
        '      "class_id": 49,\n'
        '      "source_id": 65,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "51ee7d1dfb8a79b003979a7055e2691ef957799b7dd71ab81683ffb5b475436f",\n'
        '      "mask_sha256": "a2d3ba2445749664b7ae02b70949941ce727b2ced218d072ce169e9307f5a1fd"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-558",\n'
        '      "prompt": "strawberry",\n'
        '      "class_id": 30,\n'
        '      "source_id": 558,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "381f80eb38ce41529098fa7359696b6c21929820015833e2b6c598be9ab3a3a1",\n'
        '      "mask_sha256": "78804ec71ca43c3ec42fd72de0cd239a310bc5424e42fcf3878548b211fad2e9"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-310",\n'
        '      "prompt": "pork",\n'
        '      "class_id": 47,\n'
        '      "source_id": 310,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "e6548bb8270f4600cd8d69013791d51efc486c21ebe51c58f9f2c1004d0acfbc",\n'
        '      "mask_sha256": "b0536bfc614385cdef29c71ad262b1ec04f0f84372dc59b098f33ce647568f20"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-667",\n'
        '      "prompt": "French beans",\n'
        '      "class_id": 96,\n'
        '      "source_id": 667,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "9e208780dd66f11b9453fcb82f21b3074b566a2969af925cdec3f067a7fb9840",\n'
        '      "mask_sha256": "87bf98bee2e4650751bae02c45cebaf03c90ed8005489c46c5186f0e53ed90ab"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-71",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 71,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "555ece99236c543cf0fe9a4b29bf117d059e96fd4f1b5647deee57fa4722f4b1",\n'
        '      "mask_sha256": "0514a58f6c1bc281df25c122c2d2a969a6ff0d889b3c54c0a46aa589189e34e6"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-298",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 298,\n'
        '      "source_row_group": 2,\n'
        '      "role": "train",\n'
        '      "image_sha256": "f792255405232e9ea9a941a9e81873da1af15cf27831363e5c7dffc90649a192",\n'
        '      "mask_sha256": "a94b34e676a4febabcd45d620b2fe9a82539e066606e0433942cec8e56a2323e"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-499",\n'
        '      "prompt": "milkshake",\n'
        '      "class_id": 12,\n'
        '      "source_id": 499,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "061c0bad6a54a13dd65892f2729a55d4241ec1421d25f5e2e944250ab207943c",\n'
        '      "mask_sha256": "b9934448929a5ad82ef66c87aec5d89e902cc5b7d733266813f8320d324eed64"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-548",\n'
        '      "prompt": "potato",\n'
        '      "class_id": 70,\n'
        '      "source_id": 548,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "9da021179b21e11e4226614b3eb2634b264b936d93e3eaacf2c459e6d1808830",\n'
        '      "mask_sha256": "d6d6d002685db3e105f5b3b301a67c917ef2afd5168f701db473a994f95f87ad"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-757",\n'
        '      "prompt": "wine",\n'
        '      "class_id": 11,\n'
        '      "source_id": 757,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "d9fae455150b58940a46e2b26e6b09f2e829ec7ac9a985a5b9a2526f9c05a2d8",\n'
        '      "mask_sha256": "2c11d7ad507b2a207a3449eaa0bad3e98249cdcb80230bf1095e6b9df17f7a09"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-395",\n'
        '      "prompt": "French beans",\n'
        '      "class_id": 96,\n'
        '      "source_id": 395,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "813bd55d0daf9e2b561e2e78dee9ff21824c5bb34a4e4d352cf97c2401bfe336",\n'
        '      "mask_sha256": "f5af4beb067054426e156154582234cd11b9a0e897c8498e780849a5da425cbb"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-350",\n'
        '      "prompt": "sauce",\n'
        '      "class_id": 52,\n'
        '      "source_id": 350,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "071e21e4050dff9c22a9b9e65a447922fe19e766a86d36f9dec9fd821b4ebecd",\n'
        '      "mask_sha256": "f1799fa57c4c1fca87009d59d9b62cd8ec1ac4dfe25a24392b7eb79528875519"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-370",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 370,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "cf0c23071f85c60e32bd46004cdcbd5ef210d3621dbfafbdd01fb6ebb6e0fd16",\n'
        '      "mask_sha256": "fa49cbeaa5157621c2c369feec673778189f88b4006cc8a26afcfef20bb96425"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-270",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 270,\n'
        '      "source_row_group": 2,\n'
        '      "role": "train",\n'
        '      "image_sha256": "a26bd0b3de5b192c164d2d95e5a2ddec1ed0422a429668fc0aabef694d523d4d",\n'
        '      "mask_sha256": "610ac4f60a570ecb11dbfc479a7dcbaed04a8368d20965e0687e7a796785bb33"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-46",\n'
        '      "prompt": "strawberry",\n'
        '      "class_id": 30,\n'
        '      "source_id": 46,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "76a71188b5b59f5494edb288390c406a1ea1a4de9c1804f1e145fb3caa60b797",\n'
        '      "mask_sha256": "e8a53be6cd657a7a4f6069f36af7c1d05e8171e8df13de524c5545d195827ece"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-383",\n'
        '      "prompt": "red beans",\n'
        '      "class_id": 18,\n'
        '      "source_id": 383,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "d1fa52519b37cbcb2b23204d9b271998134a9ec0db029cb06f4492b39c93eb51",\n'
        '      "mask_sha256": "b51b72ee8dd6d9289717a3527586e1c46c8e924f4d026b0eb396dfce92c7b339"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-169",\n'
        '      "prompt": "rice",\n'
        '      "class_id": 66,\n'
        '      "source_id": 169,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "9ee252dd654b9395e3e783a0b404471f62fbf093a2711eb7a09f4c1c49bff531",\n'
        '      "mask_sha256": "197f679134428abc571a9cc1c4ef06720f5dc5a0416ff1509883c0803c7bc1d4"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-29",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 29,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "5669bc6e25bdae38685313d175829345e80edd8e32e5b19f5272204168bcfe7a",\n'
        '      "mask_sha256": "30854d193ac07b35212f139f85800ee47096b02567f8791a7a2c0ba5f961c3a1"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-705",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 705,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "b97acfaa0302cdfb3a6ed5158cacd98662d6066c6a83999c0fbcda3e70244423",\n'
        '      "mask_sha256": "572cc8fc1df7ddcca9667e3119f4a481f0e1dc9ac7861d0e7f0ee91127bb1712"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-443",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 443,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "a8b454dc319d52cb1465201dc4d3123d1e216ce48c91197452df010361b8d9f7",\n'
        '      "mask_sha256": "c6e90106dbed5f91ba4fe47aca074ce248b3a35cb6b8d439955d788f95bfcfb1"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-761",\n'
        '      "prompt": "potato",\n'
        '      "class_id": 70,\n'
        '      "source_id": 761,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "8bc747f6a24652e9d3c4752009d5957336c06afb9b7cc57dcb0e2d7780031c1a",\n'
        '      "mask_sha256": "93782a4b67bf9620e6de3cf58541b89385d8caa4cd24cdcd0145e1134ff6794f"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-450",\n'
        '      "prompt": "fish",\n'
        '      "class_id": 54,\n'
        '      "source_id": 450,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "887370a803738ebc8033427b3de9698e8ef5776ae9d40e73703b0f911b7d8222",\n'
        '      "mask_sha256": "f5794906ca2a0cd1bb90ad28cf613503fb892e7ed3ec791275992fcc5037d8f2"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-69",\n'
        '      "prompt": "tomato",\n'
        '      "class_id": 73,\n'
        '      "source_id": 69,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "35f52faddea006245db088909d11473541f1772a372afa366e008876a8458b45",\n'
        '      "mask_sha256": "0c832776c200ace53c49c81ee0a71c63aa0638dbf1270901f0ba5230df36035f"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-774",\n'
        '      "prompt": "rice",\n'
        '      "class_id": 66,\n'
        '      "source_id": 774,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "de7f32a2a155cd5365b2da3bd1c6f2423f7834dbc075a401e4036b046311ca93",\n'
        '      "mask_sha256": "4713e2d987ade4503a1644fd24b5e42a4b522cda2d24616cf4d6630323e8ac8a"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-37",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 37,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "0647a12c196c9877e140265c74ca790c8e8ac9f44b19046678956c8fc32452d6",\n'
        '      "mask_sha256": "b5ea28a0be91a1c44a343b42b5495529a7b32370f9b8db3788da8fc8e08015cc"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-233",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 233,\n'
        '      "source_row_group": 2,\n'
        '      "role": "train",\n'
        '      "image_sha256": "3deb6f74871c4bbfdbbe928e2746bc2adbbd3c31b710934783bc55c644b53818",\n'
        '      "mask_sha256": "66a96a60bfbc39960aee99bdab39a77e67b9018ae941c28b5620ecaa9db549a0"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-583",\n'
        '      "prompt": "carrot",\n'
        '      "class_id": 84,\n'
        '      "source_id": 583,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "ddc2f8c13447bcdc74279f4e5df6bf9d20961e83994a994e7320d5fb1fe84bb1",\n'
        '      "mask_sha256": "8d48a78c5f99d3caa75817dceb7c87a95c8d786532528e5abe0d25cc5ef459b4"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-130",\n'
        '      "prompt": "milkshake",\n'
        '      "class_id": 12,\n'
        '      "source_id": 130,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "808cb5118b8e4bfcf92c6efefacd760d235812b96a8d71f1605c8c19539c7d37",\n'
        '      "mask_sha256": "0ec8d572d6d2c2e6355c3baeea882a4353b75ad1784a42d15f08c8a2bc2f62ae"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-640",\n'
        '      "prompt": "cucumber",\n'
        '      "class_id": 82,\n'
        '      "source_id": 640,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "692a1a8b36de0b08e0c48b871e701eea9e557fa706ef03c0b21ac0a9f28b80bb",\n'
        '      "mask_sha256": "e2cfda584013528fe956474fde33af7a2dc14ccc050649eb2ae1efca401979b6"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-265",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 265,\n'
        '      "source_row_group": 2,\n'
        '      "role": "train",\n'
        '      "image_sha256": "3613085e8e4049027431cc190a37efc56dba9d5aa68e59c3ba436661c528bc34",\n'
        '      "mask_sha256": "66066c77147aa829ac708c4993aec41d9f452c2d919062c77771f9aeb9509b98"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-253",\n'
        '      "prompt": "carrot",\n'
        '      "class_id": 84,\n'
        '      "source_id": 253,\n'
        '      "source_row_group": 2,\n'
        '      "role": "train",\n'
        '      "image_sha256": "864223f8de4be57aa2a3146bddb5d95a5794a8c0abe29fee6c96a98640bddb31",\n'
        '      "mask_sha256": "892d3d6bcd168e8b17b1f785810ebd1afbcec9ea3cf62296974323c056357cf5"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-260",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 260,\n'
        '      "source_row_group": 2,\n'
        '      "role": "train",\n'
        '      "image_sha256": "cf3c0613c550d8f75a6d81705bb8c82f284b2df412817cbe5be696ce2f888b1f",\n'
        '      "mask_sha256": "89e18ab0d268d419b6e3c7069c536fa2a406f6f8fe7c417b8a8c3fb0df9acb14"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-181",\n'
        '      "prompt": "cheese butter",\n'
        '      "class_id": 9,\n'
        '      "source_id": 181,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "9070232b2faf89a18808e1a0662e905ebefd14f0d8f6a5613a8f7d6c57d33e00",\n'
        '      "mask_sha256": "c25049c002b54d3464f6e84ab00a67aa9be5aa91cebd8b9f39fbba65e5e6c0ae"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-106",\n'
        '      "prompt": "milkshake",\n'
        '      "class_id": 12,\n'
        '      "source_id": 106,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "3f7aa2ad7b8246c136a266d42d996dbf439087da5441d5e14baf7467865d2ba9",\n'
        '      "mask_sha256": "3613c8a3b32ac8e74437bd5dc53da0bd8407fa5a25c242f367acad7f70c58de2"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-42",\n'
        '      "prompt": "pork",\n'
        '      "class_id": 47,\n'
        '      "source_id": 42,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "630fe8692c7004e8ac77eb13b3fd9d5d64dfdb0b474f2f7a5ba98dfd2d0a580e",\n'
        '      "mask_sha256": "01f8931967450ff00f1d03bbc8ff2a67e39eb4da951e9ae6cf48e0148afa5ac0"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-143",\n'
        '      "prompt": "lamb",\n'
        '      "class_id": 51,\n'
        '      "source_id": 143,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "374e24286ae679fd823fecf28fb4fb327b357b55ae2513573da8ad2d92c062ed",\n'
        '      "mask_sha256": "7961a4e9c6e89ad71eeb67ebf19861f4ea73b37c6c7f322a152f37662b287e1c"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-515",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 515,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "3ef9243ebe3ee357e10ecf4b9af379c991e5013ac155b687373c494b2013a52f",\n'
        '      "mask_sha256": "319292f923c026e1bbbde95d5c956d0e734f3cbfdc90c2eda9a3ece84252c13b"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-165",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 165,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "37cb85ac2a2c04fd63cbc2863ba76ee18a2a16bc239ac343700014fffac0953c",\n'
        '      "mask_sha256": "dada7f8b19ed39fd12f6721dceec49210fbfd3125260af22fb82ba9f62411df1"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-699",\n'
        '      "prompt": "cucumber",\n'
        '      "class_id": 82,\n'
        '      "source_id": 699,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "f01f2d399a21431359b3d19d028d59d74c8f6577868b3a228762a53cef305041",\n'
        '      "mask_sha256": "5c8ef0f6a6cb6dc3215b489ef3bf302b8970481839b491f6754cca2af9ce93f2"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-11",\n'
        '      "prompt": "pie",\n'
        '      "class_id": 67,\n'
        '      "source_id": 11,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "038e5a485c18d2090277ddf63d9cb385a6ac66d7a5ad22a608ca99b934e66bc1",\n'
        '      "mask_sha256": "04ab8b9ee8fc76626b16ae19f5ad9340b63e6466b0635eacbaa665aa380210e8"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-763",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 763,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "960719c280759b96279b5f18fcc1037ab11e7b08ec180c249f1bd97025dff191",\n'
        '      "mask_sha256": "88cb4d8bde4cc3d87f8947a4d7296db2f10f2c4deb6ccabe89062db35a10a2da"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-255",\n'
        '      "prompt": "pork",\n'
        '      "class_id": 47,\n'
        '      "source_id": 255,\n'
        '      "source_row_group": 2,\n'
        '      "role": "train",\n'
        '      "image_sha256": "0a0a80208edf6564a205ec9b5d12f2d27b6af30fdf6db31301215c99ac457171",\n'
        '      "mask_sha256": "c20b66fa389d7e2baaede3e6a4f5e258ce463dc5b179f17ff285dd940d7ecca9"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-791",\n'
        '      "prompt": "fish",\n'
        '      "class_id": 54,\n'
        '      "source_id": 791,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "83bf9a4dfe91a8f9a141c204cfd700658cd24ce1cbf41139df899822fc07b371",\n'
        '      "mask_sha256": "5adae0f46f8459916f5674eba7a7d5860eee5bf05dfa599d8e90846c5433b23a"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-584",\n'
        '      "prompt": "blueberry",\n'
        '      "class_id": 32,\n'
        '      "source_id": 584,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "36879f2723b18ab701bebd696068a53c56841b063a9bb7d69a546c1c05f2cbf3",\n'
        '      "mask_sha256": "b9dff3468a44399215afdeb2ae133d0a9cb346424df6fa9722975357e33d2058"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-460",\n'
        '      "prompt": "onion",\n'
        '      "class_id": 93,\n'
        '      "source_id": 460,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "ac6fd6c69f06a4debd1d4fa4d1432fe1fbddbccc339867b7aae40b51f8a54551",\n'
        '      "mask_sha256": "b021a5df6f7038829d5f370be1a641c0d269eea678366040ef83ed8d8ed98934"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-504",\n'
        '      "prompt": "french fries",\n'
        '      "class_id": 3,\n'
        '      "source_id": 504,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "85320d8a557f9cd1f3d930373db8cc1e218d11b34565a63765867a38df1b517f",\n'
        '      "mask_sha256": "1476f57e963ae7bc569ac4a4beaf01303633a6bac7258ac80c778dba46fbe807"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-348",\n'
        '      "prompt": "tomato",\n'
        '      "class_id": 73,\n'
        '      "source_id": 348,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "937af62314f3b31b6bf7661fd607116324ca27e5edac6b9f65a8b095b2ffb332",\n'
        '      "mask_sha256": "87a98854d5ceeee1fd52df6cf5dff5f7a1fb0bedcc2036cd794edfbdd8613259"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-337",\n'
        '      "prompt": "pepper",\n'
        '      "class_id": 94,\n'
        '      "source_id": 337,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "b166bc844016c0185c78ebdcb44597da53d805a7a50499fab75da8a3aec98166",\n'
        '      "mask_sha256": "3e152337dfb8a5f2f7e72cf1edf56f717692e34cc3ea52f8792ee907e071fca4"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-421",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 421,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "ccc067232388d15e24649f9515fb95c83ab519c6ea43812f9c083361ecd973fe",\n'
        '      "mask_sha256": "3549ca4a8169b41f115f96049397c19e135ee0585d6f53a196530aff5195d7c0"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-616",\n'
        '      "prompt": "potato",\n'
        '      "class_id": 70,\n'
        '      "source_id": 616,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "7b7d65d5615ba35b538d4cc0b16d04622db8355008046c08fe0a8bc865fb7de1",\n'
        '      "mask_sha256": "85b206b1cf316494630a5cbac2c7f873bad441210c852fefde6516ea49435db7"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-387",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 387,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "cc3265f72b7d6e04a1a7eb1f5fac1eac3334ede7dc5508a19e1a24b48bcc5839",\n'
        '      "mask_sha256": "89cc6cf16d2d5e5ab12f3c3bbafc0db8b83440942574aef3883d04772ac0c889"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-664",\n'
        '      "prompt": "lemon",\n'
        '      "class_id": 37,\n'
        '      "source_id": 664,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "d9b979685147d0e15e4336f3430896fa4b8f77fd8c4c246aa58777c89e2f24da",\n'
        '      "mask_sha256": "94b6a8827221c7287b31b283ae658274fa0df1c66e3d8670157c8136582735a0"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-307",\n'
        '      "prompt": "lemon",\n'
        '      "class_id": 37,\n'
        '      "source_id": 307,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "ffc2af096834031049920f57f2845a6ba23d57970f2812edc59525bc32d90375",\n'
        '      "mask_sha256": "2778d06ac3ca8409fe26d515a08787ad75d2518871e3f4a4abe4e54df468e83f"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-83",\n'
        '      "prompt": "pork",\n'
        '      "class_id": 47,\n'
        '      "source_id": 83,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "74e19a8f96c8a4ed0bed78777e98513cbfd9a4db4d939b0f105e0c23150b3eae",\n'
        '      "mask_sha256": "59245017373b4b9e96f35cd7e0f7c80352c5da2be90ba8b91066a04072edb625"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-49",\n'
        '      "prompt": "pasta",\n'
        '      "class_id": 64,\n'
        '      "source_id": 49,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "a992b3df2894fad50d857b723fb8125c9a41774945277d275efa18ca1d11f62c",\n'
        '      "mask_sha256": "bf65171264b6bb10e5449aef765ef2a54fa659cac5e525c5498e2c3c65e9514b"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-419",\n'
        '      "prompt": "sausage",\n'
        '      "class_id": 49,\n'
        '      "source_id": 419,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "3a408263aacae0122d7af60f0320e94447bfd3c8447a0c9306128fdaa53c839b",\n'
        '      "mask_sha256": "b9ba31514f990d441d70bb344d594607e90d1b852c20a88c9b7cf94f00ec9f35"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-731",\n'
        '      "prompt": "grape",\n'
        '      "class_id": 41,\n'
        '      "source_id": 731,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "ccfb47f57d68ba4a359d701f58661f337bc62cc89c96431e7cb78d0f04df8ae9",\n'
        '      "mask_sha256": "34856b2061d4258a44506074be5b1ada4e4d1a2af2fa6baf45dc6ed98a295814"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-555",\n'
        '      "prompt": "milkshake",\n'
        '      "class_id": 12,\n'
        '      "source_id": 555,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "9afe7b89130b44f361a92f3930ab2b4e242f98913e4be6493cefa29cc7b5b472",\n'
        '      "mask_sha256": "338e554518d33022f8f3530a198eba98384e416309d3745d52a33aea058ffdf2"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-290",\n'
        '      "prompt": "pie",\n'
        '      "class_id": 67,\n'
        '      "source_id": 290,\n'
        '      "source_row_group": 2,\n'
        '      "role": "train",\n'
        '      "image_sha256": "407ed586723c79bb67e96058a760a9d037d4347510fa79f84e57d72c681ed3ce",\n'
        '      "mask_sha256": "5818053ccb1ecf21ad3d22eb1749e5cd8f34334bf301eb558e5743887f9846d9"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-675",\n'
        '      "prompt": "lettuce",\n'
        '      "class_id": 80,\n'
        '      "source_id": 675,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "42a0db428625316208e1aaa78aa6cf39d246ddfc5d5f423aefb9eee5bb2c2db0",\n'
        '      "mask_sha256": "190e233237590c0e48df8aeae70414dd85d83c7ef77f8a221ee7c94889193e7d"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-719",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 719,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "830a8c6a22a94414e6945fe77a721ac6232e7191181425c30a977dd8d3eddf24",\n'
        '      "mask_sha256": "7568d3ed65f5be096bca24712ebed71ed4e6295ed9753c31ec1c9283a318572c"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-320",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 320,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "5e8426a7ef012ac688359e388196a63903119b812f3ed00ec890b07b8f4f5de5",\n'
        '      "mask_sha256": "4525c2a4681a46982fd23334882ccfed4b5a92f18a88572f20a64f1dc103399b"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-105",\n'
        '      "prompt": "milkshake",\n'
        '      "class_id": 12,\n'
        '      "source_id": 105,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "4ed178240328687bad84fe96bc2b7ee5fffe9453321ccfc1c848a791bdf489d9",\n'
        '      "mask_sha256": "ad0a96aec0d47cef1ee22c669d269428706966f4b1ad3bd35604cacd847e8904"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-417",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 417,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "cc4c67eada18712a84ba33d1b6b13fa14717932c6c060a6b6626f97ef72c1be2",\n'
        '      "mask_sha256": "2612f5c612bc1f0333323133917cfb1cd655c39fd508364b635aba3b2819622d"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-123",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 123,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "33b2be58058cd7639a9c2a99c02281c6cc0b64a0b003c15f7ead80837bda9cd1",\n'
        '      "mask_sha256": "ed2ad84ae2fc076ed93dac5de9b7bbb4e631937de99ee4632561f3060f4d58d7"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-771",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 771,\n'
        '      "source_row_group": 7,\n'
        '      "role": "train",\n'
        '      "image_sha256": "ed26ed6728c4d6477773285db6b80b9a6fad49cfa0a1c2165c181d53cfe90599",\n'
        '      "mask_sha256": "89fed0b060912a2694ea6dab152df05ba6ba6ccd2b67f6e44e112bd0ec22e598"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-680",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 680,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "ad3a793997c96558e868dade228c8b2967e25cb30e9708d4ad425ae5b9b7432e",\n'
        '      "mask_sha256": "891df9ecccb84a15786e599326ac06b669fc06a5bb8280e4e3a0c6669cd9bc9e"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-456",\n'
        '      "prompt": "lemon",\n'
        '      "class_id": 37,\n'
        '      "source_id": 456,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "df71a114053b1f13cbcb739ec914c5d5723f652ee82812b4a9a455aca1886682",\n'
        '      "mask_sha256": "67092492fcb59e76c2fe2b405fed4b0939da06c2f9927b82c36a4db859e59f93"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-543",\n'
        '      "prompt": "potato",\n'
        '      "class_id": 70,\n'
        '      "source_id": 543,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "fb09b110498aaba6bd571b76109aec0f0cc60a7285e61c97a40811acdf04e64c",\n'
        '      "mask_sha256": "92efe31fa9a753b3c903c71516a6956fe28255740b893b1e05edc6630e87118f"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-57",\n'
        '      "prompt": "rice",\n'
        '      "class_id": 66,\n'
        '      "source_id": 57,\n'
        '      "source_row_group": 0,\n'
        '      "role": "train",\n'
        '      "image_sha256": "9d3d20a4d867344e85eee5b0e73cbcddabe6373462d23c012abe3f3a3a1b7284",\n'
        '      "mask_sha256": "c65574044bc64a9e7c444c568a87563bbc1485d30f11610faab6ad034c770f04"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-391",\n'
        '      "prompt": "sauce",\n'
        '      "class_id": 52,\n'
        '      "source_id": 391,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "00480f1f00285ea01ccca8ddac4ee94a5c277ebac65ea3ff6886a168cfd6881f",\n'
        '      "mask_sha256": "2a46a6c212bfdf2999d88bf12573a75e26a526dd7fc793e88b4c232e03323606"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-107",\n'
        '      "prompt": "noodles",\n'
        '      "class_id": 65,\n'
        '      "source_id": 107,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "38b188248f319ecc81dd9d2369f70914fcd56562063c7458c3868b9ed99f199f",\n'
        '      "mask_sha256": "f73c54cadd02e85e0fb7b8308df89334358406252eba3f6f50de280ad60b0d97"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-627",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 627,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "a3b319f882fc731bf9e2d778cad643bb00e8ab226a8604a8e3037f8035040fdf",\n'
        '      "mask_sha256": "308aa218c212a8f23f947192eb459364909809347c30c0343f94692f1cce5a8d"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-445",\n'
        '      "prompt": "pizza",\n'
        '      "class_id": 61,\n'
        '      "source_id": 445,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "36f5a5c950cbce314332632a36b85d050513dbf8242e93fe0f76be6c5e059b81",\n'
        '      "mask_sha256": "445929df9e3042460c427b4610cfc8a5d0203096c773368c3f6368be5805ee42"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-479",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 479,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "e4abc9e1bb32f24245ce4e608f19a8696cf17448f7a91f9174b3b88d991ba058",\n'
        '      "mask_sha256": "4bbd93d680143ad583563c27cd95db1a4a38bca2b246dd227ec3188d30291120"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-336",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 336,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "75d82ea4f15cc29994a9146279ccf9b7e4698c57f1fbff784b286f16abcaa6bc",\n'
        '      "mask_sha256": "864004d2184834c8c8b28e02e902079dfbb24e10d755d07e34fa12f24045bc7a"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-643",\n'
        '      "prompt": "carrot",\n'
        '      "class_id": 84,\n'
        '      "source_id": 643,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "2491ae5fd61d8d1c0c0dd0d78793c52b83000c252ff05142e4b1809fd86abf90",\n'
        '      "mask_sha256": "ab77ecf5ceb9239f9d0c33be4fa8a197cd0ab053f020c56ff7d9cab940d238f3"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-254",\n'
        '      "prompt": "cabbage",\n'
        '      "class_id": 91,\n'
        '      "source_id": 254,\n'
        '      "source_row_group": 2,\n'
        '      "role": "train",\n'
        '      "image_sha256": "1d7f78c570ea7238cf60e1c50e3e696035ca08faa7c5c20fbef4519be2a7c462",\n'
        '      "mask_sha256": "fd49b30feba898f494d781f4bfa2a7578f72c4cb06e4bb892a1323cfa832d09a"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-462",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 462,\n'
        '      "source_row_group": 4,\n'
        '      "role": "train",\n'
        '      "image_sha256": "4852d47426b13e7a5b63b10aa2029a3edc96e1c7faddc2663f890b07d4c1c282",\n'
        '      "mask_sha256": "ea6cfb5813fd7b2c81947e272fc5c2088c4f3e565af49d45d83f06811c32ffc7"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-672",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 672,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "d4cf3073652eb2e48611838a87c0291475b6a9a65f9884240d96c045b31cd1a6",\n'
        '      "mask_sha256": "f255c9e5052d361a9a38992e83c23aa1bd931406cff629b17a2dc46d95f16760"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-687",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 687,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "346ca2d78ac51a98fa1ab234afd380a17e6bbd04d2512c47e762c01bae437012",\n'
        '      "mask_sha256": "f1b54b5a01574bc2c3b5b8be79aad4552c6911239cfedd115eaab816bb0f274f"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-334",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 334,\n'
        '      "source_row_group": 3,\n'
        '      "role": "train",\n'
        '      "image_sha256": "0f9bf51e87eaab5457b89a5df72d84129c9ad422720d1c574d2b933df6ea11c9",\n'
        '      "mask_sha256": "1060e4c33fc91721d2c0c8820c984b16d4645c5c4bf02c0dd0e8ed13094fd5c1"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-662",\n'
        '      "prompt": "fish",\n'
        '      "class_id": 54,\n'
        '      "source_id": 662,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "6b39b2c967cc66416dc3c8911b0ac4c7075f4d32e96bf45645b8694c0e77e0d6",\n'
        '      "mask_sha256": "7a24711bcdfa9110f1d5c22deb211c2b920852f02d79af89188eedb7d1ff5514"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-150",\n'
        '      "prompt": "biscuit",\n'
        '      "class_id": 5,\n'
        '      "source_id": 150,\n'
        '      "source_row_group": 1,\n'
        '      "role": "train",\n'
        '      "image_sha256": "f792befbb077ea892929af8d43d969a492a056248d624fc68c4ec47415ddff7e",\n'
        '      "mask_sha256": "51a043bcf35130c62e1705b2d22c65ab91d353be598a55ef1a4c641089e538cc"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-623",\n'
        '      "prompt": "fried meat",\n'
        '      "class_id": 50,\n'
        '      "source_id": 623,\n'
        '      "source_row_group": 6,\n'
        '      "role": "train",\n'
        '      "image_sha256": "9123b7f7e5ed706e9949c5cfaec50f8edcb3f68c18a1529a247143538f394dac",\n'
        '      "mask_sha256": "08467287925587683328578c7a9ed1e55d61023d569e5f91ab10c0ca2a5f6a8e"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-528",\n'
        '      "prompt": "fried meat",\n'
        '      "class_id": 50,\n'
        '      "source_id": 528,\n'
        '      "source_row_group": 5,\n'
        '      "role": "train",\n'
        '      "image_sha256": "9f57b3722646159d0e0665fe3d4f14d242d83eea294bc3c0caa91c3135a1b117",\n'
        '      "mask_sha256": "cedbb99f28aa53ebd00c17492dc835133055483264f46e7c0bdfa51bd3a22090"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-794",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 794,\n'
        '      "source_row_group": 7,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "5f5ce516f279f11a1dcd75f6a985ccccb3ebd93161f274f5dc63dc7cafc4b16e",\n'
        '      "mask_sha256": "0b10f6f1153f0450838e926eaf69bae54f87126f628d79a39e324f61e2f28cf5"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-128",\n'
        '      "prompt": "shiitake",\n'
        '      "class_id": 98,\n'
        '      "source_id": 128,\n'
        '      "source_row_group": 1,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "15ca297a01d8186822d60c3a803b7f914f79907b64d6a91397c323f6bc864e62",\n'
        '      "mask_sha256": "04b8bce59179ab16a0870c3dfc6392b44c3c826e8e5258cb77e16617eab48378"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-492",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 492,\n'
        '      "source_row_group": 4,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "ac085f5e875c09526913c0dc3be6125b6d84d31bad5cbeac53204af020ec949f",\n'
        '      "mask_sha256": "15c8f484ee27b679214717ad6ce63efb2ad32008171e11bb061cb8537081ec09"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-119",\n'
        '      "prompt": "corn",\n'
        '      "class_id": 59,\n'
        '      "source_id": 119,\n'
        '      "source_row_group": 1,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "3b5eee3f8bcc56c032c24edc4f720f21bbe1ab99febe49b3525e2cf3c261e367",\n'
        '      "mask_sha256": "d40e95dc97e387d3585dd5e1fbacf4d7b5da765999b24aad1c60bf9d05828c8f"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-569",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 569,\n'
        '      "source_row_group": 5,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "1e1b83b97c80e1682f6d78814892bffbbb1704b8d086e05e32c322e773bbcf3b",\n'
        '      "mask_sha256": "0f45325f99a5ebbbd9e7ccdb839b2c6061ff81538b7f4e6f64e56c871470a7bb"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-198",\n'
        '      "prompt": "egg",\n'
        '      "class_id": 24,\n'
        '      "source_id": 198,\n'
        '      "source_row_group": 1,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "fa980a9dfe4c173721b8bf06c4b704f3079929720051a90b97d2286f66d07b89",\n'
        '      "mask_sha256": "bb073c2f117e2fb66cea8e63ddf54b01ffa993dc1be53afdb13694e5dc232872"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-113",\n'
        '      "prompt": "ice cream",\n'
        '      "class_id": 8,\n'
        '      "source_id": 113,\n'
        '      "source_row_group": 1,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "2e66ac744298cf8a5c2d5dec7920405f9d9ef8fac68a8780bd416333e2467aa7",\n'
        '      "mask_sha256": "e847459464798f1ebfd35f8651cd55949e6cfb570089867995e6c74305d09071"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-277",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 277,\n'
        '      "source_row_group": 2,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "4cd602fb65f654493d5283cb6115c5259ed1ec2f25373ec7a8b22ce0aacb4b66",\n'
        '      "mask_sha256": "dd6881aa4ec1f888716fd031e681c57a8c63b19e33a60ac4e0b6b7605936adc4"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-717",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 717,\n'
        '      "source_row_group": 7,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "0b89c7eecb003172b1d6c7fa2b3feb8ea80e01e0bb0120e102f0dce49dfee812",\n'
        '      "mask_sha256": "4507b4b332f846d1773859f2e1e2f5f4ab284297e037e6d597a40855842a3efa"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-645",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 645,\n'
        '      "source_row_group": 6,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "c5c0dee58f5575737d6fc75d44db650abe87230416a7483c9904e67580b6d5d3",\n'
        '      "mask_sha256": "ccf07b2bf87405b2fb7252a8a2d67aa4b72953ac8a77aaabb441d9060c2081d7"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-531",\n'
        '      "prompt": "fish",\n'
        '      "class_id": 54,\n'
        '      "source_id": 531,\n'
        '      "source_row_group": 5,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "e7d14f50c881558bde379cbf77d8119d4e14d2babf7999e10eb3b5f3def04ceb",\n'
        '      "mask_sha256": "a70bbcede391525bd3b1fd800006a78c13a8c5c945e10e4501d3aa89b83000f4"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-63",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 63,\n'
        '      "source_row_group": 0,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "676d9dfa027984cdfd661759e859933a679d4e1d047059aeb5a7f20889f58337",\n'
        '      "mask_sha256": "0dc2cb11636315bf6181f2322521293bb37fdcb78f3647a6c6d645d358bbdd2d"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-618",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 618,\n'
        '      "source_row_group": 6,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "aba421a74dbb5f5db8e0b89d31bc953ebd5dee7120a53fd70fbed3dd28e61fac",\n'
        '      "mask_sha256": "23e6bfacf746a50f4c883c84c86129dc502556cefc196dc7203f4ad97caec291"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-244",\n'
        '      "prompt": "french fries",\n'
        '      "class_id": 3,\n'
        '      "source_id": 244,\n'
        '      "source_row_group": 2,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "f29c0cd92d37f085f3f48cca622f8ffcfb392fb2ccbe3b5090940437347fd3db",\n'
        '      "mask_sha256": "64426efc5dcfbac77ff1466b66d60c7018dc6a24eabe63a9d0eab1d4eeb4a6a5"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-136",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 136,\n'
        '      "source_row_group": 1,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "c3db89e8375784d2639e1abff4bd7073f7fcacc1a676a241970c7198414a5f62",\n'
        '      "mask_sha256": "ed0932c42fde6ff59c95c9ce2d624a3592e30f606566e69834c52cbf569db272"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-752",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 752,\n'
        '      "source_row_group": 7,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "8e30afc88d3dd697b5c52de187977a0f6e2803e3a35348f22cc19d5508680c5e",\n'
        '      "mask_sha256": "2a44ac8b6d7d975c6ce0fb869f22aa18f831d8a40dfad3cd4ccf7b3f71d0c486"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-335",\n'
        '      "prompt": "potato",\n'
        '      "class_id": 70,\n'
        '      "source_id": 335,\n'
        '      "source_row_group": 3,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "88dee314bc8afa0a8f9fe41d6f56ce0cf7f6645f8245f6a511c84dc424bf9307",\n'
        '      "mask_sha256": "c838f889980d23757aaf17788a9852082860eadf4e11deb6370c846fb5a81ef4"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-231",\n'
        '      "prompt": "pie",\n'
        '      "class_id": 67,\n'
        '      "source_id": 231,\n'
        '      "source_row_group": 2,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "9bc60cdae9349f8f1458eca250e62ca66f85473cc489ac3d264f28bdc1c7f8b3",\n'
        '      "mask_sha256": "1d647f8bde8467bd03791bb4dd2a7fbd7b71b67d18cd50d540d70f5504d5fa6a"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-622",\n'
        '      "prompt": "tomato",\n'
        '      "class_id": 73,\n'
        '      "source_id": 622,\n'
        '      "source_row_group": 6,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "ad1202ff72a03e5cf51122cf9713d52fec1ebe6b374198ef356cafff9f615c29",\n'
        '      "mask_sha256": "882bc93eb963f519ce3a7c76b97ea08cf8630ac29e68f9ca6e7bca6f7e8b9f7d"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-750",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 750,\n'
        '      "source_row_group": 7,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "1011b0c2e0f064f04ace1f3e37c1fa85acfcec40359cac6d0ccb6b2d472d0551",\n'
        '      "mask_sha256": "198a1085bd5f6681493db024b428763017918b5c8da0ac014e5f6e56323ed563"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-594",\n'
        '      "prompt": "coffee",\n'
        '      "class_id": 13,\n'
        '      "source_id": 594,\n'
        '      "source_row_group": 5,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "25c969428b8b22ecce53a0ada55519697bc9b6df19961f467dbc7d1509e15fd5",\n'
        '      "mask_sha256": "05c95427764ef58e22538b9cb3e98839c49e193a27561777725d7e630d0a17e8"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-73",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 73,\n'
        '      "source_row_group": 0,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "9973d7f6b31b071820e8aa5fe9044372a6b8f50e129a50e640495dd59f5e3f7f",\n'
        '      "mask_sha256": "bc325c8010fa9d1ea0558a34d69ba62fe8c57b8c6a10460ce6d8bd4cd5018a7f"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-487",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 487,\n'
        '      "source_row_group": 4,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "49c7d5d426580b34f3243e0819364282bfe8800ac86d16a5f7ff763856b505e1",\n'
        '      "mask_sha256": "a0d847acd63bd0940da6b7c16b4951d5db4d4f07bd55891b6004944ce7731b81"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-530",\n'
        '      "prompt": "eggplant",\n'
        '      "class_id": 69,\n'
        '      "source_id": 530,\n'
        '      "source_row_group": 5,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "eee9908bf6e242c4d40ee57a56160b6def559cbe05e0ba8dda2e2b80b7ae45f9",\n'
        '      "mask_sha256": "f1cdc2ab9a0492ad1bef9896f889d22c4e464f0794c096bbfbfa53b559b79bb2"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-379",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 379,\n'
        '      "source_row_group": 3,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "467355419d896bd2257c88dc9a6cdd533e9ab9be080f886355a4fb31a0c6db7d",\n'
        '      "mask_sha256": "286c5e1feb178598f1de13c43d438fe8b4de3af508d58084a422ac997019597a"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-471",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 471,\n'
        '      "source_row_group": 4,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "e35379133063fdf387bd066e6f6885f9a9fffc402bae95141f79ed40db1f68fc",\n'
        '      "mask_sha256": "0402b968e51bef8523f187947f1f28f410ac67c1814d80cdb01578099e26c298"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-221",\n'
        '      "prompt": "potato",\n'
        '      "class_id": 70,\n'
        '      "source_id": 221,\n'
        '      "source_row_group": 2,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "b064e0d4d35ef5df4bdde1951f70ee08fccbfee627090ec026b5b8ee217c7d4b",\n'
        '      "mask_sha256": "3866d2f163c4992c87edaed1f55ee3f589491cd79e2a29b9d6f8972783a44762"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-552",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 552,\n'
        '      "source_row_group": 5,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "3d7087253cdae1cee908c515e3246a5f040d1e7fa073ef336e5929ce0d400199",\n'
        '      "mask_sha256": "61207d2eda2e2b9eb0283af907044b6de4291407b0655821a0d390922a6c0fef"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-430",\n'
        '      "prompt": "lamb",\n'
        '      "class_id": 51,\n'
        '      "source_id": 430,\n'
        '      "source_row_group": 4,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "ca51619a50d5b0e7d0d4818a5e8b48901d39525b7b1602d7657c7ce2676f09a8",\n'
        '      "mask_sha256": "b66920402e4a69e861a67bb6a449c8bbc67880b2d9ca08825bb78d7075e3116d"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-751",\n'
        '      "prompt": "cilantro mint",\n'
        '      "class_id": 89,\n'
        '      "source_id": 751,\n'
        '      "source_row_group": 7,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "e6d92bdb41c35c78aa56fb16967c90518415454d7442ed23dd43e98e11134e6f",\n'
        '      "mask_sha256": "dd5dabbfee4751fe2369b9efb503996a860065ec8b8651f1aa0ae9efc167f75b"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-474",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 474,\n'
        '      "source_row_group": 4,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "af59d01f1a6723643d14479dc4ca64631b372f4ba2eeba62df81e55c3d06767f",\n'
        '      "mask_sha256": "58587d8ce848ff638c519ffe7f4327e9b8aec3c0968b32b4ba6c023b5578aec9"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-629",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 629,\n'
        '      "source_row_group": 6,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "524964610e12d2849e027ebd4e437c11db18f27b8a017dad77f8472e4136b21a",\n'
        '      "mask_sha256": "5e3897f716a4511c3545d827f0b505117de105a04a928a701b3e968d6310fce1"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-567",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 567,\n'
        '      "source_row_group": 5,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "e1de5481fd7771e7188ea0bbf9db2a5d10592f3ab39fd1371cc9f12a6a2507fa",\n'
        '      "mask_sha256": "9fdaa328e24fa6de802a3e60f8d1d9eca326fece27ef19c4052fc4653a2d85a7"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-561",\n'
        '      "prompt": "potato",\n'
        '      "class_id": 70,\n'
        '      "source_id": 561,\n'
        '      "source_row_group": 5,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "a65a071751d3a7a713c01770bc48c8840c7d3f7e41723814ec7175685f5dd4ae",\n'
        '      "mask_sha256": "7cacbb588f8c270a4360e0fb4a52f2cb8b98b81259b99c8f475c6e6b9b6dca3e"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-315",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 315,\n'
        '      "source_row_group": 3,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "e0cead11dcb0eba74dc4c15ef36fe097d3449ef0226478a652ea9e9efe63384d",\n'
        '      "mask_sha256": "472ec6970f58c3c255ae2aefde3e38c368789f71c4094736801c539e75dd472d"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-102",\n'
        '      "prompt": "broccoli",\n'
        '      "class_id": 87,\n'
        '      "source_id": 102,\n'
        '      "source_row_group": 1,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "189caf45afef24c10267e61d77e6b5e44d9fff9087969561ae9cf93933c232df",\n'
        '      "mask_sha256": "e64a5f3ce121a4538812b6ab67ecd428ae544048bffad8d3366771a30f2305df"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-232",\n'
        '      "prompt": "pork",\n'
        '      "class_id": 47,\n'
        '      "source_id": 232,\n'
        '      "source_row_group": 2,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "660b961c49d32565ab40357ccea035c5b1d42e97aa3099e1ae58a2c0600a1659",\n'
        '      "mask_sha256": "66b44da41728516ef583031a509ea54893705d49974f09cfc21131e8d0a281f1"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-603",\n'
        '      "prompt": "rice",\n'
        '      "class_id": 66,\n'
        '      "source_id": 603,\n'
        '      "source_row_group": 6,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "e7d6cd0615095e8192aed107f1e567a1a8bcedcb58ab4dcdf6c5600e16dde34e",\n'
        '      "mask_sha256": "9dac8b238a030a0cc156d615a9c0b9b0fad7297ea8358969f62118c5d772eb12"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-343",\n'
        '      "prompt": "cabbage",\n'
        '      "class_id": 91,\n'
        '      "source_id": 343,\n'
        '      "source_row_group": 3,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "c3ef9f143e77f318d186a66ec291217bd8fb49e2bff7f8b98f6da8f875285058",\n'
        '      "mask_sha256": "abf090f907db00408b14704f01171f399b587c57603e0a14479fa2a58824e8f0"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-446",\n'
        '      "prompt": "ice cream",\n'
        '      "class_id": 8,\n'
        '      "source_id": 446,\n'
        '      "source_row_group": 4,\n'
        '      "role": "validation",\n'
        '      "image_sha256": "e73503d9a589bfca708654bd98fd4d8e640fd90f0b6ecbc2c7b38d3038c0d3e1",\n'
        '      "mask_sha256": "07a02b9ca3271150b519f2ede133038faa3e3752ccc2fa5c73dfb80089d457c2"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-580",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 580,\n'
        '      "source_row_group": 5,\n'
        '      "role": "test",\n'
        '      "image_sha256": "4afe29b6d3d15640f02b62ba1908cb98cfe2ac1b55dbc34c8eabb0203d72292e",\n'
        '      "mask_sha256": "6d6e2d5ab4089802f1d07833e326924559d3d6d53067deaabb90b5b69f19ea33"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-747",\n'
        '      "prompt": "white button mushroom",\n'
        '      "class_id": 101,\n'
        '      "source_id": 747,\n'
        '      "source_row_group": 7,\n'
        '      "role": "test",\n'
        '      "image_sha256": "672cc1f1181f77f62750f8343fc1d628b5f371eb0b2044719f3072097ad5ffd6",\n'
        '      "mask_sha256": "69ec672192fc207fa105ce4e734b337c773bdabd804da8dca42ab34e72abbca5"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-205",\n'
        '      "prompt": "potato",\n'
        '      "class_id": 70,\n'
        '      "source_id": 205,\n'
        '      "source_row_group": 2,\n'
        '      "role": "test",\n'
        '      "image_sha256": "3b80229d54f51a9566e01f57e7d3b092b10f4e17672b31fdd3d0dc432bd41f3f",\n'
        '      "mask_sha256": "7830176eb0cbf3bc51881b548aba98d6f798351f9b2f6b91bf8341265a6a829b"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-467",\n'
        '      "prompt": "ice cream",\n'
        '      "class_id": 8,\n'
        '      "source_id": 467,\n'
        '      "source_row_group": 4,\n'
        '      "role": "test",\n'
        '      "image_sha256": "77a2379a6e62f137d75f9528e180af503d8ceeec91c4d1753df8b4271adf6897",\n'
        '      "mask_sha256": "3facbea3e81bd055b7e28a30d18934db470e3d2d4f8a0183731087c8264df3d7"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-174",\n'
        '      "prompt": "egg",\n'
        '      "class_id": 24,\n'
        '      "source_id": 174,\n'
        '      "source_row_group": 1,\n'
        '      "role": "test",\n'
        '      "image_sha256": "ac42337fe0a97628c1810314a20ca15849b1b38f2a19817224662bd421a64326",\n'
        '      "mask_sha256": "e722b03190f71366870a7352857a758417fddcbdc7da9f97c2be1549875e9691"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-635",\n'
        '      "prompt": "broccoli",\n'
        '      "class_id": 87,\n'
        '      "source_id": 635,\n'
        '      "source_row_group": 6,\n'
        '      "role": "test",\n'
        '      "image_sha256": "b527e3d26f1e8aa8d1d8f1bd13adf3c3ccf8d87930edc8e74fe327fc078d7764",\n'
        '      "mask_sha256": "5e963361a4955adafba25955df7e892d4cd33b2e869c0556e52ce3afca0d130d"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-7",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 7,\n'
        '      "source_row_group": 0,\n'
        '      "role": "test",\n'
        '      "image_sha256": "3d752bacc7cea2891e001846fa6a2008430941a1eacdc757411d0524dba95747",\n'
        '      "mask_sha256": "5f9b10017272607632e81318127a5d9e18a7747591da2f5d7239d98fbee82caa"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-112",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 112,\n'
        '      "source_row_group": 1,\n'
        '      "role": "test",\n'
        '      "image_sha256": "4f6678faa6e125acd609cc267494b495b6740d5263c5da02c7b6af6264833205",\n'
        '      "mask_sha256": "f53281c6bdab93fed09a2bd3fcd570650606c18e586ed932a12347d01ce00ef9"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-550",\n'
        '      "prompt": "french fries",\n'
        '      "class_id": 3,\n'
        '      "source_id": 550,\n'
        '      "source_row_group": 5,\n'
        '      "role": "test",\n'
        '      "image_sha256": "8410a5cbb2217bf661c13106348c61517f6c8c15683fffc06b130ee738c213c6",\n'
        '      "mask_sha256": "7f4205598b5b63f119d672b8fd202994506cbcafd31f3d27b16a168bc53712f2"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-133",\n'
        '      "prompt": "tomato",\n'
        '      "class_id": 73,\n'
        '      "source_id": 133,\n'
        '      "source_row_group": 1,\n'
        '      "role": "test",\n'
        '      "image_sha256": "57413f112c7e9eb7ca323ebf3ccabecf59503639a904dcc3c79845e87ce4d56b",\n'
        '      "mask_sha256": "cface38977b45de70f52d8dad152800e21fb2f1efb502b4f5a1e436c26066c9f"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-311",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 311,\n'
        '      "source_row_group": 3,\n'
        '      "role": "test",\n'
        '      "image_sha256": "54412f5bf14b78ac973c2794ac47049e537dfb9322af71b5b8477b9f534c37f5",\n'
        '      "mask_sha256": "e71d5f87beb4a3ad40794c309d4ab927ac28f35fbdaf752f76a65160e6e43de0"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-374",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 374,\n'
        '      "source_row_group": 3,\n'
        '      "role": "test",\n'
        '      "image_sha256": "1a83d95bad9311bd7c8e35e223b986623a45be4556ae793040a4d743e1970303",\n'
        '      "mask_sha256": "1ee59c7a9e13396e64f5bf1693e249bb36d3b7c3bbe8442203ecdc88600094f1"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-441",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 441,\n'
        '      "source_row_group": 4,\n'
        '      "role": "test",\n'
        '      "image_sha256": "b4259e6083662dcedc72a4eb4c2b4444239b040e34b1e7c0784e6dde01a94201",\n'
        '      "mask_sha256": "b84cd12ef17a8008c9dbb19b022f7ac3e1c89019e6dd93c7d4df269ec88aaf36"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-362",\n'
        '      "prompt": "cilantro mint",\n'
        '      "class_id": 89,\n'
        '      "source_id": 362,\n'
        '      "source_row_group": 3,\n'
        '      "role": "test",\n'
        '      "image_sha256": "1afcf89e1cf6a5f4dc2d034f1789e0256e66e3a941426b464481b2945899d590",\n'
        '      "mask_sha256": "fe79d45decbb8b7ba252c6020febf98fc5d50e6cb38cd9afc56ee34e065f3b4a"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-573",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 573,\n'
        '      "source_row_group": 5,\n'
        '      "role": "test",\n'
        '      "image_sha256": "c60aabaee78f87cf53186dc7ef245948aef3cc101ba7022835e19335be664c1a",\n'
        '      "mask_sha256": "29ba78d6f8336b19e89a3ae07bebbf8c2e42ca0eb0320c5e029e246ed36c481a"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-222",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 222,\n'
        '      "source_row_group": 2,\n'
        '      "role": "test",\n'
        '      "image_sha256": "2190f75fcd9e2fcef4e631c8a62ecadf1a23f09e1c2e5e2bbbae5b40dfc4903b",\n'
        '      "mask_sha256": "943e421bd38e515bb04beae72f5e28f13a8cddea161dd7a4260c576693a4f0de"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-724",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 724,\n'
        '      "source_row_group": 7,\n'
        '      "role": "test",\n'
        '      "image_sha256": "03f5d28bee12bbb7e6215b861093604835fdaf75df0816b985147659313b67c3",\n'
        '      "mask_sha256": "984668f2e357dfad258748b6c94d44b5123ceb76be32ad4c7ce3daabe676a507"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-175",\n'
        '      "prompt": "tomato",\n'
        '      "class_id": 73,\n'
        '      "source_id": 175,\n'
        '      "source_row_group": 1,\n'
        '      "role": "test",\n'
        '      "image_sha256": "536d317c5ce461cd1defc306927076bd50c62cc1b74fff561035a2cc97942cc7",\n'
        '      "mask_sha256": "5799de5924fab2bd43a159dc18e89a77097c0d2e59d556b11fa92dd83976d5ea"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-619",\n'
        '      "prompt": "potato",\n'
        '      "class_id": 70,\n'
        '      "source_id": 619,\n'
        '      "source_row_group": 6,\n'
        '      "role": "test",\n'
        '      "image_sha256": "b8945e2156dd7cba36ae230c57a76b5558b54bb940b2582d9a414f905dde1724",\n'
        '      "mask_sha256": "148ddeaec1de77ff9e0c54c7a1e34074c04c40a60023e288c3da26233a451ac0"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-709",\n'
        '      "prompt": "fish",\n'
        '      "class_id": 54,\n'
        '      "source_id": 709,\n'
        '      "source_row_group": 7,\n'
        '      "role": "test",\n'
        '      "image_sha256": "81a9328b42c2383e6ee29ac9fe4eddc346859f443772e36101ccef0ecf110b98",\n'
        '      "mask_sha256": "436ecac0febdfa63e00c0914c09e40b58b1784842604d34847df6d0a9907e645"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-74",\n'
        '      "prompt": "cake",\n'
        '      "class_id": 10,\n'
        '      "source_id": 74,\n'
        '      "source_row_group": 0,\n'
        '      "role": "test",\n'
        '      "image_sha256": "ca2036454c0dffcd0654003fc00b81ff41976dc3d3c14e18ae2249ceb1c49341",\n'
        '      "mask_sha256": "4c37fc20951730ec9dc0d42918024247ac8704c5abae25cf6d902f42f2dec27f"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-773",\n'
        '      "prompt": "fish",\n'
        '      "class_id": 54,\n'
        '      "source_id": 773,\n'
        '      "source_row_group": 7,\n'
        '      "role": "test",\n'
        '      "image_sha256": "60e037e03fa8f76ae111cbdfb58fafbb74594487b999c0d60ec8b204a713b377",\n'
        '      "mask_sha256": "cc3910e1f62d423b574feb9b22ed24a2ec59e2dd67d75605e7ba4dd8f4dd1336"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-697",\n'
        '      "prompt": "pie",\n'
        '      "class_id": 67,\n'
        '      "source_id": 697,\n'
        '      "source_row_group": 6,\n'
        '      "role": "test",\n'
        '      "image_sha256": "b29d1c2ebabbd6f1cc3f58a58a08654be58f4465f07cc881c85c7546010893c9",\n'
        '      "mask_sha256": "cf883e79d8f1f3e6f46d1eed4e0967f58733cc2725a3007a8335de83933db289"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-268",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 268,\n'
        '      "source_row_group": 2,\n'
        '      "role": "test",\n'
        '      "image_sha256": "cb821485d4baa9a8bf71c745b277321772e1d2d7199145d623191fb1d5412fb6",\n'
        '      "mask_sha256": "7f704c5f5ff34d58fa71f2423905bb1f3e1a555e486994bf48ba24d43c769494"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-644",\n'
        '      "prompt": "pork",\n'
        '      "class_id": 47,\n'
        '      "source_id": 644,\n'
        '      "source_row_group": 6,\n'
        '      "role": "test",\n'
        '      "image_sha256": "d3fa527052bb9a126d9017192c3ee36f22efb3f828a7be64f31baabcad416dfa",\n'
        '      "mask_sha256": "48506aa73194d85d1a0f93f40d4a7b593c04363e42872dab85918139ab0006e2"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-533",\n'
        '      "prompt": "fish",\n'
        '      "class_id": 54,\n'
        '      "source_id": 533,\n'
        '      "source_row_group": 5,\n'
        '      "role": "test",\n'
        '      "image_sha256": "96147915e0574bed3e405abd74e375ca417f356451306075c4111f793a064d2d",\n'
        '      "mask_sha256": "371187e239f4d2d5b6a49fc2d1ca7868e90adb5e5b4eea0e8f942dc65e58386a"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-21",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 21,\n'
        '      "source_row_group": 0,\n'
        '      "role": "test",\n'
        '      "image_sha256": "124e7ef6061ccae152cf7116af02dbca48a8ffbe5bf32eed9ce51597b537f1e4",\n'
        '      "mask_sha256": "c138263770131a14e3d7966a18cb90401893a9deb7f682aa8526595ae8cab12b"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-505",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 505,\n'
        '      "source_row_group": 5,\n'
        '      "role": "test",\n'
        '      "image_sha256": "728144a22aeb123f34465af71cfc6e94faad1b4b39b909f28b9c30847d897bc2",\n'
        '      "mask_sha256": "3364f3083cd701c7e78cfc2d115fe2ddbe1942de97be830a0e94b3965927d822"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-730",\n'
        '      "prompt": "ice cream",\n'
        '      "class_id": 8,\n'
        '      "source_id": 730,\n'
        '      "source_row_group": 7,\n'
        '      "role": "test",\n'
        '      "image_sha256": "d769b793ba1f88433577774e96177dc87e67ef903056ecb7675c25ac8e87b515",\n'
        '      "mask_sha256": "10d4eb07665e43a949ea86aa754233499b2270c0c1c5bc243da62950546c55f2"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-694",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 694,\n'
        '      "source_row_group": 6,\n'
        '      "role": "test",\n'
        '      "image_sha256": "f9fe94c13e5db90534fcb9a6b3c1069fa8282f2ac8f86a78240fd1ebf1f0f1ee",\n'
        '      "mask_sha256": "06dca6937d1ba9b11912780b5f0465cb450276451ace3e2c319f95c7d54691e4"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-527",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 527,\n'
        '      "source_row_group": 5,\n'
        '      "role": "test",\n'
        '      "image_sha256": "cd8718f8410c53ee4f7fd387d8d62c9c304367edcad1fc00e753ef161c717d31",\n'
        '      "mask_sha256": "fd3443eb62c40f570f5e8a275c862c1597fe1ce51288574b58fd49ce17fed0b0"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-452",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 452,\n'
        '      "source_row_group": 4,\n'
        '      "role": "test",\n'
        '      "image_sha256": "2b336197ffcc2cd7b975a1e067dadba4df94ad44fde7274a332971a3c6b7c40a",\n'
        '      "mask_sha256": "4bd8cacb17f1967d45d86fa1ce5f20e90c5b9e3f923c8e337056aef57685afb4"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-329",\n'
        '      "prompt": "fish",\n'
        '      "class_id": 54,\n'
        '      "source_id": 329,\n'
        '      "source_row_group": 3,\n'
        '      "role": "test",\n'
        '      "image_sha256": "fc379e5f95ae647b613a87048e3f24f8a41aeb3abe3f1faec031bd71a9e4c2da",\n'
        '      "mask_sha256": "337fd5d0b64a003bf2551cfa51191e59ec3bb3e3a6ca0b8eb359ea8ccc54f663"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-461",\n'
        '      "prompt": "bread",\n'
        '      "class_id": 58,\n'
        '      "source_id": 461,\n'
        '      "source_row_group": 4,\n'
        '      "role": "test",\n'
        '      "image_sha256": "d2376e86e9d4cecc845b636c556d8ab22b3e1dc3d1f8de32f90c2429d6e3579b",\n'
        '      "mask_sha256": "4718d4cdb7ff9f4a503c7b88d3f64d5f08cfb7d9b10c85493a89a33439e8faad"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-27",\n'
        '      "prompt": "fried meat",\n'
        '      "class_id": 50,\n'
        '      "source_id": 27,\n'
        '      "source_row_group": 0,\n'
        '      "role": "test",\n'
        '      "image_sha256": "77f052b99f449a0869024d3085a3d49a30c90f0db0bc183cb88f2c94f9ad2428",\n'
        '      "mask_sha256": "5171d9f0f8c7b729871a76c96900240b535f6023958878e167992cc95eab2cb2"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-134",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 134,\n'
        '      "source_row_group": 1,\n'
        '      "role": "test",\n'
        '      "image_sha256": "cf71dd486e19064eddef4d5d068d6fa6743be4f1a8974748d052425f5270cc01",\n'
        '      "mask_sha256": "0b7fc47fcf8c70a55609fd909839cc41d6b067eac8ad36a0fa5f89ec3f7a3439"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-399",\n'
        '      "prompt": "soup",\n'
        '      "class_id": 57,\n'
        '      "source_id": 399,\n'
        '      "source_row_group": 3,\n'
        '      "role": "test",\n'
        '      "image_sha256": "6803dfadc1fc155e718991a9a06dec785dae04870cdc9506aba2b4844827d24a",\n'
        '      "mask_sha256": "4ea73256718212470031c2a684868b53746f59f5604bbe325ff75225e5e6a620"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-224",\n'
        '      "prompt": "milkshake",\n'
        '      "class_id": 12,\n'
        '      "source_id": 224,\n'
        '      "source_row_group": 2,\n'
        '      "role": "test",\n'
        '      "image_sha256": "95e212b99552fe140ab3727ef4f7327ceb1e5320b8137ad4f6920c300aeabf33",\n'
        '      "mask_sha256": "a7f5eabea6b321129b5c4f47e828beba78f437c0d04a8972b1bab9600c174bcc"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-436",\n'
        '      "prompt": "steak",\n'
        '      "class_id": 46,\n'
        '      "source_id": 436,\n'
        '      "source_row_group": 4,\n'
        '      "role": "test",\n'
        '      "image_sha256": "7eef5e351aded155b9d00dc7b31d43d54c45ef298117b6d00c8858eed41c6110",\n'
        '      "mask_sha256": "7717af26df9d2254bdffb80af878fd968759fb9e4ebc4604735eb0a6101e4ef4"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-214",\n'
        '      "prompt": "pork",\n'
        '      "class_id": 47,\n'
        '      "source_id": 214,\n'
        '      "source_row_group": 2,\n'
        '      "role": "test",\n'
        '      "image_sha256": "b3cb6351b41bb21c500c71eb49b02c07d3034d95f4eba0bffd86d95e85bc014a",\n'
        '      "mask_sha256": "9e6f60349929d181817be8a188f6273dcda8a58a9ae9ac782e52b393890ca932"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-264",\n'
        '      "prompt": "corn",\n'
        '      "class_id": 59,\n'
        '      "source_id": 264,\n'
        '      "source_row_group": 2,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "40d728c91ba8b515716079ec529b1ae013cec4ef23c515d150cb0371d1f4e765",\n'
        '      "mask_sha256": "a833a7b6d6e44b5b8fe56eedef1ba043d8cb4bbeb79ea01f5efb7b5a77d97402"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-322",\n'
        '      "prompt": "strawberry",\n'
        '      "class_id": 30,\n'
        '      "source_id": 322,\n'
        '      "source_row_group": 3,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "16a9df9163de10c02c8a0a58cf45fa5c3df96967e64c4d9d7fe3ffa197bb3711",\n'
        '      "mask_sha256": "45427f5a39e1dfcaff45480aee44f530eeace75e26cfc2dab9e9d42997b25bee"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-48",\n'
        '      "prompt": "potato",\n'
        '      "class_id": 70,\n'
        '      "source_id": 48,\n'
        '      "source_row_group": 0,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "31d0010386d6c2f1af43e9aa06b98ec6d8bcae56db6978109df9ab1c16d1ef5e",\n'
        '      "mask_sha256": "c4672584fb1af414a7e25c88100865bd2573208e7bf18d606af04ae79e60d6b0"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-272",\n'
        '      "prompt": "corn",\n'
        '      "class_id": 59,\n'
        '      "source_id": 272,\n'
        '      "source_row_group": 2,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "f60ffcc9f84d24dc655f202df5dafa2212409ccf78bc0924887b37eeac2ab267",\n'
        '      "mask_sha256": "1a17dcf8a5c35620321b7305522a4710e8f66609b0dce884a8a617ac1dd80ef2"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-78",\n'
        '      "prompt": "biscuit",\n'
        '      "class_id": 5,\n'
        '      "source_id": 78,\n'
        '      "source_row_group": 0,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "de4d2c07fd91c317af8543c9e9ed2ec6380b83a14c61896430f1e5f5b26704bb",\n'
        '      "mask_sha256": "28ee3e463bc898fe6424c66e95c74cd576ac67cc55db698cf9112dcac1686a54"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-235",\n'
        '      "prompt": "pie",\n'
        '      "class_id": 67,\n'
        '      "source_id": 235,\n'
        '      "source_row_group": 2,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "05a8e46b3753d37071ebe4b2a5af235f89ce2bf22ff6921e2f216091a8e3067e",\n'
        '      "mask_sha256": "e7a37e75a198ec3620206d5b122c6614f7ca88440a4bff27ac4ccd585ae398fc"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-647",\n'
        '      "prompt": "pie",\n'
        '      "class_id": 67,\n'
        '      "source_id": 647,\n'
        '      "source_row_group": 6,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "392d768de8ee3b4681b4d4ee3ba0bf005e5a3c3fb970ea177560561ccc02c8ec",\n'
        '      "mask_sha256": "1dc79e42a9c7086ab5757f994de87f4a1b66323328b94768a1b72e3f3f4dc093"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-382",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 382,\n'
        '      "source_row_group": 3,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "29c4d004d0c1604c8cf16d49f27e437d74a119a125643d31ac318a65e251baf9",\n'
        '      "mask_sha256": "68705c548e62bb9a80e88a5e4d5ee24a6a88ee5b4e0a054009bbcad257980c1f"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-426",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 426,\n'
        '      "source_row_group": 4,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "f85662c129edf10d0e4299ca45c83f36f7f4e3cc5f5109dfe4b69d0319a9b6d1",\n'
        '      "mask_sha256": "527b3799b3f92aaac9f1eec3430757d54d060c5dec22e7fa656de17d37c14407"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-685",\n'
        '      "prompt": "pork",\n'
        '      "class_id": 47,\n'
        '      "source_id": 685,\n'
        '      "source_row_group": 6,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "9eeffdf46da1b4b3bdbefc31d29a534d0bff83c3776c97cf89f53a2ed489756d",\n'
        '      "mask_sha256": "05c98b70866140188a60fd42f2d79bcd50f64af9f9bde3996afbca084bb6eaf5"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-302",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 302,\n'
        '      "source_row_group": 3,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "cc63081be625192ec60f2e5aaa7c16d0b1b77fb4e27f44431fa1b419adebee87",\n'
        '      "mask_sha256": "4cab3d9b7ede5635e5af1bc2d62b7fb2a12eeed7fc13206fb97a73e4e9a6b8d9"\n'
        '    },\n'
        '    {\n'
        '      "id": "foodseg103-val-8",\n'
        '      "prompt": "chicken duck",\n'
        '      "class_id": 48,\n'
        '      "source_id": 8,\n'
        '      "source_row_group": 0,\n'
        '      "role": "activity",\n'
        '      "image_sha256": "c2eafa052d8a5dd938a60cbffd7374523ea4d2531fc1945f4487dc5dd81d83e3",\n'
        '      "mask_sha256": "3dfff3f77f2f0c40e6f2d2b79c467c57df07c3e4b4339f8e8328884b1152f8af"\n'
        '    }\n'
        '  ]\n'
        '}\n'
    ),
    'requirements.txt': (
        '# This file was autogenerated by uv via the following command:\n'
        '#    uv pip compile tools/segmentation-workshop-requirements.in --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --output-file tools/segmentation-workshop-requirements.lock\n'
        'certifi==2026.7.22 \\\n'
        '    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \\\n'
        '    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55\n'
        '    # via requests\n'
        'charset-normalizer==3.5.1 \\\n'
        '    --hash=sha256:00668ebb0609751758682eb0b5857e7c35b9f00e84dfdef062e103244ec94d45 \\\n'
        '    --hash=sha256:012a22b88a77ca2e59b98ac5889b0deb604147666032f45e6d6e217634d2550d \\\n'
        '    --hash=sha256:01e93745f7f219b703b60ba7afead36cfc4242782be5af484673fc500df12da5 \\\n'
        '    --hash=sha256:04368edf83514385ffc3e1cfd4546e595f4f1272dd23ba437a93a9cc3741d47b \\\n'
        '    --hash=sha256:0722590aabf9dc6a6c0343d523c05458fa2b5047dbe6302fd526bb570600753f \\\n'
        '    --hash=sha256:07ffd07412fc5d5e84cd8952acf9ff7e4ed7a708e69d1bada19d8ba91711353f \\\n'
        '    --hash=sha256:09a7bba9f739468c8e78c36a75c33768e53cb1959fc638f510454c14683f00d5 \\\n'
        '    --hash=sha256:0b2b1b3fa5670c127b246df1d0c059defd41f689a868a3b9d79df9b1cac42d22 \\\n'
        '    --hash=sha256:0c6dfb5ca6723eeed15aa8e564a014d69fcb8812f94eef11fe3631e0508199f5 \\\n'
        '    --hash=sha256:0d929fc574b4d6fd9e7c0f5c2ede8716a41911923aa7fa5fce38e0818aa4a1ac \\\n'
        '    --hash=sha256:13e3afe97712e8887cd516e960c63f0b93122971e5b5e4b2622fe7701771e838 \\\n'
        '    --hash=sha256:15f024313246a4ed976c60f440bb8d257815513a681d212ff74fd46f7d715a90 \\\n'
        '    --hash=sha256:195ce897c6153c0700078142cf8efe3e6454ca4cf4357499e4078dfd83396626 \\\n'
        '    --hash=sha256:19a3dd5aa73cef1c99687c4fc57db016a9c17104ae1185da88ba566a5d3bebe4 \\\n'
        '    --hash=sha256:1d1c7a53a6c2103925cdd6d7229f8c567379f211c869793df679f2e9f738c369 \\\n'
        '    --hash=sha256:1f5883d77fd409a261abb5dc8ccbe335720d798b1de4abb3b1d47ccbbc76b53b \\\n'
        '    --hash=sha256:21b82d8082f6f5e7f456ef0bd16323d08de1266efbfeb476e64b2a91d1471a4e \\\n'
        '    --hash=sha256:252d099029bcbea642f2a06c4ed5046bdf8b5a8150b64afa5e027e88b106e5ee \\\n'
        '    --hash=sha256:256dd4d85d9e4dc595e2bc983c980e73f62ddeb3165c58b4c3dfe78c5c8548c1 \\\n'
        '    --hash=sha256:26422d45fd13551cf564c58932f7d72b4f58b93b0fcf18c35ba6be12b46bb102 \\\n'
        '    --hash=sha256:2679de311c7946dde5d3b6f44941844133ff5c7cb86099c0061ab1e8901c20a8 \\\n'
        '    --hash=sha256:29880d17a8eb0b5cfdfd8944b468322928059aa35f1f5fa8ff22b149ec0b42f8 \\\n'
        '    --hash=sha256:2bced4061f000f7187254a02ad3433ae17eaf991747ceea2f478422590a5bba9 \\\n'
        '    --hash=sha256:2e9cf9253119d8e5d111f05d71626786fd3d6193817316eab1ca088cdb8593cf \\\n'
        '    --hash=sha256:2f06b7eae9dbe77fe1d644ca244dad508de8d302870a43f3c559b521270938a0 \\\n'
        '    --hash=sha256:2f293479cce755c75f1697e87c409b7ae4c555c7dfecb6e988ad13abba943031 \\\n'
        '    --hash=sha256:329fc3ccb63ad22d867d84c2adea759a64079a37ba4a343433b02c7a2816871e \\\n'
        '    --hash=sha256:343fb4f2821043bd87095f7b08a1a181febc8e36ac64212143bbfd0a0e1bc235 \\\n'
        '    --hash=sha256:3588e376b3ea2eea84976f67273d679f229e24c66dce7b82ae45aef04ff6e072 \\\n'
        '    --hash=sha256:35aea775dc2bd5f54cd84a1cd2696cc3207c479cb9cf0bd346f0d343e4300ddb \\\n'
        '    --hash=sha256:35fe081843b35aad20ffeccec3eeffbe637b15d14f3fb22cc1b59cd8ec17e93c \\\n'
        '    --hash=sha256:36047af20e17097c3bb9476c2b7655f2f7aa51322c0ba58c07695bedf755a950 \\\n'
        '    --hash=sha256:3617ac3cfd8b9888f145ad89dd6e692285834b0201c6074a5eeaad3fd4d668c2 \\\n'
        '    --hash=sha256:366ec70f5547c640d3ce1985722490f23faf4eb5216a7eeba78277490e78dacb \\\n'
        '    --hash=sha256:394fea06235c8543390050ed5f529187074b029fb027213f6c46ac11ab5d950e \\\n'
        '    --hash=sha256:3d27167433c0d5f18dc850f07d0b3816221984fecdc405d6c157a6f0b8f8e9e6 \\\n'
        '    --hash=sha256:3e5e1224c0a6a90e05843e07adfec669edebec17801c67072f51e59561d63c0b \\\n'
        '    --hash=sha256:41876ee62a3dddf48ff1121ad8f0798032aa03f2fd35f21f34a4cab14f18d8d2 \\\n'
        '    --hash=sha256:433c5a81eade63b47e522303bad236f59dba55ea6951746f5558355eeed8c75d \\\n'
        '    --hash=sha256:4582c27e8c889d64811987b5967fbd3ae0c823fe1fd933b543d55ac20bb475fa \\\n'
        '    --hash=sha256:485a0d363cafefcd2538a73c7c838daa2035f09b2c9f9b5e3133f80c6aeb84c2 \\\n'
        '    --hash=sha256:494b70049a4d69aec6e8137c13af4cf8db8c9f9820a1392ac293b0dd2987a818 \\\n'
        '    --hash=sha256:496846868fea80e479324862fa877f02411f2fd0f83b79ccee2607aa68b2a032 \\\n'
        '    --hash=sha256:4abdc5f9ad448c1ecbfae2974b820535d6bc6e7eef63babbab3d81cf46968c71 \\\n'
        '    --hash=sha256:4b599739b93b2cbeded49645ae3c8d1405c29ddfbceac1545c87a3f9580a9e96 \\\n'
        '    --hash=sha256:4bea7f8ebe90bbd7f0e4a2de42ca6924ba23e3e76418c408ff82f1d46fabd687 \\\n'
        '    --hash=sha256:4c4fb141a727957c93edfe5c32a26ceb6b5f6461d67146e2d39f51e16170bea8 \\\n'
        '    --hash=sha256:4c9548dc78002099910abaebc0a72ac58b7d30931869e0351c09b507dff4ece3 \\\n'
        '    --hash=sha256:4d26f14f041e83dd8edfd61f4cd4fa7285d31798b5bf1f28e70c367ba6c41d61 \\\n'
        '    --hash=sha256:4f298bdadb8f0b9e5672877f647d1be9373ef5320c9e2f049795e26cad28b6a9 \\\n'
        '    --hash=sha256:52ec005752a56ae79547a05c0139ca2501a0c866390b6115008456b9f0e7cde1 \\\n'
        '    --hash=sha256:55261ac0d2941c42f196dd576f543d87a8ee03cd6f5e30dfb4d807b2e3b9121a \\\n'
        '    --hash=sha256:56490c595a28b1bb27dfc583e816152a9767721ef58b2c03b13f954d2f707420 \\\n'
        '    --hash=sha256:58d3e12c88e0950bca850ae1f7c256055c097639c2edb9eb123af9807d8b15e4 \\\n'
        '    --hash=sha256:58d4aa13a59c969dbfdf9e6a9560e242cbfd9e8a8f50c2747714df1a423adf65 \\\n'
        '    --hash=sha256:59171c6e45bf07d0d5cab3b0bf81d945035530f6873398b3b531c31184d46663 \\\n'
        '    --hash=sha256:5b6d1386bf0096d26d3a863dc0a487a5b4eb9aa93cf5ba69683d29dde6b9d60f \\\n'
        '    --hash=sha256:5c0ea61a470e070686aa30892fed79e297d2c8d0ab46b8bcdf027d38c51da591 \\\n'
        '    --hash=sha256:5c84bec0ab5ae0c64bfe73a7d2adcb5ce73b467523fc27fd6a28ab2aa6cbe35a \\\n'
        '    --hash=sha256:5ca0555312ae2fe82715cada7fac375530c2f3349e1eaa1bcb33d0283ac79a18 \\\n'
        '    --hash=sha256:5d8531a6569d025f68e2321e7638fb7978f23db58e5f69f56913837aae03816e \\\n'
        '    --hash=sha256:5e2d0e146dcb57034f8b97dc58d2d512cb90aba253960ce449f695fec6a82c6f \\\n'
        '    --hash=sha256:5fc45d653ea8c9a20479167e11d4a0f8cb2fa3470737ab6f9c827532313187b7 \\\n'
        '    --hash=sha256:6117b84ea48435e5356dc737f5121485c30920ba43375fa7b434fd753df0eac3 \\\n'
        '    --hash=sha256:6199d5606e2bbf2b096cf64d03f8b6790c91081d5ac866b8e7bb6422738cc60c \\\n'
        '    --hash=sha256:62b55f6722735a6c472f88361cde6640608773d9443cebdbb51abf436a1fcdd3 \\\n'
        '    --hash=sha256:687c9ca3035544b113bea2055e180af96fb63c0c476e22a9180f51925186e7b7 \\\n'
        '    --hash=sha256:6b7430cf5728e68f6c462254009a6ef4086e1bea43cf2f57aa9c55fb4f50ff96 \\\n'
        '    --hash=sha256:6ba32c4d2abf1d2fe7cf27d280f4cca5664233b0f885549c7761719eb977f486 \\\n'
        '    --hash=sha256:6c9cdde8becb25a7fde49924511aa2644d6f8081cc8df8e9452724303348d8e3 \\\n'
        '    --hash=sha256:6df0ec430f9a831772c23ca5a224cba36517a58a84bb32c32bb59a9fa67c47f6 \\\n'
        '    --hash=sha256:6e2912d4babbc65196ac13c2f53468dc57fb8b9c25ef913e8c59ddf7c6dc0e1b \\\n'
        '    --hash=sha256:6e5e4d73d588ca5ed09df1b7dcd1b203d1df3c542e3f50d126c947d432b10731 \\\n'
        '    --hash=sha256:70055ff39b97c99e7ae40ea3e393fb62aa2e44dbd9b29f8d14f42fb0025c3959 \\\n'
        '    --hash=sha256:706bfd38730a5ac7a365793269a00f4e988178cec121391f4248d84ad8c972e9 \\\n'
        '    --hash=sha256:7235dc28fc6dd9d832ac7c7bce95367dedb85929f17368a0c2bee1e080b9acbf \\\n'
        '    --hash=sha256:774d157f112367ff4abd29019f38f023c24e00e56edc7829c20e358a5a913ad8 \\\n'
        '    --hash=sha256:77efcff2b23071c349402ac1066667a3d011f62398d81408c9b88ad991747c9e \\\n'
        '    --hash=sha256:789b8982559ae28dad2356519f841655756cdcd96616410590ae0b17454ee64f \\\n'
        '    --hash=sha256:7ac76cf9afd34929d76eb7fcb63be476a4853d8a96f0dcf2d0db68a0cbdf9885 \\\n'
        '    --hash=sha256:7c0c10730342b0c9b35dd1d619beb8214e520bd96a1f870f452680b238aab3e0 \\\n'
        '    --hash=sha256:823f82903d189af463d7df250ef1f7f696f3cee08cc8d91deb565e8d425f6506 \\\n'
        '    --hash=sha256:838648accb3a7fd9803fd45c87bce8509648eb0c11bc34e216141300977244f2 \\\n'
        '    --hash=sha256:854066be00447fa8de2ccbbe893e2ffc4b123ef16d897af794c1e18bd4a714b0 \\\n'
        '    --hash=sha256:85d5855daafc240cc045c026d7a15fd198a09b0fc8ff6f5ecbb5297b509cb11e \\\n'
        '    --hash=sha256:85de3134b5379856e323ba37c19c9256d39425f7b76a63af52b09fb4664c2e8f \\\n'
        '    --hash=sha256:87e4f41d375c0b9be2fb5251aee4b8a689169e134535aed81bf085c3b647451e \\\n'
        '    --hash=sha256:88ca277405c2d3b71c4e1c2ee0e7966e807bcba86a69d11e19ba199d18ae4491 \\\n'
        '    --hash=sha256:88e85ab89cb822c1e635f51d6d32e488f94e002e70e2f492bdb8b945543f345a \\\n'
        '    --hash=sha256:8ac8c94b6539074e0f40899301273ac8402b9b3e01c7b7ba269ff30340aaaf20 \\\n'
        '    --hash=sha256:8fe532b3c966d1fb794e0698e4589d0444017ae77fc0b31edea13c0e35bcc449 \\\n'
        '    --hash=sha256:9085f87b0e38a2b92b8923059b4e8789fe40d9279712d15dcc670048d77079af \\\n'
        '    --hash=sha256:90b7481fb62fbe172c558bc6fd1c4c98d82004a54a7551f20e11ac9bf0b8708c \\\n'
        '    --hash=sha256:92caef967d287a407085d61176fce4012b1dd62daed4eb6d5ceb26d3d2538712 \\\n'
        '    --hash=sha256:9362dd90aa7dab48c0054a21187791ccf05473f7dba5d92b8033ae62164675e7 \\\n'
        '    --hash=sha256:94d78ecec2605a8d0398b0f365d5f12a63248438516f5dac536a5eff7337df4a \\\n'
        '    --hash=sha256:94fbf1c0c6cc0d3d5e50f9a9313a8cdca90dd696d34b381cd1704f8c9e939f20 \\\n'
        '    --hash=sha256:950f23cb393f85543777b0433f082cddd25b51ab398eac7971146495679efe5f \\\n'
        '    --hash=sha256:96eefc178f8636b9c760c5829345307fd81cfae9ab1e80997dbddeb0f54ee9a3 \\\n'
        '    --hash=sha256:96fef3e886d6a9874b14f27fc193fbdc69d5d8035783d86aa4e1cea594e695f9 \\\n'
        '    --hash=sha256:977cdbd483a9cff38179bea4fd754289a6f2195c7abd414aba85410b3e66cc5e \\\n'
        '    --hash=sha256:978eab16f55b4ab2c2a745be9a0a840bf8f09a7f227d9c76eb30214d078865a5 \\\n'
        '    --hash=sha256:994e883d17c559cdfd38c84003c8b27d25424a1077272a17e7cd27bfe0bf57b2 \\\n'
        '    --hash=sha256:9ac4444d8d4fd4c4bd08bf451ed3167aa9e7ec6cdb41b648794f1d1103652e36 \\\n'
        '    --hash=sha256:9b5db6052055d34d41230fb78d7c439c23dc536a9896f6cb039e8dd92cfc1263 \\\n'
        '    --hash=sha256:9d9a0dc7cbe9bec24c3f767c9122c41fe5a1bc43f47cd099d00d393e09769de4 \\\n'
        '    --hash=sha256:9dbdd9205662134957cf0c324f639bdc5031c0ca056e2369e238db75187c0f11 \\\n'
        '    --hash=sha256:9eea3ab2597a5e65fe65296e2d6a84570845a6b55532d90333d740d48bbc850a \\\n'
        '    --hash=sha256:a2028475ba855475b8b4d3cfeb4994269c967aea8b9892dfba907f4263a863a3 \\\n'
        '    --hash=sha256:a3a370082ce34d0612f421e15fe011c53bb1feff21a26d06ad4fb244dab5a375 \\\n'
        '    --hash=sha256:a545775cfe815855ea32d7c27731d79da358ef2055b4a25830231b1622dd18aa \\\n'
        '    --hash=sha256:a5cbd90ecf0fc62e64726917ad083b73001f0563657a87ec3c0b504e277dc90d \\\n'
        '    --hash=sha256:a6d095662e73e74f0a49988e0593373e243e3a52e27bfeea0a859e88acf4a0f5 \\\n'
        '    --hash=sha256:a6dac12ff6b846103483683f60c5f8fee205121adc58ffd87e90a90a3af69e99 \\\n'
        '    --hash=sha256:a951ad59cad9145664a730d3036b40b844e74d2d3683da40111463cd3a83845d \\\n'
        '    --hash=sha256:aa1099b956fb795e686d073568f6dc002a0bb89765ea6d5b055dd7d9bf1b116c \\\n'
        '    --hash=sha256:aa2bb0b37202dca27175591f761108b5d34096ade1191ffe4808bdf6b1571488 \\\n'
        '    --hash=sha256:aae2ee51122d3ae968a3837d97dc24a0aeebb0dea23694422cd172bd30017cd6 \\\n'
        '    --hash=sha256:ab743e9bc90c1f73552ec33e10e3331315acd2c397b36065b591b0181de533cc \\\n'
        '    --hash=sha256:ac00177c4831ffa650f8609e4bdddd5fe09c03b1c0c47acece7e6ea20421598b \\\n'
        '    --hash=sha256:ac13b004224fb341e1e25a1ed5e19d32f57cdb2a403e01f003b46f051a550f6f \\\n'
        '    --hash=sha256:acaf604462bf330b0d07e7a07c1d6e4adac79e5fb13e9c5140590542cafacc00 \\\n'
        '    --hash=sha256:ae31a1a1db2ee6cc2942fccaf695c934bc7f3db9f2133a3fef1f367cf1a4ab10 \\\n'
        '    --hash=sha256:ae4a097991662cd4fff0ddc74e0fe7874f82e00042fa0ea00855645ed0c79598 \\\n'
        '    --hash=sha256:aea996a6aba25260827c9ea511d1addfde2da9eb686ac961838509086188b7e6 \\\n'
        '    --hash=sha256:b39b69b347e5e47a3b5b8cfc005c68c1ba347474e3960236c4944a8ecd174962 \\\n'
        '    --hash=sha256:b54e7e13267d49ffbfe68e25b3cbd774dab38fa37238f71265e91b36146eb21c \\\n'
        '    --hash=sha256:b9af956078716df40d985fb0dfeb2c2120c5ca92ba4ff4b388acfd01cdc14d08 \\\n'
        '    --hash=sha256:ba2f37ee79e6338845261a3c5b1784e5d1acdff2c0785b284f1b633033d136ab \\\n'
        '    --hash=sha256:ba501e667c17d8411f98e67a022d9604ef179aff0e459b7e292c796837c13573 \\\n'
        '    --hash=sha256:baf3775a2635e5a11fbd5e4e64ee69c7e86875d224a5c72aca4c141064589a90 \\\n'
        '    --hash=sha256:bb57753e36e4855b8ca375069482250a6246372331a3e4f3407eaebb007443f5 \\\n'
        '    --hash=sha256:bd6c173f04743d483881bffa1478d5a4624475b8cd1d2194956a75548e191c18 \\\n'
        '    --hash=sha256:be47f99644b208bff7766314013f9acf57b056b04191d570d68ad14022cf5b1d \\\n'
        '    --hash=sha256:c010f5581d9c612804cc59fcf7b524b707fbcb72828551237ab545bb5c7034af \\\n'
        '    --hash=sha256:c1dcc36dcb96abc02236e182d17e0f71430152a6c2c7447421da2d2dc144edea \\\n'
        '    --hash=sha256:c428c6c31eb5f4277d7f8eccaf767fbd548ddd5ce3c8b4f4cbbfab3d96b5904c \\\n'
        '    --hash=sha256:c658c50ac0c98cd755a2dd50b7977d3bca7df401dcc47fbdfa87db53ef7d4e8b \\\n'
        '    --hash=sha256:c71fb0d56c920c269cd3e2e3fe7c610e3f1fdb21a6ce60efa6430ff63676cea6 \\\n'
        '    --hash=sha256:c7b742bf31c88566b4bb6335a7f393bb322e580b6bb98df7bd0c25e6e3519ce8 \\\n'
        '    --hash=sha256:cc0329df4caaceb950d2f580b5ac716a377f7059624a0bafaeaf8a218c6ed774 \\\n'
        '    --hash=sha256:cc5d36d96478aa9c60654bd932525bf32964c62a7281eafdf16d85003a8d6004 \\\n'
        '    --hash=sha256:ce854f5f478050ade5a238731c4ca985a7d3b3cb53ff600a9b5c3b689b5f0a7a \\\n'
        '    --hash=sha256:ced3fdd71aaa83ce593746c2edb42b7a59cb4c19c8b5c407781c72e493aae55a \\\n'
        '    --hash=sha256:cee5dd7c6fb5dd52a0fe2a740f9bc6e3593f5f8b1788bde49de02086f30182b2 \\\n'
        '    --hash=sha256:cfa1c0cc3a8f9f53f1243a5a99ac36fd003880199383b37672e86ddda9cb07e2 \\\n'
        '    --hash=sha256:d1ee1e296209fdce05b81b663250eefa02213a2da7b41bf26f7829b8ba3545aa \\\n'
        '    --hash=sha256:d59b75732e9b6f27388e10c14b0259cc5f2e48c78627d185e6a177b58ad3cffe \\\n'
        '    --hash=sha256:d63600d620ad0064c3a748b950ac5ea38a80190e5498532efefa4b7b3f1da1f3 \\\n'
        '    --hash=sha256:dd732602a7009217f658d5863d12d79d373a4de0eebc111094bcdd3bb8e0a6cc \\\n'
        '    --hash=sha256:e06efa066f7dbadbc84ebc126a97c452a6451dfcf589d89d788484949e1cf795 \\\n'
        '    --hash=sha256:e199fb99720074809a7720f1c0b4d919eea8b87e88713e0f8f602f7bef543d9d \\\n'
        '    --hash=sha256:e4b018dc5a0eee4676e38fe84a47a427816c590b93b55d9025274ec4d6ffc2dc \\\n'
        '    --hash=sha256:e6621fb2a4988d6e53eedc455e5903e2679f3967b8acb3d639f1b63c14a2e893 \\\n'
        '    --hash=sha256:e71c909f353863b2b89c83de2ebed71ea6d0df8a6ef65a128193c5e650766bef \\\n'
        '    --hash=sha256:e90251c0c7bdd54a100a0dce3c07b7e637278c93af29dbf78ebb89a58c4bac7d \\\n'
        '    --hash=sha256:e9fbdce1e47394b09bc9f26ab117dfc8d6491977a11d86f592bb42c779db2fda \\\n'
        '    --hash=sha256:eb12fb2ba69ffa05f8695f61c69e591dc4b4a12ac3757ac8af8adb259bf56d17 \\\n'
        '    --hash=sha256:eda059b6bc8bc0812d626fd91a7ce01bf583df0a61296eff390fd94141a34e30 \\\n'
        '    --hash=sha256:f03ac127268b43ef4fe9e6ab6794a6794b49485a0cc0c1db79876d2f33f75bc7 \\\n'
        '    --hash=sha256:f298e218441525d3794428b4c8b8fb8662c6d3ea79925d4807ee6b9a96a3bca5 \\\n'
        '    --hash=sha256:f5542f9b941279d82d41eb0aa9f98eba36fe4df5c7086c651df7944935b37182 \\\n'
        '    --hash=sha256:f6f7deae3feb4edfa2efaf7c574fe88cbf055038a6abdb40188e4fff66d5699f \\\n'
        '    --hash=sha256:f9b1e28d0e8dbfa858abdba91d6b547beaf2df1a59bec6da6faae7b96a4991a9 \\\n'
        '    --hash=sha256:f9f8405c2c758532c74fed975dbee57be1f31a6e865c031870c79a6ed3212ada \\\n'
        '    --hash=sha256:fa48b1b63d639f9483e0633e092f5851e2348c352f1f9bb6c8182f87884ef876 \\\n'
        '    --hash=sha256:fb78f6e7fcd8ad785d28cd577168bc1aaee827b25bb8755638f694794ea98f0a \\\n'
        '    --hash=sha256:fbc597639158fd7c14d55e808718848319540f51b0e6746e3eefa59723a4a348 \\\n'
        '    --hash=sha256:fce8cbd4997efeb450bd298b54f755dcdff18d496f7a5ddbb4867c6d7c88fdc3 \\\n'
        '    --hash=sha256:fd0350afdc3aabd5576f60ea109228bd5538139713c7b094c5cd27c73a98bc6f \\\n'
        '    --hash=sha256:fd0a274c0e5f9a21565cd9d3dd749b61f96b7aa1e20a93aa1ba4029518f2e5c0 \\\n'
        '    --hash=sha256:fdb8a068947befafba9952162645dc2fecaeb400e64584829ed5e9b2fbe21a7f\n'
        '    # via requests\n'
        'cuda-bindings==13.4.3 \\\n'
        '    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \\\n'
        '    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \\\n'
        '    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \\\n'
        '    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \\\n'
        '    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \\\n'
        '    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \\\n'
        '    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \\\n'
        '    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \\\n'
        '    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \\\n'
        '    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \\\n'
        '    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \\\n'
        '    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \\\n'
        '    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \\\n'
        '    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \\\n'
        '    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \\\n'
        '    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \\\n'
        '    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \\\n'
        '    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \\\n'
        '    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \\\n'
        '    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \\\n'
        '    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \\\n'
        '    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \\\n'
        '    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368\n'
        '    # via torch\n'
        'cuda-pathfinder==1.8.2 \\\n'
        '    --hash=sha256:4e65059febdb4d19d5cbc4798677e19db2b582f2f702f457b609e571690d357e\n'
        '    # via cuda-bindings\n'
        'cuda-toolkit==13.0.3.0 \\\n'
        '    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f\n'
        '    # via torch\n'
        'filelock==4.0.4 \\\n'
        '    --hash=sha256:0df72be195ca7892216d16f2edce8d9b93a571f02402972020a8cff84c594c7b \\\n'
        '    --hash=sha256:90999ed63a26ccf86b93b959ab10cf1017f422d816be454ed54cbed263e71ab5\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   torch\n'
        '    #   transformers\n'
        'fsspec==2026.9.0 \\\n'
        '    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \\\n'
        '    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   torch\n'
        'hf-xet==1.6.0 \\\n'
        '    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \\\n'
        '    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \\\n'
        '    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \\\n'
        '    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \\\n'
        '    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \\\n'
        '    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \\\n'
        '    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \\\n'
        '    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \\\n'
        '    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \\\n'
        '    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \\\n'
        '    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \\\n'
        '    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \\\n'
        '    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \\\n'
        '    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \\\n'
        '    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \\\n'
        '    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \\\n'
        '    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b\n'
        '    # via huggingface-hub\n'
        'huggingface-hub==0.36.2 \\\n'
        '    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \\\n'
        '    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270\n'
        '    # via\n'
        '    #   -r tools/segmentation-workshop-requirements.in\n'
        '    #   tokenizers\n'
        '    #   transformers\n'
        'idna==3.20 \\\n'
        '    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n'
        '    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n'
        '    # via requests\n'
        'jinja2==3.1.6 \\\n'
        '    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n'
        '    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\n'
        '    # via torch\n'
        'markupsafe==3.0.3 \\\n'
        '    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \\\n'
        '    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \\\n'
        '    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \\\n'
        '    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \\\n'
        '    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \\\n'
        '    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \\\n'
        '    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \\\n'
        '    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \\\n'
        '    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \\\n'
        '    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \\\n'
        '    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \\\n'
        '    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \\\n'
        '    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \\\n'
        '    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \\\n'
        '    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \\\n'
        '    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \\\n'
        '    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \\\n'
        '    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \\\n'
        '    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \\\n'
        '    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \\\n'
        '    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \\\n'
        '    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \\\n'
        '    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \\\n'
        '    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \\\n'
        '    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \\\n'
        '    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \\\n'
        '    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \\\n'
        '    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \\\n'
        '    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \\\n'
        '    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \\\n'
        '    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \\\n'
        '    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \\\n'
        '    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \\\n'
        '    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \\\n'
        '    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \\\n'
        '    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \\\n'
        '    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \\\n'
        '    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \\\n'
        '    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \\\n'
        '    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \\\n'
        '    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \\\n'
        '    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \\\n'
        '    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \\\n'
        '    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \\\n'
        '    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \\\n'
        '    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \\\n'
        '    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \\\n'
        '    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \\\n'
        '    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \\\n'
        '    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \\\n'
        '    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \\\n'
        '    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \\\n'
        '    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \\\n'
        '    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \\\n'
        '    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \\\n'
        '    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \\\n'
        '    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \\\n'
        '    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \\\n'
        '    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \\\n'
        '    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \\\n'
        '    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \\\n'
        '    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \\\n'
        '    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \\\n'
        '    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \\\n'
        '    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \\\n'
        '    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \\\n'
        '    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \\\n'
        '    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \\\n'
        '    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \\\n'
        '    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \\\n'
        '    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \\\n'
        '    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \\\n'
        '    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \\\n'
        '    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \\\n'
        '    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \\\n'
        '    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \\\n'
        '    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \\\n'
        '    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \\\n'
        '    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \\\n'
        '    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \\\n'
        '    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \\\n'
        '    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \\\n'
        '    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \\\n'
        '    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \\\n'
        '    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \\\n'
        '    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \\\n'
        '    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \\\n'
        '    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \\\n'
        '    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50\n'
        '    # via jinja2\n'
        'mpmath==1.3.0 \\\n'
        '    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n'
        '    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n'
        '    # via sympy\n'
        'networkx==3.7 \\\n'
        '    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \\\n'
        '    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a\n'
        '    # via torch\n'
        'numpy==2.5.3 \\\n'
        '    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \\\n'
        '    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \\\n'
        '    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \\\n'
        '    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \\\n'
        '    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \\\n'
        '    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \\\n'
        '    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \\\n'
        '    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \\\n'
        '    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \\\n'
        '    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \\\n'
        '    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \\\n'
        '    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \\\n'
        '    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \\\n'
        '    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \\\n'
        '    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \\\n'
        '    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \\\n'
        '    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \\\n'
        '    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \\\n'
        '    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \\\n'
        '    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \\\n'
        '    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \\\n'
        '    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \\\n'
        '    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \\\n'
        '    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \\\n'
        '    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \\\n'
        '    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \\\n'
        '    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \\\n'
        '    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \\\n'
        '    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \\\n'
        '    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \\\n'
        '    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \\\n'
        '    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \\\n'
        '    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \\\n'
        '    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \\\n'
        '    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \\\n'
        '    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \\\n'
        '    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \\\n'
        '    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \\\n'
        '    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \\\n'
        '    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \\\n'
        '    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \\\n'
        '    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \\\n'
        '    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \\\n'
        '    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \\\n'
        '    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \\\n'
        '    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \\\n'
        '    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \\\n'
        '    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \\\n'
        '    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \\\n'
        '    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \\\n'
        '    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \\\n'
        '    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \\\n'
        '    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \\\n'
        '    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \\\n'
        '    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \\\n'
        '    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \\\n'
        '    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \\\n'
        '    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \\\n'
        '    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \\\n'
        '    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \\\n'
        '    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \\\n'
        '    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \\\n'
        '    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \\\n'
        '    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \\\n'
        '    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \\\n'
        '    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab\n'
        '    # via\n'
        '    #   -r tools/segmentation-workshop-requirements.in\n'
        '    #   scipy\n'
        '    #   torchvision\n'
        '    #   transformers\n'
        'nvidia-cublas==13.1.1.3 \\\n'
        '    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \\\n'
        '    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \\\n'
        '    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5\n'
        '    # via\n'
        '    #   cuda-toolkit\n'
        '    #   nvidia-cudnn-cu13\n'
        '    #   nvidia-cusolver\n'
        'nvidia-cuda-cupti==13.0.85 \\\n'
        '    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \\\n'
        '    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \\\n'
        '    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151\n'
        '    # via cuda-toolkit\n'
        'nvidia-cuda-nvrtc==13.0.88 \\\n'
        '    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \\\n'
        '    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \\\n'
        '    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b\n'
        '    # via\n'
        '    #   cuda-toolkit\n'
        '    #   nvidia-cublas\n'
        'nvidia-cuda-runtime==13.0.96 \\\n'
        '    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \\\n'
        '    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \\\n'
        '    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492\n'
        '    # via cuda-toolkit\n'
        'nvidia-cudnn-cu13==9.24.0.43 \\\n'
        '    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \\\n'
        '    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \\\n'
        '    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f\n'
        '    # via torch\n'
        'nvidia-cufft==12.0.0.61 \\\n'
        '    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \\\n'
        '    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \\\n'
        '    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3\n'
        '    # via cuda-toolkit\n'
        'nvidia-cufile==1.15.1.6 \\\n'
        '    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \\\n'
        '    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1\n'
        '    # via cuda-toolkit\n'
        'nvidia-curand==10.4.0.35 \\\n'
        '    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \\\n'
        '    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \\\n'
        '    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f\n'
        '    # via cuda-toolkit\n'
        'nvidia-cusolver==12.0.4.66 \\\n'
        '    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \\\n'
        '    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \\\n'
        '    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65\n'
        '    # via cuda-toolkit\n'
        'nvidia-cusparse==12.6.3.3 \\\n'
        '    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \\\n'
        '    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \\\n'
        '    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79\n'
        '    # via\n'
        '    #   cuda-toolkit\n'
        '    #   nvidia-cusolver\n'
        'nvidia-cusparselt-cu13==0.8.1 \\\n'
        '    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \\\n'
        '    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \\\n'
        '    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215\n'
        '    # via torch\n'
        'nvidia-nccl-cu13==2.30.7 \\\n'
        '    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \\\n'
        '    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50\n'
        '    # via torch\n'
        'nvidia-nvjitlink==13.4.92 \\\n'
        '    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \\\n'
        '    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \\\n'
        '    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \\\n'
        '    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323\n'
        '    # via\n'
        '    #   cuda-toolkit\n'
        '    #   nvidia-cufft\n'
        '    #   nvidia-cusolver\n'
        '    #   nvidia-cusparse\n'
        'nvidia-nvshmem-cu13==3.4.5 \\\n'
        '    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \\\n'
        '    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9\n'
        '    # via torch\n'
        'nvidia-nvtx==13.0.85 \\\n'
        '    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \\\n'
        '    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \\\n'
        '    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519\n'
        '    # via cuda-toolkit\n'
        'packaging==26.3 \\\n'
        '    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n'
        '    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'pillow==11.3.0 \\\n'
        '    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \\\n'
        '    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \\\n'
        '    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \\\n'
        '    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \\\n'
        '    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \\\n'
        '    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \\\n'
        '    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \\\n'
        '    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \\\n'
        '    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \\\n'
        '    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \\\n'
        '    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \\\n'
        '    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \\\n'
        '    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \\\n'
        '    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \\\n'
        '    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \\\n'
        '    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \\\n'
        '    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \\\n'
        '    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \\\n'
        '    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \\\n'
        '    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \\\n'
        '    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \\\n'
        '    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \\\n'
        '    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \\\n'
        '    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \\\n'
        '    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \\\n'
        '    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \\\n'
        '    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \\\n'
        '    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \\\n'
        '    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \\\n'
        '    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \\\n'
        '    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \\\n'
        '    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \\\n'
        '    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \\\n'
        '    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \\\n'
        '    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \\\n'
        '    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \\\n'
        '    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \\\n'
        '    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \\\n'
        '    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \\\n'
        '    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \\\n'
        '    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \\\n'
        '    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \\\n'
        '    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \\\n'
        '    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \\\n'
        '    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \\\n'
        '    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \\\n'
        '    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \\\n'
        '    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \\\n'
        '    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \\\n'
        '    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \\\n'
        '    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \\\n'
        '    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \\\n'
        '    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \\\n'
        '    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \\\n'
        '    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \\\n'
        '    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \\\n'
        '    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \\\n'
        '    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \\\n'
        '    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \\\n'
        '    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \\\n'
        '    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \\\n'
        '    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \\\n'
        '    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \\\n'
        '    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \\\n'
        '    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \\\n'
        '    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \\\n'
        '    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \\\n'
        '    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \\\n'
        '    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \\\n'
        '    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \\\n'
        '    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \\\n'
        '    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \\\n'
        '    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \\\n'
        '    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \\\n'
        '    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \\\n'
        '    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \\\n'
        '    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \\\n'
        '    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \\\n'
        '    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \\\n'
        '    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \\\n'
        '    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \\\n'
        '    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \\\n'
        '    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \\\n'
        '    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \\\n'
        '    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \\\n'
        '    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \\\n'
        '    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \\\n'
        '    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \\\n'
        '    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \\\n'
        '    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \\\n'
        '    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \\\n'
        '    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \\\n'
        '    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \\\n'
        '    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \\\n'
        '    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \\\n'
        '    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \\\n'
        '    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \\\n'
        '    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \\\n'
        '    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \\\n'
        '    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \\\n'
        '    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \\\n'
        '    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \\\n'
        '    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \\\n'
        '    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \\\n'
        '    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \\\n'
        '    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3\n'
        '    # via\n'
        '    #   -r tools/segmentation-workshop-requirements.in\n'
        '    #   torchvision\n'
        'pyarrow==25.0.1 \\\n'
        '    --hash=sha256:0b1edbb2f385a6a65e9711b62ba86ac54a7816a3f8d17bb3e8a5929d65fb2485 \\\n'
        '    --hash=sha256:0b726ad7e7b669be982b0c71c07fe4b037d654354130da79a7902a669e93a66b \\\n'
        '    --hash=sha256:0befcf816e45a1af33ac775a9970b749e4868a230c7372f0ae5e932bee27039f \\\n'
        '    --hash=sha256:0fe7c8b6c03969b49c8c66182e4a18e3819ab92d07cfab5d8370c531b9369ef0 \\\n'
        '    --hash=sha256:119297a6dc197e45d9c6d4415f7814a67ffa36c180d26f68c154c58067ae782d \\\n'
        '    --hash=sha256:169d3429d5be7c752125890620f75a60776d38b0035eddae939651640822332e \\\n'
        '    --hash=sha256:25f8720bf6387d5dc2ebd2622112de630760419e4b66134405dd24110d15f37e \\\n'
        '    --hash=sha256:31e49a7888fcdf3a835da33ae777f6bb9a866334e5a789282fc26dcf426f7f15 \\\n'
        '    --hash=sha256:35935cd5de130aa5cf4dea052a63e6bf2e17006c35c3a468194242b9b2bf5956 \\\n'
        '    --hash=sha256:38a9a4b4b9613380e200641891495a56c3d5a98a092db4a870af9975e220471d \\\n'
        '    --hash=sha256:3f89685964f46e4216103c75483aac0c0692a5f72212d7ca835adba5ede56ce3 \\\n'
        '    --hash=sha256:4288f27577352d608ca08553b0865e4a9b3aa14820c5d95b53337218d609835b \\\n'
        '    --hash=sha256:4340f0ba6c1d2e13f21658de1d7c662ca2545018568d0030a1e9afca159d87e3 \\\n'
        '    --hash=sha256:44a9120ce5bd81936b8ab9a88076e3fd47c2c6838e0e43630fed83626aca81d9 \\\n'
        '    --hash=sha256:4facd65742a024a4a366328a1d2292062d72d6e023c1b7dda8d4c37544933a25 \\\n'
        '    --hash=sha256:51093dd9e10325fbdb3c10a2ae7c4806e5c822d94e74ae4938b26524a3323fee \\\n'
        '    --hash=sha256:514ddb60285631af068875550c90eddc181db3e8e63a032b1559be189e82f056 \\\n'
        '    --hash=sha256:5389cdf79447ed1515c9e31620e6e1e2302249564d603f2ad727d4f6d313e4c3 \\\n'
        '    --hash=sha256:59a2de54c0cbd954da861eee4d1d330f8e909c45b53455baef696380f2c55033 \\\n'
        '    --hash=sha256:60e89d8f13861a1f7f8d950fa54aebb8023b30734d0ac51ffa80beabe2df4bba \\\n'
        '    --hash=sha256:6109c94d8b9f3b17a041daca16cacb2f651ad8f1ef70a4232c2c0f37a23da2a8 \\\n'
        '    --hash=sha256:62cd0d785b8aa6675ee355f9fc02252a340f4441257c42674937826fd7594325 \\\n'
        '    --hash=sha256:6943e2fe7954d29d84de45d29d34c8dc36ce96570e67d89aa9976e650a4a9138 \\\n'
        '    --hash=sha256:6a1fdfc6659b6b19022f2e50627fb5cf7156a66c46bf4299379955cbe742382a \\\n'
        '    --hash=sha256:880523be3d29efcf83d3998835d206118ccf35e3871dbd2fb60408cf6b007a80 \\\n'
        '    --hash=sha256:8858d7bfc22e3f51529aeaa4077225029724623e4595dc9eff8c793935c34140 \\\n'
        '    --hash=sha256:9150a83248bfed9813ea3c3af74c3856c1984d444aa28e58bf7733b9750ddf6a \\\n'
        '    --hash=sha256:9171748cdf796972d85a4b60157c279913e242992e350c90c7450182a9838b2a \\\n'
        '    --hash=sha256:a4d6d5e9a3d1879a97c08ded0c797579b7965eafd0f0c26c30b45ccc06db939b \\\n'
        '    --hash=sha256:a4dd8bf99a8fac133efc0ed6a92f5fddbe2adba0d0f6dd720e39ba9855cea85c \\\n'
        '    --hash=sha256:aa0559502e1cd6254d6814614085dd9c5a3dd0419362978a936a3f68a9e5c3df \\\n'
        '    --hash=sha256:b7a296aac7a71fa0886c08e155ddb6c636a50013f801f6178daafa0f9e726188 \\\n'
        '    --hash=sha256:bddd0c4f7630c2a3ddf6347c1bdaa79d97bcf6bd445f9e60c816b7d77c85a5ae \\\n'
        '    --hash=sha256:bf0b672390cdcb640d7288f96b826d71ff4e9abb254a86c89890baf51a29cee6 \\\n'
        '    --hash=sha256:c7c534ec03c358a76ea3e505e74c1b6aef290af90c444dfd092dbfe23e755b85 \\\n'
        '    --hash=sha256:cab40b1edfef0262e0e5251aa2c58d75630f24d06dd7794480243acc001a1d7d \\\n'
        '    --hash=sha256:cc4aa407fde9fc660be3939e49ea31f50f3e9fec17c0ec63159f7711edd3efc9 \\\n'
        '    --hash=sha256:d51592cb7561e87877c506113e7adbf1342ab579e6c21f0ef44b8ba41cb74c80 \\\n'
        '    --hash=sha256:dda9470024204d7bbf2042b47c6e8a0e47a3eeb8e34405882dfaea6577e0c153 \\\n'
        '    --hash=sha256:df961f2e7ae9cf496459259d798652c70625f6c080650d6952f8c04053c58ee9 \\\n'
        '    --hash=sha256:eb6203482ff3746a5632303a7279ae0b5a304c46985b49ed1378cb350ea6728d \\\n'
        '    --hash=sha256:f3831aaa25c67a99f99dc8b05873cb9d64560390372e2aa197ce9dd4a3f06a44 \\\n'
        '    --hash=sha256:f729cfdbd36fd99d543b67a914d2de044c84ebe45be8b34902b299b608c15c8f\n'
        '    # via -r tools/segmentation-workshop-requirements.in\n'
        'pyyaml==6.0.3 \\\n'
        '    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \\\n'
        '    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \\\n'
        '    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \\\n'
        '    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \\\n'
        '    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \\\n'
        '    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \\\n'
        '    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \\\n'
        '    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \\\n'
        '    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \\\n'
        '    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \\\n'
        '    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \\\n'
        '    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \\\n'
        '    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \\\n'
        '    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \\\n'
        '    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \\\n'
        '    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \\\n'
        '    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \\\n'
        '    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \\\n'
        '    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \\\n'
        '    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \\\n'
        '    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \\\n'
        '    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \\\n'
        '    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \\\n'
        '    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \\\n'
        '    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \\\n'
        '    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \\\n'
        '    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \\\n'
        '    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \\\n'
        '    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \\\n'
        '    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \\\n'
        '    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \\\n'
        '    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \\\n'
        '    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \\\n'
        '    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \\\n'
        '    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \\\n'
        '    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \\\n'
        '    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \\\n'
        '    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \\\n'
        '    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \\\n'
        '    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \\\n'
        '    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \\\n'
        '    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \\\n'
        '    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \\\n'
        '    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \\\n'
        '    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \\\n'
        '    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \\\n'
        '    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \\\n'
        '    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \\\n'
        '    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \\\n'
        '    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \\\n'
        '    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \\\n'
        '    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \\\n'
        '    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \\\n'
        '    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \\\n'
        '    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \\\n'
        '    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \\\n'
        '    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \\\n'
        '    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \\\n'
        '    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \\\n'
        '    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \\\n'
        '    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \\\n'
        '    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \\\n'
        '    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \\\n'
        '    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \\\n'
        '    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \\\n'
        '    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \\\n'
        '    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \\\n'
        '    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \\\n'
        '    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \\\n'
        '    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \\\n'
        '    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \\\n'
        '    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \\\n'
        '    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'regex==2026.9.10 \\\n'
        '    --hash=sha256:030fa9e23624e39b3b94e46b90a5abd1a1678eb2f58fcdd3fd6c27526bf91c7e \\\n'
        '    --hash=sha256:032da15431c890d376f53547f0a6219f4f4cd19f3e4f11bdc321453b5bd207e4 \\\n'
        '    --hash=sha256:044bd4639b6bb409ec9e5d8b7accd57e02b4c4a4e2eafde916f8ae8006b3e40b \\\n'
        '    --hash=sha256:048a89ee797db10160bd2bd519286577a6b43a100279bd4b7d8456a3d69c80a0 \\\n'
        '    --hash=sha256:05fb018cfe7144585fc83882405906ff84994a2d154afc2509ecc7752c51f864 \\\n'
        '    --hash=sha256:07b45ba5c94b8fcb30cb6c56a11f715c57533a3017964504322ea52690a27b72 \\\n'
        '    --hash=sha256:0aa7589394230e0f0a422ab6b90841ff12c87e855e7aaf75d192a54a5f124548 \\\n'
        '    --hash=sha256:0acee94b480dd853e39434aa9a575f95385b1b4b8fa3feae56db363ca5cad782 \\\n'
        '    --hash=sha256:0b9ba3b2765cdfe18f0f561a69f78a69701f2896654a81c711108d35d14e5099 \\\n'
        '    --hash=sha256:0c32480f3371b75068decaf9e5da72c224e953830dd71e36e06cf80e30ea39d8 \\\n'
        '    --hash=sha256:1270cdec69248592bbe38a0b263ed58d907b891bd2b93703e225c317e421bda1 \\\n'
        '    --hash=sha256:13c52fc377792675f604a207a2ae5958c080f6854f7698d40d9ff034d95b1e76 \\\n'
        '    --hash=sha256:14caa05ce39ec70437af5aac8814c50ee6628f4a90353871c059692f448a164f \\\n'
        '    --hash=sha256:1562aabd9d4eb09bd88a62ad97ed06800094b529ac43419e43020b9cefec79b0 \\\n'
        '    --hash=sha256:175cf49ce7a994c88b8f15e3cb17cdb66a48ebb2d36de736b8205033db950f89 \\\n'
        '    --hash=sha256:1aa309ab7ba89a62d6cf70dbd38d4176440bce3c7001ab86256704cf4c18c6eb \\\n'
        '    --hash=sha256:1ad10a135fa0b4e4a462a61d07c6654d7518cfdb5cb8da08f9ff7d61384af1fe \\\n'
        '    --hash=sha256:1b891f77554bff991804cee24b78b40789f7d5993a24c7907bc7025fd2a70c8d \\\n'
        '    --hash=sha256:1e321e2c84f0e52c457f5ea5944f796d6e8e09cb99738ea98dcc1bfe402a128d \\\n'
        '    --hash=sha256:1e954e246466d5a1a78f563ce8364b5d7cb19e7adb0ccdec8f9c9610083187bc \\\n'
        '    --hash=sha256:1f0a8b4928823bc8b217a1ab7bf3d90598909dec9a70fbbfe9a52cc4eca55990 \\\n'
        '    --hash=sha256:1fbc8314436353e097c050e11b01a6c11433579437ed0579730157676ef59e2f \\\n'
        '    --hash=sha256:20e8bfb07ad79a282f8b95b56fe67f9750b1b7f775724e4ba1f23cb296115ce4 \\\n'
        '    --hash=sha256:217e98ba5fc8908ed8ffd4ebac04753a0c831067cbfb495b9821b94cc61eaa76 \\\n'
        '    --hash=sha256:239620b0e0681669367c0e218c8eb2551d9f8fe3b9fccfc8d0003377804e8348 \\\n'
        '    --hash=sha256:23ac9a28180f274d7dd7651fa131ad5b02d343b75df4b040737f0356223895dd \\\n'
        '    --hash=sha256:2479171edccced52ef02b899558f88ab2c235fe05b93180fdcae1670aacd89e1 \\\n'
        '    --hash=sha256:24d12a625a37c89c2b09303402a06942f55f071b95a7916a49c17034c3d47cd5 \\\n'
        '    --hash=sha256:2dd9286093c71afc8f55ef035c5b9d2776641fd72c6535f1febc92d0b0be9666 \\\n'
        '    --hash=sha256:2e67f8843f0e4b931f1fa860bf3bbe4134b714c0155cc5c7c0d7ea450230aae0 \\\n'
        '    --hash=sha256:31e4df2b11d48f61d511019bc1ee9b477055f17c352b68fe72db7a98b14d603c \\\n'
        '    --hash=sha256:3264132d576847ab5f88bb83e7debe67854bf165b3ea613bd467312b6099536a \\\n'
        '    --hash=sha256:3540734dbe241ebb3b87d5713781f6749a3e4d45480f506aa5fb5cbb0c37d249 \\\n'
        '    --hash=sha256:35ba3bab0c45079735f55ac61526774de1d84bc4a0333cc554e1a4ab74913924 \\\n'
        '    --hash=sha256:3a66e40a1a20de96a2fee00ed67e11012b62d85b277688258677fd19997addb7 \\\n'
        '    --hash=sha256:3bdeed3318a8eb2bbadc9c56347e0ff651639e934a47e168d05a3b12929fd0e7 \\\n'
        '    --hash=sha256:3fb4ae8cf83ef4e9addd43b2da31a9f45be816a8036fae8af59c8998b72718e2 \\\n'
        '    --hash=sha256:4971776b4f2bd7fd9a83eceb2cb2592cbe2924f639fe8045e6a9de5ba4bfcf25 \\\n'
        '    --hash=sha256:4a761ea45f2ad74c575ef5850ea514cef97302a552d3c7c9d1a1a870d4661d6c \\\n'
        '    --hash=sha256:4c66d54042a14a503907d81861b8a5235e6d1f03d4fbc1d8767f652eaf957ac1 \\\n'
        '    --hash=sha256:4db7d00c4afbfbb55b8e17b1e371da11418ea9389b030acec63c1fa4c7ad4b86 \\\n'
        '    --hash=sha256:4f0407474ffac8e5e89d93ca41d60891e29f0ab8423eb66ff292d850a86a0843 \\\n'
        '    --hash=sha256:53e182b6b04d0011909b47d51a2d72d908de07c7b1c7f16b3adda2204d723bc1 \\\n'
        '    --hash=sha256:5847e22bbf959764d776937d791d034cc2d19b787e361c88d97e859e8dc68502 \\\n'
        '    --hash=sha256:58c01f7b81079cf0817ba831ff4d9eff5d28be4a3ac76c353e6f09bd63f4c386 \\\n'
        '    --hash=sha256:58da726d3e766c0b3f5a3997dfaf0275898a1107b8191cdd6b0437fe45fd817d \\\n'
        '    --hash=sha256:5bef622850cf760154719d4e0d74b0a855962432995168e250069899ae12fe8f \\\n'
        '    --hash=sha256:5ccd139b2061132e7b265cfb4b4721baeb9f8928b81415304abf1ec7e3181c26 \\\n'
        '    --hash=sha256:5cef9f3d14796500ea834c41dbe688f1f6b23c7024dc23e8a794d7ebaf5d71d0 \\\n'
        '    --hash=sha256:63bb62cf62217dc38c8a6b2b61b165b0e4eb8fa93b0aba12139251c0986a8fa3 \\\n'
        '    --hash=sha256:681ed38664b64c6617d3c3c332018d1948c77e139c5ea667c1886efa671e426f \\\n'
        '    --hash=sha256:6888065672b341e5246f391ec16dc258a29218ac784172fd67c30d941544755b \\\n'
        '    --hash=sha256:6aebdd9a946de328b3f6f61dbf48dd064a36eb6dddf96e34ae6651d37f6e9383 \\\n'
        '    --hash=sha256:6afcad14310f1311d077553ed374b42a5e538f85a8c884b4e38e52de091c8077 \\\n'
        '    --hash=sha256:6b34a778c695d24e77c140e3b4c95da69282e34f2f6b02b55656aa4a0379f643 \\\n'
        '    --hash=sha256:6fd555fc9abef50c530869690b2daca054c8811a7aff632d11f9a7b2590b2742 \\\n'
        '    --hash=sha256:71879292c9c7ac67b1680345b16daba1be937cb027362cfa04e68f65db2dcfdd \\\n'
        '    --hash=sha256:75242f44a3e283106077be4ab717bc535e4701c9d54ad69e195945c22f137a1d \\\n'
        '    --hash=sha256:75aa39d3f4f1650eea84e46b0d8cefe77dd5478c10e3d0aaf0b0f00493475a7a \\\n'
        '    --hash=sha256:75f9297b16fcb588a1f8d8a55dabef3c0c20b0c7bac43c87ceaaaf1a825c12f4 \\\n'
        '    --hash=sha256:79e9432995e14c749d34209413de5e621ec8e67789bf4f46dbfabea9d06a2406 \\\n'
        '    --hash=sha256:7abb38b8c40f3a235235a44da452c64b7b5c1d650ec6351027db0e090804f2e5 \\\n'
        '    --hash=sha256:7dcad477c49c4c626a6c4fcd71b39a971aa217060cc40a6569fd24edcc0fa509 \\\n'
        '    --hash=sha256:7e6c0b5ec6ddee4032247585dc491b0fa58627745b66a705728703a3f0331231 \\\n'
        '    --hash=sha256:7f8f10015866608fe4c043cec2e4fe4c39a94bb50e45091de4cdf4004b9ae4b0 \\\n'
        '    --hash=sha256:866de9f98df0611d7b62b3a8729d3284a64c0cc6edd90bb95a533e443a4939cb \\\n'
        '    --hash=sha256:87f5f75c109f08f5c602d68e1af54cead8165189c727b6ac946b30b9833a3ba4 \\\n'
        '    --hash=sha256:880ac684c27176464c00c3fdc456116364f5ebc70da07aad0c2d4a7ba45e98db \\\n'
        '    --hash=sha256:88b02aa8d0ec9b6189fe933d425775882271c23700ac11fd26d1779b0f56fde3 \\\n'
        '    --hash=sha256:8ba1f78bd4fef2d8f84b894ec28ac3481afe6cc07aaa253ad4717ef7b3fe6bcb \\\n'
        '    --hash=sha256:8c07021a4faa3f092869adbd1f35cdc7a592276c807aeebc3ceb8ff1a638f0b4 \\\n'
        '    --hash=sha256:8d5c4518235a2ec1611e57af85fa488d529c1106aacff12adadcedf8687012cd \\\n'
        '    --hash=sha256:8e127d9a80cbf1c3276bb465c6d047e8705e97b58c2b8f2f0c0a69c336b44b37 \\\n'
        '    --hash=sha256:94c5ce3bc41d226b4eb89ca3f842b2e28c031487fb1f34eb2153d98235831325 \\\n'
        '    --hash=sha256:94d096369b7cd96d15343fef5257fe39eff9d0e8758b92a0e15e358b92cdb2fc \\\n'
        '    --hash=sha256:968c1e33edd9a104d1bf24c8d476c72de7e3839ae7f894b37e9e4f4739fdeeca \\\n'
        '    --hash=sha256:990797e765d89a423880052c68b61c31afe701de94a8c060f61c40605ca6c727 \\\n'
        '    --hash=sha256:9ce239acb15843ab03976626af810a4424b0409689ec2bbc52088ab5479ab487 \\\n'
        '    --hash=sha256:9d772586951d7d6a5d162d48f414065e483b1c81ab38fd8ed97c78b05883421a \\\n'
        '    --hash=sha256:9fbd2e5d8002dc49a6129fb321ec51c57a025e752ed525ddce0ba9223c4350a7 \\\n'
        '    --hash=sha256:a41693eb3fc4b92e6127d113813c6c395237f7edd3224abf67609af48c690d11 \\\n'
        '    --hash=sha256:abbfc1c33bf8efddcc43844aba61e036d74a918680dc3ce8ce2538b004eda0f9 \\\n'
        '    --hash=sha256:b298cdc33c5cc6969ff07f0fba19cc73e0fd8576373c50935feadaca2f6b4405 \\\n'
        '    --hash=sha256:b43456de605c8ee77eb75f07bc1ee44ba27f9cee22207deb77d495e954b7d953 \\\n'
        '    --hash=sha256:b71649169a9fcf30b395ee01047fa7ad6654a4c900ca75b23c04dedcce6a1f8c \\\n'
        '    --hash=sha256:b91c37551bf39d75116c02b146956f65b9aa0337a4a652f4ae186983789d4001 \\\n'
        '    --hash=sha256:b9d36b03dc362aa40ffaaec9d9bd75e87763529563ec008c43b0e07782f5be7a \\\n'
        '    --hash=sha256:bafa41b0dd63669e5c0f8adf3d24819efeb73c847f492eb011212eb352e69041 \\\n'
        '    --hash=sha256:bb7774924f8cd69f49cba0b3c2d679a6326f777e0e67d130ad5203e4df53f0d3 \\\n'
        '    --hash=sha256:bf29611e5376fec8f795879bb5c6153a76c3a292573d173c26784042b01eb840 \\\n'
        '    --hash=sha256:c014641157e9049b0603b8daa5343bd408d9b757b709aaa0f373cd3fab2d7944 \\\n'
        '    --hash=sha256:c103b3b14e011774af4fb7e4617ad4d72b9171905cd3b231a70a4efd76e477d7 \\\n'
        '    --hash=sha256:c22df8dd6373bbe3898e77429ffc85594300e39d752fd0e68a31e59d37899376 \\\n'
        '    --hash=sha256:c25a754bb81a2edcfc3b65eda50f017d736f818112ed43e8aafd595cb00678ae \\\n'
        '    --hash=sha256:c32818b28bcd153b25b63038348a9fe9b9fbcddb60df43f204c3ab55eeb57f77 \\\n'
        '    --hash=sha256:c37fa93bf18bf4f90b01c0fa9f11ea567ee4b7dd8bf96e63663e5edc37aa38cf \\\n'
        '    --hash=sha256:c3d95d7d9538b5b726dd6fcd7b6117a71e6565202f6d64f5845fb4d8f203f533 \\\n'
        '    --hash=sha256:c8fbd9cb30c68c1686b94029b9ef845d5870d3d65baf66cb126b676849b9d72b \\\n'
        '    --hash=sha256:cb76a9c4e07a6a47849726af0ed14c41741a182f097f134a8cf29c1bc0f4dde8 \\\n'
        '    --hash=sha256:ce7c118cb102975f974585688357a717ffbf9dddd64ab0bb1bc93eb5b367cf95 \\\n'
        '    --hash=sha256:cf377960d2ac37d987394a9dbaa75e91338c41a46d41e1d25e90125e7b3ee2dc \\\n'
        '    --hash=sha256:d278ad30ec83b6b9202685b0f80b741a51ea3ca7f0595ebda96e7628b6398876 \\\n'
        '    --hash=sha256:d2d377fd1cad611b806cdd732d86b65f536c768209890cb442556548daa65a23 \\\n'
        '    --hash=sha256:d414c411c06fe0009eac33488fb1591c66b5c2673e342e452e7bb2fe63da8194 \\\n'
        '    --hash=sha256:d8c668af8f7bdb1d18739c27d30cd9f4b371495a883f75a002fb7a39d740fecd \\\n'
        '    --hash=sha256:dce932f8e3ba936475ea3d0d8b59f7b050a9e206e994f53f8fd80299871e87da \\\n'
        '    --hash=sha256:debc629e98b95abaea1cf3057ca296151f348c697c9b8a59d18013adb302c0dd \\\n'
        '    --hash=sha256:e0dc78251154b66dc60211563fc115345da332eaa881e4e2523fb1edae3772f4 \\\n'
        '    --hash=sha256:e5e4a6e0734a685d13b9685622bb503bdbb2927f8b0df025a5085f0ea067475b \\\n'
        '    --hash=sha256:e6b99181d184d0f5c7b36b8d12b94d1e9499cce6246594331f9edc5d2ea9fceb \\\n'
        '    --hash=sha256:e7327795089ddb44912dce1434e1d7244be2e9fb48fcc2d6782936af7a3062db \\\n'
        '    --hash=sha256:ebb2ba68e4641a994061f70bf44ed448fba0b9b1d18c94ffb9efc1cca805b39b \\\n'
        '    --hash=sha256:ec8855f08c17895a26fbf5f19ed829722e19b34a96629e49a43c92974924026b \\\n'
        '    --hash=sha256:ecb2e7acb18f8cc4a67f0ad986c0af291ea4dd385d0614ba9bc09d7f8bbb478c \\\n'
        '    --hash=sha256:ef4c0a9dfdc90581b90b1b95a8c3d1557f8ff8f5a2a53536d26314de699d1468 \\\n'
        '    --hash=sha256:ef4ce69ff97fbb44b46751cfea5e859ad0b66d1a50abf34954f0645f51e81671 \\\n'
        '    --hash=sha256:ef5a059ea1c6ee5d1c7e99a2484e628608d010921efe876c6f0e2029d2f35eca \\\n'
        '    --hash=sha256:f0e2e5d23448b660d60a6ed85c46cc03b4b48bd276b8f4041d4a5fe2a4a0626b \\\n'
        '    --hash=sha256:f2374c27deb189b282ec7e16106752c22ad39b056bbd8018960b1e4cc95d67a1 \\\n'
        '    --hash=sha256:f2f43bf4e47ff7ce9e585558706d698c6204d0f80bf2207766382ed817c8e9f4 \\\n'
        '    --hash=sha256:f5c629df03adec31ee505dda3c8988f106c9390e4cbd343600036eb8b3d6724f \\\n'
        '    --hash=sha256:f70b9f0e39c2dba1d9da6bf7ef7c377cad7277f8440e9a69be05ede529ff024c \\\n'
        '    --hash=sha256:f7d4656e17ab736e9415a6442a345bfc97bb8b7dcce47884bb74a37f70f08d0c \\\n'
        '    --hash=sha256:f8bdec659a8fa7af51a32b224b3b7c02bc415d54ffd35187b1d224176b17d607 \\\n'
        '    --hash=sha256:faa911fbbcf8ac90bda0e0657d60768e3390954ef0588211d63a22add1cb1cd1 \\\n'
        '    --hash=sha256:fbc4e2f3cb7ce8436154e6483079e7d35eeb321a952fa936e180300630d8b873 \\\n'
        '    --hash=sha256:fd6bd89b9fc06018d35851cab0240adb7dd84d51941b19f6574ac90cd54e3ae5 \\\n'
        '    --hash=sha256:ff4d7b14ea19e50c8d9d6d83f45bd9b45cbb624c07ac1fa54db0a019049abed7 \\\n'
        '    --hash=sha256:ff6b3267318661dfddf6b3628663e00e5946bd0a5c8fa678537a1401f0388f91 \\\n'
        '    --hash=sha256:ffc2da104e43db716ce30cef9f28049a1faa6aca385dd8771b033268d0730b07\n'
        '    # via transformers\n'
        'requests==2.34.2 \\\n'
        '    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n'
        '    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'safetensors==0.8.0 \\\n'
        '    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \\\n'
        '    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \\\n'
        '    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \\\n'
        '    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \\\n'
        '    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \\\n'
        '    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \\\n'
        '    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \\\n'
        '    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \\\n'
        '    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \\\n'
        '    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \\\n'
        '    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \\\n'
        '    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \\\n'
        '    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \\\n'
        '    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \\\n'
        '    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \\\n'
        '    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \\\n'
        '    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\n'
        '    # via\n'
        '    #   -r tools/segmentation-workshop-requirements.in\n'
        '    #   transformers\n'
        'scipy==1.17.1 \\\n'
        '    --hash=sha256:010f4333c96c9bb1a4516269e33cb5917b08ef2166d5556ca2fd9f082a9e6ea0 \\\n'
        '    --hash=sha256:02ae3b274fde71c5e92ac4d54bc06c42d80e399fec704383dcd99b301df37458 \\\n'
        '    --hash=sha256:08b900519463543aa604a06bec02461558a6e1cef8fdbb8098f77a48a83c8118 \\\n'
        '    --hash=sha256:131f5aaea57602008f9822e2115029b55d4b5f7c070287699fe45c661d051e39 \\\n'
        '    --hash=sha256:158dd96d2207e21c966063e1635b1063cd7787b627b6f07305315dd73d9c679e \\\n'
        '    --hash=sha256:1cc682cea2ae55524432f3cdff9e9a3be743d52a7443d0cba9017c23c87ae2f6 \\\n'
        '    --hash=sha256:1f95b894f13729334fb990162e911c9e5dc1ab390c58aa6cbecb389c5b5e28ec \\\n'
        '    --hash=sha256:200e1050faffacc162be6a486a984a0497866ec54149a01270adc8a59b7c7d21 \\\n'
        '    --hash=sha256:2040ad4d1795a0ae89bfc7e8429677f365d45aa9fd5e4587cf1ea737f927b4a1 \\\n'
        '    --hash=sha256:2b64ca7d4aee0102a97f3ba22124052b4bd2152522355073580bf4845e2550b6 \\\n'
        '    --hash=sha256:2ceb2d3e01c5f1d83c4189737a42d9cb2fc38a6eeed225e7515eef71ad301dce \\\n'
        '    --hash=sha256:35c3a56d2ef83efc372eaec584314bd0ef2e2f0d2adb21c55e6ad5b344c0dcb8 \\\n'
        '    --hash=sha256:37425bc9175607b0268f493d79a292c39f9d001a357bebb6b88fdfaff13f6448 \\\n'
        '    --hash=sha256:3877ac408e14da24a6196de0ddcace62092bfc12a83823e92e49e40747e52c19 \\\n'
        '    --hash=sha256:3fd1fcdab3ea951b610dc4cef356d416d5802991e7e32b5254828d342f7b7e0b \\\n'
        '    --hash=sha256:41b71f4a3a4cab9d366cd9065b288efc4d4f3c0b37a91a8e0947fb5bd7f31d87 \\\n'
        '    --hash=sha256:43af8d1f3bea642559019edfe64e9b11192a8978efbd1539d7bc2aaa23d92de4 \\\n'
        '    --hash=sha256:45abad819184f07240d8a696117a7aacd39787af9e0b719d00285549ed19a1e9 \\\n'
        '    --hash=sha256:4b400bdc6f79fa02a4d86640310dde87a21fba0c979efff5248908c6f15fad1b \\\n'
        '    --hash=sha256:4eb6c25dd62ee8d5edf68a8e1c171dd71c292fdae95d8aeb3dd7d7de4c364082 \\\n'
        '    --hash=sha256:581b2264fc0aa555f3f435a5944da7504ea3a065d7029ad60e7c3d1ae09c5464 \\\n'
        '    --hash=sha256:5cf36e801231b6a2059bf354720274b7558746f3b1a4efb43fcf557ccd484a87 \\\n'
        '    --hash=sha256:5e3c5c011904115f88a39308379c17f91546f77c1667cea98739fe0fccea804c \\\n'
        '    --hash=sha256:6609bc224e9568f65064cfa72edc0f24ee6655b47575954ec6339534b2798369 \\\n'
        '    --hash=sha256:6e3dcd57ab780c741fde8dc68619de988b966db759a3c3152e8e9142c26295ad \\\n'
        '    --hash=sha256:6fac755ca3d2c3edcb22f479fceaa241704111414831ddd3bc6056e18516892f \\\n'
        '    --hash=sha256:744b2bf3640d907b79f3fd7874efe432d1cf171ee721243e350f55234b4cec4c \\\n'
        '    --hash=sha256:74cbb80d93260fe2ffa334efa24cb8f2f0f622a9b9febf8b483c0b865bfb3475 \\\n'
        '    --hash=sha256:766e0dc5a616d026a3a1cffa379af959671729083882f50307e18175797b3dfd \\\n'
        '    --hash=sha256:7bdf2da170b67fdf10bca777614b1c7d96ae3ca5794fd9587dce41eb2966e866 \\\n'
        '    --hash=sha256:7ff200bf9d24f2e4d5dc6ee8c3ac64d739d3a89e2326ba68aaf6c4a2b838fd7d \\\n'
        '    --hash=sha256:844e165636711ef41f80b4103ed234181646b98a53c8f05da12ca5ca289134f6 \\\n'
        '    --hash=sha256:8a604bae87c6195d8b1045eddece0514d041604b14f2727bbc2b3020172045eb \\\n'
        '    --hash=sha256:94055a11dfebe37c656e70317e1996dc197e1a15bbcc351bcdd4610e128fe1ca \\\n'
        '    --hash=sha256:95d8e012d8cb8816c226aef832200b1d45109ed4464303e997c5b13122b297c0 \\\n'
        '    --hash=sha256:9cdc1a2fcfd5c52cfb3045feb399f7b3ce822abdde3a193a6b9a60b3cb5854ca \\\n'
        '    --hash=sha256:9ecb4efb1cd6e8c4afea0daa91a87fbddbce1b99d2895d151596716c0b2e859d \\\n'
        '    --hash=sha256:a3472cfbca0a54177d0faa68f697d8ba4c80bbdc19908c3465556d9f7efce9ee \\\n'
        '    --hash=sha256:a4328d245944d09fd639771de275701ccadf5f781ba0ff092ad141e017eccda4 \\\n'
        '    --hash=sha256:a48a72c77a310327f6a3a920092fa2b8fd03d7deaa60f093038f22d98e096717 \\\n'
        '    --hash=sha256:a720477885a9d2411f94a93d16f9d89bad0f28ca23c3f8daa521e2dcc3f44d49 \\\n'
        '    --hash=sha256:a77cbd07b940d326d39a1d1b37817e2ee4d79cb30e7338f3d0cddffae70fcaa2 \\\n'
        '    --hash=sha256:a9956e4d4f4a301ebf6cde39850333a6b6110799d470dbbb1e25326ac447f52a \\\n'
        '    --hash=sha256:adb2642e060a6549c343603a3851ba76ef0b74cc8c079a9a58121c7ec9fe2350 \\\n'
        '    --hash=sha256:beeda3d4ae615106d7094f7e7cef6218392e4465cc95d25f900bebabfded0950 \\\n'
        '    --hash=sha256:c80be5ede8f3f8eded4eff73cc99a25c388ce98e555b17d31da05287015ffa5b \\\n'
        '    --hash=sha256:cc90d2e9c7e5c7f1a482c9875007c095c3194b1cfedca3c2f3291cdc2bc7c086 \\\n'
        '    --hash=sha256:cd96a1898c0a47be4520327e01f874acfd61fb48a9420f8aa9f6483412ffa444 \\\n'
        '    --hash=sha256:d2650c1fb97e184d12d8ba010493ee7b322864f7d3d00d3f9bb97d9c21de4068 \\\n'
        '    --hash=sha256:d30e57c72013c2a4fe441c2fcb8e77b14e152ad48b5464858e07e2ad9fbfceff \\\n'
        '    --hash=sha256:d59c30000a16d8edc7e64152e30220bfbd724c9bbb08368c054e24c651314f0a \\\n'
        '    --hash=sha256:dbc12c9f3d185f5c737d801da555fb74b3dcfa1a50b66a1a93e09190f41fab50 \\\n'
        '    --hash=sha256:e18f12c6b0bc5a592ed23d3f7b891f68fd7f8241d69b7883769eb5d5dfb52696 \\\n'
        '    --hash=sha256:e19ebea31758fac5893a2ac360fedd00116cbb7628e650842a6691ba7ca28a21 \\\n'
        '    --hash=sha256:e30bdeaa5deed6bc27b4cc490823cd0347d7dae09119b8803ae576ea0ce52e4c \\\n'
        '    --hash=sha256:eb092099205ef62cd1782b006658db09e2fed75bffcae7cc0d44052d8aa0f484 \\\n'
        '    --hash=sha256:eee2cfda04c00a857206a4330f0c5e3e56535494e30ca445eb19ec624ae75118 \\\n'
        '    --hash=sha256:f4115102802df98b2b0db3cce5cb9b92572633a1197c77b7553e5203f284a5b3 \\\n'
        '    --hash=sha256:f590cd684941912d10becc07325a3eeb77886fe981415660d9265c4c418d0bea \\\n'
        '    --hash=sha256:f8885db0bc2bffa59d5c1b72fad7a6a92d3e80e7257f967dd81abb553a90d293 \\\n'
        '    --hash=sha256:fcb310ddb270a06114bb64bbe53c94926b943f5b7f0842194d585c65eb4edd76\n'
        '    # via -r tools/segmentation-workshop-requirements.in\n'
        'setuptools==84.0.0 \\\n'
        '    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \\\n'
        '    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73\n'
        '    # via torch\n'
        'sympy==1.14.0 \\\n'
        '    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \\\n'
        '    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5\n'
        '    # via torch\n'
        'tokenizers==0.22.2 \\\n'
        '    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \\\n'
        '    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \\\n'
        '    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \\\n'
        '    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \\\n'
        '    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \\\n'
        '    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \\\n'
        '    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \\\n'
        '    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \\\n'
        '    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \\\n'
        '    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \\\n'
        '    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \\\n'
        '    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \\\n'
        '    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \\\n'
        '    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \\\n'
        '    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \\\n'
        '    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \\\n'
        '    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \\\n'
        '    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \\\n'
        '    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \\\n'
        '    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \\\n'
        '    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \\\n'
        '    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \\\n'
        '    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \\\n'
        '    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5\n'
        '    # via transformers\n'
        'torch==2.14.0 \\\n'
        '    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \\\n'
        '    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \\\n'
        '    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \\\n'
        '    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \\\n'
        '    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \\\n'
        '    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \\\n'
        '    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \\\n'
        '    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \\\n'
        '    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \\\n'
        '    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \\\n'
        '    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \\\n'
        '    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \\\n'
        '    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \\\n'
        '    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \\\n'
        '    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \\\n'
        '    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \\\n'
        '    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \\\n'
        '    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \\\n'
        '    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \\\n'
        '    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \\\n'
        '    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \\\n'
        '    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \\\n'
        '    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \\\n'
        '    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963\n'
        '    # via\n'
        '    #   -r tools/segmentation-workshop-requirements.in\n'
        '    #   torchvision\n'
        'torchvision==0.29.0 \\\n'
        '    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \\\n'
        '    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \\\n'
        '    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \\\n'
        '    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \\\n'
        '    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \\\n'
        '    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \\\n'
        '    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \\\n'
        '    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \\\n'
        '    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \\\n'
        '    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \\\n'
        '    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \\\n'
        '    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \\\n'
        '    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \\\n'
        '    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \\\n'
        '    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \\\n'
        '    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \\\n'
        '    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \\\n'
        '    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \\\n'
        '    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \\\n'
        '    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \\\n'
        '    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \\\n'
        '    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \\\n'
        '    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \\\n'
        '    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e\n'
        '    # via -r tools/segmentation-workshop-requirements.in\n'
        'tqdm==4.70.1 \\\n'
        '    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n'
        '    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'transformers==4.57.6 \\\n'
        '    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \\\n'
        '    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3\n'
        '    # via -r tools/segmentation-workshop-requirements.in\n'
        'triton==3.8.0 \\\n'
        '    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \\\n'
        '    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \\\n'
        '    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \\\n'
        '    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \\\n'
        '    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \\\n'
        '    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \\\n'
        '    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \\\n'
        '    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \\\n'
        '    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \\\n'
        '    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \\\n'
        '    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \\\n'
        '    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3\n'
        '    # via torch\n'
        'typing-extensions==4.16.0 \\\n'
        '    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n'
        '    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   torch\n'
        'urllib3==2.8.0 \\\n'
        '    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n'
        '    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n'
        '    # via requests\n'
    ),
    'source.json': (
        '{\n'
        '  "repository": "kurtvalcorza/clipseg-segmentation-pipeline",\n'
        '  "base_revision": "2b004834303160e19ad2aef37b9ac7eeb996476c",\n'
        '  "generator": "build_segmentation_workshop.py/1",\n'
        '  "files": {\n'
        '    "clipseg_reference/__init__.py": "c3d9057e5782891b61ef400c4ef4e9a277dec19202785f81c898e6be59ee1caf",\n'
        '    "clipseg_reference/metrics.py": "7b3b8c1a35e8c29997408a9ac733ac9cf595f7fd67a9ee56fb48664301222e71",\n'
        '    "clipseg_reference/pipeline.py": "9c48e2d8ca9fa365d264deec9aba97dcc570843dec803e49576e1d46bdd7023c",\n'
        '    "clipseg_reference/samples.py": "8666e9a23613ae463001e5ffb783acea42d831b675f296981559abebe52949db",\n'
        '    "sam_reference.py": "ecdf92a7985ba182d716648545c7ba76223eac91cdd1753487d0350daf37d714",\n'
        '    "weights/sam/dimer-base-manifest.json": "3b51600ae3133cf24fe8ffaed771d94185f3935c9803d2c57c8b4a8c74c71dd4",\n'
        '    "licenses/sam.txt": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138",\n'
        '    "sam2_reference.py": "8c2a3aac5444f2bebe2a56202fd42ae80e539e1a1921e5eb9f7b8c12c98d0a3b",\n'
        '    "weights/sam2/dimer-base-manifest.json": "ccbf7408250a591a6dbef9782ef2a262997bdab4ee97512516b54797769b9a11",\n'
        '    "licenses/sam2.txt": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138",\n'
        '    "vendor_provenance.json": "8ecb238ba235b46bf0744d4b9557880cfc184d1ceaa5e3da371afa9e141ab6e3",\n'
        '    "weights/clipseg/dimer-base-manifest.json": "69a97d2c6aa74059642376ae3b3de52f1f08cd1eb35bcc0dcb493c347bedf032",\n'
        '    "workshop.py": "bc97f0ad87c43674f7b24b70b9e8e81ea33852768cb76f73e3528e7dab88e7b1",\n'
        '    "sample.json": "232829c2009243ced9edb7fe7c2e4f92ee9d5f143c3b553e56b76266f6ede737",\n'
        '    "requirements.txt": "18af71cf7809134f0453e5c4229b580eb106ce1758f90f2ad7f9a73cae270351"\n'
        '  }\n'
        '}'
    ),
}
CARRIED_HASHES = {'clipseg_reference/__init__.py': 'c3d9057e5782891b61ef400c4ef4e9a277dec19202785f81c898e6be59ee1caf', 'clipseg_reference/metrics.py': '7b3b8c1a35e8c29997408a9ac733ac9cf595f7fd67a9ee56fb48664301222e71', 'clipseg_reference/pipeline.py': '9c48e2d8ca9fa365d264deec9aba97dcc570843dec803e49576e1d46bdd7023c', 'clipseg_reference/samples.py': '8666e9a23613ae463001e5ffb783acea42d831b675f296981559abebe52949db', 'sam_reference.py': 'ecdf92a7985ba182d716648545c7ba76223eac91cdd1753487d0350daf37d714', 'weights/sam/dimer-base-manifest.json': '3b51600ae3133cf24fe8ffaed771d94185f3935c9803d2c57c8b4a8c74c71dd4', 'licenses/sam.txt': 'f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138', 'sam2_reference.py': '8c2a3aac5444f2bebe2a56202fd42ae80e539e1a1921e5eb9f7b8c12c98d0a3b', 'weights/sam2/dimer-base-manifest.json': 'ccbf7408250a591a6dbef9782ef2a262997bdab4ee97512516b54797769b9a11', 'licenses/sam2.txt': 'f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138', 'vendor_provenance.json': '8ecb238ba235b46bf0744d4b9557880cfc184d1ceaa5e3da371afa9e141ab6e3', 'weights/clipseg/dimer-base-manifest.json': '69a97d2c6aa74059642376ae3b3de52f1f08cd1eb35bcc0dcb493c347bedf032', 'workshop.py': 'bc97f0ad87c43674f7b24b70b9e8e81ea33852768cb76f73e3528e7dab88e7b1', 'sample.json': '232829c2009243ced9edb7fe7c2e4f92ee9d5f143c3b553e56b76266f6ede737', 'requirements.txt': '18af71cf7809134f0453e5c4229b580eb106ce1758f90f2ad7f9a73cae270351', 'source.json': '6038dc94daddebc0ae82f290980e2342fa3875b58574d21d16e946344c8d05bb'}
for name, text in CARRIED_FILES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
    if hashlib.sha256(path.read_bytes()).hexdigest() != CARRIED_HASHES[name]:
        raise RuntimeError('Carried source integrity failure: ' + name)
print('Separate CLIPSeg, SAM and SAM 2 source namespaces materialized and verified.')

In [ ]:
# @title Infrastructure: isolated environment, stage runner and display helpers
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
with urllib.request.urlopen(UV_URL, timeout=90) as response:
    wheel = response.read(20081405)
if len(wheel) != 20081404 or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
    raise RuntimeError('uv wheel size/hash mismatch')
import io
with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
    member = next(n for n in archive.namelist() if n.endswith('.data/scripts/uv'))
    UV = ROOT / 'uv'
    UV.write_bytes(archive.read(member))
UV.chmod(0o700)
ENV = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN='1', HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1')
ENV.pop('HF_TOKEN', None)
ENV.pop('HUGGING_FACE_HUB_TOKEN', None)
subprocess.run([str(UV), 'venv', '--managed-python', '--python', '3.12.12', str(ROOT / 'env')], env=ENV, check=True)
PYTHON = ROOT / 'env/bin/python'
subprocess.run([str(UV), 'pip', 'install', '--python', str(PYTHON), '--require-hashes', '--only-binary', ':all:',
                '--index-url', 'https://pypi.org/simple', '-r', str(ROOT / 'requirements.txt')], env=ENV, check=True)
subprocess.run([str(PYTHON), '-c', 'import sys, torch, transformers; print("Python", sys.version); '
                'print("torch", torch.__version__, "built for CUDA", torch.version.cuda); '
                'print("transformers", transformers.__version__); assert torch.cuda.is_available()'], env=ENV, check=True)
BOOTSTRAP_SECONDS = time.perf_counter() - SESSION_START

def run_stage(stage, *options, root=None):
    selected_root = ROOT if root is None else root
    log = selected_root / (stage + '.log')
    print('Running', stage, '— one model per process; log:', log, flush=True)
    with log.open('w', encoding='utf-8') as output:
        result = subprocess.Popen([str(PYTHON), '-u', str(ROOT / 'workshop.py'), '--root', str(selected_root), '--stage', stage, *options],
                                  env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in result.stdout:
            output.write(line)
            output.flush()
            if line.strip() and len(line) < 2000:
                print(line.rstrip(), flush=True)
        result.wait()
    if result.returncode:
        # Lines are streamed above; on failure repeat the end of the log, which holds the traceback.
        print(log.read_text(encoding='utf-8')[-7000:])
        raise RuntimeError(f'{stage} failed with exit {result.returncode}. Read the diagnostics above.')

def show_json(relative, root=None):
    value = json.loads(((ROOT if root is None else root) / relative).read_text())
    print(json.dumps(value, indent=2))
    return value

def show_training(root=None):
    history = json.loads(((ROOT if root is None else root) / 'outputs/training_history.json').read_text())
    changes = {u['epoch']: u['max_abs_decoder_change'] for u in history.get('observed_epoch_updates', [])}
    print('epoch | train loss | validation mean IoU | max decoder change')
    for entry in history['history']:
        loss = '-' if entry['train_loss'] is None else format(entry['train_loss'], '.4f')
        val = (entry.get('val') or {}).get('miou')
        val = '-' if val is None else format(val, '.4f')
        print(f"{entry['epoch']:>5} | {loss:>10} | {val:>18} | {changes.get(entry['epoch'], 0.0):.2e}")
    frozen = ' (the frozen decoder)' if history['best_epoch'] == 0 else ''
    print(f"Selected epoch: {history['best_epoch']}{frozen}; optimizer steps: {history.get('optimizer_steps')}; "
          f"trainable decoder parameters: {history.get('n_trainable')} of {history.get('n_total')}")
    return history

def show_reload(root=None):
    parity = json.loads(((ROOT if root is None else root) / 'outputs/reload_parity.json').read_text())
    records = parity['records']
    worst = max(r['max_abs_probability_difference'] for r in records)
    print(f"Fresh reload {'passed' if parity['passed'] else 'FAILED'}: {len(records)} records scored by a new process "
          f"(PID {parity.get('pid')}); maximum probability difference {worst:.2e} (tolerance {parity['atol']}); "
          f"identical masks on every record: {all(r['identical_masks'] for r in records)}; "
          f"metric parity: {parity.get('metric_parity')}.")
    return parity

def show_previews(pattern):
    from IPython.display import Image, display
    for path in sorted((ROOT / 'outputs').glob(pattern))[:6]:
        print(path.name)
        display(Image(filename=str(path)))

def show_score_table(filenames, models=None, root=None):
    from IPython.display import Markdown, display
    groups = {}
    for filename in filenames:
        for row in json.loads(((ROOT if root is None else root) / 'outputs' / filename).read_text()):
            if row['role'] != 'test' or (models is not None and row['model'] not in models):
                continue
            key = (row['model'], row['protocol'])
            groups.setdefault(key, []).append(row)

    def mean(rows, key):
        values = [r[key] for r in rows if r[key] is not None]
        return format(sum(values) / len(values), '.4f') if values else 'n/a'

    table = ['| Model | Prompt protocol | Test images | Mean target IoU | Dice | Precision* | Recall | Predicted area |',
             '|---|---|---:|---:|---:|---:|---:|---:|']
    for (model, protocol), rows in groups.items():
        table.append(f"| {model} | {protocol} | {len(rows)} | {mean(rows, 'iou')} | {mean(rows, 'dice')} | "
                     f"{mean(rows, 'precision')} | {mean(rows, 'recall')} | {mean(rows, 'predicted_area_fraction')} |")
    table += ['', '*Per-image means. Precision is averaged over images with any predicted foreground; '
              'an empty prediction has no precision (n/a). Predicted area is the fraction of the image predicted as target.']
    display(Markdown('\n'.join(table)))

def show_box_experiment(root=None):
    from IPython.display import Markdown, display
    entries = json.loads(((ROOT if root is None else root) / 'outputs/box_experiment.json').read_text())
    lines = ['| Expansion per side | Images | Improved | Unchanged | Worsened | Clipped by an image edge | Whole-image box |',
             '|---:|---:|---:|---:|---:|---:|---:|']
    per_image = {}
    for entry in entries:
        lines.append(f"| {entry['expansion']:.0%} | {len(entry['records'])} | {entry['improved']} | {entry['unchanged']} | "
                     f"{entry['worsened']} | {entry['clipped_records']} | {entry['whole_image_records']} |")
        for record in entry['records']:
            per_image.setdefault(record['id'], {'phrase': record['phrase'], 'tight': record['iou_tight_box']})[entry['expansion']] = record
    lines += ['', '| Image | Phrase | IoU, tight box | IoU, +10% | Clipped sides, +10% | IoU, +25% | Clipped sides, +25% |',
              '|---|---|---:|---:|---|---:|---|']
    for image_id, value in per_image.items():
        cells = []
        for fraction in (0.1, 0.25):
            record = value[fraction]
            sides = ', '.join(record['clipped_sides']) or 'none'
            if record['whole_image_box']:
                sides += ' (whole image)'
            cells += [format(record['iou_expanded_box'], '.3f'), sides]
        lines.append(f"| {image_id} | {value['phrase']} | {value['tight']:.3f} | " + ' | '.join(cells) + ' |')
    display(Markdown('\n'.join(lines)))
    return entries

## 2. Understand the image and target

**Input:** digest-pinned FoodSeg103 photographs and ingredient labels. **System:** decode, validate and assign frozen roles. **Output:** aligned RGB images, class phrases and binary reference masks on the original pixel grid.

The target is the **union of every pixel of the largest eligible ingredient class**, excluding background and “other ingredients”. Disconnected regions remain in the target. This is a semantic class mask, not one object instance. Selecting only its largest connected region would change the question.

The build deduplicates decoded RGB images before assigning 120/40/40/12 roles with seed 42. Training changes decoder weights; validation selects the epoch; the final test scores the locked choice. The activity images support exploration. These roles are carved from the source's validation slice; this is not an official FoodSeg103 test benchmark, and upstream pretraining overlap is unknown.

**Predict:** If there are two pieces of the same ingredient, should both be white in this target mask? Would one click necessarily tell a model to include both?

In [ ]:
run_stage('prepare')
print('Frozen roles:', json.loads((ROOT / 'sample.json').read_text())['roles'])
show_previews('reference*.png')

**What to notice:** the reference mask answers a particular phrase. It may contain holes and disconnected regions. Model logits/maps are resized using each wrapper's postprocessing, then thresholded; references are never bilinearly resized for scoring.

<details><summary>Worked interpretation</summary>The target asks for all pixels of an ingredient. SAM prompting may select a single object or part. Missing a second disconnected piece can therefore reflect both the interaction protocol and model behavior. Keep that limitation visible.</details>

## 3. Measure overlap without hiding background errors

**IoU = TP / (TP + FP + FN)**. **Dice = 2TP / (2TP + FP + FN)**. TP means correctly predicted foreground, FP means extra foreground, and FN means missed foreground.

**Predict:** a four-pixel reference overlaps a four-pixel prediction at two pixels. What are IoU and Dice? Why might an all-background mask have high pixel accuracy but fail our task?

In [ ]:
reference = {0, 1, 2, 3}
prediction = {2, 3, 4, 5}
tp = len(reference & prediction)
fp = len(prediction - reference)
fn = len(reference - prediction)
print({'TP': tp, 'FP': fp, 'FN': fn, 'IoU': tp / (tp + fp + fn), 'Dice': 2 * tp / (2 * tp + fp + fn)})

<details><summary>Worked interpretation</summary>TP=2, FP=2, FN=2, so IoU=1/3 and Dice=1/2. Background dominates many images, so pixel accuracy can conceal a useless mask. An empty prediction against a positive reference scores zero IoU/Dice; its precision is undefined. Both-empty references/predictions have undefined overlap, tracked separately. The core sample has positive reference masks.</details>

Our primary metric is **mean per-image target IoU**, not mean IoU over all 103 classes. Pooled IoU weights large regions more heavily. The score tables also show Dice, foreground precision and recall, and the predicted area fraction; pooled IoU and complete denominators are in the exported report. Metrics describe this small sample; they are not calibrated confidence or proof of deployment reliability.

## 4. Adapt CLIPSeg and lock the selection

**Input:** images + canonical ingredient phrases, 120 training masks and 40 validation masks. **Model:** frozen CLIP towers with a trainable CLIPSeg decoder. **Output:** frozen baseline, four-epoch training history, validation-selected decoder and adapter artifact.

The run uses learning rate 0.0003, batch size 4, seed 42 and threshold 0.5. It selects the highest validation mean target IoU, keeping the earliest epoch on ties, including frozen epoch zero. Nothing is selected using final test labels. If epoch zero wins, the notebook says so; a positive improvement is not required.

**Predict:** Will a lower training loss necessarily improve held-out IoU? What evidence would show that training occurred even if the frozen checkpoint wins?

In [ ]:
run_stage('clipseg')
show_training()
show_score_table(['baselines.json', 'clipseg_frozen.json', 'clipseg_selected_live.json'],
                 models={'empty', 'clipseg_frozen', 'clipseg_selected_live'})
show_previews('clipseg*.png')

**What to notice:** the loss/validation history, selected epoch, actual update evidence and frozen-versus-selected test results. Training loss measures optimization; it is not segmentation quality. Compare unchanged phrases and thresholds on identical images.

<details><summary>Worked interpretation</summary>Improved validation IoU motivated selection, but final test change may be smaller, absent or negative. Report that difference. Epoch zero is an honest selected result, and its exported decoder must be labelled as frozen rather than improved.</details>

## 5. Verify the exported model in a fresh process

**Input:** pinned base snapshot, safetensors adapter and manifest. **System:** verify bytes, tensor schema and base identity, reconstruct locally. **Output:** probability/mask parity on fixed probes and test metrics from the persisted selected model.

The next cell starts a new Python process. The old live model cannot supply hidden state. A checksum or parity mismatch fails; tolerances are fixed before execution. Export/reload proves reproducibility of this artifact, not correctness of the model's predictions.

In [ ]:
run_stage('reload')
show_reload()

## 6. Compare SAM and SAM 2 with the same assistance

**Input:** identical original images and identical point/box coordinates. **Models:** SAM ViT-B and SAM 2.1 Hiera-Small, both frozen. **Output:** separate point-prompt and box-prompt scores, masks and overlays.

The point is an interior pixel in the largest target component, deterministically chosen by distance from its boundary. The box tightly surrounds the full target-class union. Both come from the reference mask: **oracle-assisted localization**, simulating an informed user. This is not autonomous recognition. The full semantic reference, including disconnected regions, remains the scoring target.

Each model chooses its candidate using its own predicted quality score, with a stable tie rule. We never use reference IoU to choose the best candidate. Predicted quality is not measured IoU or calibrated certainty.

**Predict:** Why might a box help more than a point? Can you fairly rank a text-prompted CLIPSeg score against a model that received the reference object's location?

In [ ]:
run_stage('sam')
run_stage('sam2')
show_score_table(['baselines.json', 'sam_metrics.json', 'sam2_metrics.json'], models={'oracle_filled_box', 'sam', 'sam2'})
show_previews('sam*.png')

**What to notice:** compare SAM versus SAM 2 within the same prompt protocol. The `oracle_filled_box` row fills the same oracle box with foreground; it shows how much of a box-prompted score comes from the box alone, and what the model adds beyond localization. Compare precision with recall to see whether a model over- or under-segments. CLIPSeg frozen/adapted results form a separate comparison. There is no overall three-model leaderboard.

<details><summary>Worked interpretation</summary>Different prompt information can explain apparent performance differences. A stronger model may still select a different object or part. Inspect the worst masks and disconnected targets, not only average scores.</details>

## 7. Controlled activity: make the box less precise

**Input:** the 12 reserved activity images. **System:** SAM 2 with the same model and candidate-selection rule. **Change one thing:** expand each box by 0%, 10% or 25% of its width/height on each side, clipped at the image edges. Exact clipped coordinates are exported.

Many targets in this sample are large and already touch an image edge. A box cannot grow past the edge, so it grows only on its free sides; at +25% some boxes cover the whole image. The table below shows which sides were clipped for every image.

**Predict:** Will every wider box worsen overlap? What does a box that covers the whole image tell SAM 2 about which object you mean? A wider box can help, hurt or leave the result unchanged. These activity results do not tune the test protocol.

In [ ]:
show_box_experiment()
show_previews('activity*.png')

**What to notice:** the improved/unchanged/worsened counts, how many boxes were clipped by an image edge or became the whole image, and the per-image rows. Compare images whose boxes were clipped with those that were not before attributing a change to imprecision alone.

<details><summary>Worked interpretation</summary>Localization precision affects the interaction. A box that reaches the image edges, or covers the whole image, no longer says which region is the target; SAM 2 may then pick the plate, the background or the inverse of the target. A case that improves with a larger box is a valid observation. With 12 images, most of them clipped, this experiment supports a narrow conclusion about these images, not a universal ranking or significance claim.</details>

## 8. Read the report and conclude

The report separates CLIPSeg adaptation from SAM prompt comparisons. Its summary states the selected epoch, the fresh-reload result, how many test images the adaptation improved or worsened, and the box-activity counts. The bundle contains all per-image metrics, prompt coordinates, role/source identities, training and reload evidence, selected adapter and checksums; download it from Colab's Files sidebar using the path the cell prints. The default sample report can include representative overlays; private BYOD images are excluded from its export.

In [ ]:
run_stage('report')
from IPython.display import FileLink, Markdown, display
display(Markdown((ROOT / 'outputs/summary.md').read_text()))
show_previews('figures/*.png')
bundle = os.path.relpath(ROOT / 'outputs/segmentation_results.zip')
print('Results bundle:', bundle)
print('In Colab: open the Files sidebar (folder icon), browse to that path, then choose Download.')
display(FileLink(bundle))
session = {'bootstrap_seconds': BOOTSTRAP_SECONDS, 'elapsed_seconds': time.perf_counter() - SESSION_START,
           'source_files': CARRIED_HASHES, 'qualification': 'Candidate until exact hosted run reviewed'}
(ROOT / 'session_summary.json').write_text(json.dumps(session, indent=2))
print(session['elapsed_seconds'], 'seconds since setup began')

Use your own results to complete this optional note:

> On [sample and prompt protocol], [model/state] obtained [metric], compared with [appropriate baseline]. Changing only [training or box expansion] produced [paired observation]. One failure was [example]. This does not establish [generalization claim]. Before using it for [application], I would test [specific requirement].

**Self-check:** Can you distinguish a semantic union from an instance? Explain why equal prompts matter? Identify which images selected the epoch? Explain why verified reload does not establish accuracy?

## 9. Optional: bring your own labelled data

Use images you are permitted to process in Colab, a third-party hosted environment. Data stays inside this runtime; no DIMER worker or external inference API receives it. Reports include labels/metrics, so review them before sharing.

Provide a local directory containing `manifest.json` with records naming unique `id`, relative `image`, relative `mask`, `prompt`, `role` and optional `group_id`. Roles are `train`, `validation`, `test`, `activity`. RGB images and single-channel binary masks must align, with sides 16–4096 pixels and positive target masks. Related images must share one group and role; decoded duplicates across roles are refused. Use at least 8/2/2/1 records respectively. These minimums support execution, not a generalization claim.

This is the same full adaptation/evaluation/export route, not an inference-only replacement. Limits: 13–256 records, 20 MiB per file, 1 MiB manifest, and 64 million decoded image pixels in total. Oversized or misoriented inputs fail before model execution, and each validation error names the record to fix (`records[i]` and its `id`). After the run, the cell shows your run's summary, training history and score tables. The optional path uses a separate run directory and does not overwrite canonical reports. Set a directory path from mounted storage or Colab's Files sidebar; the default run never opens an upload dialog.

Example record inside `{"records": [...]}`: `{"id":"train01","image":"images/01.png","mask":"masks/01.png","prompt":"bread","role":"train","group_id":"source01"}`. Use separate source groups for the other roles.

In [ ]:
RUN_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
if RUN_BYOD:
    if not BYOD_PATH.strip():
        raise ValueError('Set BYOD_PATH to the directory containing manifest.json, images and masks.')
    run_stage('byod', '--byod', str(Path(BYOD_PATH).resolve()))
    byod_root = Path(json.loads((ROOT / 'byod_output.json').read_text())['path'])
    from IPython.display import FileLink, Markdown, display
    display(Markdown((byod_root / 'outputs/summary.md').read_text()))
    show_training(byod_root)
    show_score_table(['baselines.json', 'clipseg_frozen.json', 'clipseg_reloaded.json'],
                     models={'empty', 'clipseg_frozen', 'clipseg_reloaded'}, root=byod_root)
    show_score_table(['baselines.json', 'sam_metrics.json', 'sam2_metrics.json'],
                     models={'oracle_filled_box', 'sam', 'sam2'}, root=byod_root)
    bundle = os.path.relpath(byod_root / 'outputs/segmentation_results.zip')
    print('Your results bundle (metrics and manifests; no images):', bundle)
    print('In Colab: open the Files sidebar (folder icon), browse to that path, then choose Download.')
    display(FileLink(bundle))
else:
    print('Optional BYOD disabled; default workflow complete.')

## 10. Optional: segment one image without a reference

After the canonical run, use the selected CLIPSeg adapter and the frozen SAM models on a new image. Supply a text phrase plus either a positive point `[x,y]` or a box `[x0,y0,x1,y1]` in original-image coordinates. Those are different information sources, so these masks do not form a model ranking. This inference-only option supplements the full labelled adaptation route above.

No reference mask means **no IoU, Dice or accuracy claim**. Output includes local masks, area fractions, input identity and the reason quality is not measurable. Private images and masks are not automatically added to the canonical report.

In [ ]:
RUN_UNLABELLED = False  # @param {type:"boolean"}
IMAGE_PATH = ''  # @param {type:"string"}
TEXT_PROMPT = 'bread'  # @param {type:"string"}
GEOMETRIC_PROMPT_KIND = 'point'  # @param ['point', 'box']
GEOMETRIC_PROMPT = '[100, 100]'  # @param {type:"string"}
if RUN_UNLABELLED:
    if not IMAGE_PATH.strip():
        raise ValueError('Set IMAGE_PATH and choose coordinates inside the original image.')
    run_stage('infer', '--image', str(Path(IMAGE_PATH).resolve()), '--prompt', TEXT_PROMPT,
              '--' + GEOMETRIC_PROMPT_KIND, GEOMETRIC_PROMPT)
    inference_root = Path(json.loads((ROOT / 'unlabelled_output.json').read_text())['path'])
    print((inference_root / 'results.json').read_text())
    from IPython.display import Image as DisplayImage
    for mask_path in sorted(inference_root.glob('*.png')):
        print(mask_path.name)
        display(DisplayImage(filename=str(mask_path)))
else:
    print('Optional unlabelled inference disabled.')

## Troubleshooting, boundaries and attribution

| Observation | Action |
|---|---|
| GPU unavailable / memory exhausted | Start a fresh T4 runtime and close competing workloads; do not silently skip a model or training. |
| Package or download failure | Read the stage log; preserve pins and retry acquisition. Never bypass a digest mismatch. |
| Empty mask | Inspect the phrase, threshold, candidate and area fraction; an empty result is not removed from the score. |
| Misaligned mask | Check original dimensions/orientation and binary labels; never resize a reference just to make validation pass. |
| Poor selected result | Inspect validation selection and test examples. Improvement is not a release requirement. |
| Reload mismatch | Retain the report and investigate; do not widen tolerance merely to pass. |
| BYOD rejected | Fix the named schema, path, group, mask or dimension issue before model execution. |

This is image-only segmentation. It does not demonstrate SAM video tracking, instance counting, panoptic segmentation, clinical validity or geospatial generalization. Food images are a narrow domain. The masks and oracle prompts do not establish performance when a real user or detector supplies inaccurate localization.

**Weight-license provenance:** the CLIPSeg checkpoint's Hub tag states Apache-2.0, while original project notes exclude weights from its code's MIT license; an earlier fleet entry also records MIT. This discrepancy remains a release-review item, documented in `docs/WEIGHTS.md`. The notebook does not resolve those terms by assumption. SAM weights are recorded as Apache-2.0. No model weights are bundled here.

**AI Assistance Disclosure:** Generative AI assisted with this notebook's code and instructional content under maintainer direction. The maintainer remains responsible for review, validation, and release decisions. AI-generated material may contain errors; validation claims are limited to documented runs and configurations. AI use does not imply independent verification, provider endorsement, or release approval.

References: [CLIPSeg checkpoint](https://huggingface.co/CIDAS/clipseg-rd64-refined), [SAM ViT-B](https://huggingface.co/facebook/sam-vit-base), [SAM 2.1](https://huggingface.co/facebook/sam2.1-hiera-small), [FoodSeg103](https://huggingface.co/datasets/EduardoPacheco/FoodSeg103) (Wu et al., ACM MM 2021; dataset metadata records Apache-2.0; underlying photographs curated from Recipe1M), [DIMER notebook specification](https://github.com/kurtvalcorza/ml-worker/blob/main/integrations/dimer/fleet-specs/NOTEBOOK_SPEC.md). Immutable model/data revisions, source hashes and modifications are recorded in the carried manifests and exported provenance.